# 03 · Transfer Learning and Fine-Tuning of Persian Transformers

> **Project 3 · Intelligent Analysis of Digikala Reviews**
> Phase goal: fine-tune pretrained encoders on `body` only, to predict **recommendation status** and **rating band** with one shared model.

---

## 🎯 Objective

Fine-tune a pretrained Transformer with **two classification heads** on top of one **shared encoder** (multi-task learning):

| Head | Task | Classes |
| --- | --- | --- |
| `rec` | Recommendation status | `recommended`, `not_recommended`, `no_idea` |
| `band` | Ordered rating band | `low`, `medium`, `high` |

The total loss is the sum of the two head losses. Rows with a missing target are masked out of that head's loss only.

---

## 🧭 Phase Overview

1. **Fine-tune TookaBERT** in two variants:
   - **B**: Step-1 corrected labels, no sample weights.
   - **C**: Step-1 corrected labels plus Step-2 sample weights.
2. **Choose B or C** with a rule fixed before seeing results.
3. **Fine-tune two more encoders** with the winning variant.
4. **Select the primary model** for the semi-supervised phase.
5. **Compare** against the dummy and TF-IDF baselines under the same split and protocol.

---

## 🤖 Encoders

| Model | Tokenizer | `max_length` |
| --- | --- | --- |
| `PartAI/TookaBERT-Base` | BPE (Persian) | 128 |
| `HooshvareLab/roberta-fa-zwnj-base` | Byte-level BPE (Persian) | 128 |
| `microsoft/mdeberta-v3-base` | SentencePiece (multilingual) | 160 |

---

## 📐 Ground Rules

- **Input:** `body` only. No `title`, `rate`, `product_id`, or any other metadata.
- **Labels for training and checkpoint selection:** corrected Step-1 labels only.
- **Data:** full `train_supervised` for training, full validation set for evaluation every epoch.
- **Training:** at most 4 epochs, early stopping with patience 1 on the mean Macro F1 of both heads.
- **Fair comparison:** B and C use identical seeds, batch order, and settings. Only the sample weights differ.
- **No decision uses the test set.** Test metrics are reported on both raw labels (comparable with classmates) and corrected labels.

---

## 🖥️ Where Things Run

| Where | What |
| --- | --- |
| **Training machine** (friend's PC, RTX 4060 Ti, 16 GB) | Fine-tuning and prediction scripts only |
| **This notebook** (local machine) | Analysis of the saved outputs: curves, tables, decisions, charts |

Training is run by `scripts/train_transformer.py` from YAML configs. It resumes automatically from the last saved epoch if interrupted.

---

## 📦 Main Outputs

- Training history, logs, and run summaries for every run
- Validation and test probabilities for both heads
- Saved tokenizer and encoder config (for offline inference)
- Settings table with the reason for each choice
- B vs C decision, encoder comparison, and primary-model selection
- Test metrics, confusion matrices, and four report charts

---

## 🗂️ Notebook Map

| Step | Content |
| --- | --- |
| 1 | Imports and parameters |
| 2 | Load run outputs |
| 3 | Training settings and run comparison tables |
| 4 | Learning curves and overfitting check |
| 5 | B vs C decision |
| 6 | Test metrics and baseline comparison |
| 7 | Confusion matrices |
| 8 | Encoder comparison and primary model |
| 9 | Closeout facts |

---

# Step 0: preparing

In [12]:
# ── Read-only inspection, part 1: project setup, git, ignore rules ──
import json
import subprocess
import tomllib
from pathlib import Path

from digikala_review_transformer import config

root = Path(config.ROOT)


def show(title):
    print(f"\n{'─' * 8} {title} {'─' * 8}")


def run_git(*args):
    result = subprocess.run(["git", "-C", str(root), *args], check=False, capture_output=True, text=True)
    return (result.stdout or result.stderr).strip()


# ── 1) pyproject.toml (full text) ──
show("pyproject.toml")
print((root / "pyproject.toml").read_text(encoding="utf-8"))

# ── 1b) Versions locked in uv.lock for the packages we care about ──
show("uv.lock: selected package versions")
lock = tomllib.loads((root / "uv.lock").read_text(encoding="utf-8"))
wanted = [
    "torch", "transformers", "tokenizers", "huggingface-hub", "safetensors",
    "sentencepiece", "protobuf", "accelerate", "pyyaml", "tqdm", "pandas",
    "pyarrow", "scikit-learn", "numpy", "duckdb", "matplotlib",
]
locked = {pkg["name"]: pkg.get("version", "-") for pkg in lock["package"]}
for name in wanted:
    print(f"{name:18s} {locked.get(name, 'NOT IN LOCK')}")

# ── 2) Package __init__.py ──
show("src/digikala_review_transformer/__init__.py")
init_text = (root / "src/digikala_review_transformer/__init__.py").read_text(encoding="utf-8")
print(init_text if init_text.strip() else "(empty file)")

# ── 6) .gitignore and ignore status of the paths this phase will create ──
show(".gitignore")
print((root / ".gitignore").read_text(encoding="utf-8"))

show("git check-ignore on future paths")
sample_paths = [
    "data/training_pack/train.parquet",
    "data/interim/token_cache/tooka_128/tokens.npy",
    "models/pretrained/TookaBERT-Base/model.safetensors",
    "artifacts/transformer_runs/tooka_B/best_model.pt",
    "artifacts/transformer_runs/tooka_B/last.pt",
    "outputs_to_send/tooka_B.zip",
]
for path in sample_paths:
    verdict = run_git("check-ignore", "-v", path)
    print(f"{path}\n    -> {verdict if verdict else 'NOT IGNORED'}")

# ── 7) scripts/ and configs/ ──
show("scripts/ and configs/")
for folder_name in ["scripts", "configs", "src/digikala_review_transformer/training"]:
    folder = root / folder_name
    if folder.exists():
        print(f"{folder_name}/ exists:", sorted(p.name for p in folder.iterdir()))
    else:
        print(f"{folder_name}/ does not exist")
print("tracked in git (scripts, configs):", run_git("ls-files", "scripts", "configs") or "(none)")

# ── 8) Branch and remote (hide any token if the URL contains one) ──
show("git branch and remote")
print("branch:", run_git("branch", "--show-current"))
print(run_git("remote", "-v"))

# ── 9) Notebook numbering ──
show("notebooks/")
for notebook in sorted((root / "notebooks").glob("*.ipynb")):
    print(notebook.name)

# ── 10) Where is a text-normalization function? (source cells only, no outputs) ──
show("'normaliz' in src/*.py")
hits = 0
for py_file in sorted((root / "src").rglob("*.py")):
    for line_number, line in enumerate(py_file.read_text(encoding="utf-8").splitlines(), 1):
        if "normaliz" in line.lower() and hits < 25:
            print(f"{py_file.relative_to(root)}:{line_number}: {line.strip()[:140]}")
            hits += 1
print("matches shown:", hits)

show("'normaliz' in notebook code cells")
hits = 0
# for notebook in sorted((root / "notebooks").glob("*.ipynb")):
#     cells = json.loads(notebook.read_text(encoding="utf-8")).get("cells", [])
#     for cell_index, cell in enumerate(cells):
#         if cell.get("cell_type") != "code":
#             continue
#         source = "".join(cell.get("source", []))
#         matching = [ln.strip() for ln in source.splitlines() if "normaliz" in ln.lower()]
#         if matching and hits < 25:
#             print(f"{notebook.name} | cell {cell_index}: {matching[0][:140]}")
#             hits += 1
print("matches shown:", hits)


──────── pyproject.toml ────────
[project]
name = "digikala-review-transformer"
version = "0.1.0"
description = "Persian Digikala review classification: TF-IDF baseline vs fine-tuned Transformer"
readme = "README.md"
authors = [
    { name = "AliirezzaaZiarati", email = "arzxii1998@gmail.com" }
]
requires-python = ">=3.12"

# Training dependencies only: this is everything the training machine needs
dependencies = [
    "huggingface-hub>=2.4.0",
    "numpy>=2.5.4",
    "pandas>=3.1.1",
    "protobuf",
    "pyarrow>=26.0.1",
    "pyyaml>=6.0.3",
    "scikit-learn>=1.9.1",
    "sentencepiece",
    "torch>=2.15.0",
    "tqdm>=4.70.1",
    "transformers>=5.22.2",
]

[build-system]
requires = ["uv_build>=0.13.6,<0.14.0"]
build-backend = "uv_build"

[dependency-groups]
dev = [
    "ipykernel>=7.4.0",
]
# Analysis, charts and data-preparation tools (not needed on the training machine)
analysis = [
    "arabic-reshaper>=3.1.0",
    "duckdb>=2.0.0",
    "joblib>=1.6.0",
    "matplotlib>=3.11.2"

In [13]:
# ── Read-only inspection, part 1b: normalization search in notebook code cells ──
import json  # noqa: F811
from pathlib import Path

from digikala_review_transformer import config

root = Path(config.ROOT)

hits = 0
unreadable = []
for notebook in sorted((root / "notebooks").glob("*.ipynb")):
    # Skip notebooks that are empty or not valid JSON, and remember them
    try:
        cells = json.loads(notebook.read_text(encoding="utf-8")).get("cells", [])
    except json.JSONDecodeError:
        unreadable.append((notebook.name, notebook.stat().st_size))
        continue
    for cell_index, cell in enumerate(cells):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        matching = [ln.strip() for ln in source.splitlines() if "normaliz" in ln.lower()]
        if matching and hits < 25:
            print(f"{notebook.name} | cell {cell_index}: {matching[0][:140]}")
            hits += 1

print("matches shown:", hits)
print("unreadable notebooks (name, size in bytes):", unreadable or "none")

00_data_ingest_quality_gate.ipynb | cell 42: "body text normalization; original text stays in raw"
00_data_ingest_quality_gate.ipynb | cell 50: # Total number of rows before normalization
00_data_ingest_quality_gate.ipynb | cell 53: # Replace the old body with the normalized one
00_data_ingest_quality_gate.ipynb | cell 87: fig.text(0.04, 0.865, "Rows whose exact (normalized) review text appears more than once in clean_base",
00_data_ingest_quality_gate.ipynb | cell 91: bg.text(M, 0.17, f"Base: {total:,} reviews, all of clean_base. Words are whitespace-separated tokens of the normalized body. "
01_more_data_ingest_analysis.ipynb | cell 16: # from parsivar import Normalizer
01_more_data_ingest_analysis.ipynb | cell 22: # # Spacing fixes that only the SpellCheck+guard route found, not the Normalizer
01_more_data_ingest_analysis.ipynb | cell 43: print(df["recommendation_status"].value_counts(dropna=False, normalize=True).round(4))
04_label_correction_step1.ipynb | cell 15: # Kneedle core: 

# Inspect: Imports, Labels, Config, Data Columns

In [ ]:
%load_ext autoreload
%autoreload 2

# ── Read-only inspection, part 2: imports, labels, config, data columns ──
import ast
import inspect
import subprocess  # noqa: F811
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

from digikala_review_transformer import config, data_io, labels, utils

root = Path(config.ROOT)
package_dir = root / "src" / "digikala_review_transformer"
processed_dir = Path(config.DATA_PROCESSED)
predictions_dir = root / "reports" / "predictions"


def show(title):
    print(f"\n{'─' * 8} {title} {'─' * 8}")


def run_git(*args):
    result = subprocess.run(["git", "-C", str(root), *args], check=False, capture_output=True, text=True)
    return (result.stdout or result.stderr).strip()


# ── A) Which third-party packages does each module import? (light-import check) ──
show("A) imports per module (top-level / inside functions)")
training_ok = {
    "torch", "transformers", "huggingface_hub", "pandas", "pyarrow", "sklearn", "yaml",
    "tqdm", "sentencepiece", "google", "numpy", "tokenizers", "safetensors",
    "digikala_review_transformer",
}


def root_names(nodes):
    names = set()
    for node in nodes:
        if isinstance(node, ast.Import):
            names.update(alias.name.split(".")[0] for alias in node.names)
        elif isinstance(node, ast.ImportFrom) and node.level == 0 and node.module:
            names.add(node.module.split(".")[0])
    return names


for module_name in ["config", "labels", "utils", "data_io", "metrics", "predictions", "plot_style"]:
    tree = ast.parse((package_dir / f"{module_name}.py").read_text(encoding="utf-8"))
    top_level = root_names(tree.body)
    everywhere = root_names(ast.walk(tree))
    nested_only = everywhere - top_level
    outside = sorted(n for n in everywhere if n not in sys.stdlib_module_names and n not in training_ok)
    print(f"{module_name:12s} top={sorted(top_level)} nested_only={sorted(nested_only)}")
    print(f"{'':12s} NOT in training set: {outside or 'none'}")

# ── B) Seed-like constants and simple values in config ──
show("B) config: simple constants")
for name in sorted(vars(config)):
    value = getattr(config, name)
    if name.isupper() and isinstance(value, (int, float, str, bool)):
        print(f"{name} = {value!r}")
path_names = [n for n in vars(config) if n.isupper() and isinstance(getattr(config, n), Path)]
print("number of Path constants:", len(path_names))
print("lines containing 'seed' in config.py:")
for line_number, line in enumerate((package_dir / "config.py").read_text(encoding="utf-8").splitlines(), 1):
    if "seed" in line.lower():
        print(f"  {line_number}: {line.strip()[:140]}")

# ── C) labels: constants and function signatures; data_io.read_split ──
show("C) labels module")
for name in sorted(n for n in vars(labels) if not n.startswith("_")):
    obj = getattr(labels, name)
    if inspect.isfunction(obj) and obj.__module__ == labels.__name__:
        print(f"def {name}{inspect.signature(obj)}")
    elif not (inspect.ismodule(obj) or inspect.isfunction(obj) or inspect.isclass(obj)):
        print(f"{name} = {repr(obj)[:300]}")

show("C2) data_io.read_split and utils.timer")
print("read_split", inspect.signature(data_io.read_split))
print((inspect.getdoc(data_io.read_split) or "(no docstring)")[:400])
print("timer", inspect.signature(utils.timer))

# ── D) Parquet schemas (no data loaded) ──
show("D) parquet schemas")
parquet_paths = [predictions_dir / f"tfidf12_{head}_B__{split}.parquet"
                 for head in ("rec", "band") for split in ("validation", "test")]
parquet_paths.append(processed_dir / "step2_sample_weights.parquet")
for path in parquet_paths:
    parquet_file = pq.ParquetFile(path)
    columns = [f"{field.name}:{field.type}" for field in parquet_file.schema_arrow]
    print(f"{path.name}  rows={parquet_file.metadata.num_rows:,}")
    print("   ", columns)

# ── E) CSV headers and unique label values (label columns only) ──
show("E) CSV headers and unique label values")
for split_name in ["train_supervised", "validation", "test"]:
    csv_path = processed_dir / f"{split_name}.csv"
    header = list(pd.read_csv(csv_path, nrows=0).columns)
    print(f"\n{split_name}.csv columns: {header}")
    label_columns = [c for c in ["recommendation_status", "step1_rec_label",
                                 "rating_band", "step1_rate_band_label"] if c in header]
    label_frame = pd.read_csv(csv_path, usecols=label_columns, dtype=str, keep_default_na=False)
    print(f"  rows: {len(label_frame):,}")
    for column in label_columns:
        print(f"  {column}: {label_frame[column].value_counts(dropna=False).to_dict()}")

# ── F) Git: the empty notebook 06 and tracked files under models/ ──
show("F) git: notebook 06 and models/")
notebook_06 = "notebooks/06_masking_and_artifacts.ipynb"
print("size on disk:", (root / notebook_06).stat().st_size)
print("size in HEAD:", run_git("cat-file", "-s", f"HEAD:{notebook_06}"))
print("last commits touching it:", run_git("log", "--oneline", "-3", "--", notebook_06) or "(none)")
print("git status of it:", run_git("status", "--short", "--", notebook_06) or "(clean)")
print("tracked files under models/:", run_git("ls-files", "models")[:600] or "(none)")


──────── A) imports per module (top-level / inside functions) ────────
config       top=['pathlib'] nested_only=[]
             NOT in training set: none
labels       top=['__future__', 'numpy', 'pandas'] nested_only=[]
             NOT in training set: none
utils        top=['__future__', 'contextlib', 'csv', 'datetime', 'hashlib', 'json', 'pathlib', 'time'] nested_only=[]
             NOT in training set: none
data_io      top=['__future__', 'itertools', 'numpy', 'pandas'] nested_only=[]
             NOT in training set: none
metrics      top=['numpy', 'pandas', 'sklearn'] nested_only=[]
             NOT in training set: none
predictions  top=['datetime', 'numpy', 'pandas', 'pathlib', 'zoneinfo'] nested_only=[]
             NOT in training set: none
plot_style   top=['__future__', 'arabic_reshaper', 'colorsys', 'matplotlib', 'numpy', 'pathlib', 're', 'struct'] nested_only=['bidi']
             NOT in training set: ['arabic_reshaper', 'bidi', 'matplotlib']

──────── B) config: simple

# Inspect: Labels and Metrics Source

In [ ]:
# ── Read-only inspection, part 3: labels and metrics source and behavior ──
import inspect

import numpy as np
import pandas as pd

from digikala_review_transformer import labels, metrics


def show(title):
    print(f"\n{'─' * 8} {title} {'─' * 8}")


def print_source(obj_name, module):
    # Print the source of a function, or a note if it does not exist
    obj = getattr(module, obj_name, None)
    show(f"{module.__name__.split('.')[-1]}.{obj_name}")
    print(inspect.getsource(obj) if obj is not None else "NOT FOUND")


print_source("class_weights", labels)
print_source("encode", labels)
print_source("_valid", metrics)
print_source("evaluate", metrics)

show("metrics.per_class_table signature and doc")
print(inspect.signature(metrics.per_class_table))
print(inspect.getdoc(metrics.per_class_table))

# ── Toy calls: one missing label and a prediction that is wrong once ──
show("toy calls")
toy_true = pd.array([0, 1, 2, 1, None], dtype="Int8")
toy_pred = np.array([0, 1, 1, 1, 2])
toy_calls = {
    "class_weights": lambda: labels.class_weights(toy_true),
    "evaluate": lambda: metrics.evaluate(toy_true, toy_pred),
    "per_class_table": lambda: metrics.per_class_table(toy_true, toy_pred, "rec"),
}
for name, call in toy_calls.items():
    try:
        print(f"{name}:\n{call()}\n")
    except Exception as error:  # noqa: BLE001
        print(f"{name}: {type(error).__name__}: {error}\n")


──────── labels.class_weights ────────
def class_weights(y, n_classes=3):
    """Balanced weights n / (k * n_c) as {code: weight}, computed from training labels only."""
    y = np.asarray(y, dtype=int)
    counts = np.bincount(y, minlength=n_classes)
    return {i: float(len(y) / (n_classes * c)) for i, c in enumerate(counts)}


──────── labels.encode ────────
def encode(series, head):
    """Map label strings to int codes (nullable Int8). Missing stays <NA>; unknown values raise."""
    out = series.map(label_map(head))
    bad = series.notna() & out.isna()
    if bad.any():
        raise ValueError(f"unknown labels for head '{head}': {series[bad].unique()[:10]}")
    return out.astype("Int8")


──────── metrics._valid ────────
def _valid(y_true, y_pred):
    """Keep only rows with a valid true label; return int arrays."""
    yt = pd.Series(y_true).reset_index(drop=True)
    yp = np.asarray(y_pred)
    assert len(yt) == len(yp), "y_true and y_pred length differ"
    m = yt.notna().

# U2: pyproject.toml، .gitignore و git

In [5]:
# terminal prompts

# Inspect: Windows Wheels in uv.lock and config.py Paths

In [6]:
# ── Read-only inspection, part 4: Windows CUDA wheels in the lock file and config.py ──
import tomllib
from pathlib import Path

from digikala_review_transformer import config

root = Path(config.ROOT)

# ── A) torch / torchvision entries in uv.lock and their Windows cp312 wheels ──
lock = tomllib.loads((root / "uv.lock").read_text(encoding="utf-8"))
for package in lock["package"]:
    if package["name"] in ("torch", "torchvision"):
        wheel_urls = [w.get("url", "") for w in package.get("wheels", [])]
        windows_wheels = [u.rsplit("/", 1)[-1] for u in wheel_urls if "win_amd64" in u and "cp312" in u]
        print(package["name"], package["version"], "| source:", package.get("source"))
        print("   windows cp312 wheels:", windows_wheels or "NONE")

# ── B) How config.py defines paths (first 60 lines) ──
print("\n--- config.py (first 60 lines) ---")
config_path = root / "src" / "digikala_review_transformer" / "config.py"
for line_number, line in enumerate(config_path.read_text(encoding="utf-8").splitlines()[:60], 1):
    print(f"{line_number:3d}: {line}")

torch 2.15.0 | source: {'registry': 'https://pypi.org/simple'}
   windows cp312 wheels: NONE
torch 2.15.0+cu134 | source: {'registry': 'https://download.pytorch.org/whl/cu134'}
   windows cp312 wheels: ['torch-2.15.0%2Bcu134-cp312-cp312-win_amd64.whl']
torchvision 0.29.2 | source: {'registry': 'https://pypi.org/simple'}
   windows cp312 wheels: NONE
torchvision 0.29.2+cu134 | source: {'registry': 'https://download.pytorch.org/whl/cu134'}
   windows cp312 wheels: NONE

--- config.py (first 60 lines) ---
  1: from pathlib import Path
  2: 
  3: # Project root: src/digikala_review_transformer/config.py -> three levels up
  4: ROOT = Path(__file__).resolve().parents[2]
  5: 
  6: # Folders (data and artifacts contents are not tracked by Git)
  7: DATA_RAW = ROOT / "data" / "raw"
  8: DATA_INTERIM = ROOT / "data" / "interim"
  9: DATA_PROCESSED = ROOT / "data" / "processed"
 10: ARTIFACTS = ROOT / "artifacts"
 11: REPORTS = ROOT / "reports"
 12: 
 13: # Cleaned data: DuckDB file lives direc

# Test: Run Config Loader, Hash and Validation Errors

In [7]:
# ── Test: run_config loading, hashing and validation errors ──
import copy
import tempfile
from pathlib import Path

import yaml

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

configs_dir = Path(config.ROOT) / "configs" / "transformer"
base_path = configs_dir / "base.yaml"
tooka_path = configs_dir / "tooka_B.yaml"
failed = []


def report(name, passed, detail=""):
    print(f"[{'PASS' if passed else 'FAIL'}] {name} {detail}")
    if not passed:
        failed.append(name)


# ── 1) A valid config loads, with the project seed and normalized floats ──
cfg = run_config.load_run_config(tooka_path)
report("tooka_B loads", cfg["run_name"] == "tooka_B" and cfg["training"]["variant"] == "B")
report("seed comes from project config", cfg["seed"] == config.SPLIT_SEED, f"(seed={cfg['seed']})")
report("'extends' is removed", "extends" not in cfg)
report("floats are floats", isinstance(cfg["training"]["lr_encoder"], float))
print("run dir:", run_config.run_dir(cfg))

# ── 2) Hash: deterministic, ignores runtime-only keys, reacts to real changes ──
hash_a = run_config.config_hash(cfg)
report("hash is deterministic", hash_a == run_config.config_hash(run_config.load_run_config(tooka_path))
       and len(hash_a) == 64, f"({hash_a[:12]})")
runtime_changed = copy.deepcopy(cfg)
runtime_changed["training"]["num_workers"] = 0
runtime_changed["training"]["eval_batch_size"] = 64
report("hash ignores runtime-only keys", run_config.config_hash(runtime_changed) == hash_a)
batch_changed = copy.deepcopy(cfg)
batch_changed["training"]["batch_size"] = 16
report("hash changes with batch_size", run_config.config_hash(batch_changed) != hash_a)
variant_changed = copy.deepcopy(cfg)
variant_changed["training"]["variant"] = "C"
report("hash changes with variant", run_config.config_hash(variant_changed) != hash_a)

# ── 3) Resolved config can be written and read back unchanged ──
with tempfile.TemporaryDirectory() as tmp_name:
    tmp = Path(tmp_name)
    run_config.dump_resolved_config(cfg, tmp / "sub" / "config_resolved.yaml")
    reloaded = yaml.safe_load((tmp / "sub" / "config_resolved.yaml").read_text(encoding="utf-8"))
    report("resolved config round-trips", reloaded == cfg)

    # ── 4) Bad configs must be rejected with a clear message ──
    def write_child(name, training=None, top=None):
        training_lines = {"variant": "B"} | (training or {})
        lines = [f"extends: {base_path.as_posix()}", "run_name: t", "model:",
                 "  pretrained_path: p", "  max_length: 128", "training:"]
        lines += [f"  {key}: {value}" for key, value in training_lines.items()]
        lines += [f"{key}: {value}" for key, value in (top or {}).items()]
        path = tmp / name
        path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        return path

    # Control: the helper itself produces a valid config
    try:
        run_config.load_run_config(write_child("control.yaml"))
        report("control child config loads", True)
    except run_config.ConfigError as error:
        report("control child config loads", False, f"\n{error}")

    bad_cases = [
        ("seed in YAML is rejected", {"top": {"seed": 7}}, "seed"),
        ("unknown key is rejected", {"training": {"lr_encodr": "1.0e-5"}}, "lr_encodr"),
        ("encoder LR above head LR is rejected", {"training": {"lr_encoder": "3.0e-4"}}, "lr_encoder"),
        ("unknown variant is rejected", {"training": {"variant": "D"}}, "variant"),
        ("1e-4 read as text is rejected", {"training": {"lr_heads": "1e-4"}}, "lr_heads"),
    ]
    for index, (name, kwargs, snippet) in enumerate(bad_cases):
        path = write_child(f"bad_{index}.yaml", **kwargs)
        try:
            run_config.load_run_config(path)
            report(name, False, "(no error was raised)")
        except run_config.ConfigError as error:
            message = str(error)
            report(name, "[FAIL]" in message and snippet in message)

print("\nALL PASSED" if not failed else f"\nFAILED: {failed}")

[PASS] tooka_B loads 
[PASS] seed comes from project config (seed=2026)
[PASS] 'extends' is removed 
[PASS] floats are floats 
run dir: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/artifacts/transformer_runs/tooka_B
[PASS] hash is deterministic (4d353853f2f5)
[PASS] hash ignores runtime-only keys 
[PASS] hash changes with batch_size 
[PASS] hash changes with variant 
[PASS] resolved config round-trips 
[PASS] control child config loads 
[PASS] seed in YAML is rejected 
[PASS] unknown key is rejected 
[PASS] encoder LR above head LR is rejected 
[PASS] unknown variant is rejected 
[PASS] 1e-4 read as text is rejected 

ALL PASSED


# Inspect: Source Files for the Training Pack

In [ ]:
# ---- Section: Imports ----
import inspect
import subprocess  # noqa: F811
from pathlib import Path

import pyarrow.parquet as pq

from digikala_review_transformer import config, data_io, labels

# ---- Section: Helpers ----
SKIP_DIR_NAMES = {"tfidf_cache", "token_cache", "__pycache__", ".ipynb_checkpoints"}


def list_files(base, suffixes):
    """Return (path, size_mb) for files under base, skipping cache folders."""
    found = []
    if not base.exists():
        return found
    for p in sorted(base.rglob("*")):
        if p.is_file() and p.suffix.lower() in suffixes:  # noqa: SIM102
            if SKIP_DIR_NAMES.isdisjoint(p.parts):
                found.append((p, p.stat().st_size / 1e6))
    return found


# ---- Section: CSV and parquet files in data/ ----
print("=== data/ (csv, parquet) ===")
for base in (config.DATA_RAW, config.DATA_INTERIM, config.DATA_PROCESSED):
    for p, mb in list_files(base, {".csv", ".parquet"}):
        print(f"{p.relative_to(config.ROOT)}  [{mb:,.1f} MB]")

# ---- Section: CSV headers of split-like files ----
print("\n=== CSV header lines ===")
for base in (config.DATA_INTERIM, config.DATA_PROCESSED):
    for p, _ in list_files(base, {".csv"}):
        with open(p, encoding="utf-8") as f:
            print(f"{p.relative_to(config.ROOT)}: {f.readline().strip()}")

# ---- Section: Step 2 weight files anywhere in the project ----
print("\n=== Files with 'weight' or 'step2' in the name ===")
for base in (config.DATA_INTERIM, config.DATA_PROCESSED, config.ARTIFACTS, config.REPORTS):
    for p, mb in list_files(base, {".parquet", ".csv", ".json"}):
        low = p.name.lower()
        if "weight" in low or "step2" in low:
            line = f"{p.relative_to(config.ROOT)}  [{mb:,.1f} MB]"
            if p.suffix == ".parquet":
                pf = pq.ParquetFile(p)
                line += f"  rows={pf.metadata.num_rows:,}  cols={pf.schema_arrow.names}"
            print(line)

# ---- Section: Function signatures I will call ----
print("\n=== Signatures ===")
print("data_io.read_split", inspect.signature(data_io.read_split))
print("labels.encode", inspect.signature(labels.encode))
print("labels.class_weights", inspect.signature(labels.class_weights))

# ---- Section: Git commit (for the manifest) ----
print("\n=== Git ===")
try:
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=config.ROOT,
                          capture_output=True, text=True, check=True).stdout.strip()
    print("HEAD:", head)
except Exception as exc:  # noqa: BLE001
    print("git not usable:", exc)

=== data/ (csv, parquet) ===
data/raw/digikala-comments.csv  [1,278.5 MB]
data/raw/digikala-products.csv  [258.7 MB]
data/interim/body_groups_rating_band.parquet  [0.2 MB]
data/interim/body_groups_recommendation_status.parquet  [0.2 MB]
data/interim/cap_dropped_rows.parquet  [6.1 MB]
data/interim/cap_overcap_bodies.parquet  [0.0 MB]
data/interim/dropped_tables_archive/body_backup_latin.parquet  [0.1 MB]
data/interim/dropped_tables_archive/defect_manifest.parquet  [10.3 MB]
data/interim/dropped_tables_archive/dup_plan.parquet  [0.1 MB]
data/interim/dropped_tables_archive/dup_plan_s3.parquet  [0.0 MB]
data/interim/dropped_tables_archive/len_sample.parquet  [17.7 MB]
data/interim/dropped_tables_archive/rule_summary.parquet  [0.0 MB]
data/interim/explorer_candidates_tfidf12/tfidf12_band_B.parquet  [0.8 MB]
data/interim/explorer_candidates_tfidf12/tfidf12_rec_B.parquet  [0.9 MB]
data/interim/step2_oof/oof_band_fold0.parquet  [12.9 MB]
data/interim/step2_oof/oof_band_fold1.parquet  [12.9 MB]

# Inspect: Local Models and Tokenizer Behavior

In [ ]:
# ---- Section: Imports ----
import gc
import time

import numpy as np
import pandas as pd
from transformers import AutoConfig, AutoTokenizer

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

# ---- Section: Parameters ----
MODELS = [
    ("TookaBERT-Base", 128),
    ("roberta-fa-zwnj-base", 128),
    ("mdeberta-v3-base", 160),
]
SAMPLE_ROWS = 20_000
PERCENTILES = [50, 90, 95, 99, 99.9]
TOTAL_TRAIN_ROWS = 2_860_236

models_dir = config.ROOT / "models" / "pretrained"
cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])

# ---- Section: What is on disk ----
print("=== models/pretrained ===")
if not models_dir.exists():
    print("MISSING:", models_dir)
else:
    for d in sorted(p for p in models_dir.iterdir() if p.is_dir()):
        files = sorted(f for f in d.iterdir() if f.is_file())
        total_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f"{d.name}: {len(files)} files, {total_mb:,.0f} MB")
        print("   ", [f.name for f in files])

# ---- Section: Sample of training bodies ----
train_bodies = pd.read_parquet(pack_dir / "train.parquet", columns=["body"])["body"]
sample = train_bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
del train_bodies
gc.collect()
print(f"\nsample size: {len(sample):,}")

# ---- Section: Per-model tokenizer and config facts ----
for name, max_length in MODELS:
    path = models_dir / name
    print(f"\n=== {name} (max_length={max_length}) ===")
    if not path.exists():
        print("MISSING -> model not downloaded on this machine")
        continue
    try:
        tok = AutoTokenizer.from_pretrained(path)
        mcfg = AutoConfig.from_pretrained(path)
    except Exception as exc:  # noqa: BLE001
        print("LOAD ERROR:", type(exc).__name__, str(exc)[:300])
        continue

    print("tokenizer class:", type(tok).__name__, "| is_fast:", tok.is_fast)
    print("len(tokenizer):", len(tok), "| model_max_length:", tok.model_max_length,
          "| padding_side:", tok.padding_side)
    print("special ids -> cls:", tok.cls_token_id, "sep:", tok.sep_token_id, "pad:", tok.pad_token_id,
          "bos:", tok.bos_token_id, "eos:", tok.eos_token_id, "unk:", tok.unk_token_id)
    print("model_type:", mcfg.model_type, "| architectures:", getattr(mcfg, "architectures", None))
    print("hidden_size:", mcfg.hidden_size, "| config.vocab_size:", mcfg.vocab_size,
          "| max_position_embeddings:", getattr(mcfg, "max_position_embeddings", None),
          "| pad_token_id:", getattr(mcfg, "pad_token_id", None))

    # Keys and ids of one truncated example (ids only, no text).
    one = tok(sample[0], truncation=True, max_length=max_length)
    ids = one["input_ids"]
    print("encoding keys:", list(one.keys()))
    print("first 8 ids:", ids[:8], "| last 3 ids:", ids[-3:], "| len:", len(ids))
    print("starts with cls:", ids[0] == tok.cls_token_id, "| ends with sep:", ids[-1] == tok.sep_token_id)

    # Length distribution without truncation, to see how much gets cut.
    start = time.perf_counter()
    enc = tok(sample, truncation=False, add_special_tokens=True, return_attention_mask=False)
    seconds = time.perf_counter() - start
    lengths = np.array([len(x) for x in enc["input_ids"]])
    pct = np.percentile(lengths, PERCENTILES)
    print("length percentiles", dict(zip(PERCENTILES, pct.round(1))), "| max:", lengths.max())
    print(f"truncated at {max_length}: {(lengths > max_length).mean() * 100:.2f}% of rows")
    print(f"max id seen: {max(max(x) for x in enc['input_ids'])} (must be < vocab size)")
    print(f"tokenize speed: {SAMPLE_ROWS / seconds:,.0f} rows/s -> "
          f"~{TOTAL_TRAIN_ROWS / (SAMPLE_ROWS / seconds) / 60:.1f} min for the full train (single call)")

    del tok, enc
    gc.collect()

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== models/pretrained ===
TookaBERT-Base: 9 files, 986 MB
    ['.gitattributes', 'README.md', 'config.json', 'generation_config.json', 'model.safetensors', 'pytorch_model.bin', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']
mdeberta-v3-base: 8 files, 2,286 MB
    ['.gitattributes', 'README.md', 'config.json', 'generator_config.json', 'pytorch_model.bin', 'pytorch_model.generator.bin', 'spm.model', 'tokenizer_config.json']
roberta-fa-zwnj-base: 9 files, 478 MB
    ['.gitattributes', 'README.md', 'config.json', 'merges.txt', 'pytorch_model.bin', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']

sample size: 20,000

=== TookaBERT-Base (max_length=128) ===
tokenizer class: TokenizersBackend | is_fast: True
len(tokenizer): 48000 | model_max_length: 512 | padding_side: right
special ids -> cls: None sep: 3 pad: 0 bos: 2 eos: 3 unk: 1
model_type: bert | architectures: ['BertForMaskedLM']
hidden_size: 768 | config.vocab_size: 48000 | ma

[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


tokenizer class: DebertaV2Tokenizer | is_fast: True
len(tokenizer): 250102 | model_max_length: 1000000000000000019884624838656 | padding_side: right
special ids -> cls: 1 sep: 2 pad: 0 bos: 1 eos: 2 unk: 3
model_type: deberta-v2 | architectures: None
hidden_size: 768 | config.vocab_size: 251000 | max_position_embeddings: 512 | pad_token_id: 0
encoding keys: ['input_ids', 'token_type_ids', 'attention_mask']
first 8 ids: [1, 260, 22464, 2144, 44146, 13019, 1506, 12989] | last 3 ids: [49123, 377, 2] | len: 44
starts with cls: True | ends with sep: True
length percentiles {50: np.float64(17.0), 90: np.float64(48.0), 95: np.float64(66.0), 99: np.float64(118.0), 99.9: np.float64(280.0)} | max: 694
truncated at 160: 0.46% of rows
max id seen: 249556 (must be < vocab size)
tokenize speed: 100,734 rows/s -> ~0.5 min for the full train (single call)


# Inspect: Tokenizer Correctness and the Mistral Regex Warning

In [ ]:
# ---- Section: Imports ----
import gc

import numpy as np
import pandas as pd
import sentencepiece as spm
from transformers import AutoConfig, AutoTokenizer

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

# ---- Section: Parameters ----
MODELS = [
    ("TookaBERT-Base", 128),
    ("roberta-fa-zwnj-base", 128),
    ("mdeberta-v3-base", 160),
]
SAMPLE_ROWS = 20_000
SHOW_MISMATCHES = 3
PROBE_TEXT = "x"

models_dir = config.ROOT / "models" / "pretrained"
cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])

# ---- Section: Same sample as the previous inspection ----
train_bodies = pd.read_parquet(pack_dir / "train.parquet", columns=["body"])["body"]
sample = train_bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
del train_bodies
gc.collect()


# ---- Section: Helpers ----
def encode_ids(tok, texts, max_length):
    """Return input_ids lists; max_length=None means no truncation."""
    kwargs = {"return_attention_mask": False, "return_token_type_ids": False}
    if max_length is None:
        return tok(texts, truncation=False, **kwargs)["input_ids"]
    return tok(texts, truncation=True, max_length=max_length, **kwargs)["input_ids"]


def report(ok, name, hint=""):
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


# ---- Section: Checks per model ----
for name, max_length in MODELS:
    path = models_dir / name
    print(f"\n=== {name} (max_length={max_length}) ===")
    tok = AutoTokenizer.from_pretrained(path)
    mcfg = AutoConfig.from_pretrained(path)

    # First/last ids come from a probe text, never from cls_token_id.
    probe = tok(PROBE_TEXT, return_token_type_ids=False)["input_ids"]
    bare = tok(PROBE_TEXT, add_special_tokens=False, return_token_type_ids=False)["input_ids"]
    first_id, last_id = probe[0], probe[-1]
    print(f"probe ids: {probe} | special tokens added: {len(probe) - len(bare)} "
          f"| first_id={first_id} last_id={last_id}")
    print("config.type_vocab_size:", getattr(mcfg, "type_vocab_size", None))

    full = encode_ids(tok, sample, None)
    cut = encode_ids(tok, sample, max_length)
    full_len = np.array([len(x) for x in full])
    cut_len = np.array([len(x) for x in cut])
    needs_cut = full_len > max_length

    report(all(x[0] == first_id for x in cut), "every row starts with first_id", "tokenizer adds no start token on some rows")
    report(all(x[-1] == last_id for x in cut), "every row (also truncated ones) ends with last_id",
           "truncation removed the end token")
    report(bool((cut_len <= max_length).all()
                and (cut_len[needs_cut] == max_length).all()
                and (cut_len[~needs_cut] == full_len[~needs_cut]).all()),
           f"truncation gives exactly min(len, {max_length}); {int(needs_cut.sum())} rows were cut",
           "length logic differs from expectation")
    report(tok.pad_token_id == getattr(mcfg, "pad_token_id", None),
           f"tokenizer pad id ({tok.pad_token_id}) equals model config pad id",
           "padding would hit a real token")

    core = np.concatenate([np.asarray(x[1:-1]) for x in full])
    unk_rate = float((core == tok.unk_token_id).mean()) * 100
    print(f"unk rate: {unk_rate:.4f}% of {len(core):,} tokens | mean length: {full_len.mean():.2f}")

    # ---- Section: mdeberta only: warning flag and reference tokenizer ----
    if name == "mdeberta-v3-base":
        try:
            tok_fix = AutoTokenizer.from_pretrained(path, fix_mistral_regex=True)
            cut_fix = encode_ids(tok_fix, sample, max_length)
            same = sum(a == b for a, b in zip(cut, cut_fix))
            report(same == len(cut), f"ids identical with fix_mistral_regex=True ({same:,}/{len(cut):,} rows)",
                   "the flag changes tokenization; I must decide which one is correct")
            del tok_fix
        except Exception as exc:  # noqa: BLE001
            print("flag test could not run:", type(exc).__name__, str(exc)[:200])

        sp = spm.SentencePieceProcessor(model_file=str(path / "spm.model"))
        sp_ids = sp.encode(sample)
        fast_core = [x[1:-1] for x in full]
        bad = [i for i, (a, b) in enumerate(zip(fast_core, sp_ids)) if a != b]
        report(not bad, f"fast tokenizer equals raw sentencepiece ({len(sample) - len(bad):,}/{len(sample):,} rows)",
               "see mismatches below")
        for i in bad[:SHOW_MISMATCHES]:
            a, b = fast_core[i], sp_ids[i]
            pos = next((k for k in range(min(len(a), len(b))) if a[k] != b[k]), min(len(a), len(b)))
            print(f"  row {i}: fast_len={len(a)} sp_len={len(b)} first_diff_pos={pos} "
                  f"fast={a[max(0, pos - 2):pos + 3]} sp={b[max(0, pos - 2):pos + 3]}")
        del sp, sp_ids, fast_core

    del tok, full, cut
    gc.collect()


=== TookaBERT-Base (max_length=128) ===
probe ids: [2, 46517, 3] | special tokens added: 2 | first_id=2 last_id=3
config.type_vocab_size: 2
[PASS] every row starts with first_id
[PASS] every row (also truncated ones) ends with last_id
[PASS] truncation gives exactly min(len, 128); 48 rows were cut
[PASS] tokenizer pad id (0) equals model config pad id
unk rate: 0.0819% of 268,710 tokens | mean length: 15.44

=== roberta-fa-zwnj-base (max_length=128) ===
probe ids: [0, 98, 2] | special tokens added: 2 | first_id=0 last_id=2
config.type_vocab_size: 1
[PASS] every row starts with first_id
[PASS] every row (also truncated ones) ends with last_id
[PASS] truncation gives exactly min(len, 128); 50 rows were cut
[PASS] tokenizer pad id (1) equals model config pad id
unk rate: 0.0000% of 279,352 tokens | mean length: 15.97

=== mdeberta-v3-base (max_length=160) ===


[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


probe ids: [1, 260, 330, 2] | special tokens added: 2 | first_id=1 last_id=2
config.type_vocab_size: 0
[PASS] every row starts with first_id
[PASS] every row (also truncated ones) ends with last_id
[PASS] truncation gives exactly min(len, 160); 92 rows were cut
[PASS] tokenizer pad id (0) equals model config pad id
unk rate: 0.2480% of 447,595 tokens | mean length: 24.38
[FAIL] ids identical with fix_mistral_regex=True (0/20,000 rows) -> the flag changes tokenization; I must decide which one is correct
[FAIL] fast tokenizer equals raw sentencepiece (19,226/20,000 rows) -> see mismatches below
  row 19: fast_len=310 sp_len=309 first_diff_pos=122 fast=[377, 823, 3, 11490, 2409] sp=[377, 823, 260, 11490, 2409]
  row 27: fast_len=20 sp_len=19 first_diff_pos=15 fast=[17761, 377, 3, 1374, 260] sp=[17761, 377, 1374, 260, 11486]
  row 28: fast_len=45 sp_len=41 first_diff_pos=12 fast=[17036, 377, 3, 914, 7736] sp=[17036, 377, 1876, 7736, 260]


#  Inspect: Why Fast mDeBERTa Differs from SentencePiece

In [11]:
# ---- Section: Imports ----
import gc
import unicodedata
from collections import Counter

import pandas as pd
import sentencepiece as spm
from transformers import AutoTokenizer

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

# ---- Section: Parameters ----
SAMPLE_ROWS = 20_000
TOP_CHARS = 15
SHOW_ROWS = 3
HEAD_IDS = 10

path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])

# ---- Section: Same sample as before ----
train_bodies = pd.read_parquet(pack_dir / "train.parquet", columns=["body"])["body"]
sample = train_bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
del train_bodies
gc.collect()

# ---- Section: Three tokenizers, no special tokens, no truncation ----
tok_default = AutoTokenizer.from_pretrained(path)
tok_flag = AutoTokenizer.from_pretrained(path, fix_mistral_regex=True)
sp = spm.SentencePieceProcessor(model_file=str(path / "spm.model"))
UNK = tok_default.unk_token_id
print("fast unk id:", UNK, "| sentencepiece unk id:", sp.unk_id())

kwargs = {"add_special_tokens": False, "truncation": False,
          "return_attention_mask": False, "return_token_type_ids": False}
enc_default = tok_default(sample, return_offsets_mapping=True, **kwargs)
ids_default = enc_default["input_ids"]
ids_flag = tok_flag(sample, **kwargs)["input_ids"]
ids_sp = sp.encode(sample)

# ---- Section: How the three versions compare ----
bad = [i for i in range(SAMPLE_ROWS) if ids_default[i] != ids_sp[i]]
bad_with_unk = [i for i in bad if UNK in ids_default[i]]
rows_with_unk = sum(UNK in x for x in ids_default)
sp_unk_rows = sum(sp.unk_id() in x for x in ids_sp)
print(f"\ndefault != sentencepiece: {len(bad):,} rows")
print(f"  of these, fast output contains unk: {len(bad_with_unk):,}")
print(f"  of these, no unk in fast output:    {len(bad) - len(bad_with_unk):,}")
print(f"rows with unk -> fast: {rows_with_unk:,} | sentencepiece: {sp_unk_rows:,}")
print(f"unk tokens -> fast: {sum(x.count(UNK) for x in ids_default):,} "
      f"| sentencepiece: {sum(x.count(sp.unk_id()) for x in ids_sp):,}")

same_flag_sp = sum(a == b for a, b in zip(ids_flag, ids_sp))
same_flag_default = sum(a == b for a, b in zip(ids_flag, ids_default))
print(f"\nflag == sentencepiece: {same_flag_sp:,}/{SAMPLE_ROWS:,}")
print(f"flag == default:       {same_flag_default:,}/{SAMPLE_ROWS:,}")
print(f"unk tokens with flag: {sum(x.count(UNK) for x in ids_flag):,}")
shortest = min(range(SAMPLE_ROWS), key=lambda i: len(ids_sp[i]))
for label, row in (("row 0", 0), ("shortest row", shortest)):
    print(f"{label}: default={ids_default[row][:HEAD_IDS]} flag={ids_flag[row][:HEAD_IDS]} "
          f"sp={ids_sp[row][:HEAD_IDS]} (lens {len(ids_default[row])}/{len(ids_flag[row])}/{len(ids_sp[row])})")

# ---- Section: Characters inside fast unk spans (code points only) ----
char_counter = Counter()
empty_spans = 0
for text, ids, offsets in zip(sample, ids_default, enc_default["offset_mapping"]):
    for token_id, (start, end) in zip(ids, offsets):
        if token_id != UNK:
            continue
        if start == end:
            empty_spans += 1
        for ch in text[start:end]:
            char_counter[ch] += 1
print(f"\nunk spans with empty offsets: {empty_spans:,}")
print(f"distinct characters inside unk spans: {len(char_counter)}")
print("code point | category | unicode name | count | fast ids alone | sentencepiece ids alone")
for ch, count in char_counter.most_common(TOP_CHARS):
    name = unicodedata.name(ch, "?")
    fast_alone = tok_default(ch, **{k: v for k, v in kwargs.items()})["input_ids"]
    print(f"U+{ord(ch):04X} | {unicodedata.category(ch)} | {name} | {count:,} | {fast_alone} | {sp.encode(ch)}")

# ---- Section: A few mismatched rows without unk in fast output ----
others = [i for i in bad if i not in set(bad_with_unk)]
for i in others[:SHOW_ROWS]:
    a, b = ids_default[i], ids_sp[i]
    pos = next((k for k in range(min(len(a), len(b))) if a[k] != b[k]), min(len(a), len(b)))
    print(f"row {i}: fast_len={len(a)} sp_len={len(b)} first_diff_pos={pos} "
          f"fast={a[max(0, pos - 2):pos + 3]} sp={b[max(0, pos - 2):pos + 3]}")

[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


fast unk id: 3 | sentencepiece unk id: 3

default != sentencepiece: 774 rows
  of these, fast output contains unk: 774
  of these, no unk in fast output:    0
rows with unk -> fast: 774 | sentencepiece: 0
unk tokens -> fast: 1,110 | sentencepiece: 0

flag == sentencepiece: 0/20,000
flag == default:       0/20,000
unk tokens with flag: 220,793
row 0: default=[260, 22464, 2144, 44146, 13019, 1506, 12989, 260, 11389, 3573] flag=[22464, 3, 1506, 44146, 3, 1441, 11805, 3, 377, 2409] sp=[260, 22464, 2144, 44146, 13019, 1506, 12989, 260, 11389, 3573] (lens 42/61/42)
shortest row: default=[555] flag=[5624] sp=[555] (lens 1/1/1)

unk spans with empty offsets: 0
distinct characters inside unk spans: 20
code point | category | unicode name | count | fast ids alone | sentencepiece ids alone
U+200C | Cf | ZERO WIDTH NON-JOINER | 1,096 | [260, 3] | []
U+1F979 | So | FACE HOLDING BACK TEARS | 3 | [260, 3] | [260, 244, 163, 169, 189]
U+1FAE1 | So | SALUTING FACE | 2 | [260, 3] | [260, 244, 163, 175, 1

# Inspect: mDeBERTa SentencePiece Special Pieces

In [12]:
# ---- Section: Imports ----
import json

import sentencepiece as spm

from digikala_review_transformer import config

# ---- Section: Parameters ----
path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
SPECIAL_NAMES = ["[PAD]", "[CLS]", "[SEP]", "[UNK]", "[MASK]"]
FIRST_IDS = 6

# ---- Section: Raw sentencepiece facts ----
sp = spm.SentencePieceProcessor(model_file=str(path / "spm.model"))
print("piece size:", sp.get_piece_size())
print("bos/eos/pad/unk ids:", sp.bos_id(), sp.eos_id(), sp.pad_id(), sp.unk_id())
for i in range(FIRST_IDS):
    print(i, sp.id_to_piece(i), "| control:", sp.is_control(i), "| unknown:", sp.is_unknown(i))
for name in SPECIAL_NAMES:
    print(name, "->", sp.piece_to_id(name))
print("empty text ->", sp.encode(""))

# ---- Section: HF config files (no text, only structure) ----
tok_cfg = json.loads((path / "tokenizer_config.json").read_text(encoding="utf-8"))
print("\ntokenizer_config keys (without added_tokens_decoder):")
print({k: v for k, v in tok_cfg.items() if k != "added_tokens_decoder"})
print("added tokens:", {k: v.get("content") for k, v in tok_cfg.get("added_tokens_decoder", {}).items()})
model_cfg = json.loads((path / "config.json").read_text(encoding="utf-8"))
print("\nconfig.json:", {k: model_cfg.get(k) for k in
      ("vocab_size", "pad_token_id", "type_vocab_size", "max_position_embeddings", "position_buckets", "relative_attention")})

piece size: 250101
bos/eos/pad/unk ids: 1 2 0 3
0 [PAD] | control: True | unknown: False
1 [CLS] | control: True | unknown: False
2 [SEP] | control: True | unknown: False
3 [UNK] | control: False | unknown: True
4 <0x00> | control: False | unknown: False
5 <0x01> | control: False | unknown: False
[PAD] -> 0
[CLS] -> 1
[SEP] -> 2
[UNK] -> 3
[MASK] -> 3
empty text -> []

tokenizer_config keys (without added_tokens_decoder):
{'do_lower_case': False, 'vocab_type': 'spm'}
added tokens: {}

config.json: {'vocab_size': 251000, 'pad_token_id': None, 'type_vocab_size': 0, 'max_position_embeddings': 512, 'position_buckets': 256, 'relative_attention': True}


#  Test: tokenizer_backend Config Key

In [1]:
# ---- Section: Imports ----
import shutil
import tempfile
from pathlib import Path

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

# ---- Section: Helpers ----
CONFIG_DIR = config.ROOT / "configs" / "transformer"
CHILD_TEMPLATE = """extends: base.yaml
run_name: tooka_B
model:
  pretrained_path: models/pretrained/TookaBERT-Base
  max_length: 128
  tokenizer_backend: {backend}
training:
  variant: B
"""
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def load_child(tmp_dir, backend):
    """Write a child config next to a copy of base.yaml and load it."""
    child = tmp_dir / "child.yaml"
    child.write_text(CHILD_TEMPLATE.format(backend=backend), encoding="utf-8")
    return run_config.load_run_config(child)


# ---- Section: Checks ----
reference = run_config.load_run_config(CONFIG_DIR / "tooka_B.yaml")
check(reference["model"]["tokenizer_backend"] == "hf_fast", "tooka_B uses hf_fast from base.yaml",
      "add 'tokenizer_backend: hf_fast' under model: in base.yaml")

with tempfile.TemporaryDirectory() as tmp:
    tmp_dir = Path(tmp)
    shutil.copy(CONFIG_DIR / "base.yaml", tmp_dir / "base.yaml")

    same = load_child(tmp_dir, "hf_fast")
    check(run_config.config_hash(same) == run_config.config_hash(reference),
          "control: identical settings give identical hash",
          "the child template differs from tooka_B.yaml; fix the test before trusting the next check")

    other = load_child(tmp_dir, "sentencepiece")
    check(other["model"]["tokenizer_backend"] == "sentencepiece", "sentencepiece value is accepted", "check schema edit")
    check(run_config.config_hash(other) != run_config.config_hash(same),
          "tokenizer_backend changes the config hash", "key is missing from the hash; check RUNTIME_ONLY_KEYS")

    try:
        load_child(tmp_dir, "fast")
        check(False, "unknown backend is rejected", "add the check in _check_values")
    except run_config.ConfigError as error:
        check("tokenizer_backend" in str(error), "unknown backend is rejected with a clear message",
              "the error does not name the key")

print("\nALL PASSED" if all(results) else "\nFAILED")

[PASS] tooka_B uses hf_fast from base.yaml
[PASS] control: identical settings give identical hash
[PASS] sentencepiece value is accepted
[PASS] tokenizer_backend changes the config hash
[PASS] unknown backend is rejected with a clear message

ALL PASSED


# Test: Tokenization Backends, Cache and Round Trip

In [ ]:
# ---- Section: Imports ----
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config
from digikala_review_transformer.training import tokenization as tk

# ---- Section: Parameters ----
SAMPLE_ROWS = 20_000
SHORT_ROWS = 200
CACHE_ROWS = 5_000
MODELS = {
    # name: (backend, max_length, (first, last, pad, unk)) taken from the inspection cells
    "TookaBERT-Base": ("hf_fast", 128, (2, 3, 0, 1)),
    "roberta-fa-zwnj-base": ("hf_fast", 128, (0, 2, 1, 3)),
    "mdeberta-v3-base": ("sentencepiece", 160, (1, 2, 0, 3)),
}
models_dir = config.ROOT / "models" / "pretrained"
cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def raises(function, error_type=tk.TokenizationError):
    try:
        function()
    except error_type:
        return True
    return False


# ---- Section: Sample (validation split, same rows for every model) ----
bodies = pd.read_parquet(pack_dir / "validation.parquet", columns=["body"])["body"]
sample = bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
short = sample[:SHORT_ROWS]
del bodies

# ---- Section: Per-model checks ----
bundles = {}
for name, (backend, max_length, expected_ids) in MODELS.items():
    print(f"\n=== {name} ({backend}, max_length={max_length}) ===")
    bundle = tk.load_tokenizer(models_dir / name, backend, max_length)
    bundles[name] = bundle
    got = (bundle.first_id, bundle.last_id, bundle.pad_id, bundle.unk_id)
    check(got == expected_ids, f"special ids first/last/pad/unk = {got}", f"expected {expected_ids}")

    arrays = tk.encode_texts(bundle, sample)
    check(len(arrays) == SAMPLE_ROWS, "one row of ids per text", "row count differs")
    try:
        tk.validate_arrays(arrays, bundle)
        check(True, "all rows start/end with special ids and fit max_length")
    except tk.TokenizationError as error:
        check(False, "row structure", str(error))

    direct = bundle.encode(short)
    check(all(arrays.row(i).tolist() == direct[i] for i in range(SHORT_ROWS)),
          "flat arrays equal bundle.encode output", "chunk concatenation or offsets are wrong")
    stats = tk.token_stats(arrays, bundle)
    print("   stats:", {k: stats[k] for k in ("mean_length", "rows_at_max_length_pct", "unk_pct_of_tokens",
                                              "rows_with_unk", "max_token_id")})

# ---- Section: sentencepiece vs the default Hugging Face tokenizer (rows without unk) ----
name, (backend, max_length, _) = "mdeberta-v3-base", MODELS["mdeberta-v3-base"]
hf_default = AutoTokenizer.from_pretrained(models_dir / name)
hf_ids = hf_default(sample, truncation=True, max_length=max_length, return_attention_mask=False,
                    return_token_type_ids=False)["input_ids"]
sp_arrays = tk.encode_texts(bundles[name], sample)
hf_unk = bundles[name].unk_id
clean_rows = [i for i, row in enumerate(hf_ids) if hf_unk not in row]
same = sum(sp_arrays.row(i).tolist() == hf_ids[i] for i in clean_rows)
check(same == len(clean_rows),
      f"sentencepiece == HF fast on rows without unk ({same:,}/{len(clean_rows):,})",
      "the two backends differ even without unk; stop and inspect")
print(f"[INFO] rows where HF fast has unk (excluded): {SAMPLE_ROWS - len(clean_rows):,}; "
      f"unk tokens from sentencepiece: {tk.token_stats(sp_arrays, bundles[name])['unk_tokens']:,}")

# ---- Section: Cache ----
bundle = bundles["TookaBERT-Base"]
texts = sample[:CACHE_ROWS]
reference = tk.encode_texts(bundle, texts)
reference_stats = tk.token_stats(reference, bundle)
old_chunks = (tk.ENCODE_CHUNK_ROWS, tk.STATS_CHUNK_ROWS)
tk.ENCODE_CHUNK_ROWS, tk.STATS_CHUNK_ROWS = 1_500, 700  # force several chunks
try:
    chunked = tk.encode_texts(bundle, texts)
    chunked_stats = tk.token_stats(chunked, bundle)
finally:
    tk.ENCODE_CHUNK_ROWS, tk.STATS_CHUNK_ROWS = old_chunks
check(chunked.equals(reference), "chunked encoding equals one-shot encoding", "chunk boundaries break rows")
check(chunked_stats == reference_stats, "chunked stats equal one-shot stats", "stats loop is wrong")


def no_texts():
    raise AssertionError("load_texts must not be called when the cache is valid")


with tempfile.TemporaryDirectory() as tmp:
    cache_dir = Path(tmp) / "cache"
    first, stats1, origin1 = tk.get_split_tokens(bundle, "validation", "sha_a", lambda: texts, CACHE_ROWS, cache_dir,
                                                 show_progress=False)
    check(origin1 == "built", "first call builds the cache", f"origin was {origin1}")
    second, stats2, origin2 = tk.get_split_tokens(bundle, "validation", "sha_a", no_texts, CACHE_ROWS, cache_dir)
    check(origin2 == "cache" and second.equals(first) and stats2 == stats1,
          "second call reads the identical arrays from cache", "cache round trip changed the data")
    _, _, origin3 = tk.get_split_tokens(bundle, "validation", "sha_b", lambda: texts, CACHE_ROWS, cache_dir,
                                        show_progress=False)
    check(origin3 == "built", "a different source sha256 invalidates the cache", "stale cache was reused")
    check(raises(lambda: tk.get_split_tokens(bundle, "validation", "sha_b", no_texts, CACHE_ROWS + 1, cache_dir,
                                             show_progress=False), AssertionError),
          "expected_rows mismatch is not served from cache", "cache ignored the row count")

    # Damage the id file on purpose: the cache must be rejected and rebuilt (a [WARN] line is expected).
    np.save(cache_dir / "validation_ids.npy", first.ids[:-5], allow_pickle=False)
    rebuilt, _, origin4 = tk.get_split_tokens(bundle, "validation", "sha_b", lambda: texts, CACHE_ROWS, cache_dir,
                                              show_progress=False)
    check(origin4 == "built" and rebuilt.equals(first), "damaged cache is rebuilt, not trusted", "bad cache was used")
    _, _, origin5 = tk.get_split_tokens(bundle, "validation", "sha_b", no_texts, CACHE_ROWS, cache_dir)
    check(origin5 == "cache", "rebuilt cache is valid again", "cache was not rewritten")

    _, _, origin6 = tk.get_split_tokens(bundle, "validation", "sha_a", lambda: texts, CACHE_ROWS, None,
                                        show_progress=False)
    check(origin6 == "built", "cache_dir=None builds without a disk cache", "unexpected origin")
    check(raises(lambda: tk.get_split_tokens(bundle, "validation", "sha_a", lambda: texts[:-1], CACHE_ROWS, None,
                                             show_progress=False)),
          "built row count must match expected_rows", "length mismatch not detected")

# ---- Section: Save and reload ----
with tempfile.TemporaryDirectory() as tmp:
    for name, bundle in bundles.items():
        out_dir = Path(tmp) / name
        bundle.save(out_dir)
        reloaded = tk.load_saved_tokenizer(out_dir)
        check(reloaded.encode(short) == bundle.encode(short) and reloaded.info()["special_ids"] == bundle.info()["special_ids"],
              f"{name}: saved tokenizer reloads and encodes identically", "save/load changed the tokenizer")
        check((out_dir / tk.BACKEND_INFO_NAME).is_file(), f"{name}: tokenizer_backend.json written")
    check((Path(tmp) / "mdeberta-v3-base" / tk.SP_MODEL_NAME).is_file(), "mdeberta: spm.model copied", "copy failed")

# ---- Section: Error handling ----
check(raises(lambda: tk.load_tokenizer(models_dir / "TookaBERT-Base", "unknown", 128)), "unknown backend is rejected")
check(raises(lambda: tk.load_tokenizer(models_dir / "TookaBERT-Base", "sentencepiece", 128)),
      "sentencepiece backend on a folder without spm.model is rejected")
check(raises(lambda: tk.load_tokenizer(models_dir / "mdeberta-v3-base", "sentencepiece", 2)),
      "max_length < 3 is rejected")

print("\nALL PASSED" if all(results) else "\nFAILED")

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== TookaBERT-Base (hf_fast, max_length=128) ===
[PASS] special ids first/last/pad/unk = (2, 3, 0, 1)
[PASS] one row of ids per text
[PASS] all rows start/end with special ids and fit max_length
[PASS] flat arrays equal bundle.encode output
   stats: {'mean_length': 13.097, 'rows_at_max_length_pct': 0.16, 'unk_pct_of_tokens': 0.06223, 'rows_with_unk': 157, 'max_token_id': 47982}

=== roberta-fa-zwnj-base (hf_fast, max_length=128) ===
[PASS] special ids first/last/pad/unk = (0, 2, 1, 3)
[PASS] one row of ids per text
[PASS] all rows start/end with special ids and fit max_length
[PASS] flat arrays equal bundle.encode output
   stats: {'mean_length': 13.566, 'rows_at_max_length_pct': 0.16, 'unk_pct_of_tokens': 0.0, 'rows_with_unk': 0, 'max_token_id': 41947}

=== mdeberta-v3-base (sentencepiece, max_length=160) ===
[PASS] special ids first/last/pad/unk = (1, 2, 0, 3)
[PASS] one row of ids per text
[PASS] all rows start/end with special ids and fit max_length
[PASS] flat arrays equal bundl

[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[PASS] sentencepiece == HF fast on rows without unk (19,372/19,372)
[INFO] rows where HF fast has unk (excluded): 628; unk tokens from sentencepiece: 0
[PASS] chunked encoding equals one-shot encoding
[PASS] chunked stats equal one-shot stats
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] first call builds the cache
[PASS] second call reads the identical arrays from cache
[WARN] validation: cache key differs (new pack, tokenizer or max_length) -> rebuilding
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] a different source sha256 invalidates the cache
[WARN] validation: cache unreadable: cache has 5,000 rows, expected 5,001 -> rebuilding
[PASS] expected_rows mismatch is not served from cache
[WARN] validation: cache unreadable: token count differs from meta.json -> rebuilding
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] damaged cache is rebuilt, not trusted
[PASS] rebuilt cache is valid again
[timer] tokenize_TookaBERT-Base_validation: 0.1s
[PASS] cache_dir=

#  Test: Data Module, Batch Order, Padding and Spawn Loader

In [ ]:
# ---- Section: Imports ----
import numpy as np
import pandas as pd
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import data, pack, run_config
from digikala_review_transformer.training import tokenization as tk

# ---- Section: Parameters ----
SUBSET_ROWS = 20_000
MONITOR_ROWS = 5_000
BATCH = 64
EVAL_BATCH = 512
MIN_SPREAD_RATIO = 0.5
KNOWN_TRAIN_COUNTS = {"rec": [1_834_524, 266_638, 290_995], "band": [549_788, 500_933, 1_809_515]}
SEED = config.SPLIT_SEED

cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])
model_dir = run_config.project_path(cfg["model"]["pretrained_path"])
train_path = pack_dir / "train.parquet"
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def raises(function, error_type=data.DataError):
    try:
        function()
    except error_type:
        return True
    return False


def same_batch(a, b):
    return a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a)


# ---- Section: Pack and class weights on the full train labels ----
manifest = data.open_pack(pack_dir)
check(True, "open_pack verified sha256, rows and columns")
labels_full = pd.read_parquet(train_path, columns=["y_rec", "y_band"])
for head in data.HEADS:
    codes = data.to_codes(labels_full[f"y_{head}"], head)
    counts = np.bincount(codes[codes >= 0], minlength=3).tolist()
    check(counts == KNOWN_TRAIN_COUNTS[head], f"{head}: train class counts equal the handoff {counts}",
          f"expected {KNOWN_TRAIN_COUNTS[head]}")
    vector = data.class_weight_vector(codes, head)
    expected = np.array([sum(counts) / (3 * c) for c in counts], dtype=np.float32)
    check(np.allclose(vector, expected, rtol=1e-6), f"{head}: class weights = n/(k*n_c) -> {vector.round(4).tolist()}",
          "labels.class_weights differs from the plan formula")
del labels_full, codes

# ---- Section: Subsets of train for B and C ----
bundle = tk.load_tokenizer(model_dir, cfg["model"]["tokenizer_backend"], cfg["model"]["max_length"])
split_b = data.load_split("train", bundle, pack_dir, manifest, "B", subset_rows=SUBSET_ROWS, show_progress=False)
split_c = data.load_split("train", bundle, pack_dir, manifest, "C", subset_rows=SUBSET_ROWS, show_progress=False)
n = split_b.n_rows
check(n == SUBSET_ROWS and split_b.origin == "built", "subset has the requested rows and is built without cache")
check(np.array_equal(split_b.source_rows, split_c.source_rows) and split_b.arrays.equals(split_c.arrays),
      "B and C use identical rows and tokens", "subset selection depends on the variant")

pack_labels = pd.read_parquet(train_path, columns=["id"] + pack.LABEL_COLUMNS).iloc[split_b.source_rows]
check(np.array_equal(split_b.ids, pack_labels["id"].to_numpy()), "ids follow source_rows")
for head in data.HEADS:
    raw_codes = data.to_codes(pack_labels[f"y_{head}_raw"], head)
    check(np.array_equal(split_b.y_raw[head], raw_codes), f"{head}: raw labels match the pack")
    check(np.array_equal(split_b.y[head], split_c.y[head]), f"{head}: corrected labels do not depend on the variant")
    check(int((split_b.y[head] == data.IGNORE_INDEX).sum()) == int(pack_labels[f"y_{head}"].isna().sum()),
          f"{head}: missing labels become {data.IGNORE_INDEX}")

pack_weights = pd.read_parquet(train_path, columns=data.WEIGHT_COLUMNS).iloc[split_b.source_rows]
for head in data.HEADS:
    check(bool((split_b.w[head] == 1).all()), f"{head}: variant B weights are all 1")
    original = pack_weights[f"w_{head}"].to_numpy(dtype="float32", na_value=np.nan)
    finite = ~np.isnan(original)
    check(np.isfinite(split_c.w[head]).all(), f"{head}: variant C has no NaN after replacement")
    check(np.array_equal(split_c.w[head][finite], original[finite]) and bool((split_c.w[head][~finite] == 1).all()),
          f"{head}: variant C keeps Step 2 weights and uses 1 for NaN ({int((~finite).sum()):,} NaN rows)",
          "weights were altered")
    check(0.1 - 1e-6 <= split_c.w[head][finite].min() and split_c.w[head].max() <= 1.0 + 1e-6,
          f"{head}: variant C weights lie in [0.1, 1]")

# ---- Section: Batch order ----
sampler_b = data.LengthBucketedBatches(split_b.lengths, BATCH, SEED)
sampler_c = data.LengthBucketedBatches(split_c.lengths, BATCH, SEED)
epoch0 = sampler_b.batches_for_epoch(0)
check(np.array_equal(np.sort(np.concatenate(epoch0)), np.arange(n)), "every row appears exactly once per epoch")
check(len(epoch0) == len(sampler_b) == -(-n // BATCH), "number of batches = ceil(rows / batch_size)")
check(all(len(b) == BATCH for b in epoch0[:-1]) or sum(len(b) != BATCH for b in epoch0) == 1,
      "at most one batch is smaller than batch_size")
check(data.batch_fingerprint(epoch0) == data.batch_fingerprint(sampler_b.batches_for_epoch(0)),
      "same seed and epoch give the same order")
check(data.batch_fingerprint(epoch0) != data.batch_fingerprint(sampler_b.batches_for_epoch(1)),
      "a different epoch gives a different order", "epoch is ignored")
check(data.batch_fingerprint(epoch0) == data.batch_fingerprint(sampler_c.batches_for_epoch(0)),
      "B and C see identical batches", "order depends on the variant")
check(data.batch_fingerprint(epoch0) != data.batch_fingerprint(
    data.LengthBucketedBatches(split_b.lengths, BATCH, SEED + 1).batches_for_epoch(0)),
      "a different seed gives a different order")

spread = np.mean([split_b.lengths[b].max() - split_b.lengths[b].min() for b in epoch0])
random_order = np.random.default_rng(0).permutation(n)
random_spread = np.mean([
    np.ptp(split_b.lengths[random_order[i:i + BATCH]])
    for i in range(0, n, BATCH)
])
check(spread < MIN_SPREAD_RATIO * random_spread,
      f"bucketing shrinks length spread inside batches ({spread:.1f} vs {random_spread:.1f} random)",
      "length sorting is not working")

# ---- Section: Padding and targets ----
dataset = split_c.dataset()
indices = epoch0[0]
batch = dataset[indices]
lengths = split_c.lengths[indices]
check(batch["input_ids"].shape == (len(indices), int(lengths.max())) and batch["input_ids"].dtype == torch.int64,
      "batch shape is (rows, longest row) with int64 ids")
rows_ok = pads_ok = mask_ok = True
for i, row in enumerate(indices):
    length = int(lengths[i])
    rows_ok &= batch["input_ids"][i, :length].tolist() == split_c.arrays.row(row).tolist()
    pads_ok &= bool((batch["input_ids"][i, length:] == split_c.pad_id).all())
    mask_ok &= int(batch["attention_mask"][i].sum()) == length and bool(batch["attention_mask"][i, :length].all())
check(rows_ok, "padded rows keep the exact token ids")
check(pads_ok, f"padding uses pad id {split_c.pad_id}")
check(mask_ok, "attention mask is 1 on tokens and 0 on padding")
check(torch.equal(batch["index"], torch.from_numpy(indices)), "batch carries the row indices")
check(all(torch.equal(batch[f"y_{h}"], torch.from_numpy(split_c.y[h][indices])) for h in data.HEADS),
      "targets are aligned with rows")
check(all(torch.equal(batch[f"w_{h}"], torch.from_numpy(split_c.w[h][indices])) for h in data.HEADS),
      "weights are aligned with rows")

# Rows without a band label (future semi-supervised use) must collate without any rewrite.
no_band = data.BatchedTokenDataset(split_c.arrays, {"rec": split_c.y["rec"], "band": np.full(n, data.IGNORE_INDEX)},
                                   split_c.w, split_c.pad_id)
check(bool((no_band[indices]["y_band"] == data.IGNORE_INDEX).all()), "rows without band label collate fine")

# ---- Section: Spawn loader with workers (also checks set_epoch with persistent workers) ----
sampler = data.LengthBucketedBatches(split_c.lengths, BATCH, SEED)
loader = data.make_loader(dataset, sampler, num_workers=2, persistent=True)
sampler.set_epoch(0)
got0 = list(loader)
check(len(got0) == len(sampler), "loader yields every batch")
check(all(same_batch(got0[i], dataset[sampler.batches_for_epoch(0)[i]]) for i in (0, 1, len(got0) - 1)),
      "spawn workers return the same batches as direct indexing", "pickling or collate differs in workers")
sampler.set_epoch(1)
got1 = list(loader)
check(same_batch(got1[0], dataset[sampler.batches_for_epoch(1)[0]]) and not same_batch(got0[0], got1[0]),
      "set_epoch changes the order of a persistent loader", "workers cache the old order")
del loader, got0, got1

# ---- Section: Evaluation batches ----
monitor = data.subset_row_indices(n, MONITOR_ROWS, SEED, "monitor")
smoke = data.subset_row_indices(n, MONITOR_ROWS, SEED, "smoke")
check(len(monitor) == MONITOR_ROWS and (np.diff(monitor) > 0).all(), "monitor subset: right size, sorted, unique")
check(np.array_equal(monitor, data.subset_row_indices(n, MONITOR_ROWS, SEED, "monitor")), "monitor subset is reproducible")
check(not np.array_equal(monitor, smoke), "monitor and smoke subsets use different streams")
eval_batches = data.sorted_eval_batches(split_c.lengths, monitor, EVAL_BATCH)
flat = np.concatenate(eval_batches)
check(np.array_equal(np.sort(flat), monitor) and max(len(b) for b in eval_batches) <= EVAL_BATCH,
      "eval batches cover the subset once and respect the batch size")
check(bool((np.diff(split_c.lengths[flat]) >= 0).all()), "eval rows are sorted by length")
eval_loader = data.make_loader(dataset, eval_batches, num_workers=0)
seen = np.concatenate([b["index"].numpy() for b in eval_loader])
check(np.array_equal(seen, flat), "eval loader returns rows in the planned order")

# ---- Section: Full validation split (real size, variant C must still give weight 1) ----
validation = data.load_split("validation", bundle, pack_dir, manifest, "C", show_progress=False)
entry = manifest["files"]["validation.parquet"]
check(validation.n_rows == entry["rows"] and pd.Series(validation.ids).is_unique,
      f"validation: {validation.n_rows:,} rows with unique ids")
check(all(int((validation.y[h] == data.IGNORE_INDEX).sum()) == entry["label_missing"][f"y_{h}"] for h in data.HEADS),
      "validation: missing label counts equal the manifest")
check(all(bool((validation.w[h] == 1).all()) for h in data.HEADS), "validation: weights are 1 (no weight columns)")
print("   validation token stats:", {k: validation.stats[k] for k in ("mean_length", "rows_at_max_length_pct", "unk_pct_of_tokens")})

# ---- Section: Error handling ----
check(raises(lambda: data.class_weight_vector(np.array([0, 0, 1, 0, data.IGNORE_INDEX]), "rec")),
      "class weights with a missing class are rejected")
check(raises(lambda: data.load_split("dev", bundle, pack_dir, manifest, "B")), "unknown split is rejected")
check(raises(lambda: data.load_split("validation", bundle, pack_dir, manifest, "D")), "unknown variant is rejected")
check(raises(lambda: data.load_split("validation", bundle, pack_dir, manifest, "B", subset_rows=10, cache_dir=tk.CACHE_ROOT)),
      "subset together with disk cache is rejected")
check(raises(lambda: data.subset_row_indices(10, 5, SEED, "other")), "unknown subset purpose is rejected")

print("\nALL PASSED" if all(results) else "\nFAILED")

[PASS] open_pack verified sha256, rows and columns
[PASS] rec: train class counts equal the handoff [1834524, 266638, 290995]
[PASS] rec: class weights = n/(k*n_c) -> [0.43470001220703125, 2.990499973297119, 2.7402000427246094]
[PASS] band: train class counts equal the handoff [549788, 500933, 1809515]
[PASS] band: class weights = n/(k*n_c) -> [1.7340999841690063, 1.9033000469207764, 0.5268999934196472]
[timer] tokenize_TookaBERT-Base_train: 0.2s
[timer] tokenize_TookaBERT-Base_train: 0.2s
[PASS] subset has the requested rows and is built without cache
[PASS] B and C use identical rows and tokens
[PASS] ids follow source_rows
[PASS] rec: raw labels match the pack
[PASS] rec: corrected labels do not depend on the variant
[PASS] rec: missing labels become -100
[PASS] band: raw labels match the pack
[PASS] band: corrected labels do not depend on the variant
[PASS] band: missing labels become -100
[PASS] rec: variant B weights are all 1
[PASS] rec: variant C has no NaN after replacement
[P

# Test: Model, Heads, Parameter Groups and Padding Invariance

In [ ]:
# ---- Section: Imports ----
import copy
import gc
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import data, run_config
from digikala_review_transformer.training import model as mdl
from digikala_review_transformer.training import tokenization as tk

# ---- Section: Parameters ----
MODELS = {
    # name: (backend, max_length)
    "TookaBERT-Base": ("hf_fast", 128),
    "roberta-fa-zwnj-base": ("hf_fast", 128),
    "mdeberta-v3-base": ("sentencepiece", 160),
}
SAMPLE_ROWS = 2_000
FORWARD_ROWS = 4
PAD_ATOL = 1e-3
STD_TOLERANCE = 0.2
LR_ENCODER, LR_HEADS, WEIGHT_DECAY = 2.0e-5, 1.0e-4, 0.01

base_cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(base_cfg["data"]["pack_dir"])
models_dir = config.ROOT / "models" / "pretrained"
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def raises(function, error_type=mdl.ModelError):
    try:
        function()
    except error_type:
        return True
    return False


def cfg_for(name, seed=None):
    cfg = copy.deepcopy(base_cfg)
    cfg["model"]["pretrained_path"] = str(models_dir / name)
    if seed is not None:
        cfg["seed"] = seed
    return cfg


def heads_equal(a, b):
    return all(torch.equal(x, y) for x, y in zip(a.heads.state_dict().values(), b.heads.state_dict().values()))


# ---- Section: Sample texts (validation, fixed seed) ----
bodies = pd.read_parquet(pack_dir / "validation.parquet", columns=["body"])["body"]
sample = bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
del bodies

# ---- Section: Per-model checks ----
for name, (backend, max_length) in MODELS.items():
    print(f"\n=== {name} ===")
    cfg = cfg_for(name)
    bundle = tk.load_tokenizer(models_dir / name, backend, max_length)
    model, load_info = mdl.build_model(cfg)
    print("   load_info:", load_info, "| encoder:", type(model.encoder).__name__)

    hidden = model.encoder.config.hidden_size
    check(all(p.dtype == torch.float32 for p in model.parameters()), "all parameters are float32",
          "dtype was not forced to float32")
    check(getattr(model.encoder, "pooler", None) is None, "pooler is removed")
    check(model.dropout.p == cfg["model"]["head_dropout"], f"head dropout = {model.dropout.p}")

    counts = mdl.count_parameters(model)
    expected_heads = sum(hidden * len(mdl.labels.CLASSES[h]) + len(mdl.labels.CLASSES[h]) for h in mdl.HEADS)
    check(counts["heads"] == expected_heads and counts["total"] == counts["encoder"] + counts["heads"]
          and counts["trainable"] == counts["total"],
          f"parameter counts are consistent {counts}", "counts do not add up")

    stds = [layer.weight.std().item() for layer in model.heads.values()]
    check(all(abs(s - mdl.HEAD_INIT_STD) < STD_TOLERANCE * mdl.HEAD_INIT_STD for s in stds)
          and all(bool((layer.bias == 0).all()) for layer in model.heads.values()),
          f"head init: std {[round(s, 4) for s in stds]}, zero bias", "init differs from the plan")

    # Token ids vs embedding rows.
    arrays = tk.encode_texts(bundle, sample)
    stats = tk.token_stats(arrays, bundle)
    try:
        mdl.check_token_ids_fit(model, stats["max_token_id"], bundle.pad_id)
        check(True, f"token ids fit the embedding (max id {stats['max_token_id']})")
    except mdl.ModelError as error:
        check(False, "token ids fit the embedding", str(error))
    check(raises(lambda: mdl.check_token_ids_fit(model, 10 ** 9, bundle.pad_id)), "oversized token id is rejected")  # noqa: B023
    check(raises(lambda: mdl.check_token_ids_fit(model, 1, bundle.pad_id + 1))  # noqa: B023
          or getattr(model.encoder.config, "pad_token_id", None) is None,
          "wrong pad id is rejected (when the config has a pad id)")

    # Forward: padding invariance proves attention mask and position 0 are used correctly.
    zeros = {h: np.full(SAMPLE_ROWS, data.IGNORE_INDEX) for h in data.HEADS}
    ones = {h: np.ones(SAMPLE_ROWS, dtype=np.float32) for h in data.HEADS}
    dataset = data.BatchedTokenDataset(arrays, zeros, ones, bundle.pad_id)
    order = np.argsort(arrays.lengths)
    rows = np.array([order[0], order[SAMPLE_ROWS // 3], order[2 * SAMPLE_ROWS // 3], order[-1]])[:FORWARD_ROWS]
    model.eval()
    with torch.no_grad():
        batch = dataset[rows]
        together = model(batch["input_ids"], batch["attention_mask"])
        alone = [model(*(lambda b: (b["input_ids"], b["attention_mask"]))(dataset[rows[i:i + 1]])) for i in range(len(rows))]  # noqa: PLC3002
    check(all(together[h].shape == (len(rows), 3) and together[h].dtype == torch.float32
              and bool(torch.isfinite(together[h]).all()) for h in mdl.HEADS),
          "forward gives finite float32 logits of shape (rows, 3) per head")
    gap = max((together[h][i] - alone[i][h][0]).abs().max().item() for h in mdl.HEADS for i in range(len(rows)))
    check(gap < PAD_ATOL, f"logits do not depend on padding (max gap {gap:.2e}; lengths {arrays.lengths[rows].tolist()})",
          "attention mask or feature position is wrong")

    # Backward: heads get gradients; report encoder tensors without gradient.
    model.train()
    model.zero_grad(set_to_none=True)
    logits = model(batch["input_ids"], batch["attention_mask"])
    (logits["rec"].sum() + logits["band"].sum()).backward()
    check(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.heads.parameters()),
          "heads receive finite gradients")
    no_grad = [n for n, p in model.encoder.named_parameters() if p.grad is None]
    check(len(no_grad) / counts["encoder"] < 1e-3 or len(no_grad) < 5,
          f"almost every encoder tensor receives a gradient ({len(no_grad)} without: {no_grad[:3]})",
          "large part of the encoder is disconnected from the loss")

    # Parameter groups and one optimizer step.
    groups = mdl.parameter_groups(model, LR_ENCODER, LR_HEADS, WEIGHT_DECAY)
    flat = [p for g in groups for p in g["params"]]
    check(len(flat) == len({id(p) for p in flat}) == len(list(model.parameters())),
          f"every parameter is in exactly one group ({[g['name'] for g in groups]})")
    check(all((g["weight_decay"] > 0) == all(p.ndim >= 2 for p in g["params"]) for g in groups)
          and all(p.ndim < 2 for g in groups if g["weight_decay"] == 0 for p in g["params"]),
          "weight decay only on tensors with ndim >= 2")
    check(all(g["lr"] == (LR_ENCODER if g["name"].startswith("encoder") else LR_HEADS) for g in groups),
          "encoder and head learning rates are applied")
    before = model.heads["rec"].weight.detach().clone()
    torch.optim.AdamW(groups).step()
    check(not torch.equal(before, model.heads["rec"].weight.detach()), "an AdamW step changes the head weights")

    # Saved config.
    with tempfile.TemporaryDirectory() as tmp:
        target = Path(tmp) / "encoder_config.json"
        mdl.save_encoder_config(model, target)
        check(target.is_file() and f'"hidden_size": {hidden}' in target.read_text(encoding="utf-8"),
              "encoder_config.json is written")
    summary = mdl.model_summary(model, load_info)
    check(summary["parameters"] == counts and summary["dtype"] == "torch.float32", "model_summary is consistent")

    del model, bundle, arrays, dataset, batch, together, alone, logits, groups, flat
    gc.collect()

# ---- Section: Same seed -> same heads (variants B and C) ----
print("\n=== seeds ===")
first, _ = mdl.build_model(cfg_for("TookaBERT-Base"))
second, _ = mdl.build_model(cfg_for("TookaBERT-Base"))
third, _ = mdl.build_model(cfg_for("TookaBERT-Base", seed=config.SPLIT_SEED + 1))
check(heads_equal(first, second), "same seed gives identical head initialisation")
check(not heads_equal(first, third), "a different seed gives different heads")
check(all(torch.equal(a, b) for a, b in zip(first.encoder.state_dict().values(), second.encoder.state_dict().values())),
      "encoder weights are identical across builds")
del first, second, third
gc.collect()

# ---- Section: Error handling ----
bad_cfg = cfg_for("TookaBERT-Base")
bad_cfg["model"]["pretrained_path"] = str(models_dir / "does_not_exist")
check(raises(lambda: mdl.build_model(bad_cfg)), "missing model folder is rejected")

print("\nALL PASSED" if all(results) else "\nFAILED")


=== TookaBERT-Base ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8076.61it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream ta

   load_info: {'unexpected_keys': 5, 'pooler_keys_missing': 2} | encoder: BertModel
[PASS] all parameters are float32
[PASS] pooler is removed
[PASS] head dropout = 0.1
[PASS] parameter counts are consistent {'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[PASS] head init: std [0.0197, 0.0199], zero bias
[PASS] token ids fit the embedding (max id 47187)
[PASS] oversized token id is rejected
[PASS] wrong pad id is rejected (when the config has a pad id)
[PASS] forward gives finite float32 logits of shape (rows, 3) per head
[PASS] logits do not depend on padding (max gap 1.05e-06; lengths [3, 6, 12, 128])
[PASS] heads receive finite gradients
[PASS] almost every encoder tensor receives a gradient (0 without: [])
[PASS] every parameter is in exactly one group (['encoder_decay', 'encoder_no_decay', 'heads_decay', 'heads_no_decay'])
[PASS] weight decay only on tensors with ndim >= 2
[PASS] encoder and head learning rates are applied
[PASS] an AdamW step cha

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 50475.13it/s]
[transformers] RobertaModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/roberta-fa-zwnj-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   load_info: {'unexpected_keys': 7, 'pooler_keys_missing': 2} | encoder: RobertaModel
[PASS] all parameters are float32
[PASS] pooler is removed
[PASS] head dropout = 0.1
[PASS] parameter counts are consistent {'total': 117712134, 'trainable': 117712134, 'encoder': 117707520, 'heads': 4614}
[PASS] head init: std [0.0197, 0.0199], zero bias
[PASS] token ids fit the embedding (max id 41944)
[PASS] oversized token id is rejected
[PASS] wrong pad id is rejected (when the config has a pad id)
[PASS] forward gives finite float32 logits of shape (rows, 3) per head
[PASS] logits do not depend on padding (max gap 5.36e-07; lengths [3, 7, 12, 128])
[PASS] heads receive finite gradients
[PASS] almost every encoder tensor receives a gradient (0 without: [])
[PASS] every parameter is in exactly one group (['encoder_decay', 'encoder_no_decay', 'heads_decay', 'heads_no_decay'])
[PASS] weight decay only on tensors with ndim >= 2
[PASS] encoder and head learning rates are applied
[PASS] an AdamW step 

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 865.30it/s]
[transformers] DebertaV2Model LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_prediction

   load_info: {'unexpected_keys': 12, 'pooler_keys_missing': 0} | encoder: DebertaV2Model
[PASS] all parameters are float32
[PASS] pooler is removed
[PASS] head dropout = 0.1
[PASS] parameter counts are consistent {'total': 278223366, 'trainable': 278223366, 'encoder': 278218752, 'heads': 4614}
[PASS] head init: std [0.0197, 0.0199], zero bias
[PASS] token ids fit the embedding (max id 242648)
[PASS] oversized token id is rejected
[PASS] wrong pad id is rejected (when the config has a pad id)
[PASS] forward gives finite float32 logits of shape (rows, 3) per head
[PASS] logits do not depend on padding (max gap 2.68e-06; lengths [3, 10, 19, 160])
[PASS] heads receive finite gradients
[PASS] almost every encoder tensor receives a gradient (0 without: [])
[PASS] every parameter is in exactly one group (['encoder_decay', 'encoder_no_decay', 'heads_decay', 'heads_no_decay'])
[PASS] weight decay only on tensors with ndim >= 2
[PASS] encoder and head learning rates are applied
[PASS] an AdamW 

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16316.70it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[PASS] same seed gives identical head initialisation
[PASS] a different seed gives different heads
[PASS] encoder weights are identical across builds
[PASS] missing model folder is rejected

ALL PASSED


#  Inspect: mDeBERTa Embedding Keys in the Checkpoint

In [7]:
# ---- Section: Imports ----
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import model as mdl

# ---- Section: Parameters ----
path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
KEY_FILTER = "word_embeddings"

# ---- Section: Embedding-related keys in the raw checkpoint ----
state = torch.load(path / "pytorch_model.bin", map_location="cpu", weights_only=True)
keys = [k for k in state if KEY_FILTER in k]
print("keys containing", KEY_FILTER, ":")
for k in keys:
    t = state[k]
    print(f"  {k} | shape={tuple(t.shape)} | dtype={t.dtype} | mean={t.float().mean():.5f} | std={t.float().std():.5f}")

weight_key = next((k for k in keys if k.endswith("word_embeddings.weight")), None)
extra_key = next((k for k in keys if k.endswith("word_embeddings._weight")), None)
print("\nweight key:", weight_key, "| extra key:", extra_key)

# ---- Section: Compare the two tensors ----
if weight_key and extra_key:
    a, b = state[weight_key].float(), state[extra_key].float()
    if a.shape == b.shape:
        diff = (a - b).abs()
        print(f"max |weight - _weight| = {diff.max():.6f} | mean = {diff.mean():.6f} | identical: {torch.equal(a, b)}")
        print(f"norm(weight) = {a.norm():.2f} | norm(_weight) = {b.norm():.2f}")
    else:
        print("shapes differ:", tuple(a.shape), tuple(b.shape))

# ---- Section: What the loaded model actually uses ----
encoder, info = mdl.load_encoder(path)
loaded = encoder.get_input_embeddings().weight.detach().float()
if weight_key:
    print("\nmodel embedding == checkpoint .weight:", torch.equal(loaded, state[weight_key].float()))
if extra_key and state[extra_key].shape == loaded.shape:
    print("model embedding == checkpoint ._weight:", torch.equal(loaded, state[extra_key].float()))
print("model embedding shape:", tuple(loaded.shape), "| load_info:", info)

keys containing word_embeddings :
  deberta.embeddings.word_embeddings.weight | shape=(251000, 768) | dtype=torch.float16 | mean=-0.01234 | std=0.14396
  deberta.embeddings.word_embeddings._weight | shape=(251000, 768) | dtype=torch.float32 | mean=0.01802 | std=0.15646

weight key: deberta.embeddings.word_embeddings.weight | extra key: deberta.embeddings.word_embeddings._weight
max |weight - _weight| = 1.315626 | mean = 0.054561 | identical: False
norm(weight) = 1957.61 | norm(_weight) = 2129.99


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1042.28it/s]
[transformers] DebertaV2Model LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.L


model embedding == checkpoint .weight: True
model embedding == checkpoint ._weight: False
model embedding shape: (251000, 768) | load_info: {'unexpected_keys': 12, 'pooler_keys_missing': 0}


# Inspect: mDeBERTa Embedding vs Generator Embedding

In [8]:
# ---- Section: Imports ----
from collections import Counter

import torch

from digikala_review_transformer import config

# ---- Section: Parameters ----
path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
EMB = "deberta.embeddings.word_embeddings"
ROW_QUANTILES = torch.tensor([0.5, 0.9, 0.99, 1.0])
SAMPLE_PREVIEW = 3


# ---- Section: Helpers ----
def describe(tensor):
    t = tensor.float()
    return f"shape={tuple(tensor.shape)} dtype={tensor.dtype} mean={t.mean():.5f} std={t.std():.5f} norm={t.norm():.2f}"


def compare(name_a, a, name_b, b):
    """Print numeric closeness of two tensors of the same shape."""
    if a.shape != b.shape:
        print(f"{name_a} vs {name_b}: shapes differ {tuple(a.shape)} {tuple(b.shape)}")
        return
    a, b = a.float(), b.float()
    diff = (a - b).abs()
    cosine = torch.nn.functional.cosine_similarity(a, b, dim=1)
    row_gap = (a - b).norm(dim=1) / a.norm(dim=1).clamp_min(1e-8)
    print(f"{name_a} vs {name_b}: identical={torch.equal(a, b)} | max|diff|={diff.max():.6f} | mean|diff|={diff.mean():.6f}")
    print(f"   row cosine mean={cosine.mean():.4f} min={cosine.min():.4f} | "
          f"relative row gap quantiles (50/90/99/100%)={[round(v, 4) for v in torch.quantile(row_gap, ROW_QUANTILES).tolist()]}")


# ---- Section: Discriminator checkpoint ----
disc = torch.load(path / "pytorch_model.bin", map_location="cpu", weights_only=True)
print("discriminator tensors by dtype:", dict(Counter(str(t.dtype) for t in disc.values())))
weight, extra = disc[f"{EMB}.weight"], disc[f"{EMB}._weight"]
print("weight :", describe(weight))
print("_weight:", describe(extra))
print("weight has non-finite values:", not bool(torch.isfinite(weight.float()).all()),
      "| _weight has non-finite values:", not bool(torch.isfinite(extra).all()))

# ---- Section: Generator checkpoint ----
gen = torch.load(path / "pytorch_model.generator.bin", map_location="cpu", weights_only=True)
print("\ngenerator tensors by dtype:", dict(Counter(str(t.dtype) for t in gen.values())))
gen_keys = [k for k in gen if "word_embeddings" in k]
print("generator embedding keys:", gen_keys)
for key in gen_keys:
    print(" ", key, describe(gen[key]))

# ---- Section: Comparisons ----
print()
compare("discriminator .weight", weight, "discriminator ._weight", extra)
for key in gen_keys:
    compare("discriminator .weight", weight, f"generator {key.split('.')[-1]}", gen[key])
    compare("discriminator ._weight", extra, f"generator {key.split('.')[-1]}", gen[key])

discriminator tensors by dtype: {'torch.float16': 210, 'torch.float32': 2}
weight : shape=(251000, 768) dtype=torch.float16 mean=-0.01234 std=0.14396 norm=1957.61
_weight: shape=(251000, 768) dtype=torch.float32 mean=0.01802 std=0.15646 norm=2129.99
weight has non-finite values: False | _weight has non-finite values: False

generator tensors by dtype: {'torch.float32': 108}
generator embedding keys: ['deberta.embeddings.word_embeddings.weight']
  deberta.embeddings.word_embeddings.weight shape=(251000, 768) dtype=torch.float32 mean=-0.03036 std=0.06194 norm=933.96

discriminator .weight vs discriminator ._weight: identical=False | max|diff|=1.315626 | mean|diff|=0.054561
   row cosine mean=0.8913 min=0.3470 | relative row gap quantiles (50/90/99/100%)=[0.4706, 0.5431, 2.219, 2.3229]
discriminator .weight vs generator weight: identical=False | max|diff|=1.957709 | mean|diff|=0.124575
   row cosine mean=0.0402 min=-0.1387 | relative row gap quantiles (50/90/99/100%)=[1.0881, 1.1209, 2.40

# Test: Losses, Masking, Class Weights and Accumulator

In [9]:
# ---- Section: Imports ----
import numpy as np
import torch
import torch.nn.functional as F

from digikala_review_transformer.training import data, losses

# ---- Section: Parameters ----
N_ROWS = 12
SMOOTHING = 0.1
TOLERANCE = 1e-5
CLASS_WEIGHTS = {"rec": [0.6, 1.7, 2.1], "band": [1.8, 1.3, 0.5]}
IGN = data.IGNORE_INDEX
HEADS = data.HEADS
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def raises(function, error_type=losses.LossError):
    try:
        function()
    except error_type:
        return True
    return False


def make_batch(n=N_ROWS, rec_missing=(1, 5), seed=0):
    g = torch.Generator().manual_seed(seed)
    logits = {h: torch.randn(n, 3, generator=g).requires_grad_() for h in HEADS}
    y_rec = torch.randint(0, 3, (n,), generator=g)
    y_rec[list(rec_missing)] = IGN
    batch = {
        "y_rec": y_rec,
        "y_band": torch.randint(0, 3, (n,), generator=g),
        "w_rec": 0.1 + 0.9 * torch.rand(n, generator=g),
        "w_band": 0.1 + 0.9 * torch.rand(n, generator=g),
    }
    return logits, batch


def reference_head_loss(logits, target, w, class_weight, eps):
    """Slow row-by-row version of the plan formula (double precision)."""
    logp = F.log_softmax(logits.detach().double(), dim=1)
    numerator = denominator = 0.0
    for i in range(len(target)):
        t = int(target[i])
        if t == IGN:
            continue
        ce = (1 - eps) * (-logp[i, t]) + eps * (-logp[i].mean())
        e = float(w[i]) * class_weight[t]
        numerator += e * float(ce)
        denominator += e
    return numerator / denominator if denominator > 0 else 0.0


def make_loss(smoothing=SMOOTHING, lam=1.0, weights=CLASS_WEIGHTS):
    return losses.TwoHeadLoss(weights, smoothing, lam, "cpu")


# ---- Section: Formula against a hand-written reference ----
logits, batch = make_batch()
output = make_loss(lam=2.0)(logits, batch)
for head in HEADS:
    expected = reference_head_loss(logits[head], batch[f"y_{head}"], batch[f"w_{head}"], CLASS_WEIGHTS[head], SMOOTHING)
    check(abs(float(output.head_loss[head]) - expected) < TOLERANCE,
          f"{head}: loss equals the row-by-row reference ({float(output.head_loss[head]):.6f})",
          "formula differs from the plan")
check(abs(float(output.total) - (float(output.head_loss["rec"]) + 2.0 * float(output.head_loss["band"]))) < TOLERANCE,
      "total = loss_rec + loss_weight_band * loss_band", "head combination is wrong")
check(output.total.dtype == torch.float32 and output.total.requires_grad, "total is float32 and keeps the graph")
check(not any(v.requires_grad for v in output.head_loss.values()), "per-head values are detached")

# ---- Section: Special cases of the weighting ----
ones = {h: [1.0, 1.0, 1.0] for h in HEADS}
flat = make_loss(weights=ones)
batch_ones = {**batch, "w_rec": torch.ones(N_ROWS), "w_band": torch.ones(N_ROWS)}
out_flat = flat(logits, batch_ones)
valid = batch["y_rec"] != IGN
mean_ce = F.cross_entropy(logits["rec"][valid], batch["y_rec"][valid], label_smoothing=SMOOTHING)
check(abs(float(out_flat.head_loss["rec"]) - float(mean_ce)) < TOLERANCE,
      "w=1 and class weights 1 give the plain smoothed mean CE over valid rows", "normalisation is wrong")

scaled = make_loss()(logits, {**batch, "w_rec": batch["w_rec"] * 3.7, "w_band": batch["w_band"] * 3.7})
check(all(abs(float(scaled.head_loss[h]) - float(output.head_loss[h])) < TOLERANCE for h in HEADS),
      "multiplying all sample weights by a constant does not change the loss", "denominator is not normalising")

# ---- Section: Masking ----
grad_rec = torch.autograd.grad(make_loss()(logits, batch).total, logits["rec"])[0]
masked_rows = (batch["y_rec"] == IGN).nonzero().flatten()
check(bool((grad_rec[masked_rows] == 0).all()) and bool((grad_rec[~(batch["y_rec"] == IGN)] != 0).any()),
      f"rows without rec label get zero gradient ({len(masked_rows)} masked rows)", "masked rows leak into the loss")
band_same = torch.allclose(
    make_loss()(logits, {**batch, "y_rec": torch.full((N_ROWS,), IGN)}).head_loss["band"], output.head_loss["band"])
check(band_same, "band loss is unchanged when rec is fully masked", "heads are not independent")

logits_empty, batch_empty = make_batch()
batch_empty["y_rec"] = torch.full((N_ROWS,), IGN)
batch_empty["y_band"] = torch.full((N_ROWS,), IGN)
out_empty = make_loss()(logits_empty, batch_empty)
out_empty.total.backward()
check(float(out_empty.total) == 0.0 and all(bool(torch.isfinite(l.grad).all()) and bool((l.grad == 0).all())
                                            for l in logits_empty.values()),
      "batch with no valid label gives loss 0 and zero finite gradients", "empty batch produces NaN or garbage")
check(all(int(out_empty.n_valid[h]) == 0 and float(out_empty.plain_ce_sum[h]) == 0.0 for h in HEADS),
      "empty batch: plain CE sum 0 and count 0")

# ---- Section: Plain cross entropy ----
for head in HEADS:
    target = batch[f"y_{head}"]
    ok_rows = target != IGN
    expected = float(F.cross_entropy(logits[head][ok_rows], target[ok_rows]))
    got = float(output.plain_ce_sum[head]) / int(output.n_valid[head])
    check(abs(got - expected) < TOLERANCE, f"{head}: plain CE ignores weights and smoothing ({got:.5f})")

# ---- Section: Numeric types ----
bf16_out = make_loss()({h: l.detach().bfloat16() for h, l in logits.items()}, batch)
check(bf16_out.total.dtype == torch.float32 and bool(torch.isfinite(bf16_out.total)),
      "bf16 logits are handled in float32")

# ---- Section: Accumulator ----
acc = losses.LossAccumulator("cpu")
loss_fn = make_loss()
parts = []
for seed in (1, 2):
    l, b = make_batch(seed=seed)
    out = loss_fn(l, b)
    acc.add(out)
    parts.append((l, b, out))
summary = acc.summary()
check(abs(summary["objective"] - np.mean([float(p[2].total) for p in parts])) < TOLERANCE and summary["steps"] == 2,
      "accumulator objective = mean of step losses")
pooled = {h: float(sum(p[2].plain_ce_sum[h] for p in parts)) / int(sum(p[2].n_valid[h] for p in parts)) for h in HEADS}
check(all(abs(summary[f"plain_ce_{h}"] - pooled[h]) < TOLERANCE for h in HEADS),
      "accumulator plain CE is the pooled mean over valid rows (not a mean of batch means)")
acc_eval = losses.LossAccumulator("cpu")
for l, b, _ in parts:
    acc_eval.add_logits(l, b)
eval_summary = acc_eval.summary()
check(all(abs(eval_summary[f"plain_ce_{h}"] - pooled[h]) < TOLERANCE for h in HEADS) and np.isnan(eval_summary["objective"]),
      "add_logits gives the same plain CE and no objective")
acc.reset()
fresh = acc.summary()
check(fresh["steps"] == 0 and all(np.isnan(fresh[k]) for k in ("objective", "plain_ce_rec", "plain_ce_band")),
      "reset clears the sums (empty summary is NaN, not 0)")

# ---- Section: build_loss from config and labels ----
cfg = {"training": {"use_class_weights": True, "label_smoothing": SMOOTHING, "loss_weight_band": 1.0}}
y_toy = {"rec": np.array([0, 0, 0, 0, 1, 2, IGN]), "band": np.array([0, 1, 2, 2, 2, 2, 1])}
built = losses.build_loss(cfg, y_toy, "cpu")
check(np.allclose(built.class_weights["rec"].numpy(), [6 / (3 * 4), 6 / (3 * 1), 6 / (3 * 1)]),
      "build_loss: class weights come from the labels of each head (masked rows excluded)", "wrong counts")
cfg_off = {"training": {**cfg["training"], "use_class_weights": False}}
check(bool((losses.build_loss(cfg_off, y_toy, "cpu").class_weights["band"] == 1).all()),
      "use_class_weights=false gives all-ones class weights")

# ---- Section: Error handling ----
check(raises(lambda: make_loss(lam=0.0)), "loss_weight_band = 0 is rejected")
check(raises(lambda: make_loss(smoothing=1.0)), "label_smoothing = 1 is rejected")
check(raises(lambda: make_loss(weights={"rec": [1, 1, 1]})), "missing head weights are rejected")
check(raises(lambda: make_loss(weights={"rec": [1, 1], "band": [1, 1, 1]})), "wrong weight vector length is rejected")
check(raises(lambda: make_loss(weights={"rec": [1, 0, 1], "band": [1, 1, 1]})), "zero class weight is rejected")
check(raises(lambda: make_loss(weights={"rec": [1, float("nan"), 1], "band": [1, 1, 1]})), "NaN class weight is rejected")

print("\nALL PASSED" if all(results) else "\nFAILED")

[PASS] rec: loss equals the row-by-row reference (1.074777)
[PASS] band: loss equals the row-by-row reference (1.487795)
[PASS] total = loss_rec + loss_weight_band * loss_band
[PASS] total is float32 and keeps the graph
[PASS] per-head values are detached
[PASS] w=1 and class weights 1 give the plain smoothed mean CE over valid rows
[PASS] multiplying all sample weights by a constant does not change the loss
[PASS] rows without rec label get zero gradient (2 masked rows)
[PASS] band loss is unchanged when rec is fully masked
[PASS] batch with no valid label gives loss 0 and zero finite gradients
[PASS] empty batch: plain CE sum 0 and count 0
[PASS] rec: plain CE ignores weights and smoothing (1.00999)
[PASS] band: plain CE ignores weights and smoothing (1.52831)
[PASS] bf16 logits are handled in float32
[PASS] accumulator objective = mean of step losses
[PASS] accumulator plain CE is the pooled mean over valid rows (not a mean of batch means)
[PASS] add_logits gives the same plain CE a

/tmp/ipykernel_237443/3684321815.py:72: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  check(abs(float(output.total) - (float(output.head_loss["rec"]) + 2.0 * float(output.head_loss["band"]))) < TOLERANCE,


# Test: Checkpoint Save, Resume, RNG and Run Status

In [10]:
# Test: Checkpoint Save, Resume, RNG and Run Status

import copy
import json
import random
import tempfile
from collections.abc import Mapping
from pathlib import Path
from unittest.mock import patch

import numpy as np
import torch
from torch import nn

from digikala_review_transformer.training import checkpoint

# ---- Section: Test Helpers ----
# Keep the notebook's RNG state unchanged after running this test cell.
_checkpoint_initial_rng = checkpoint.capture_rng()
_checkpoint_failures = []


def _checkpoint_check(name, test_function, fix):
    try:
        test_function()
        print(f"[PASS] {name}")
    except Exception as error:  # noqa: BLE001
        message = f"{type(error).__name__}: {error}"
        _checkpoint_failures.append((name, message))
        print(f"[FAIL] {name}: {message} -> {fix}")


def _assert_nested_equal(left, right, path="root"):
    """Assert equality for nested checkpoint state, including tensors."""
    if isinstance(left, torch.Tensor):
        assert isinstance(right, torch.Tensor), f"{path}: tensor type mismatch"
        assert left.dtype == right.dtype, f"{path}: tensor dtype mismatch"
        assert torch.equal(left.cpu(), right.cpu()), f"{path}: tensor value mismatch"
        return

    if isinstance(left, np.ndarray):
        assert isinstance(right, np.ndarray), f"{path}: array type mismatch"
        assert np.array_equal(left, right), f"{path}: array value mismatch"
        return

    if isinstance(left, Mapping):
        assert isinstance(right, Mapping), f"{path}: mapping type mismatch"
        assert set(left) == set(right), f"{path}: mapping keys differ"
        for key in left:
            _assert_nested_equal(left[key], right[key], f"{path}.{key}")
        return

    if isinstance(left, (list, tuple)):
        assert isinstance(right, (list, tuple)), f"{path}: sequence type mismatch"
        assert len(left) == len(right), f"{path}: sequence lengths differ"
        for index, (left_item, right_item) in enumerate(zip(left, right)):
            _assert_nested_equal(left_item, right_item, f"{path}[{index}]")
        return

    assert left == right, f"{path}: values differ ({left!r} != {right!r})"


def _build_training_bundle(seed):
    """Build matching CPU model, optimizer, and scheduler objects."""
    torch.manual_seed(seed)

    model = nn.Sequential(
        nn.Linear(3, 5),
        nn.Tanh(),
        nn.Linear(5, 2),
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1.0e-3,
        weight_decay=0.01,
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: max(0.0, 1.0 - step / 10.0),
    )
    return model, optimizer, scheduler


_checkpoint_tmp = tempfile.TemporaryDirectory(
    prefix="transformer_checkpoint_test_"
)
_checkpoint_root = Path(_checkpoint_tmp.name)
_checkpoint_fixture = {}


# ---- Section: Last Checkpoint Round-Trip ----
# Create optimizer state, save a checkpoint, and restore it into fresh objects.
def _test_last_checkpoint_roundtrip():
    source_model, source_optimizer, source_scheduler = _build_training_bundle(101)

    inputs = torch.tensor(
        [
            [0.2, -0.4, 0.8],
            [1.0, 0.3, -0.5],
        ],
        dtype=torch.float32,
    )
    source_optimizer.zero_grad(set_to_none=True)
    loss = source_model(inputs).square().mean()
    loss.backward()
    source_optimizer.step()
    source_scheduler.step()

    state = checkpoint.TrainState(
        epochs_done=1,
        global_step=1,
        best_score=0.41,
        best_epoch=1,
        bad_epochs=0,
        history=[
            {
                "epoch": 1,
                "train_objective": float(loss.detach().item()),
            }
        ],
        train_seconds=2.5,
    )

    expected_model = {
        name: value.detach().cpu().clone()
        for name, value in source_model.state_dict().items()
    }
    expected_optimizer = copy.deepcopy(source_optimizer.state_dict())
    expected_scheduler = copy.deepcopy(source_scheduler.state_dict())

    last_path = _checkpoint_root / "roundtrip" / "last.pt"
    checkpoint.save_last(
        last_path,
        source_model,
        source_optimizer,
        source_scheduler,
        state,
        config_hash="checkpoint-test-hash",
    )

    payload = checkpoint.load_last(
        last_path,
        expected_hash="checkpoint-test-hash",
    )

    target_model, target_optimizer, target_scheduler = _build_training_bundle(202)
    restored_state = checkpoint.restore_last(
        payload,
        target_model,
        target_optimizer,
        target_scheduler,
    )

    for name, expected_value in expected_model.items():
        actual_value = target_model.state_dict()[name].detach().cpu()
        assert torch.equal(actual_value, expected_value), (
            f"Model tensor {name!r} was not restored exactly."
        )

    _assert_nested_equal(
        expected_optimizer,
        target_optimizer.state_dict(),
        "optimizer_state_dict",
    )
    _assert_nested_equal(
        expected_scheduler,
        target_scheduler.state_dict(),
        "scheduler_state_dict",
    )
    _assert_nested_equal(
        state.to_dict(),
        restored_state.to_dict(),
        "train_state",
    )

    assert payload["config_hash"] == "checkpoint-test-hash"
    _checkpoint_fixture["last_path"] = last_path


_checkpoint_check(
    "last.pt round-trip restores model, optimizer, scheduler, and TrainState",
    _test_last_checkpoint_roundtrip,
    "check save_last/load_last/restore_last and resolve the first reported mismatch",
)


# ---- Section: Configuration Hash Guard ----
# A checkpoint made for another resolved configuration must not be resumed.
def _test_config_hash_mismatch():
    last_path = _checkpoint_fixture.get("last_path")
    assert last_path is not None, "The round-trip checkpoint was not created."

    try:
        checkpoint.load_last(
            last_path,
            expected_hash="deliberately-wrong-hash",
        )
    except checkpoint.CheckpointError as error:
        assert "config hash" in str(error).lower(), (
            "The error should clearly identify the configuration hash mismatch."
        )
    else:
        raise AssertionError("A mismatched config hash was accepted.")


_checkpoint_check(
    "load_last rejects a mismatched configuration hash",
    _test_config_hash_mismatch,
    "ensure the run uses the original config hash or a new run directory",
)


# ---- Section: Best Weights ----
# Check fp32 floating weights, preserved integer buffers, and strict loading.
def _test_best_weights():
    source_model = nn.Linear(3, 2)
    source_model.register_buffer(
        "integer_marker",
        torch.tensor([17], dtype=torch.int64),
    )
    source_model = source_model.half()

    target_model = nn.Linear(3, 2)
    target_model.register_buffer(
        "integer_marker",
        torch.tensor([0], dtype=torch.int64),
    )

    best_path = _checkpoint_root / "best_model.pt"
    checkpoint.save_best_weights(best_path, source_model)

    saved_state = torch.load(
        best_path,
        map_location="cpu",
        weights_only=True,
    )

    for name, value in saved_state.items():
        if torch.is_floating_point(value):
            assert value.dtype == torch.float32, (
                f"Floating tensor {name!r} was not saved as fp32."
            )

    assert saved_state["integer_marker"].dtype == torch.int64
    checkpoint.load_best_weights(best_path, target_model)

    for name, source_value in source_model.state_dict().items():
        expected = source_value.detach().cpu()
        if torch.is_floating_point(expected):
            expected = expected.to(dtype=torch.float32)

        actual = target_model.state_dict()[name].detach().cpu()
        assert torch.equal(actual, expected), (
            f"Best-model tensor {name!r} did not load as expected."
        )


_checkpoint_check(
    "best_model.pt stores floating weights as fp32 and loads strictly",
    _test_best_weights,
    "check save_best_weights/load_best_weights and the model state_dict keys",
)


# ---- Section: Atomic Replacement and Temporary Files ----
# Simulate transient Windows file locks and verify failed saves preserve old data.
def _test_atomic_replace_retry():
    target_path = _checkpoint_root / "retry" / "checkpoint.pt"
    real_replace = checkpoint.os.replace
    attempts = {"count": 0}

    def fail_twice_then_replace(source, destination):
        attempts["count"] += 1
        if attempts["count"] <= 2:
            raise PermissionError("simulated temporary file lock")
        return real_replace(source, destination)

    with patch.object(
        checkpoint.os,
        "replace",
        new=fail_twice_then_replace,
    ):
        checkpoint.atomic_torch_save(
            {"sentinel": 23},
            target_path,
            max_attempts=4,
            retry_delay_seconds=0,
        )

    assert attempts["count"] == 3, (
        f"Expected 3 replacement attempts; got {attempts['count']}."
    )
    saved = torch.load(
        target_path,
        map_location="cpu",
        weights_only=True,
    )
    assert saved == {"sentinel": 23}


_checkpoint_check(
    "atomic_torch_save retries transient replacement errors",
    _test_atomic_replace_retry,
    "check the retry policy and same-directory temporary-file replacement",
)


def _test_failed_replace_preserves_existing_checkpoint():
    target_path = _checkpoint_root / "preserve" / "checkpoint.pt"
    checkpoint.atomic_torch_save(
        {"version": "old"},
        target_path,
        retry_delay_seconds=0,
    )

    real_replace = checkpoint.os.replace  # noqa: F841
    attempts = {"count": 0}

    def always_fail_replace(source, destination):
        attempts["count"] += 1
        raise PermissionError("simulated persistent file lock")

    try:
        with patch.object(
            checkpoint.os,
            "replace",
            new=always_fail_replace,
        ):
            checkpoint.atomic_torch_save(
                {"version": "new"},
                target_path,
                max_attempts=2,
                retry_delay_seconds=0,
            )
    except checkpoint.CheckpointError:
        pass
    else:
        raise AssertionError("A permanently blocked replacement did not fail.")

    assert attempts["count"] == 2
    saved = torch.load(
        target_path,
        map_location="cpu",
        weights_only=True,
    )
    assert saved == {"version": "old"}, (
        "The existing checkpoint was changed after replacement failed."
    )

    temporary_files = list(
        target_path.parent.glob(f".{target_path.name}.*.tmp")
    )
    assert not temporary_files, (
        f"Temporary files were left behind: {temporary_files}"
    )


_checkpoint_check(
    "failed replacement preserves the old checkpoint and removes the temp file",
    _test_failed_replace_preserves_existing_checkpoint,
    "check atomic replacement cleanup and ensure the original file is never overwritten on failure",
)


def _test_serialization_failure_cleans_temp_file():
    target_path = _checkpoint_root / "serialization_failure" / "broken.pt"

    try:
        checkpoint.atomic_torch_save(
            {"not_serializable": lambda value: value},
            target_path,
            retry_delay_seconds=0,
        )
    except checkpoint.CheckpointError:
        pass
    else:
        raise AssertionError("An unserializable payload did not raise CheckpointError.")

    assert not target_path.exists(), "A target checkpoint was created unexpectedly."
    temporary_files = list(
        target_path.parent.glob(f".{target_path.name}.*.tmp")
    )
    assert not temporary_files, (
        f"Temporary files were left behind: {temporary_files}"
    )


_checkpoint_check(
    "serialization failure removes the incomplete temporary file",
    _test_serialization_failure_cleans_temp_file,
    "check atomic_torch_save cleanup when torch.save fails",
)


# ---- Section: RNG Round-Trip ----
# Verify Python, NumPy, PyTorch CPU, and available CUDA RNG states are restored.
def _test_rng_roundtrip():
    original_state = checkpoint.capture_rng()

    try:
        expected_state = checkpoint.capture_rng()

        expected_python = random.random()
        expected_numpy = np.random.random(4)
        expected_cpu = torch.rand(5, device="cpu")

        expected_cuda = None
        if torch.cuda.is_available():
            expected_cuda = torch.rand(5, device="cuda").cpu()

        checkpoint.restore_rng(expected_state)

        actual_python = random.random()
        actual_numpy = np.random.random(4)
        actual_cpu = torch.rand(5, device="cpu")

        assert actual_python == expected_python
        assert np.array_equal(actual_numpy, expected_numpy)
        assert torch.equal(actual_cpu, expected_cpu)

        if expected_cuda is not None:
            actual_cuda = torch.rand(5, device="cuda").cpu()
            assert torch.equal(actual_cuda, expected_cuda)

    finally:
        checkpoint.restore_rng(original_state)


_checkpoint_check(
    "capture_rng/restore_rng reproduce Python, NumPy, CPU, and available CUDA draws",
    _test_rng_roundtrip,
    "check capture_rng/restore_rng and run the test on the same CUDA device setup",
)


# ---- Section: Run Status and Guarded Cleanup ----
# Only a complete run may remove its resumable last.pt checkpoint.
def _test_run_status_and_cleanup():
    run_dir = _checkpoint_root / "status_test"
    last_path = run_dir / "last.pt"
    summary_path = run_dir / "run_summary.json"

    assert checkpoint.run_status(run_dir) == "fresh"

    run_dir.mkdir(parents=True, exist_ok=True)
    last_path.write_bytes(b"resume marker")
    assert checkpoint.run_status(run_dir) == "resumable"

    try:
        checkpoint.cleanup_after_complete(run_dir)
    except checkpoint.CheckpointError:
        pass
    else:
        raise AssertionError("Cleanup was allowed before the run was complete.")

    assert last_path.is_file(), "last.pt was removed from an incomplete run."

    summary_path.write_text(
        json.dumps({"status": "complete"}),
        encoding="utf-8",
    )
    assert checkpoint.run_status(run_dir) == "complete"

    checkpoint.cleanup_after_complete(run_dir)
    assert not last_path.exists(), "last.pt was not removed after completion."
    assert checkpoint.run_status(run_dir) == "complete"


_checkpoint_check(
    "run_status detects fresh/resumable/complete and guards last.pt cleanup",
    _test_run_status_and_cleanup,
    "check run_summary.json status handling and cleanup_after_complete",
)


# ---- Section: Cleanup and Final Result ----
# Restore the notebook RNG and remove all temporary test files.
_checkpoint_check(
    "notebook RNG state is restored after the tests",
    lambda: checkpoint.restore_rng(_checkpoint_initial_rng),
    "check restore_rng and the saved notebook RNG state",
)

_checkpoint_tmp.cleanup()

if _checkpoint_failures:
    print(
        f"[FAIL] {len(_checkpoint_failures)} checkpoint test(s) failed; "
        "see the failure details above and apply the listed fixes."
    )
else:
    print("[PASS] ALL CHECKPOINT TESTS PASSED")

[PASS] last.pt round-trip restores model, optimizer, scheduler, and TrainState
[PASS] load_last rejects a mismatched configuration hash
[PASS] best_model.pt stores floating weights as fp32 and loads strictly
[PASS] atomic_torch_save retries transient replacement errors
[PASS] failed replacement preserves the old checkpoint and removes the temp file
[PASS] serialization failure removes the incomplete temporary file
[PASS] capture_rng/restore_rng reproduce Python, NumPy, CPU, and available CUDA draws
[PASS] run_status detects fresh/resumable/complete and guards last.pt cleanup
[PASS] notebook RNG state is restored after the tests
[PASS] ALL CHECKPOINT TESTS PASSED


# Inspect: Training Loop Dependencies (Read-Only)

In [ ]:
# Inspect: Training Loop Dependencies (Read-Only)

import ast
from pathlib import Path

# ---- Section: Locate the Project ----
# Find the repository root without importing or modifying project modules.
_current_dir = Path.cwd()
_project_root = next(
    (
        candidate
        for candidate in (_current_dir, *_current_dir.parents)
        if (candidate / "src" / "digikala_review_transformer").is_dir()
    ),
    None,
)

if _project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing "
        "'src/digikala_review_transformer'. Run this cell from the project."
    )


# ---- Section: Inspection Targets ----
# Inspect only interfaces and constants needed to connect the training loop.
_targets = [
    (
        "src/digikala_review_transformer/config.py",
        {"ROOT", "SPLIT_SEED", "ARTIFACTS"},
    ),
    (
        "src/digikala_review_transformer/training/run_config.py",
        {
            "CONFIG_SCHEMA",
            "RUNTIME_ONLY_KEYS",
            "load_run_config",
            "config_hash",
            "project_path",
            "run_dir",
            "dump_resolved_config",
        },
    ),
    (
        "src/digikala_review_transformer/training/data.py",
        {
            "HEADS",
            "IGNORE_INDEX",
            "SplitData",
            "BatchedTokenDataset",
            "open_pack",
            "class_weight_vector",
            "subset_row_indices",
            "load_split",
            "LengthBucketedBatches",
            "sorted_eval_batches",
            "make_loader",
        },
    ),
    (
        "src/digikala_review_transformer/training/tokenization.py",
        {
            "load_tokenizer",
            "get_split_tokens",
            "default_cache_dir",
        },
    ),
    (
        "src/digikala_review_transformer/training/model.py",
        {
            "TwoHeadClassifier",
            "build_model",
            "count_parameters",
            "model_summary",
            "parameter_groups",
        },
    ),
    (
        "src/digikala_review_transformer/training/losses.py",
        {
            "LossOutput",
            "TwoHeadLoss",
            "build_loss",
            "LossAccumulator",
        },
    ),
    (
        "src/digikala_review_transformer/training/checkpoint.py",
        {
            "TrainState",
            "save_last",
            "load_last",
            "restore_last",
            "save_best_weights",
            "load_best_weights",
            "run_status",
            "cleanup_after_complete",
        },
    ),
    (
        "src/digikala_review_transformer/labels.py",
        {
            "CLASSES",
            "CORR_COL",
            "RAW_COL",
            "class_weights",
            "decode",
            "encode",
            "label_map",
        },
    ),
    (
        "src/digikala_review_transformer/metrics.py",
        {
            "evaluate",
            "per_class_table",
        },
    ),
    (
        "src/digikala_review_transformer/utils.py",
        {"timer"},
    ),
]


# ---- Section: AST Helpers ----
# Report source-level interfaces without executing any project code.
def _assigned_names(target):
    if isinstance(target, ast.Name):
        return {target.id}
    if isinstance(target, (ast.Tuple, ast.List)):
        names = set()
        for item in target.elts:
            names.update(_assigned_names(item))
        return names
    return set()


def _node_names(node):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        return {node.name}
    if isinstance(node, ast.Assign):
        names = set()
        for target in node.targets:
            names.update(_assigned_names(target))
        return names
    if isinstance(node, ast.AnnAssign):
        return _assigned_names(node.target)
    return set()


def _format_function_signature(node):
    prefix = "async def" if isinstance(node, ast.AsyncFunctionDef) else "def"
    arguments = ast.unparse(node.args)
    returns = (
        f" -> {ast.unparse(node.returns)}"
        if node.returns is not None
        else ""
    )
    decorators = [
        f"@{ast.unparse(decorator)}"
        for decorator in node.decorator_list
    ]
    signature = f"{prefix} {node.name}({arguments}){returns}"
    return "\n".join([*decorators, signature])


def _print_class_interface(node):
    decorators = [
        f"@{ast.unparse(decorator)}"
        for decorator in node.decorator_list
    ]
    bases = ", ".join(ast.unparse(base) for base in node.bases)
    class_line = f"class {node.name}({bases}):" if bases else f"class {node.name}:"

    for decorator in decorators:
        print(decorator)
    print(class_line)

    docstring = ast.get_docstring(node)
    if docstring:
        print(f'  doc: {docstring.splitlines()[0]}')

    for child in node.body:
        if isinstance(child, ast.AnnAssign):
            field_name = ast.unparse(child.target)
            annotation = ast.unparse(child.annotation)
            default = (
                f" = {ast.unparse(child.value)}"
                if child.value is not None
                else ""
            )
            print(f"  field: {field_name}: {annotation}{default}")

        elif isinstance(child, ast.Assign):
            source = ast.get_source_segment(_source_text, child)
            if source:
                print(f"  class assignment: {source.strip()}")

        elif isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
            signature = _format_function_signature(child).replace("\n", "\n  ")
            print(f"  {signature}")


# ---- Section: Print Selected Interfaces ----
# Print selected constants, function signatures, and class fields/method signatures.
for _relative_path, _selected_names in _targets:
    _file_path = _project_root / _relative_path
    print(f"\n===== {_relative_path} =====")

    if not _file_path.is_file():
        print("[MISSING] File not found at this path.")
        continue

    _source_text = _file_path.read_text(encoding="utf-8")
    _module = ast.parse(_source_text, filename=str(_file_path))
    _found_names = set()

    for _node in _module.body:
        _matched = _node_names(_node) & _selected_names
        if not _matched:
            continue

        _found_names.update(_matched)

        if isinstance(_node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print(_format_function_signature(_node))
            _docstring = ast.get_docstring(_node)
            if _docstring:
                print(f"  doc: {_docstring.splitlines()[0]}")

        elif isinstance(_node, ast.ClassDef):
            _print_class_interface(_node)

        else:
            _source_segment = ast.get_source_segment(_source_text, _node)
            if _source_segment:
                print(_source_segment.strip())

    _missing_names = _selected_names - _found_names
    if _missing_names:
        print(f"[NOT FOUND] {sorted(_missing_names)}")


===== src/digikala_review_transformer/config.py =====
ROOT = Path(__file__).resolve().parents[2]
ARTIFACTS = ROOT / "artifacts"
SPLIT_SEED = 2026

===== src/digikala_review_transformer/training/run_config.py =====
CONFIG_SCHEMA = {
    "run_name": str,
    "data": {"pack_dir": str},
    "model": {
        "pretrained_path": str,
        "max_length": int,
        "head_dropout": float,
        "tokenizer_backend": str,
    },    "training": {
        "variant": str,
        "max_epochs": int,
        "early_stopping_patience": int,
        "early_stopping_min_delta": float,
        "batch_size": int,
        "eval_batch_size": int,
        "lr_encoder": float,
        "lr_heads": float,
        "weight_decay": float,
        "warmup_ratio": float,
        "max_grad_norm": float,
        "label_smoothing": float,
        "loss_weight_band": float,
        "use_class_weights": bool,
        "bf16": bool,
        "num_workers": int,
        "log_every_steps": int,
        "train_subset_e

# Test: Training Loop, Evaluation Alignment, Resume and Early Stopping

In [2]:
# Test: Training Loop, Evaluation Alignment, Resume and Early Stopping

import copy
import json
import math
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn  # noqa: F811

from digikala_review_transformer import config, labels
from digikala_review_transformer.training import (
    checkpoint,
    data,
    loop,
    run_config,
    tokenization,
)
from digikala_review_transformer.training import (
    model as model_module,
)

# ---- Section: Test Helpers ----
# Restore the notebook RNG after exercising training, checkpointing, and resume.
_loop_initial_rng = checkpoint.capture_rng()
_loop_failures = []


def _loop_check(name, test_function, fix):
    try:
        test_function()
        print(f"[PASS] {name}")
    except Exception as error:  # noqa: BLE001
        message = f"{type(error).__name__}: {error}"
        _loop_failures.append((name, message))
        print(f"[FAIL] {name}: {message} -> {fix}")


class _ToyEncoderOutput:
    def __init__(self, last_hidden_state):
        self.last_hidden_state = last_hidden_state


class _ToyEncoder(nn.Module):
    def __init__(self, vocab_size=64, hidden_size=12):
        super().__init__()
        self.config = type("ToyConfig", (), {"hidden_size": hidden_size})()
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.projection = nn.Linear(hidden_size, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, input_ids, attention_mask=None):
        hidden = self.embeddings(input_ids)
        hidden = torch.tanh(self.projection(hidden))
        hidden = self.norm(hidden)
        return _ToyEncoderOutput(last_hidden_state=hidden)


def _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.1):
    model_module.seed_everything(seed)
    encoder = _ToyEncoder()
    model_module.seed_everything(seed)
    return model_module.TwoHeadClassifier(encoder, head_dropout=head_dropout)


def _make_token_arrays(sequences):
    offsets = [0]
    flat_ids = []
    for sequence in sequences:
        token_ids = [int(value) for value in sequence]
        flat_ids.extend(token_ids)
        offsets.append(len(flat_ids))
    return tokenization.TokenArrays(
        ids=np.asarray(flat_ids, dtype=np.int32),
        offsets=np.asarray(offsets, dtype=np.int64),
    )


def _make_split(
    split_name,
    n_rows,
    *,
    seed,
    id_start,
    variant="B",
    mask_rec_every=5,
):
    rng = np.random.default_rng(seed)
    sequences = []
    rec_corr = np.empty(n_rows, dtype=np.int64)
    band_corr = np.empty(n_rows, dtype=np.int64)
    rec_raw = np.empty(n_rows, dtype=np.int64)
    band_raw = np.empty(n_rows, dtype=np.int64)

    for row_index in range(n_rows):
        rec_class = row_index % 3
        band_class = (row_index // 3 + row_index) % 3
        extra_length = int((row_index * 5 + 3) % 9) + 2

        anchor_rec = 4 + rec_class
        anchor_band = 10 + band_class
        body_tokens = rng.integers(16, 48, size=extra_length).tolist()
        sequence = [2, anchor_rec, anchor_band, *body_tokens, 3]
        sequences.append(sequence)

        rec_corr[row_index] = rec_class
        band_corr[row_index] = band_class
        rec_raw[row_index] = (
            rec_class if row_index % 7 != 0 else (rec_class + 1) % 3
        )
        band_raw[row_index] = (
            band_class if row_index % 9 != 0 else (band_class + 1) % 3
        )

    if mask_rec_every is not None:
        for row_index in range(0, n_rows, mask_rec_every):
            rec_corr[row_index] = data.IGNORE_INDEX
            rec_raw[row_index] = data.IGNORE_INDEX

    if variant == "C":
        w_rec = np.linspace(0.6, 1.0, n_rows, dtype=np.float32)
        w_band = np.linspace(1.0, 0.7, n_rows, dtype=np.float32)
    else:
        w_rec = np.ones(n_rows, dtype=np.float32)
        w_band = np.ones(n_rows, dtype=np.float32)

    arrays = _make_token_arrays(sequences)
    ids = np.arange(id_start, id_start + n_rows, dtype=np.int64)

    return data.SplitData(
        split=split_name,
        ids=ids,
        arrays=arrays,
        pad_id=0,
        y={"rec": rec_corr, "band": band_corr},
        y_raw={"rec": rec_raw, "band": band_raw},
        w={"rec": w_rec, "band": w_band},
        stats={"rows": n_rows},
        origin="test",
        source_rows=n_rows,
    )


def _make_config(
    output_root,
    *,
    run_name="loop_test",
    variant="B",
    max_epochs=2,
    patience=2,
    min_delta=0.0,
    batch_size=8,
    eval_batch_size=7,
    lr_encoder=2.0e-4,
    lr_heads=1.0e-3,
    log_every_steps=2,
):
    return {
        "run_name": run_name,
        "seed": config.SPLIT_SEED,
        "data": {"pack_dir": "data/training_pack"},
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 32,
            "head_dropout": 0.1,
            "tokenizer_backend": "hf_fast",
        },
        "training": {
            "variant": variant,
            "max_epochs": max_epochs,
            "early_stopping_patience": patience,
            "early_stopping_min_delta": min_delta,
            "batch_size": batch_size,
            "eval_batch_size": eval_batch_size,
            "lr_encoder": lr_encoder,
            "lr_heads": lr_heads,
            "weight_decay": 0.01,
            "warmup_ratio": 0.06,
            "max_grad_norm": 1.0,
            "label_smoothing": 0.1,
            "loss_weight_band": 1.0,
            "use_class_weights": True,
            "bf16": False,
            "num_workers": 0,
            "log_every_steps": log_every_steps,
            "train_subset_eval_rows": 12,
        },
        "output": {"root": str(output_root)},
    }


def _manual_probabilities(model, split):
    model.eval()
    dataset = split.dataset()
    batch = dataset[np.arange(split.n_rows, dtype=np.int64)]
    with torch.inference_mode():
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        return {
            head: torch.softmax(logits[head].float(), dim=-1).cpu().numpy()
            for head in data.HEADS
        }


_loop_tmp = tempfile.TemporaryDirectory(prefix="transformer_loop_test_")
_loop_root = Path(_loop_tmp.name)


# ---- Section: Scheduler and Evaluation Order ----
# Check the 6% warmup/decay schedule and length-sorted evaluation restoration.
def _test_linear_scheduler():
    model = _build_toy_model()
    groups = model_module.parameter_groups(
        model,
        lr_encoder=2.0e-5,
        lr_heads=1.0e-4,
        weight_decay=0.01,
    )
    optimizer = torch.optim.AdamW(groups)
    scheduler, warmup_steps = loop.make_linear_scheduler(
        optimizer,
        total_steps=100,
        warmup_ratio=0.06,
    )

    assert warmup_steps == 6
    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 0.0, abs_tol=1e-12)
    assert math.isclose(heads_lr, 0.0, abs_tol=1e-12)

    for _ in range(warmup_steps):
        optimizer.step()
        scheduler.step()

    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 2.0e-5, rel_tol=1e-6)
    assert math.isclose(heads_lr, 1.0e-4, rel_tol=1e-6)

    for _ in range(100 - warmup_steps):
        optimizer.step()
        scheduler.step()

    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 0.0, abs_tol=1e-12)
    assert math.isclose(heads_lr, 0.0, abs_tol=1e-12)


_loop_check(
    "linear scheduler warms up for 6% of total steps and decays to zero",
    _test_linear_scheduler,
    "check make_linear_scheduler warmup_steps and decay denominator",
)


def _test_eval_restores_original_order_and_schema():
    split = _make_split("validation", 23, seed=11, id_start=5000)
    model = _build_toy_model(seed=19)

    expected = _manual_probabilities(model, split)
    evaluation = loop.evaluate_split(
        model,
        split,
        batch_size=5,
        device="cpu",
        bf16=False,
        include_raw=True,
    )

    for head in data.HEADS:
        assert np.allclose(
            evaluation.probabilities[head],
            expected[head],
            rtol=1e-5,
            atol=1e-6,
        ), f"Restored probabilities for {head} do not match row order."
        assert math.isfinite(evaluation.plain_ce[head])
        assert math.isfinite(evaluation.corrected_metrics[head]["macro_f1"])
        assert math.isfinite(evaluation.raw_metrics[head]["macro_f1"])

    frame = loop.probability_frame(split, evaluation.probabilities)
    expected_columns = [
        "id",
        *(f"p_{name}" for name in labels.CLASSES["rec"]),
        *(f"p_{name}" for name in labels.CLASSES["band"]),
    ]
    assert list(frame.columns) == expected_columns
    assert np.array_equal(frame["id"].to_numpy(), split.ids)
    for column in expected_columns[1:]:
        assert frame[column].dtype == np.float32


_loop_check(
    "sorted evaluation restores original id order and float32 probability columns",
    _test_eval_restores_original_order_and_schema,
    "check sorted_eval_batches index restoration and probability_frame columns",
)


# ---- Section: Full Fit, Pause/Resume, and Hash Guard ----
# A 1-epoch pause followed by resume must match an uninterrupted 2-epoch run.
def _test_fit_and_exact_resume():
    train_split = _make_split("train", 24, seed=101, id_start=100, variant="C")
    val_split = _make_split("validation", 18, seed=202, id_start=1000)
    subset_split = _make_split(
        "train_subset",
        12,
        seed=303,
        id_start=3000,
        mask_rec_every=6,
    )

    direct_cfg = _make_config(
        _loop_root / "direct",
        run_name="direct_two_epochs",
        variant="C",
        max_epochs=2,
        patience=2,
    )
    direct_model = _build_toy_model(seed=config.SPLIT_SEED)
    direct_result = loop.fit(
        direct_cfg,
        direct_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert direct_result.status == "trained"
    assert direct_result.epochs_done == 2
    assert direct_result.global_step == 6

    resume_cfg = _make_config(
        _loop_root / "resumed",
        run_name="paused_then_resumed",
        variant="C",
        max_epochs=2,
        patience=2,
    )
    paused_model = _build_toy_model(seed=config.SPLIT_SEED)
    paused_result = loop.fit(
        resume_cfg,
        paused_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
        max_epochs_this_call=1,
    )

    assert paused_result.status == "paused"
    assert paused_result.stop_reason == "paused_by_call_limit"
    assert paused_result.epochs_done == 1
    assert (paused_result.run_directory / "last.pt").is_file()

    # Append an uncommitted step row to simulate a crash during epoch 2.
    train_log_path = paused_result.run_directory / "train_log.csv"
    log_df = pd.read_csv(train_log_path)
    extra_row = log_df.iloc[[-1]].copy()
    extra_row["global_step"] = 999
    pd.concat([log_df, extra_row], ignore_index=True).to_csv(
        train_log_path,
        index=False,
    )

    wrong_cfg = copy.deepcopy(resume_cfg)
    wrong_cfg["training"]["lr_heads"] = 9.0e-4
    try:
        loop.fit(
            wrong_cfg,
            _build_toy_model(seed=config.SPLIT_SEED),
            train_split,
            val_split,
            subset_split,
            run_directory=paused_result.run_directory,
            device="cpu",
        )
    except checkpoint.CheckpointError as error:
        assert "config hash" in str(error).lower()
    else:
        raise AssertionError("Mismatched config hash did not fail on resume.")

    resumed_model = _build_toy_model(seed=999)
    resumed_result = loop.fit(
        resume_cfg,
        resumed_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert resumed_result.status == "trained"
    assert resumed_result.epochs_done == 2
    assert resumed_result.global_step == 6

    for name, direct_tensor in direct_model.state_dict().items():
        resumed_tensor = resumed_model.state_dict()[name]
        assert torch.allclose(
            direct_tensor,
            resumed_tensor,
            rtol=1e-6,
            atol=1e-6,
        ), f"Final best weight {name} differed after resume."

    direct_history = pd.read_csv(direct_result.run_directory / "history.csv")
    resumed_history = pd.read_csv(resumed_result.run_directory / "history.csv")
    required_history_columns = {
        "train_objective",
        "train_plain_ce_rec",
        "train_plain_ce_band",
        "train_subset_plain_ce_rec",
        "train_subset_plain_ce_band",
        "val_plain_ce_rec",
        "val_plain_ce_band",
        "val_acc_rec",
        "val_acc_band",
        "val_macro_f1_rec",
        "val_macro_f1_band",
        "val_macro_f1_rec_raw",
        "val_macro_f1_band_raw",
        "train_subset_macro_f1_rec",
        "train_subset_macro_f1_band",
        "val_mean_macro_f1",
        "lr_end",
        "epoch_seconds",
        "is_best",
    }
    assert required_history_columns.issubset(direct_history.columns)

    compared_columns = [
        column
        for column in direct_history.columns
        if column != "epoch_seconds"
    ]
    pd.testing.assert_frame_equal(
        direct_history[compared_columns],
        resumed_history[compared_columns],
        rtol=1e-5,
        atol=1e-6,
    )

    repaired_log = pd.read_csv(train_log_path)
    assert int(repaired_log["global_step"].max()) == 6
    assert (resumed_result.run_directory / "best_model.pt").is_file()
    assert (resumed_result.run_directory / "val_probs_best.parquet").is_file()
    assert (resumed_result.run_directory / "train.log").is_file()


_loop_check(
    "fit supports pause/resume, prunes uncommitted logs, and rejects hash changes",
    _test_fit_and_exact_resume,
    "check save_last/restore_last integration and epoch-boundary state checks",
)


# ---- Section: Shared Early Stopping and Best Weight Reload ----
# When epoch 2 does not beat min_delta, patience=1 stops and reloads epoch 1.
def _test_early_stopping_and_best_reload():
    train_split = _make_split("train", 16, seed=404, id_start=400)
    val_split = _make_split("validation", 15, seed=505, id_start=1400)
    subset_split = _make_split("train_subset", 10, seed=606, id_start=2400)

    cfg = _make_config(
        _loop_root / "early_stop",
        run_name="early_stop_run",
        max_epochs=4,
        patience=1,
        min_delta=2.0,
        lr_encoder=5.0e-3,
        lr_heads=1.0e-2,
    )
    model = _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.0)
    result = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert result.status == "trained"
    assert result.stop_reason == "early_stopping_patience"
    assert result.epochs_done == 2
    assert result.best_epoch == 1
    assert [row["is_best"] for row in result.history] == [True, False]

    best_state = torch.load(
        result.best_model_path,
        map_location="cpu",
        weights_only=True,
    )
    last_payload = checkpoint.load_last(
        result.run_directory / "last.pt",
        expected_hash=run_config.config_hash(cfg),
    )

    for name, best_tensor in best_state.items():
        assert torch.equal(model.state_dict()[name].cpu(), best_tensor)

    changed_after_best = any(
        not torch.equal(
            best_state[name].cpu(),
            last_payload["model_state_dict"][name].cpu(),
        )
        for name in best_state
    )
    assert changed_after_best, "Epoch 2 did not change weights from epoch 1."

    # Mark the run complete and verify a second call skips without overwriting.
    (result.run_directory / "run_summary.json").write_text(
        json.dumps({"status": "complete"}),
        encoding="utf-8",
    )
    skipped = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )
    assert skipped.skipped is True
    assert skipped.status == "complete"


_loop_check(
    "shared early stopping stops on patience and reloads best_model.pt",
    _test_early_stopping_and_best_reload,
    "check val_mean_macro_f1 improvement logic and load_best_weights at the end of fit",
)


# ---- Section: Cleanup and Final Result ----
# Restore the notebook RNG and remove temporary test artifacts.
_loop_check(
    "notebook RNG state is restored after training loop tests",
    lambda: checkpoint.restore_rng(_loop_initial_rng),
    "check capture_rng/restore_rng around the test cell",
)

_loop_tmp.cleanup()

if _loop_failures:
    print(
        f"[FAIL] {len(_loop_failures)} training loop test(s) failed; "
        "see the failure details above and apply the listed fixes."
    )
else:
    print("[PASS] ALL TRAINING LOOP TESTS PASSED")

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] linear scheduler warms up for 6% of total steps and decays to zero
[PASS] sorted evaluation restores original id order and float32 probability columns
[run] start run=direct_two_epochs device=cpu variant=C seed=2026
[model] parameter_counts={'total': 1026, 'trainable': 1026, 'encoder': 948, 'heads': 78}
[optimizer] fused=False steps_per_epoch=3 total_steps=6 warmup_steps=0
[train] run=direct_two_epochs epoch=1/2 step=2/3 global_step=2/6 progress=33.33% objective=2.208625 lr_encoder=1.333e-04 lr_heads=6.667e-04 samples_per_second=4217.61 ETA=0.0 min
[train] run=direct_two_epochs epoch=1/2 step=3/3 global_step=3/6 progress=50.00% objective=2.223766 lr_encoder=1.000e-04 lr_heads=5.000e-04 samples_per_second=4231.15 ETA=0.0 min
[timer] transformer_train_epoch_1: 0.0s
[timer] transformer_validation_epoch_1: 0.0s
[timer] transformer_train_subset_epoch_1: 0.0s
[epoch] epoch=1/2 train_objective=2.213672 val_mean_macro_f1=0.157407 best_epoch=1 is_best=True bad_epochs=0/2 epoch_seconds=0.

# Inspect: Evaluation Root Cause and Helper Bodies (Read-Only)

In [1]:
# Inspect: Evaluation Root Cause and Helper Bodies (Read-Only)

import ast
import traceback
from pathlib import Path

# ---- Section: Locate Project Root ----
# Read source files directly without changing anything on disk.
_current_dir = Path.cwd()
_project_root = next(
    (
        candidate
        for candidate in (_current_dir, *_current_dir.parents)
        if (candidate / "src" / "digikala_review_transformer").is_dir()
    ),
    None,
)

if _project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing "
        "'src/digikala_review_transformer'."
    )


# ---- Section: Print Exact Helper Bodies ----
# Inspect the bodies of the helpers called by _collect_split_outputs.
_source_targets = [
    (
        "src/digikala_review_transformer/training/data.py",
        {
            "BatchedTokenDataset",
            "SplitData",
            "sorted_eval_batches",
            "make_loader",
        },
    ),
    (
        "src/digikala_review_transformer/training/losses.py",
        {"plain_ce_sum"},
    ),
    (
        "src/digikala_review_transformer/metrics.py",
        {"evaluate"},
    ),
]

for _relative_path, _names in _source_targets:
    _path = _project_root / _relative_path
    _text = _path.read_text(encoding="utf-8")
    _tree = ast.parse(_text, filename=str(_path))
    print(f"\n===== {_relative_path} =====")
    for _node in _tree.body:
        if isinstance(_node, (ast.FunctionDef, ast.ClassDef)) and _node.name in _names:
            print(ast.get_source_segment(_text, _node))
            print()


# ---- Section: Print Underlying Evaluation Exception ----
# Run the existing toy evaluation call once and print __cause__ if it fails.
if "_make_split" in globals() and "_build_toy_model" in globals():
    from digikala_review_transformer.training import loop

    print("===== Underlying exception from evaluate_split =====")
    try:
        _debug_split = _make_split("validation", 23, seed=11, id_start=5000)
        _debug_model = _build_toy_model(seed=19)
        loop.evaluate_split(
            _debug_model,
            _debug_split,
            batch_size=5,
            device="cpu",
            bf16=False,
            include_raw=True,
        )
    except Exception as _error:  # noqa: BLE001
        print(f"Top-level: {type(_error).__name__}: {_error}")
        if _error.__cause__ is not None:
            print(
                f"Direct cause: "
                f"{type(_error.__cause__).__name__}: {_error.__cause__}"
            )
            traceback.print_exception(
                type(_error.__cause__),
                _error.__cause__,
                _error.__cause__.__traceback__,
            )
        else:
            traceback.print_exc()


===== src/digikala_review_transformer/training/data.py =====
class BatchedTokenDataset(Dataset):
    """Flat token arrays + targets + weights. dataset[row_indices] returns one padded batch."""

    def __init__(self, arrays, y, w, pad_id):
        self.ids = arrays.ids
        self.offsets = arrays.offsets
        self.lengths = arrays.lengths
        n_rows = len(self.lengths)
        self.y = {head: np.ascontiguousarray(y[head], dtype=np.int64) for head in HEADS}
        self.w = {head: np.ascontiguousarray(w[head], dtype=np.float32) for head in HEADS}
        for head in HEADS:
            if len(self.y[head]) != n_rows or len(self.w[head]) != n_rows:
                raise DataError(f"'{head}' targets/weights have a different length than the token arrays")
        self.pad_id = int(pad_id)

    def __len__(self):
        return len(self.lengths)

    def __getitem__(self, index):
        rows = np.asarray(index, dtype=np.int64)
        lengths = self.lengths[rows]
        steps = n

# test again

In [3]:
# Test: Training Loop, Evaluation Alignment, Resume and Early Stopping

import copy
import json
import math  # noqa: F811
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

from digikala_review_transformer import config, labels
from digikala_review_transformer.training import (
    checkpoint,
    data,
    loop,
    run_config,
    tokenization,  # noqa: F811
)
from digikala_review_transformer.training import (
    model as model_module,  # noqa: F811
)

# ---- Section: Test Helpers ----
# Restore the notebook RNG after exercising training, checkpointing, and resume.
_loop_initial_rng = checkpoint.capture_rng()
_loop_failures = []


def _loop_check(name, test_function, fix):
    try:
        test_function()
        print(f"[PASS] {name}")
    except Exception as error:  # noqa: BLE001
        message = f"{type(error).__name__}: {error}"
        _loop_failures.append((name, message))
        print(f"[FAIL] {name}: {message} -> {fix}")


class _ToyEncoderOutput:
    def __init__(self, last_hidden_state):
        self.last_hidden_state = last_hidden_state


class _ToyEncoder(nn.Module):
    def __init__(self, vocab_size=64, hidden_size=12):
        super().__init__()
        self.config = type("ToyConfig", (), {"hidden_size": hidden_size})()
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.projection = nn.Linear(hidden_size, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, input_ids, attention_mask=None):
        hidden = self.embeddings(input_ids)
        hidden = torch.tanh(self.projection(hidden))
        hidden = self.norm(hidden)
        return _ToyEncoderOutput(last_hidden_state=hidden)


def _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.1):
    model_module.seed_everything(seed)
    encoder = _ToyEncoder()
    model_module.seed_everything(seed)
    return model_module.TwoHeadClassifier(encoder, head_dropout=head_dropout)


def _make_token_arrays(sequences):
    offsets = [0]
    flat_ids = []
    for sequence in sequences:
        token_ids = [int(value) for value in sequence]
        flat_ids.extend(token_ids)
        offsets.append(len(flat_ids))
    return tokenization.TokenArrays(
        ids=np.asarray(flat_ids, dtype=np.int32),
        offsets=np.asarray(offsets, dtype=np.int64),
    )


def _make_split(
    split_name,
    n_rows,
    *,
    seed,
    id_start,
    variant="B",
    mask_rec_every=5,
):
    rng = np.random.default_rng(seed)
    sequences = []
    rec_corr = np.empty(n_rows, dtype=np.int64)
    band_corr = np.empty(n_rows, dtype=np.int64)
    rec_raw = np.empty(n_rows, dtype=np.int64)
    band_raw = np.empty(n_rows, dtype=np.int64)

    for row_index in range(n_rows):
        rec_class = row_index % 3
        band_class = (row_index // 3 + row_index) % 3
        extra_length = int((row_index * 5 + 3) % 9) + 2

        anchor_rec = 4 + rec_class
        anchor_band = 10 + band_class
        body_tokens = rng.integers(16, 48, size=extra_length).tolist()
        sequence = [2, anchor_rec, anchor_band, *body_tokens, 3]
        sequences.append(sequence)

        rec_corr[row_index] = rec_class
        band_corr[row_index] = band_class
        rec_raw[row_index] = (
            rec_class if row_index % 7 != 0 else (rec_class + 1) % 3
        )
        band_raw[row_index] = (
            band_class if row_index % 9 != 0 else (band_class + 1) % 3
        )

    if mask_rec_every is not None:
        for row_index in range(0, n_rows, mask_rec_every):
            rec_corr[row_index] = data.IGNORE_INDEX
            rec_raw[row_index] = data.IGNORE_INDEX

    if variant == "C":
        w_rec = np.linspace(0.6, 1.0, n_rows, dtype=np.float32)
        w_band = np.linspace(1.0, 0.7, n_rows, dtype=np.float32)
    else:
        w_rec = np.ones(n_rows, dtype=np.float32)
        w_band = np.ones(n_rows, dtype=np.float32)

    arrays = _make_token_arrays(sequences)
    ids = np.arange(id_start, id_start + n_rows, dtype=np.int64)

    return data.SplitData(
        split=split_name,
        ids=ids,
        arrays=arrays,
        pad_id=0,
        y={"rec": rec_corr, "band": band_corr},
        y_raw={"rec": rec_raw, "band": band_raw},
        w={"rec": w_rec, "band": w_band},
        stats={"rows": n_rows},
        origin="test",
        source_rows=n_rows,
    )


def _make_config(
    output_root,
    *,
    run_name="loop_test",
    variant="B",
    max_epochs=2,
    patience=2,
    min_delta=0.0,
    batch_size=8,
    eval_batch_size=7,
    lr_encoder=2.0e-4,
    lr_heads=1.0e-3,
    log_every_steps=2,
):
    return {
        "run_name": run_name,
        "seed": config.SPLIT_SEED,
        "data": {"pack_dir": "data/training_pack"},
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 32,
            "head_dropout": 0.1,
            "tokenizer_backend": "hf_fast",
        },
        "training": {
            "variant": variant,
            "max_epochs": max_epochs,
            "early_stopping_patience": patience,
            "early_stopping_min_delta": min_delta,
            "batch_size": batch_size,
            "eval_batch_size": eval_batch_size,
            "lr_encoder": lr_encoder,
            "lr_heads": lr_heads,
            "weight_decay": 0.01,
            "warmup_ratio": 0.06,
            "max_grad_norm": 1.0,
            "label_smoothing": 0.1,
            "loss_weight_band": 1.0,
            "use_class_weights": True,
            "bf16": False,
            "num_workers": 0,
            "log_every_steps": log_every_steps,
            "train_subset_eval_rows": 12,
        },
        "output": {"root": str(output_root)},
    }


def _manual_probabilities(model, split):
    model.eval()
    dataset = split.dataset()
    batch = dataset[np.arange(split.n_rows, dtype=np.int64)]
    with torch.inference_mode():
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        return {
            head: torch.softmax(logits[head].float(), dim=-1).cpu().numpy()
            for head in data.HEADS
        }


_loop_tmp = tempfile.TemporaryDirectory(prefix="transformer_loop_test_")
_loop_root = Path(_loop_tmp.name)


# ---- Section: Scheduler and Evaluation Order ----
# Check the 6% warmup/decay schedule and length-sorted evaluation restoration.
def _test_linear_scheduler():
    model = _build_toy_model()
    groups = model_module.parameter_groups(
        model,
        lr_encoder=2.0e-5,
        lr_heads=1.0e-4,
        weight_decay=0.01,
    )
    optimizer = torch.optim.AdamW(groups)
    scheduler, warmup_steps = loop.make_linear_scheduler(
        optimizer,
        total_steps=100,
        warmup_ratio=0.06,
    )

    assert warmup_steps == 6
    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 0.0, abs_tol=1e-12)
    assert math.isclose(heads_lr, 0.0, abs_tol=1e-12)

    for _ in range(warmup_steps):
        optimizer.step()
        scheduler.step()

    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 2.0e-5, rel_tol=1e-6)
    assert math.isclose(heads_lr, 1.0e-4, rel_tol=1e-6)

    for _ in range(100 - warmup_steps):
        optimizer.step()
        scheduler.step()

    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 0.0, abs_tol=1e-12)
    assert math.isclose(heads_lr, 0.0, abs_tol=1e-12)


_loop_check(
    "linear scheduler warms up for 6% of total steps and decays to zero",
    _test_linear_scheduler,
    "check make_linear_scheduler warmup_steps and decay denominator",
)


def _test_eval_restores_original_order_and_schema():
    split = _make_split("validation", 23, seed=11, id_start=5000)
    model = _build_toy_model(seed=19)

    expected = _manual_probabilities(model, split)
    evaluation = loop.evaluate_split(
        model,
        split,
        batch_size=5,
        device="cpu",
        bf16=False,
        include_raw=True,
    )

    for head in data.HEADS:
        assert np.allclose(
            evaluation.probabilities[head],
            expected[head],
            rtol=1e-5,
            atol=1e-6,
        ), f"Restored probabilities for {head} do not match row order."
        assert math.isfinite(evaluation.plain_ce[head])
        assert math.isfinite(evaluation.corrected_metrics[head]["macro_f1"])
        assert math.isfinite(evaluation.raw_metrics[head]["macro_f1"])

    frame = loop.probability_frame(split, evaluation.probabilities)
    expected_columns = [
        "id",
        *(f"p_{name}" for name in labels.CLASSES["rec"]),
        *(f"p_{name}" for name in labels.CLASSES["band"]),
    ]
    assert list(frame.columns) == expected_columns
    assert np.array_equal(frame["id"].to_numpy(), split.ids)
    for column in expected_columns[1:]:
        assert frame[column].dtype == np.float32


_loop_check(
    "sorted evaluation restores original id order and float32 probability columns",
    _test_eval_restores_original_order_and_schema,
    "check sorted_eval_batches index restoration and probability_frame columns",
)


# ---- Section: Full Fit, Pause/Resume, and Hash Guard ----
# A 1-epoch pause followed by resume must match an uninterrupted 2-epoch run.
def _test_fit_and_exact_resume():
    train_split = _make_split("train", 24, seed=101, id_start=100, variant="C")
    val_split = _make_split("validation", 18, seed=202, id_start=1000)
    subset_split = _make_split(
        "train_subset",
        12,
        seed=303,
        id_start=3000,
        mask_rec_every=6,
    )

    direct_cfg = _make_config(
        _loop_root / "direct",
        run_name="direct_two_epochs",
        variant="C",
        max_epochs=2,
        patience=2,
    )
    direct_model = _build_toy_model(seed=config.SPLIT_SEED)
    direct_result = loop.fit(
        direct_cfg,
        direct_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert direct_result.status == "trained"
    assert direct_result.epochs_done == 2
    assert direct_result.global_step == 6

    resume_cfg = _make_config(
        _loop_root / "resumed",
        run_name="paused_then_resumed",
        variant="C",
        max_epochs=2,
        patience=2,
    )
    paused_model = _build_toy_model(seed=config.SPLIT_SEED)
    paused_result = loop.fit(
        resume_cfg,
        paused_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
        max_epochs_this_call=1,
    )

    assert paused_result.status == "paused"
    assert paused_result.stop_reason == "paused_by_call_limit"
    assert paused_result.epochs_done == 1
    assert (paused_result.run_directory / "last.pt").is_file()

    # Append an uncommitted step row to simulate a crash during epoch 2.
    train_log_path = paused_result.run_directory / "train_log.csv"
    log_df = pd.read_csv(train_log_path)
    extra_row = log_df.iloc[[-1]].copy()
    extra_row["global_step"] = 999
    pd.concat([log_df, extra_row], ignore_index=True).to_csv(
        train_log_path,
        index=False,
    )

    wrong_cfg = copy.deepcopy(resume_cfg)
    wrong_cfg["training"]["lr_heads"] = 9.0e-4
    try:
        loop.fit(
            wrong_cfg,
            _build_toy_model(seed=config.SPLIT_SEED),
            train_split,
            val_split,
            subset_split,
            run_directory=paused_result.run_directory,
            device="cpu",
        )
    except checkpoint.CheckpointError as error:
        assert "config hash" in str(error).lower()
    else:
        raise AssertionError("Mismatched config hash did not fail on resume.")

    resumed_model = _build_toy_model(seed=999)
    resumed_result = loop.fit(
        resume_cfg,
        resumed_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert resumed_result.status == "trained"
    assert resumed_result.epochs_done == 2
    assert resumed_result.global_step == 6

    for name, direct_tensor in direct_model.state_dict().items():
        resumed_tensor = resumed_model.state_dict()[name]
        assert torch.allclose(
            direct_tensor,
            resumed_tensor,
            rtol=1e-6,
            atol=1e-6,
        ), f"Final best weight {name} differed after resume."

    direct_history = pd.read_csv(direct_result.run_directory / "history.csv")
    resumed_history = pd.read_csv(resumed_result.run_directory / "history.csv")
    required_history_columns = {
        "train_objective",
        "train_plain_ce_rec",
        "train_plain_ce_band",
        "train_subset_plain_ce_rec",
        "train_subset_plain_ce_band",
        "val_plain_ce_rec",
        "val_plain_ce_band",
        "val_acc_rec",
        "val_acc_band",
        "val_macro_f1_rec",
        "val_macro_f1_band",
        "val_macro_f1_rec_raw",
        "val_macro_f1_band_raw",
        "train_subset_macro_f1_rec",
        "train_subset_macro_f1_band",
        "val_mean_macro_f1",
        "lr_end",
        "epoch_seconds",
        "is_best",
    }
    assert required_history_columns.issubset(direct_history.columns)

    compared_columns = [
        column
        for column in direct_history.columns
        if column != "epoch_seconds"
    ]
    pd.testing.assert_frame_equal(
        direct_history[compared_columns],
        resumed_history[compared_columns],
        rtol=1e-5,
        atol=1e-6,
    )

    repaired_log = pd.read_csv(train_log_path)
    assert int(repaired_log["global_step"].max()) == 6
    assert (resumed_result.run_directory / "best_model.pt").is_file()
    assert (resumed_result.run_directory / "val_probs_best.parquet").is_file()
    assert (resumed_result.run_directory / "train.log").is_file()


_loop_check(
    "fit supports pause/resume, prunes uncommitted logs, and rejects hash changes",
    _test_fit_and_exact_resume,
    "check save_last/restore_last integration and epoch-boundary state checks",
)


# ---- Section: Shared Early Stopping and Best Weight Reload ----
# When epoch 2 does not beat min_delta, patience=1 stops and reloads epoch 1.
def _test_early_stopping_and_best_reload():
    train_split = _make_split("train", 16, seed=404, id_start=400)
    val_split = _make_split("validation", 15, seed=505, id_start=1400)
    subset_split = _make_split("train_subset", 10, seed=606, id_start=2400)

    cfg = _make_config(
        _loop_root / "early_stop",
        run_name="early_stop_run",
        max_epochs=4,
        patience=1,
        min_delta=2.0,
        lr_encoder=5.0e-3,
        lr_heads=1.0e-2,
    )
    model = _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.0)
    result = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert result.status == "trained"
    assert result.stop_reason == "early_stopping_patience"
    assert result.epochs_done == 2
    assert result.best_epoch == 1
    assert [row["is_best"] for row in result.history] == [True, False]

    best_state = torch.load(
        result.best_model_path,
        map_location="cpu",
        weights_only=True,
    )
    last_payload = checkpoint.load_last(
        result.run_directory / "last.pt",
        expected_hash=run_config.config_hash(cfg),
    )

    for name, best_tensor in best_state.items():
        assert torch.equal(model.state_dict()[name].cpu(), best_tensor)

    changed_after_best = any(
        not torch.equal(
            best_state[name].cpu(),
            last_payload["model_state_dict"][name].cpu(),
        )
        for name in best_state
    )
    assert changed_after_best, "Epoch 2 did not change weights from epoch 1."

    # Mark the run complete and verify a second call skips without overwriting.
    (result.run_directory / "run_summary.json").write_text(
        json.dumps({"status": "complete"}),
        encoding="utf-8",
    )
    skipped = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )
    assert skipped.skipped is True
    assert skipped.status == "complete"


_loop_check(
    "shared early stopping stops on patience and reloads best_model.pt",
    _test_early_stopping_and_best_reload,
    "check val_mean_macro_f1 improvement logic and load_best_weights at the end of fit",
)


# ---- Section: Cleanup and Final Result ----
# Restore the notebook RNG and remove temporary test artifacts.
_loop_check(
    "notebook RNG state is restored after training loop tests",
    lambda: checkpoint.restore_rng(_loop_initial_rng),
    "check capture_rng/restore_rng around the test cell",
)

_loop_tmp.cleanup()

if _loop_failures:
    print(
        f"[FAIL] {len(_loop_failures)} training loop test(s) failed; "
        "see the failure details above and apply the listed fixes."
    )
else:
    print("[PASS] ALL TRAINING LOOP TESTS PASSED")

[PASS] linear scheduler warms up for 6% of total steps and decays to zero
[PASS] sorted evaluation restores original id order and float32 probability columns
[run] start run=direct_two_epochs device=cpu variant=C seed=2026
[model] parameter_counts={'total': 1026, 'trainable': 1026, 'encoder': 948, 'heads': 78}
[optimizer] fused=False steps_per_epoch=3 total_steps=6 warmup_steps=0
[train] run=direct_two_epochs epoch=1/2 step=2/3 global_step=2/6 progress=33.33% objective=2.208625 lr_encoder=1.333e-04 lr_heads=6.667e-04 samples_per_second=5289.47 ETA=0.0 min
[train] run=direct_two_epochs epoch=1/2 step=3/3 global_step=3/6 progress=50.00% objective=2.223766 lr_encoder=1.000e-04 lr_heads=5.000e-04 samples_per_second=5274.60 ETA=0.0 min
[timer] transformer_train_epoch_1: 0.0s
[timer] transformer_validation_epoch_1: 0.0s
[timer] transformer_train_subset_epoch_1: 0.0s
[epoch] epoch=1/2 train_objective=2.213672 val_mean_macro_f1=0.157407 best_epoch=1 is_best=True bad_epochs=0/2 epoch_seconds=0.

# Inspect: Predict and Run-Summary Interfaces (Read-Only)# 

In [4]:
# Inspect: Predict and Run-Summary Interfaces (Read-Only)

import ast
from pathlib import Path

# ---- Section: Locate Project Root ----
# Read module sources directly without modifying anything on disk.
_current_dir = Path.cwd()
_project_root = next(
    (
        candidate
        for candidate in (_current_dir, *_current_dir.parents)
        if (candidate / "src" / "digikala_review_transformer").is_dir()
    ),
    None,
)

if _project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing "
        "'src/digikala_review_transformer'."
    )


# ---- Section: Print Target Definitions ----
# Inspect exact signatures and return keys used by predict.py.
_targets = [
    (
        "src/digikala_review_transformer/training/pack.py",
        {"sha256_file", "pack_fingerprint", "pack_file_name"},
    ),
    (
        "src/digikala_review_transformer/training/tokenization.py",
        {
            "HfFastBundle",
            "SentencePieceBundle",
            "load_saved_tokenizer",
            "token_stats",
        },
    ),
    (
        "src/digikala_review_transformer/training/model.py",
        {"model_summary", "save_encoder_config", "count_parameters"},
    ),
    (
        "src/digikala_review_transformer/training/run_config.py",
        {"dump_resolved_config", "config_hash", "run_dir", "project_path"},
    ),
]

for _relative_path, _selected in _targets:
    _path = _project_root / _relative_path
    _text = _path.read_text(encoding="utf-8")
    _tree = ast.parse(_text, filename=str(_path))
    print(f"\n===== {_relative_path} =====")
    for _node in _tree.body:
        if isinstance(_node, (ast.FunctionDef, ast.ClassDef)) and _node.name in _selected:
            print(ast.get_source_segment(_text, _node))
            print()


===== src/digikala_review_transformer/training/pack.py =====
def pack_file_name(split):
    return f"{split}.parquet"

def sha256_file(path, chunk_bytes=1 << 20):
    """Return the sha256 hex digest of a file, read in chunks."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            block = handle.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def pack_fingerprint(manifest):
    """Short id of the pack contents (used in token-cache keys)."""
    joined = "".join(manifest["files"][pack_file_name(s)]["sha256"] for s in SPLITS)
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()[:16]


===== src/digikala_review_transformer/training/tokenization.py =====
class HfFastBundle(TokenizerBundle):
    backend = "hf_fast"
    structure = "tokenizer adds its own special tokens; truncation keeps the end token"

    def __init__(self, source_dir, max_length):
     

# Test: Predict, Probability Alignment, Run Summary and Final Cleanup

In [5]:
# Test: Predict, Probability Alignment, Run Summary and Final Cleanup

import copy
import json
import math  # noqa: F811
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

from digikala_review_transformer import config, labels
from digikala_review_transformer.training import (
    checkpoint,
    data,
    loop,
    pack,
    predict,
    run_config,
    tokenization,  # noqa: F811
)
from digikala_review_transformer.training import (
    model as model_module,  # noqa: F811
)

# ---- Section: Test Helpers ----
# Keep the notebook RNG state unchanged after running prediction and finalization tests.
_predict_initial_rng = checkpoint.capture_rng()
_predict_failures = []


def _predict_check(name, test_function, fix):
    try:
        test_function()
        print(f"[PASS] {name}")
    except Exception as error:  # noqa: BLE001
        message = f"{type(error).__name__}: {error}"
        _predict_failures.append((name, message))
        print(f"[FAIL] {name}: {message} -> {fix}")


class _ToyEncoderConfig:
    def __init__(self, hidden_size=12, vocab_size=64):
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size
        self.model_type = "toy_encoder"

    def to_dict(self):
        return {
            "hidden_size": self.hidden_size,
            "vocab_size": self.vocab_size,
            "model_type": self.model_type,
        }


class _ToyEncoderOutput:
    def __init__(self, last_hidden_state):
        self.last_hidden_state = last_hidden_state


class _ToyEncoder(nn.Module):
    def __init__(self, vocab_size=64, hidden_size=12):
        super().__init__()
        self.config = _ToyEncoderConfig(
            hidden_size=hidden_size,
            vocab_size=vocab_size,
        )
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.projection = nn.Linear(hidden_size, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, input_ids, attention_mask=None):
        hidden = self.embeddings(input_ids)
        hidden = torch.tanh(self.projection(hidden))
        hidden = self.norm(hidden)
        return _ToyEncoderOutput(last_hidden_state=hidden)


class _ToyTokenizerBundle:
    backend = "hf_fast"
    name = "ToyBERT"
    max_length = 32
    first_id = 2
    last_id = 3
    pad_id = 0
    unk_id = 1

    def info(self):
        return {
            "name": self.name,
            "backend": self.backend,
            "max_length": self.max_length,
            "fingerprint": "toy-tokenizer-fp",
            "special_ids": {
                "first": self.first_id,
                "last": self.last_id,
                "pad": self.pad_id,
                "unk": self.unk_id,
            },
        }

    def save(self, out_dir):
        out_path = Path(out_dir)
        out_path.mkdir(parents=True, exist_ok=True)
        (out_path / "tokenizer_backend.json").write_text(
            json.dumps(self.info(), indent=2),
            encoding="utf-8",
        )
        (out_path / "vocab.txt").write_text(
            "[PAD]\n[UNK]\n[CLS]\n[SEP]\n",
            encoding="utf-8",
        )


def _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.0):
    model_module.seed_everything(seed)
    encoder = _ToyEncoder()
    model_module.seed_everything(seed)
    return model_module.TwoHeadClassifier(encoder, head_dropout=head_dropout)


def _make_token_arrays(sequences):
    offsets = [0]
    flat_ids = []
    for sequence in sequences:
        token_ids = [int(value) for value in sequence]
        flat_ids.extend(token_ids)
        offsets.append(len(flat_ids))
    return tokenization.TokenArrays(
        ids=np.asarray(flat_ids, dtype=np.int32),
        offsets=np.asarray(offsets, dtype=np.int64),
    )


def _make_split(
    split_name,
    n_rows,
    *,
    seed,
    id_start,
    variant="B",
    mask_rec_every=5,
):
    rng = np.random.default_rng(seed)
    bundle = _ToyTokenizerBundle()
    sequences = []
    rec_corr = np.empty(n_rows, dtype=np.int64)
    band_corr = np.empty(n_rows, dtype=np.int64)
    rec_raw = np.empty(n_rows, dtype=np.int64)
    band_raw = np.empty(n_rows, dtype=np.int64)

    for row_index in range(n_rows):
        rec_class = row_index % 3
        band_class = (row_index // 3 + row_index) % 3
        body_length = int((row_index * 7 + 5) % 11) + 2

        body_tokens = rng.integers(8, 48, size=body_length).tolist()
        if row_index % 6 == 0:
            body_tokens[0] = bundle.unk_id
        sequence = [
            bundle.first_id,
            4 + rec_class,
            10 + band_class,
            *body_tokens,
            bundle.last_id,
        ][: bundle.max_length]
        sequences.append(sequence)

        rec_corr[row_index] = rec_class
        band_corr[row_index] = band_class
        rec_raw[row_index] = (
            rec_class if row_index % 7 != 0 else (rec_class + 1) % 3
        )
        band_raw[row_index] = (
            band_class if row_index % 8 != 0 else (band_class + 1) % 3
        )

    if mask_rec_every is not None:
        for row_index in range(0, n_rows, mask_rec_every):
            rec_corr[row_index] = data.IGNORE_INDEX
            rec_raw[row_index] = data.IGNORE_INDEX

    if variant == "C":
        w_rec = np.linspace(0.7, 1.0, n_rows, dtype=np.float32)
        w_band = np.linspace(1.0, 0.8, n_rows, dtype=np.float32)
    else:
        w_rec = np.ones(n_rows, dtype=np.float32)
        w_band = np.ones(n_rows, dtype=np.float32)

    arrays = _make_token_arrays(sequences)
    stats = tokenization.token_stats(arrays, bundle)
    # Give splits non-monotonic, realistic ids so order bugs are caught immediately.
    ids = np.asarray(
        [id_start + ((index * 37 + 13) % 997) + index * 1000 for index in range(n_rows)],
        dtype=np.int64,
    )

    return data.SplitData(
        split=split_name,
        ids=ids,
        arrays=arrays,
        pad_id=bundle.pad_id,
        y={"rec": rec_corr, "band": band_corr},
        y_raw={"rec": rec_raw, "band": band_raw},
        w={"rec": w_rec, "band": w_band},
        stats=stats,
        origin="test",
        source_rows=n_rows,
    )


def _make_manifest():
    return {
        "format_version": 1,
        "files": {
            "train.parquet": {
                "sha256": "a" * 64,
                "rows": 24,
                "size_mb": 0.1,
            },
            "validation.parquet": {
                "sha256": "b" * 64,
                "rows": 18,
                "size_mb": 0.05,
            },
            "test.parquet": {
                "sha256": "c" * 64,
                "rows": 21,
                "size_mb": 0.06,
            },
        },
    }


def _make_config(output_root, *, run_name="predict_test_run"):
    return {
        "run_name": run_name,
        "seed": config.SPLIT_SEED,
        "data": {"pack_dir": "data/training_pack"},
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 32,
            "head_dropout": 0.0,
            "tokenizer_backend": "hf_fast",
        },
        "training": {
            "variant": "C",
            "max_epochs": 2,
            "early_stopping_patience": 1,
            "early_stopping_min_delta": 0.0,
            "batch_size": 8,
            "eval_batch_size": 6,
            "lr_encoder": 2.0e-4,
            "lr_heads": 1.0e-3,
            "weight_decay": 0.01,
            "warmup_ratio": 0.06,
            "max_grad_norm": 1.0,
            "label_smoothing": 0.1,
            "loss_weight_band": 1.0,
            "use_class_weights": True,
            "bf16": False,
            "num_workers": 0,
            "log_every_steps": 2,
            "train_subset_eval_rows": 12,
        },
        "output": {"root": str(output_root)},
    }


def _direct_probabilities(model, split):
    model.eval()
    batch = split.dataset()[np.arange(split.n_rows, dtype=np.int64)]
    with torch.inference_mode():
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        return {
            head: torch.softmax(logits[head].float(), dim=-1).cpu().numpy()
            for head in data.HEADS
        }


_predict_tmp = tempfile.TemporaryDirectory(prefix="transformer_predict_test_")
_predict_root = Path(_predict_tmp.name)


# ---- Section: Probability Schema, Order Restoration, and Corruption Guards ----
# Verify length-sorted prediction writes exact id order and rejects corrupted tables.
def _test_predict_split_and_verify_guards():
    test_split = _make_split("test", 21, seed=17, id_start=9000)
    model = _build_toy_model(seed=31)
    parquet_path = _predict_root / "standalone" / "test_probs.parquet"

    frame, probabilities = predict.predict_split_to_parquet(
        model,
        test_split,
        parquet_path,
        batch_size=4,
        device="cpu",
        bf16=False,
    )
    expected = _direct_probabilities(model, test_split)

    assert parquet_path.is_file()
    assert list(frame.columns) == [
        "id",
        "p_recommended",
        "p_not_recommended",
        "p_no_idea",
        "p_low",
        "p_medium",
        "p_high",
    ]
    assert np.array_equal(frame["id"].to_numpy(), test_split.ids)

    for head in data.HEADS:
        assert np.allclose(
            probabilities[head],
            expected[head],
            rtol=1.0e-5,
            atol=1.0e-6,
        )

    reloaded_frame, _ = predict.read_and_verify_probability_parquet(
        parquet_path,
        test_split,
    )
    for column_name in predict.expected_probability_columns()[1:]:
        assert reloaded_frame[column_name].dtype == np.float32

    swapped = reloaded_frame.copy()
    swapped.iloc[[0, 1]] = swapped.iloc[[1, 0]].to_numpy()
    try:
        predict.verify_probability_frame(swapped, test_split)
    except predict.PredictionError as error:
        assert "aligned" in str(error).lower()
    else:
        raise AssertionError("Misaligned ids were accepted.")

    wrong_dtype = reloaded_frame.copy()
    wrong_dtype["p_low"] = wrong_dtype["p_low"].astype(np.float64)
    try:
        predict.verify_probability_frame(wrong_dtype, test_split)
    except predict.PredictionError as error:
        assert "float32" in str(error).lower()
    else:
        raise AssertionError("float64 probability column was accepted.")

    bad_sum = reloaded_frame.copy()
    bad_sum.loc[0, "p_low"] = np.float32(0.95)
    bad_sum.loc[0, "p_medium"] = np.float32(0.95)
    try:
        predict.verify_probability_frame(bad_sum, test_split)
    except predict.PredictionError as error:
        assert "sum to 1" in str(error).lower()
    else:
        raise AssertionError("Probabilities not summing to 1 were accepted.")


_predict_check(
    "predict_split_to_parquet restores id order and rejects misaligned/bad probability tables",
    _test_predict_split_and_verify_guards,
    "check predict_split_to_parquet and verify_probability_frame",
)


# ---- Section: Both-View Quick Metrics and Incomplete Run Guard ----
# Both corrected and raw views must be reported; paused runs must not be finalized.
def _test_quick_metrics_and_paused_guard():
    train_split = _make_split("train", 24, seed=101, id_start=100, variant="C")
    val_split = _make_split("validation", 18, seed=202, id_start=1000)
    test_split = _make_split("test", 21, seed=303, id_start=2000)
    subset_split = _make_split("train_subset", 12, seed=404, id_start=3000)

    paused_cfg = _make_config(
        _predict_root / "paused_guard",
        run_name="paused_run",
    )
    paused_model = _build_toy_model(seed=config.SPLIT_SEED)
    paused_loop = loop.fit(
        paused_cfg,
        paused_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
        max_epochs_this_call=1,
    )

    try:
        predict.finalize_run(
            paused_cfg,
            paused_model,
            _ToyTokenizerBundle(),
            paused_loop,
            train_split=train_split,
            validation_split=val_split,
            test_split=test_split,
            train_subset_split=subset_split,
            manifest=_make_manifest(),
            load_info={"unexpected_keys": [], "missing_keys": []},
            device="cpu",
        )
    except predict.PredictionError as error:
        assert "paused" in str(error).lower()
    else:
        raise AssertionError("A paused run was allowed to finalize.")

    assert (paused_loop.run_directory / "last.pt").is_file()


_predict_check(
    "finalize_run refuses paused runs and leaves last.pt intact",
    _test_quick_metrics_and_paused_guard,
    "check finalize_run status validation before writing run_summary.json",
)


# ---- Section: Full End-to-End Finalize, Best Weight Reload, Summary, and Cleanup ----
# Corrupt in-memory weights before finalize_run to prove best_model.pt is reloaded.
def _test_full_finalize_and_idempotent_rerun():
    train_split = _make_split("train", 24, seed=501, id_start=100, variant="C")
    val_split = _make_split("validation", 18, seed=502, id_start=1000)
    test_split = _make_split("test", 21, seed=503, id_start=2000)
    subset_split = _make_split("train_subset", 12, seed=504, id_start=3000)
    manifest = _make_manifest()
    load_info = {
        "unexpected_keys": ["cls.predictions.bias"],
        "missing_keys": ["pooler.dense.weight", "pooler.dense.bias"],
        "mismatched_keys": [],
        "error_msgs": [],
    }

    cfg = _make_config(
        _predict_root / "complete_flow",
        run_name="tooka_toy_C",
    )
    model = _build_toy_model(seed=config.SPLIT_SEED)
    loop_result = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )
    run_directory = loop_result.run_directory
    assert (run_directory / "last.pt").is_file()

    reference_model = _build_toy_model(seed=777)
    checkpoint.load_best_weights(run_directory / "best_model.pt", reference_model)
    expected_test_probs = _direct_probabilities(reference_model, test_split)

    # Deliberately corrupt the in-memory model before finalization.
    with torch.no_grad():
        for parameter in model.parameters():
            parameter.fill_(0.0)

    finalized = predict.finalize_run(
        cfg,
        model,
        _ToyTokenizerBundle(),
        loop_result,
        train_split=train_split,
        validation_split=val_split,
        test_split=test_split,
        train_subset_split=subset_split,
        manifest=manifest,
        load_info=load_info,
        device="cpu",
    )

    assert not (run_directory / "last.pt").exists(), "last.pt was not deleted."
    assert checkpoint.run_status(run_directory) == "complete"

    required_files = [
        "config_resolved.yaml",
        "history.csv",
        "train_log.csv",
        "train.log",
        "run_summary.json",
        "val_probs_best.parquet",
        "test_probs.parquet",
        "best_model.pt",
        "encoder_config.json",
        "tokenizer/tokenizer_backend.json",
        "tokenizer/vocab.txt",
    ]
    for relative_path in required_files:
        assert (run_directory / relative_path).is_file(), (
            f"Missing required artifact: {relative_path}"
        )

    _, saved_test_probs = predict.read_and_verify_probability_parquet(
        finalized.test_probabilities_path,
        test_split,
    )
    for head in data.HEADS:
        assert np.allclose(
            saved_test_probs[head],
            expected_test_probs[head],
            rtol=1.0e-5,
            atol=1.0e-6,
        ), f"Test predictions for {head} did not use reloaded best_model.pt."

    summary = json.loads(
        (run_directory / "run_summary.json").read_text(encoding="utf-8")
    )
    assert summary["status"] == "complete"
    assert summary["run_name"] == "tooka_toy_C"
    assert summary["seed"] == config.SPLIT_SEED
    assert summary["variant"] == "C"
    assert summary["config_hash"] == run_config.config_hash(cfg)
    assert summary["tokenizer_backend"] == "hf_fast"
    assert summary["encoder_load_info"] == load_info
    assert summary["selection_and_stopping"]["test_used_for_any_decision"] is False
    assert summary["selection_and_stopping"]["labels_used_for_selection"] == (
        "step1_corrected_only"
    )
    assert summary["selection_and_stopping"]["best_epoch"] == loop_result.best_epoch

    for split_name in ("train", "validation", "test", "train_subset"):
        assert "unk_pct_of_tokens" in summary["token_stats"][split_name]
        assert "rows_at_max_length_pct" in summary["token_stats"][split_name]

    for block_name in ("validation_best", "test_quick"):
        block = summary[block_name]
        for view in ("corrected", "raw"):
            assert view in block
            assert math.isfinite(block[view]["mean_macro_f1"])
            for head in data.HEADS:
                assert math.isfinite(block[view][head]["macro_f1"])
                assert math.isfinite(block[view][head]["accuracy"])
                assert math.isfinite(block[view][head]["balanced_accuracy"])
            assert "mean_abs_distance" in block[view]["band"]
            assert "neighbor_error_pct" in block[view]["band"]
            assert "extreme_error_pct" in block[view]["band"]

    assert summary["pack"]["fingerprint"] == pack.pack_fingerprint(manifest)
    for relative_name, recorded_hash in summary["artifact_sha256"].items():
        actual_hash = pack.sha256_file(run_directory / relative_name)
        assert recorded_hash == actual_hash, (
            f"SHA-256 mismatch for {relative_name}"
        )

    # Simulate a second call on an already completed run.
    skipped_loop = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )
    assert skipped_loop.skipped is True
    finalized_again = predict.finalize_run(
        cfg,
        model,
        _ToyTokenizerBundle(),
        skipped_loop,
        train_split=train_split,
        validation_split=val_split,
        test_split=test_split,
        train_subset_split=subset_split,
        manifest=manifest,
        load_info=load_info,
        device="cpu",
    )
    assert finalized_again.summary["config_hash"] == summary["config_hash"]


_predict_check(
    "finalize_run reloads best_model.pt, writes complete artifacts/hashes, and deletes last.pt",
    _test_full_finalize_and_idempotent_rerun,
    "check finalize_run artifact creation, test_quick calculation, and cleanup_after_complete",
)


# ---- Section: Cleanup and Final Result ----
# Restore notebook RNG state and clean temporary files.
_predict_check(
    "notebook RNG state is restored after predict tests",
    lambda: checkpoint.restore_rng(_predict_initial_rng),
    "check capture_rng/restore_rng around the test cell",
)

_predict_tmp.cleanup()

if _predict_failures:
    print(
        f"[FAIL] {len(_predict_failures)} predict test(s) failed; "
        "see the failure details above and apply the listed fixes."
    )
else:
    print("[PASS] ALL PREDICT TESTS PASSED")

[PASS] predict_split_to_parquet restores id order and rejects misaligned/bad probability tables
[run] start run=paused_run device=cpu variant=C seed=2026
[model] parameter_counts={'total': 1026, 'trainable': 1026, 'encoder': 948, 'heads': 78}
[optimizer] fused=False steps_per_epoch=3 total_steps=6 warmup_steps=0
[train] run=paused_run epoch=1/2 step=2/3 global_step=2/6 progress=33.33% objective=2.222551 lr_encoder=1.333e-04 lr_heads=6.667e-04 samples_per_second=4585.09 ETA=0.0 min
[train] run=paused_run epoch=1/2 step=3/3 global_step=3/6 progress=50.00% objective=2.179136 lr_encoder=1.000e-04 lr_heads=5.000e-04 samples_per_second=4271.79 ETA=0.0 min
[timer] transformer_train_epoch_1: 0.0s
[timer] transformer_validation_epoch_1: 0.0s
[timer] transformer_train_subset_epoch_1: 0.0s
[epoch] epoch=1/2 train_objective=2.208079 val_mean_macro_f1=0.157407 best_epoch=1 is_best=True bad_epochs=0/1 epoch_seconds=0.0
[run] training_loop_status=paused stop_reason=paused_by_call_limit epochs_done=1 

# Cell Title: Inspect: Pack Manifest and Signatures Used by Smoke (Read-Only)

In [2]:
import inspect

from digikala_review_transformer import config
from digikala_review_transformer.training import data, losses, model, pack, tokenization  # noqa: F811

opened = data.open_pack(config.ROOT / "data" / "training_pack")
print("open_pack type & len:", type(opened), len(opened) if hasattr(opened, "__len__") else None)
if isinstance(opened, tuple):
    for i, item in enumerate(opened):
        print(f"  tuple[{i}]:", type(item), item if not isinstance(item, dict) else list(item.keys()))
elif isinstance(opened, dict):
    print("  dict keys:", list(opened.keys()))

for fn in (
    pack.verify_pack,
    tokenization.load_tokenizer,
    tokenization.get_split_tokens,
    data.load_split,
    data.subset_row_indices,
    data.LengthBucketedBatches,
    data.batch_fingerprint,
    model.build_model,
    model.check_token_ids_fit,
    model.count_parameters,
    model.parameter_groups,
    losses.build_loss,
):
    print(f"{fn.__module__.split('.')[-1]}.{fn.__name__}{inspect.signature(fn)}")

open_pack type & len: <class 'dict'> 7
  dict keys: ['format_version', 'code', 'label_codes', 'label_code_note', 'weights', 'files', 'sources']
pack.verify_pack(pack_dir, check_content=False)
tokenization.load_tokenizer(model_path, backend, max_length)
tokenization.get_split_tokens(bundle, split, source_sha256, load_texts, expected_rows, cache_dir=None, show_progress=True)
data.load_split(split, bundle, pack_dir, manifest, variant, subset_rows=None, subset_seed=2026, cache_dir=None, show_progress=True)
data.subset_row_indices(n_rows, size, seed, purpose)
data.LengthBucketedBatches(lengths, batch_size, seed, chunk_batches=50)
data.batch_fingerprint(batches, n_batches=5)
model.build_model(cfg)
model.check_token_ids_fit(model, max_token_id, pad_id)
model.count_parameters(model)
model.parameter_groups(model, lr_encoder, lr_heads, weight_decay)
losses.build_loss(cfg, y_train, device)


# Cell Title: Inspect: Manifest Files and Object Attributes for Smoke (Read-Only)

In [3]:
from digikala_review_transformer import config
from digikala_review_transformer.training import data, pack, tokenization

pack_dir = config.ROOT / "data" / "training_pack"
manifest = data.open_pack(pack_dir)
print("manifest['files']:", manifest["files"])

vp = pack.verify_pack(pack_dir, check_content=False)
print("verify_pack return:", type(vp), vp[:2] if isinstance(vp, list) else vp)

print("TokenArrays attrs:", [a for a in dir(tokenization.TokenArrays) if not a.startswith("_")])
print("SplitData attrs:", [a for a in dir(data.SplitData) if not a.startswith("_")])
print("LengthBucketedBatches attrs:", [a for a in dir(data.LengthBucketedBatches) if not a.startswith("_")])

manifest['files']: {'train.parquet': {'rows': 2860236, 'label_missing': {'y_rec': 468079, 'y_band': 0, 'y_rec_raw': 468079, 'y_band_raw': 0}, 'blank_body_rows': 0, 'columns': ['id', 'body', 'y_rec', 'y_band', 'y_rec_raw', 'y_band_raw', 'w_rec', 'w_band'], 'sha256': 'b770d123b261dc7526f3d003544029c60c95c4de3a639922a32dece275b26b4c', 'size_mb': 119.3}, 'validation.parquet': {'rows': 922794, 'label_missing': {'y_rec': 134117, 'y_band': 0, 'y_rec_raw': 134117, 'y_band_raw': 0}, 'blank_body_rows': 0, 'columns': ['id', 'body', 'y_rec', 'y_band', 'y_rec_raw', 'y_band_raw'], 'sha256': '2ff8bcf518abf62d1179449c565da322b94fb9accc38c9f27e8643770e6ebe30', 'size_mb': 29.5}, 'test.parquet': {'rows': 922805, 'label_missing': {'y_rec': 134960, 'y_band': 1, 'y_rec_raw': 134960, 'y_band_raw': 1}, 'blank_body_rows': 0, 'columns': ['id', 'body', 'y_rec', 'y_band', 'y_rec_raw', 'y_band_raw'], 'sha256': '6eee541ad46f1003fe6f8634ceddccce7399d6ce6e27afa77fce92bcd0084902', 'size_mb': 29.5}}
verify_pack return:

# Cell Title: Inspect: Remaining Interfaces Used in Smoke (Read-Only)

In [3]:
import inspect

from digikala_review_transformer.training import data, losses, tokenization

print("SentencePieceBundle attrs:", [a for a in dir(tokenization.SentencePieceBundle) if not a.startswith("_")])
print("encode_texts sig:", inspect.signature(tokenization.encode_texts))
print("LengthBucketedBatches source:\n", inspect.getsource(data.LengthBucketedBatches))
print("batch_fingerprint source:\n", inspect.getsource(data.batch_fingerprint))
print("SplitData annotations:", data.SplitData.__annotations__)
# TwoHeadLoss may not define a `forward` method; inspect the class itself and
# fall back to `__call__` if needed.
print("TwoHeadLoss source:\n", inspect.getsource(losses.TwoHeadLoss))

if hasattr(losses.TwoHeadLoss, "forward"):
    print("TwoHeadLoss.forward source:\n", inspect.getsource(losses.TwoHeadLoss.forward))
else:
    print("TwoHeadLoss has no `forward` method; checking `__call__` instead.")
    if callable(losses.TwoHeadLoss):
        try:
            print("TwoHeadLoss.__call__ source:\n", inspect.getsource(losses.TwoHeadLoss.__call__))
        except (OSError, TypeError):
            print("Could not inspect __call__ source.")

SentencePieceBundle attrs: ['backend', 'encode', 'info', 'save', 'structure']
encode_texts sig: (bundle, texts, show_progress=False)
LengthBucketedBatches source:
 class LengthBucketedBatches:
    """Training batches. The order depends only on (seed, epoch, lengths), never on the variant."""

    def __init__(self, lengths, batch_size, seed, chunk_batches=CHUNK_BATCHES):
        self.lengths = np.asarray(lengths)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.chunk_batches = int(chunk_batches)
        self.epoch = 0
        if self.batch_size < 1 or self.chunk_batches < 1:
            raise DataError("batch_size and chunk_batches must be at least 1")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)

    def batches_for_epoch(self, epoch):
        """Shuffle rows, sort each chunk by length, cut into batches, shuffle batch order."""
        rng = np.

# Cell Title: Inspect: TokenizerBundle Attributes and Info (Read-Only)

In [4]:
from digikala_review_transformer import config
from digikala_review_transformer.training import tokenization

sp_b = tokenization.load_tokenizer(config.ROOT / "models/pretrained/mdeberta-v3-base", "sentencepiece", 160)
hf_b = tokenization.load_tokenizer(config.ROOT / "models/pretrained/TookaBERT-Base", "hf_fast", 128)
print("sp vars:", {k: v for k, v in vars(sp_b).items() if k != "sp"}, "info:", sp_b.info())
print("hf vars:", {k: v for k, v in vars(hf_b).items() if k != "tokenizer"}, "info:", hf_b.info())

sp vars: {'source_dir': PosixPath('/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base'), 'max_length': 160, 'name': 'mdeberta-v3-base', 'fingerprint': 'ceae13c4186e4ca6be9c90ba02c3bbe5aba12f7c224a313a68631703354bb1b5', 'first_id': 1, 'last_id': 2, 'pad_id': 0, 'unk_id': 3, 'processor': <sentencepiece.SentencePieceProcessor object at 0x76dbed3f43b0>} info: {'backend': 'sentencepiece', 'source_model': 'mdeberta-v3-base', 'max_length': 160, 'special_ids': {'first': 1, 'last': 2, 'pad': 0, 'unk': 3}, 'structure': '[CLS] + ids[:max_length-2] + [SEP]; attention mask is all ones', 'fingerprint': 'ceae13c4186e4ca6be9c90ba02c3bbe5aba12f7c224a313a68631703354bb1b5'}
hf vars: {'source_dir': PosixPath('/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base'), 'max_length': 128, 'name': 'TookaBERT-Base', 'fingerprint': 'd1a1da1f5d72c29284a049412da210f8a4c0640dcf8acb67574e8786676d

# Cell Title: Inspect: Failed Check in Smoke Report (Read-Only)

In [2]:
for c in report.checks:
    print(smoke.format_check_line(c))

[PASS] 1. Environment & CUDA: os=Linux 7.0.0-42-generic, python=3.12.16, torch=2.15.0+cu132, gpu='NVIDIA GeForce RTX 4060 Laptop GPU' (7.62 GB), bf16=True, free_disk=15.0 GB
[PASS] 2. Training pack integrity: fingerprint=9faa2a97e4502701, rows(train=2,860,236, val=922,794, test=922,805)
[PASS] 3. Model & tokenizer load: backend=hf_fast, model=TookaBERT-Base, params(total=122,319,366, trainable=122,319,366), ignored_unexpected=[]
[FAIL] 4. Temporary token cache build & re-read: ValueError: too many values to unpack (expected 2) -> check write permissions in system temp directory and tokenization.py
[PASS] 5. Variant B/C batch order equivalence: B and C batch fingerprints match (ep0=dbe9821e8f5f3922, ep1=5325511e61ab9ce3), max_token_id=46517 fits embedding table
[PASS] 6. Worst-case batch (forward/backward/step): shape=(16, 64), loss=2.6952, peak_vram=2356.8 MB (2.30 GB)
[PASS] 7. 2-epoch training & checkpoint auto-resume: resumed epoch 1 -> 2 (steps=20), train_objective=(2.2685 -> 2.152

# Cell Title: Inspect: Return Signatures of get_split_tokens and read_and_verify_probability_parquet (Read-Only)

In [3]:
import inspect

from digikala_review_transformer.training import predict, tokenization  # noqa: F811

print("get_split_tokens:\n", inspect.getsource(tokenization.get_split_tokens))
print("read_and_verify_probability_parquet:\n", inspect.getsource(predict.read_and_verify_probability_parquet))

get_split_tokens:
 def get_split_tokens(bundle, split, source_sha256, load_texts, expected_rows, cache_dir=None, show_progress=True):
    """Return (TokenArrays, stats, origin); origin is 'cache' or 'built'.

    load_texts is only called when the cache cannot be used. cache_dir=None means no disk cache.
    """
    key = cache_key(bundle, source_sha256)
    if cache_dir is not None:
        arrays, stats, reason = _read_cache(bundle, split, key, expected_rows, cache_dir)
        if arrays is not None:
            return arrays, stats, "cache"
        if reason != "no cache entry":
            print(f"[WARN] {split}: {reason} -> rebuilding")

    with utils.timer(f"tokenize_{bundle.name}_{split}"):
        arrays = encode_texts(bundle, load_texts(), show_progress=show_progress)
    if len(arrays) != expected_rows:
        raise TokenizationError(f"{split}: got {len(arrays):,} rows, expected {expected_rows:,} -> texts and manifest differ")
    validate_arrays(arrays, bundle)
    stats =

# Cell Title: Test: Smoke Runner and mDeBERTa Tokenizer Checks

In [1]:
import copy
import importlib
import tempfile
from pathlib import Path

import pyarrow.parquet as pq
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import (
    checkpoint,
    data,
    pack,
    predict,
    run_config,
    smoke,
    tokenization,
)

importlib.reload(smoke)

results: list[tuple[str, bool, str]] = []


def record(name: str, cond: bool, detail: str = "") -> None:
    results.append((name, bool(cond), detail))
    status = "[PASS]" if cond else "[FAIL]"
    suffix = f" ({detail})" if detail else ""
    print(f"{status} {name}{suffix}")


# ---- 1. Formatting and Failure Guard ----
pass_line = smoke.format_check_line(
    smoke.SmokeCheckResult(name="sample", passed=True, detail="ok")
)
fail_line = smoke.format_check_line(
    smoke.SmokeCheckResult(name="sample", passed=False, detail="bad", fix="do x")
)
record(
    "format_check_line follows [PASS]/[FAIL] convention",
    pass_line.startswith("[PASS]") and fail_line == "[FAIL] sample: bad -> do x",
    fail_line,
)

base_cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")

with tempfile.TemporaryDirectory(prefix="dk_smoke_guard_") as guard_dir:
    guard_report = smoke.run_smoke(
        base_cfg,
        smoke_directory=Path(guard_dir) / "smoke",
        train_rows=32,
        val_rows=16,
        test_rows=16,
        monitor_rows=16,
        num_workers=0,
        require_cuda=False,
        min_free_disk_gb=1e9,
    )
    record(
        "Disk space guard halts cleanly at Check 1 when threshold is unmet",
        (not guard_report.passed)
        and len(guard_report.checks) == 1
        and (not guard_report.checks[0].passed),
        guard_report.checks[0].detail,
    )

# ---- 2. mDeBERTa 3 Extra Tokenizer Checks ----
mdeb_path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
mdeb_bundle = tokenization.load_tokenizer(
    mdeb_path,
    backend="sentencepiece",
    max_length=160,
)
pack_dir = run_config.project_path(base_cfg["data"]["pack_dir"])
manifest = data.open_pack(pack_dir)
val_total_rows = int(manifest["files"][pack.pack_file_name("validation")]["rows"])
sample_idx = data.subset_row_indices(val_total_rows, 500, config.SPLIT_SEED, purpose="smoke")
val_table = pq.read_table(pack_dir / pack.pack_file_name("validation"), columns=["body"]).take(sample_idx)
sample_texts = [t if t is not None else "" for t in val_table.column("body").to_pylist()]

mdeb_checks = smoke.run_mdeberta_tokenizer_checks(mdeb_path, mdeb_bundle, sample_texts)
record(
    "mDeBERTa check 1 (special token IDs match HF)",
    len(mdeb_checks) == 3 and mdeb_checks[0].passed,
    mdeb_checks[0].detail if mdeb_checks else "missing",
)
record(
    "mDeBERTa check 2 (100% parity with HF fast on non-UNK rows)",
    len(mdeb_checks) == 3 and mdeb_checks[1].passed,
    mdeb_checks[1].detail if len(mdeb_checks) > 1 else "missing",
)
record(
    "mDeBERTa check 3 ([UNK] row count reported as warning)",
    len(mdeb_checks) == 3
    and mdeb_checks[2].passed
    and len(mdeb_checks[2].warnings) == 1
    and "sentencepiece=" in mdeb_checks[2].warnings[0]
    and "hf_fast=" in mdeb_checks[2].warnings[0],
    mdeb_checks[2].warnings[0] if (len(mdeb_checks) > 2 and mdeb_checks[2].warnings) else "missing",
)

# ---- 3. Full 9-Step End-to-End Smoke Run (TookaBERT in Temp Directory) ----
test_cfg = copy.deepcopy(base_cfg)
test_cfg["run_name"] = "smoke_unit_test"
test_cfg["model"]["max_length"] = 64
test_cfg["training"]["batch_size"] = 16
test_cfg["training"]["eval_batch_size"] = 32

with tempfile.TemporaryDirectory(prefix="dk_smoke_e2e_") as tmp_root:
    smoke_out_dir = Path(tmp_root) / "smoke"
    report = smoke.run_smoke(
        test_cfg,
        smoke_directory=smoke_out_dir,
        train_rows=160,
        val_rows=80,
        test_rows=80,
        monitor_rows=64,
        num_workers=0,
        require_cuda=torch.cuda.is_available(),
    )

    record(
        "End-to-end smoke report passed all 9 checks",
        report.passed and len(report.checks) == 9 and all(c.passed for c in report.checks),
        f"checks={len(report.checks)}, warnings={len(report.warnings)}",
    )
    record(
        "Worst-case batch recorded peak memory on CUDA",
        (not torch.cuda.is_available()) or (report.peak_memory_mb > 0.0),
        f"peak_memory_mb={report.peak_memory_mb:.1f}",
    )
    record(
        "Throughput and full-epoch wall-time estimate computed",
        report.train_samples_per_sec > 0.0
        and report.eval_samples_per_sec > 0.0
        and report.estimated_full_epoch_minutes > 0.0
        and report.estimated_full_run_minutes > 0.0,
        f"train_sps={report.train_samples_per_sec:.1f}, "
        f"est_epoch={report.estimated_full_epoch_minutes:.1f} min",
    )
    record(
        "Smoke directory finalized to status='complete' and last.pt cleaned up",
        checkpoint.run_status(smoke_out_dir) == "complete"
        and not (smoke_out_dir / predict.LAST_CHECKPOINT_NAME).exists()
        and (smoke_out_dir / predict.RUN_SUMMARY_NAME).exists()
        and (smoke_out_dir / predict.VAL_PROBS_NAME).exists()
        and (smoke_out_dir / predict.TEST_PROBS_NAME).exists()
        and (smoke_out_dir / predict.BEST_MODEL_NAME).exists(),
        f"status={checkpoint.run_status(smoke_out_dir)}",
    )

failed = [name for name, ok, _ in results if not ok]
print("-" * 60)
if failed:
    raise AssertionError(f"Failed checks ({len(failed)}): {failed}")
print(f"ALL PASSED ({len(results)}/{len(results)} checks)")

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] format_check_line follows [PASS]/[FAIL] convention ([FAIL] sample: bad -> do x)
STARTING SMOKE TEST: run_name=tooka_B, model=TookaBERT-Base, variant=B, batch_size=128, max_length=128
Smoke output directory: /tmp/dk_smoke_guard_ua7f0lhd/smoke
[FAIL] 1. Environment & disk: free disk space is 15.00 GB (< 1000000000.00 GB required) -> free up disk space on the drive hosting the repository
------------------------------------------------------------------------
SMOKE RESULT (tooka_B): 1 FAILED out of 1 checks
  - [FAIL] 1. Environment & disk: free disk space is 15.00 GB (< 1000000000.00 GB required) -> free up disk space on the drive hosting the repository
------------------------------------------------------------------------
[PASS] Disk space guard halts cleanly at Check 1 when threshold is unmet (free disk space is 15.00 GB (< 1000000000.00 GB required))


[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[PASS] mDeBERTa check 1 (special token IDs match HF) (SentencePiece and HF special IDs match {'first_id': 1, 'last_id': 2, 'pad_id': 0, 'unk_id': 3})
[PASS] mDeBERTa check 2 (100% parity with HF fast on non-UNK rows) (100% identical across all 485 non-[UNK] rows (out of 500 sampled rows))
[PASS] mDeBERTa check 3 ([UNK] row count reported as warning) (rows with [UNK] on 500 sampled texts: sentencepiece=0 (0.00%), hf_fast=15 (3.00%))
STARTING SMOKE TEST: run_name=smoke_unit_test, model=TookaBERT-Base, variant=B, batch_size=16, max_length=64
Smoke output directory: /tmp/dk_smoke_e2e_wdufi0u8/smoke
[PASS] 1. Environment & CUDA: os=Linux 7.0.0-42-generic, python=3.12.16, torch=2.15.0+cu132, gpu='NVIDIA GeForce RTX 4060 Laptop GPU' (7.62 GB), bf16=True, free_disk=15.0 GB
[PASS] 2. Training pack integrity: fingerprint=9faa2a97e4502701, rows(train=2,860,236, val=922,794, test=922,805)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12481.16it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[PASS] 3. Model & tokenizer load: backend=hf_fast, model=TookaBERT-Base, params(total=122,319,366, trainable=122,319,366), ignored_unexpected=[]
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] 4. Temporary token cache build & re-read: origin=(built -> cache), rows=64, tokens=928
[timer] tokenize_TookaBERT-Base_train: 0.0s
[timer] tokenize_TookaBERT-Base_train: 0.0s
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[timer] tokenize_TookaBERT-Base_test: 0.0s
[timer] tokenize_TookaBERT-Base_train: 0.0s
[PASS] 5. Variant B/C batch order equivalence: B and C batch fingerprints match (ep0=dbe9821e8f5f3922, ep1=5325511e61ab9ce3), max_token_id=46517 fits embedding table
[PASS] 6. Worst-case batch (forward/backward/step): shape=(16, 64), loss=2.6952, peak_vram=2356.8 MB (2.30 GB)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14695.13it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=smoke_unit_test device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[optimizer] fused=True steps_per_epoch=10 total_steps=20 warmup_steps=1
[train] run=smoke_unit_test epoch=1/2 step=5/10 global_step=5/20 progress=25.00% objective=2.345933 lr_encoder=1.579e-05 lr_heads=7.895e-05 samples_per_second=384.30 ETA=0.0 min
[train] run=smoke_unit_test epoch=1/2 step=10/10 global_step=10/20 progress=50.00% objective=2.190993 lr_encoder=1.053e-05 lr_heads=5.263e-05 samples_per_second=394.51 ETA=0.0 min
[timer] transformer_train_epoch_1: 0.4s
[timer] transformer_validation_epoch_1: 0.0s
[timer] transformer_train_subset_epoch_1: 0.0s
[epoch] epoch=1/2 train_objective=2.268463 val_mean_macro_f1=0.222325 best_epoch=1 is_best=True bad_epochs=0/2 epoch_seconds=0.5
[run] training_loop_status=paused stop_reason=paused_by_call_limit epochs_done=1 global_step=10 best_epoch=1 best_score=0.222325 train_s

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11196.48it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=smoke_unit_test device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[resume] Rebuilding best_model.pt and val_probs_best.parquet from the committed last.pt weights.
[resume] restored epochs_done=1 global_step=10 best_epoch=1 best_score=0.222325
[optimizer] fused=True steps_per_epoch=10 total_steps=20 warmup_steps=1
[train] run=smoke_unit_test epoch=2/2 step=5/10 global_step=15/20 progress=75.00% objective=2.175960 lr_encoder=5.263e-06 lr_heads=2.632e-05 samples_per_second=416.57 ETA=0.0 min
[train] run=smoke_unit_test epoch=2/2 step=10/10 global_step=20/20 progress=100.00% objective=2.128339 lr_encoder=0.000e+00 lr_heads=0.000e+00 samples_per_second=412.76 ETA=0.0 min
[timer] transformer_train_epoch_2: 0.4s
[timer] transformer_validation_epoch_2: 0.0s
[timer] transformer_train_subset_epoch_2: 0.0s
[epoch] epoch=2/2 train_objective=2.152149 val_mean_macro_f1=0.248496 best_epoch=2 is_

# Cell Title: Inspect: Failed Check Detail in Smoke Report (Read-Only)

In [2]:
for c in report.checks:
    if not c.passed:
        print(smoke.format_check_line(c))

[FAIL] 8. Predictions & run finalization: val_ids_match=False, test_ids_match=False, status=complete, last_removed=True -> inspect predict.finalize_run and sorted_eval_batches restoration


# Cell Title: Inspect: Split IDs Dtype and Verification Logic (Read-Only)

In [3]:
import inspect

from digikala_review_transformer.training import predict

print("verify_probability_frame source:\n", inspect.getsource(predict.verify_probability_frame))

verify_probability_frame source:
 def verify_probability_frame(
    frame: pd.DataFrame,
    split: data.SplitData,
    *,
    source_name: str = "probability frame",
) -> dict[str, np.ndarray]:
    """Validate id alignment, float32 dtypes, and row sums; return probabilities."""
    if not isinstance(frame, pd.DataFrame):
        raise PredictionError(f"{source_name} is not a pandas DataFrame.")

    expected_columns = expected_probability_columns()
    actual_columns = list(frame.columns)
    if actual_columns != expected_columns:
        raise PredictionError(
            f"{source_name} has columns {actual_columns}; "
            f"expected {expected_columns}."
        )

    expected_ids = np.asarray(split.ids)
    actual_ids = frame["id"].to_numpy()
    if len(actual_ids) != len(expected_ids):
        raise PredictionError(
            f"{source_name} has {len(actual_ids)} rows; "
            f"expected {len(expected_ids)} rows for split {split.split!r}."
        )
    if not np.a

# Cell Title: Inspect: data.load_split and run_config for CLI Script (Read-Only)

In [3]:
import inspect

from digikala_review_transformer.training import data, run_config, tokenization

print("data.py source:\n", inspect.getsource(data))
print("=" * 60)
print("run_config.py source:\n", inspect.getsource(run_config))
print("=" * 60)
print("default_cache_dir source:\n", inspect.getsource(tokenization.default_cache_dir))

data.py source:
 """Data layer: pack loading, token arrays + labels + weights, batching, loaders.

Batches are built inside Dataset.__getitem__ (it receives an array of row indices), so the
DataLoader runs with batch_size=None and a sampler that yields index arrays.
Light imports only: this module also runs on the training machine.
"""

# ---- Section: Imports ----
import hashlib
import math
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

from digikala_review_transformer import config, labels
from digikala_review_transformer.training import pack, tokenization
from digikala_review_transformer.training.run_config import VARIANTS

# ---- Section: Constants ----
HEADS = pack.HEADS
IGNORE_INDEX = -100
CHUNK_BATCHES = 50  # plan: chunks of batch_size * 50 rows are sorted by length
PREFETCH_FACTOR = 2
# Separate random streams so different subsets never coincide.
SUBSET_STREAMS = 

# Cell Title: Test: CLI Script train_transformer.py

In [4]:
import importlib.util
import tempfile
from pathlib import Path

import numpy as np
import torch
import yaml

from digikala_review_transformer import config
from digikala_review_transformer.training import (
    checkpoint,
    data,
    predict,
    run_config,
    tokenization,
)

script_path = config.ROOT / "scripts" / "train_transformer.py"
spec = importlib.util.spec_from_file_location("train_transformer_cli", script_path)
cli = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cli)

results: list[tuple[str, bool, str]] = []


def record(name: str, cond: bool, detail: str = "") -> None:
    results.append((name, bool(cond), detail))
    status = "[PASS]" if cond else "[FAIL]"
    suffix = f" ({detail})" if detail else ""
    print(f"{status} {name}{suffix}")


# ---- 1. Argument Parser & Config Override ----
parser = cli.build_arg_parser()
parsed = parser.parse_args(
    ["--config", "a.yaml", "b.yaml", "--smoke", "--batch-size", "32"]
)
record(
    "CLI parser accepts multiple --config paths, --smoke, and --batch-size",
    parsed.configs == ["a.yaml", "b.yaml"]
    and parsed.smoke is True
    and parsed.batch_size == 32,
    f"configs={parsed.configs}, smoke={parsed.smoke}, bs={parsed.batch_size}",
)

tooka_b_path = config.ROOT / "configs" / "transformer" / "tooka_B.yaml"
cfg_override = cli.load_and_prepare_config(tooka_b_path, batch_size_override=48)
record(
    "load_and_prepare_config applies --batch-size override",
    cfg_override["training"]["batch_size"] == 48,
    f"batch_size={cfg_override['training']['batch_size']}",
)

bad_bs_caught = False
try:
    cli.load_and_prepare_config(tooka_b_path, batch_size_override=0)
except run_config.ConfigError:
    bad_bs_caught = True
record("Invalid --batch-size <= 0 raises ConfigError", bad_bs_caught)

# ---- 2. Zero-Copy build_monitor_subset Verification ----
pack_dir = run_config.project_path(cfg_override["data"]["pack_dir"])
manifest = data.open_pack(pack_dir)
bundle = tokenization.load_tokenizer(
    run_config.project_path(cfg_override["model"]["pretrained_path"]),
    backend=cfg_override["model"]["tokenizer_backend"],
    max_length=64,
)
small_train = data.load_split(
    "train",
    bundle,
    pack_dir,
    manifest,
    variant="B",
    subset_rows=200,
    subset_seed=config.SPLIT_SEED,
    cache_dir=None,
    show_progress=False,
)
monitor_sub = cli.build_monitor_subset(small_train, subset_rows=50, seed=config.SPLIT_SEED)
expected_monitor_idx = data.subset_row_indices(200, 50, config.SPLIT_SEED, purpose="monitor")
sample_batch = monitor_sub.dataset()[np.array([0, 1, 2], dtype=np.int64)]

record(
    "build_monitor_subset creates exact zero-copy monitor split",
    monitor_sub.n_rows == 50
    and np.array_equal(monitor_sub.source_rows, expected_monitor_idx)
    and np.array_equal(monitor_sub.ids, small_train.ids[expected_monitor_idx])
    and monitor_sub.arrays.ids is small_train.arrays.ids
    and sample_batch["input_ids"].shape[0] == 3,
    f"n_rows={monitor_sub.n_rows}, batch_shape={tuple(sample_batch['input_ids'].shape)}",
)

# ---- 3. Skip Completed Runs & Stop Sequence on Error ----
base_yaml_path = config.ROOT / "configs" / "transformer" / "base.yaml"

with tempfile.TemporaryDirectory(prefix="dk_cli_seq_") as tmp_dir_str:
    tmp_dir = Path(tmp_dir_str)
    shutil_base = tmp_dir / "base.yaml"
    shutil_base.write_text(base_yaml_path.read_text(encoding="utf-8"), encoding="utf-8")

    # Config 1: already complete run -> should skip and return 0
    cfg1_path = tmp_dir / "run_complete.yaml"
    cfg1_data = {
        "extends": "base.yaml",
        "run_name": "cli_done_run",
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 64,
            "tokenizer_backend": "hf_fast",
        },
        "training": {"variant": "B"},
        "output": {"root": str(tmp_dir / "runs")},
    }
    cfg1_path.write_text(yaml.safe_dump(cfg1_data, sort_keys=False), encoding="utf-8")

    done_run_dir = tmp_dir / "runs" / "cli_done_run"
    done_run_dir.mkdir(parents=True, exist_ok=True)
    predict.atomic_write_json({"status": "complete"}, done_run_dir / predict.RUN_SUMMARY_NAME)
    (done_run_dir / predict.BEST_MODEL_NAME).write_bytes(b"placeholder")
    (done_run_dir / predict.VAL_PROBS_NAME).write_bytes(b"placeholder")
    (done_run_dir / predict.TEST_PROBS_NAME).write_bytes(b"placeholder")

    code_skip = cli.main(["--config", str(cfg1_path)])
    record(
        "Completed run is skipped cleanly with exit code 0",
        code_skip == 0 and checkpoint.run_status(done_run_dir) == "complete",
        f"exit_code={code_skip}",
    )

    # Config 2: resumable run with mismatched config hash -> must fail with code != 0
    # and MUST NOT execute Config 3 after it.
    cfg2_path = tmp_dir / "run_bad_resume.yaml"
    cfg2_data = {
        "extends": "base.yaml",
        "run_name": "cli_bad_resume",
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 64,
            "tokenizer_backend": "hf_fast",
        },
        "training": {"variant": "B"},
        "output": {"root": str(tmp_dir / "runs")},
    }
    cfg2_path.write_text(yaml.safe_dump(cfg2_data, sort_keys=False), encoding="utf-8")
    bad_resume_dir = tmp_dir / "runs" / "cli_bad_resume"
    bad_resume_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "format_version": checkpoint.CHECKPOINT_FORMAT_VERSION,
            "config_hash": "wrong_hash_0000",
            "state": {},
        },
        bad_resume_dir / predict.LAST_CHECKPOINT_NAME,
    )

    cfg3_path = tmp_dir / "run_after_fail.yaml"
    cfg3_data = dict(cfg2_data, run_name="cli_should_not_run")
    cfg3_path.write_text(yaml.safe_dump(cfg3_data, sort_keys=False), encoding="utf-8")

    code_fail_seq = cli.main(["--config", str(cfg2_path), str(cfg3_path)])
    record(
        "Sequential runner halts with non-zero exit code on hash mismatch and does not start next config",
        code_fail_seq != 0 and not (tmp_dir / "runs" / "cli_should_not_run").exists(),
        f"exit_code={code_fail_seq}",
    )

    # ---- 4. CLI --smoke Execution with --batch-size Override ----
    smoke_out = tmp_dir / "cli_smoke_out"
    code_smoke = cli.main(
        ["--config", str(cfg1_path), "--smoke", "--batch-size", "16"],
        smoke_directory=smoke_out,
        require_cuda_in_smoke=torch.cuda.is_available(),
    )
    record(
        "CLI --smoke mode executes end-to-end and returns exit code 0",
        code_smoke == 0 and checkpoint.run_status(smoke_out) == "complete",
        f"exit_code={code_smoke}, status={checkpoint.run_status(smoke_out)}",
    )

failed = [name for name, ok, _ in results if not ok]
print("-" * 60)
if failed:
    raise AssertionError(f"Failed checks ({len(failed)}): {failed}")
print(f"ALL PASSED ({len(results)}/{len(results)} checks)")

[PASS] CLI parser accepts multiple --config paths, --smoke, and --batch-size (configs=['a.yaml', 'b.yaml'], smoke=True, bs=32)
[PASS] load_and_prepare_config applies --batch-size override (batch_size=48)
[PASS] Invalid --batch-size <= 0 raises ConfigError
[timer] tokenize_TookaBERT-Base_train: 0.0s
[PASS] build_monitor_subset creates exact zero-copy monitor split (n_rows=50, batch_shape=(3, 19))
RUN [1/1]: config=/tmp/dk_cli_seq_0e0ljo9o/run_complete.yaml (smoke=False, batch_size_override=None)
[PASS] cli_done_run: run is already complete at /tmp/dk_cli_seq_0e0ljo9o/runs/cli_done_run -> skipping
[PASS] All 1 config(s) finished successfully.
[PASS] Completed run is skipped cleanly with exit code 0 (exit_code=0)
RUN [1/2]: config=/tmp/dk_cli_seq_0e0ljo9o/run_bad_resume.yaml (smoke=False, batch_size_override=None)
[FAIL] run_bad_resume.yaml: CheckpointError: The last checkpoint is missing keys: ['model_state_dict', 'optimizer_state_dict', 'rng_state', 'scheduler_state_dict', 'train_state'

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 18876.43it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[PASS] 3. Model & tokenizer load: backend=hf_fast, model=TookaBERT-Base, params(total=122,319,366, trainable=122,319,366), ignored_unexpected=[]
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] 4. Temporary token cache build & re-read: origin=(built -> cache), rows=64, tokens=928
[timer] tokenize_TookaBERT-Base_train: 0.0s
[timer] tokenize_TookaBERT-Base_train: 0.0s
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[timer] tokenize_TookaBERT-Base_test: 0.0s
[timer] tokenize_TookaBERT-Base_train: 0.0s
[PASS] 5. Variant B/C batch order equivalence: B and C batch fingerprints match (ep0=0879763f767a4629, ep1=e94867aeb86bc3ee), max_token_id=47308 fits embedding table
[PASS] 6. Worst-case batch (forward/backward/step): shape=(16, 64), loss=2.6989, peak_vram=2356.8 MB (2.30 GB)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 21957.37it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=cli_done_run device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[optimizer] fused=True steps_per_epoch=125 total_steps=250 warmup_steps=15
[train] run=cli_done_run epoch=1/2 step=5/125 global_step=5/250 progress=2.00% objective=2.346177 lr_encoder=6.667e-06 lr_heads=3.333e-05 samples_per_second=13.44 ETA=4.9 min
[train] run=cli_done_run epoch=1/2 step=10/125 global_step=10/250 progress=4.00% objective=2.259209 lr_encoder=1.333e-05 lr_heads=6.667e-05 samples_per_second=26.08 ETA=2.5 min
[train] run=cli_done_run epoch=1/2 step=15/125 global_step=15/250 progress=6.00% objective=2.257822 lr_encoder=2.000e-05 lr_heads=1.000e-04 samples_per_second=37.88 ETA=1.7 min
[train] run=cli_done_run epoch=1/2 step=20/125 global_step=20/250 progress=8.00% objective=2.397271 lr_encoder=1.957e-05 lr_heads=9.787e-05 samples_per_second=49.17 ETA=1.2 min
[train] run=cli_done_run epoch=1/2 step=25/125 gl

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 23729.30it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=cli_done_run device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[resume] Rebuilding best_model.pt and val_probs_best.parquet from the committed last.pt weights.
[resume] restored epochs_done=1 global_step=125 best_epoch=1 best_score=0.569977
[optimizer] fused=True steps_per_epoch=125 total_steps=250 warmup_steps=15
[train] run=cli_done_run epoch=2/2 step=5/125 global_step=130/250 progress=52.00% objective=1.713451 lr_encoder=1.021e-05 lr_heads=5.106e-05 samples_per_second=126.69 ETA=0.3 min
[train] run=cli_done_run epoch=2/2 step=10/125 global_step=135/250 progress=54.00% objective=1.837259 lr_encoder=9.787e-06 lr_heads=4.894e-05 samples_per_second=130.17 ETA=0.2 min
[train] run=cli_done_run epoch=2/2 step=15/125 global_step=140/250 progress=56.00% objective=1.784875 lr_encoder=9.362e-06 lr_heads=4.681e-05 samples_per_second=133.60 ETA=0.2 min
[train] run=cli_done_run epoch=2/2 ste

# Cell Title: Test: All 6 YAML Configs, package_run_outputs.py, and download_pretrained_models.py

In [1]:
import ast
import copy
import importlib.util
import tempfile
import zipfile
from pathlib import Path

from digikala_review_transformer import config
from digikala_review_transformer.training import predict, run_config

results: list[tuple[str, bool, str]] = []


def record(name: str, cond: bool, detail: str = "") -> None:
    results.append((name, bool(cond), detail))
    status = "[PASS]" if cond else "[FAIL]"
    suffix = f" ({detail})" if detail else ""
    print(f"{status} {name}{suffix}")


# ---- 1. Validate All 6 YAML Run Configs ----
expected_specs = {
    "tooka_B": ("models/pretrained/TookaBERT-Base", 128, "hf_fast", "B"),
    "tooka_C": ("models/pretrained/TookaBERT-Base", 128, "hf_fast", "C"),
    "roberta_B": ("models/pretrained/roberta-fa-zwnj-base", 128, "hf_fast", "B"),
    "roberta_C": ("models/pretrained/roberta-fa-zwnj-base", 128, "hf_fast", "C"),
    "mdeberta_B": ("models/pretrained/mdeberta-v3-base", 160, "sentencepiece", "B"),
    "mdeberta_C": ("models/pretrained/mdeberta-v3-base", 160, "sentencepiece", "C"),
}

cfg_dir = config.ROOT / "configs" / "transformer"
loaded_cfgs: dict[str, dict] = {}
hashes: set[str] = set()

for run_name, (model_rel_path, max_len, backend, variant) in expected_specs.items():
    yaml_path = cfg_dir / f"{run_name}.yaml"
    cfg = run_config.load_run_config(yaml_path)
    loaded_cfgs[run_name] = cfg
    cfg_h = run_config.config_hash(cfg)
    hashes.add(cfg_h)

    model_disk_path = run_config.project_path(cfg["model"]["pretrained_path"])
    spec_ok = (
        cfg["run_name"] == run_name
        and cfg["seed"] == config.SPLIT_SEED
        and cfg["model"]["pretrained_path"] == model_rel_path
        and model_disk_path.is_dir()
        and cfg["model"]["max_length"] == max_len
        and cfg["model"]["tokenizer_backend"] == backend
        and cfg["training"]["variant"] == variant
    )
    record(
        f"Config '{run_name}.yaml' matches plan schema and local model folder",
        spec_ok,
        f"path={model_rel_path}, max_len={max_len}, backend={backend}, variant={variant}, hash={cfg_h[:10]}",
    )

record(
    "All 6 run configs produce pairwise distinct config hashes",
    len(hashes) == 6,
    f"unique_hashes={len(hashes)}",
)

for prefix in ("tooka", "roberta", "mdeberta"):
    cfg_b = copy.deepcopy(loaded_cfgs[f"{prefix}_B"])
    cfg_c = copy.deepcopy(loaded_cfgs[f"{prefix}_C"])
    cfg_b.pop("run_name")
    cfg_c.pop("run_name")
    cfg_b["training"].pop("variant")
    cfg_c["training"].pop("variant")
    record(
        f"'{prefix}_B' and '{prefix}_C' are identical except for run_name and training.variant",
        cfg_b == cfg_c,
    )

# ---- 2. Test scripts/package_run_outputs.py ----
pkg_script_path = config.ROOT / "scripts" / "package_run_outputs.py"
pkg_spec = importlib.util.spec_from_file_location("package_run_outputs_cli", pkg_script_path)
pkg_cli = importlib.util.module_from_spec(pkg_spec)
pkg_spec.loader.exec_module(pkg_cli)


def make_synthetic_completed_run(runs_root: Path, name: str) -> Path:
    rdir = runs_root / name
    rdir.mkdir(parents=True, exist_ok=True)
    for req_name in pkg_cli.REQUIRED_FILES:
        if req_name == predict.RUN_SUMMARY_NAME:
            predict.atomic_write_json({"run_name": name, "status": "complete"}, rdir / req_name)
        else:
            (rdir / req_name).write_text(f"sample content for {req_name}\n", encoding="utf-8")

    tok_dir = rdir / predict.TOKENIZER_DIR_NAME
    tok_dir.mkdir(parents=True, exist_ok=True)
    (tok_dir / "tokenizer_info.json").write_text('{"backend": "hf_fast"}\n', encoding="utf-8")
    (tok_dir / "tokenizer.json").write_text("{}\n", encoding="utf-8")

    # Heavy / excluded files that must exist on disk for run_status=='complete' or cleanup testing,
    # but MUST NOT be included inside the generated zip archive.
    (rdir / predict.BEST_MODEL_NAME).write_bytes(b"heavy_weights_bytes" * 100)
    (rdir / "extra_weights.ckpt").write_bytes(b"checkpoint_bytes")
    pycache_dir = rdir / "__pycache__"
    pycache_dir.mkdir(exist_ok=True)
    (pycache_dir / "temp.pyc").write_bytes(b"bytecode")
    return rdir


with tempfile.TemporaryDirectory(prefix="dk_pkg_test_") as tmp_str:
    tmp_root = Path(tmp_str)
    runs_root = tmp_root / "transformer_runs"
    out_dir = tmp_root / "outputs_to_send"

    make_synthetic_completed_run(runs_root, "tooka_B")
    make_synthetic_completed_run(runs_root, "tooka_C")

    exit_code = pkg_cli.main(
        [
            "--runs",
            "tooka_B",
            "tooka_C",
            "--runs-root",
            str(runs_root),
            "--output-dir",
            str(out_dir),
        ]
    )
    zip_b = out_dir / "tooka_B.zip"
    zip_c = out_dir / "tooka_C.zip"
    record(
        "package_run_outputs CLI packages multiple runs with exit code 0",
        exit_code == 0 and zip_b.is_file() and zip_c.is_file(),
        f"exit_code={exit_code}",
    )

    with zipfile.ZipFile(zip_b, "r") as zf:
        archived_names = set(zf.namelist())

    expected_in_zip = {f"tooka_B/{fname}" for fname in pkg_cli.REQUIRED_FILES} | {
        "tooka_B/tokenizer/tokenizer_info.json",
        "tooka_B/tokenizer/tokenizer.json",
    }
    forbidden_in_zip = {
        f"tooka_B/{predict.BEST_MODEL_NAME}",
        f"tooka_B/{predict.LAST_CHECKPOINT_NAME}",
        "tooka_B/extra_weights.ckpt",
        "tooka_B/__pycache__/temp.pyc",
    }
    has_any_pt = any(item.endswith((".pt", ".pth", ".ckpt")) for item in archived_names)

    record(
        "Archive contains all required light artifacts and tokenizer/ files under <run_name>/",
        expected_in_zip.issubset(archived_names),
        f"archived_count={len(archived_names)}",
    )
    record(
        "Archive excludes best_model.pt, last.pt, .ckpt files, and __pycache__",
        archived_names.isdisjoint(forbidden_in_zip) and (not has_any_pt),
        f"archived={sorted(archived_names)}",
    )

    # Guard 1: Incomplete run (has last.pt -> status == 'resumable') must fail
    incomplete_dir = make_synthetic_completed_run(runs_root, "incomplete_run")
    (incomplete_dir / predict.LAST_CHECKPOINT_NAME).write_bytes(b"resumable_state")
    code_incomplete = pkg_cli.main(
        [
            "--runs",
            "incomplete_run",
            "--runs-root",
            str(runs_root),
            "--output-dir",
            str(out_dir),
        ]
    )
    record(
        "package_run_outputs refuses to package an incomplete/resumable run",
        code_incomplete != 0 and not (out_dir / "incomplete_run.zip").exists(),
        f"exit_code={code_incomplete}",
    )

    # Guard 2: Completed status but missing encoder_config.json must fail
    missing_file_dir = make_synthetic_completed_run(runs_root, "missing_artifact_run")
    (missing_file_dir / predict.ENCODER_CONFIG_NAME).unlink()
    code_missing = pkg_cli.main(
        [
            "--runs",
            "missing_artifact_run",
            "--runs-root",
            str(runs_root),
            "--output-dir",
            str(out_dir),
        ]
    )
    record(
        "package_run_outputs refuses to package a run missing encoder_config.json",
        code_missing != 0 and not (out_dir / "missing_artifact_run.zip").exists(),
        f"exit_code={code_missing}",
    )

# ---- 3. Verify scripts/download_pretrained_models.py AST Structure ----
dl_script_path = config.ROOT / "scripts" / "download_pretrained_models.py"
dl_source = dl_script_path.read_text(encoding="utf-8")
dl_tree = ast.parse(dl_source, filename=str(dl_script_path))

models_dict_literal = None
for node in dl_tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id == "MODELS":
                models_dict_literal = ast.literal_eval(node.value)

expected_models_map = {
    "TookaBERT-Base": "PartAI/TookaBERT-Base",
    "roberta-fa-zwnj-base": "HooshvareLab/roberta-fa-zwnj-base",
    "mdeberta-v3-base": "microsoft/mdeberta-v3-base",
}
record(
    "download_pretrained_models.py parses cleanly and defines exact 3 pretrained repos",
    models_dict_literal == expected_models_map,
    f"MODELS={models_dict_literal}",
)

failed = [name for name, ok, _ in results if not ok]
print("-" * 60)
if failed:
    raise AssertionError(f"Failed checks ({len(failed)}): {failed}")
print(f"ALL PASSED ({len(results)}/{len(results)} checks)")

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] Config 'tooka_B.yaml' matches plan schema and local model folder (path=models/pretrained/TookaBERT-Base, max_len=128, backend=hf_fast, variant=B, hash=c5a1914e63)
[PASS] Config 'tooka_C.yaml' matches plan schema and local model folder (path=models/pretrained/TookaBERT-Base, max_len=128, backend=hf_fast, variant=C, hash=f04a343633)
[PASS] Config 'roberta_B.yaml' matches plan schema and local model folder (path=models/pretrained/roberta-fa-zwnj-base, max_len=128, backend=hf_fast, variant=B, hash=1cbd6fbb94)
[PASS] Config 'roberta_C.yaml' matches plan schema and local model folder (path=models/pretrained/roberta-fa-zwnj-base, max_len=128, backend=hf_fast, variant=C, hash=701f31bb68)
[PASS] Config 'mdeberta_B.yaml' matches plan schema and local model folder (path=models/pretrained/mdeberta-v3-base, max_len=160, backend=sentencepiece, variant=B, hash=1ce46e0d17)
[PASS] Config 'mdeberta_C.yaml' matches plan schema and local model folder (path=models/pretrained/mdeberta-v3-base, max_le

# Cell Title: Inspect: Git Status, .gitignore, pyproject.toml, and uv Lock Check (Read-Only)

In [2]:
import subprocess

from digikala_review_transformer import config


def run_cmd(cmd: list[str]) -> str:
    proc = subprocess.run(cmd, check=False, cwd=config.ROOT, capture_output=True, text=True)
    return f"$ {' '.join(cmd)} (exit={proc.returncode})\n{proc.stdout}{proc.stderr}".strip()

print(run_cmd(["git", "status", "--short", "--untracked-files=all"]))
print("=" * 72)
print(run_cmd(["uv", "lock", "--check"]))
print("=" * 72)
gitignore_path = config.ROOT / ".gitignore"
print("--- .gitignore ---")
print(gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else "<missing>")
print("=" * 72)
print("--- pyproject.toml ---")
print((config.ROOT / "pyproject.toml").read_text(encoding="utf-8"))

$ git status --short --untracked-files=all (exit=0)
 M notebooks/10_1st_transformer_finetune.ipynb
 M src/digikala_review_transformer/training/smoke.py
?? configs/transformer/mdeberta_B.yaml
?? configs/transformer/mdeberta_C.yaml
?? configs/transformer/roberta_B.yaml
?? configs/transformer/roberta_C.yaml
?? configs/transformer/tooka_C.yaml
?? scripts/download_pretrained_models.py
?? scripts/package_run_outputs.py
?? scripts/train_transformer.py
$ uv lock --check (exit=0)
Resolved 123 packages in 2ms
--- .gitignore ---
# Python-generated files
__pycache__/
*.py[oc]
build/
dist/
wheels/
*.egg-info

# Virtual environments
.venv

# data and models (large; not for Git)
.cache/
*.rar
*.parquet
*.pt
*.pth
*.safetensors
*.bin
*.joblib

# models: weights and tokenizers are downloaded by script, never committed
models/*
!models/.gitkeep

# zip files produced by package_run_outputs.py
outputs_to_send/

# notebooks and misc
.ipynb_checkpoints/
.env
notebooks/00_data_ingest_quality_gate_copy_for_re

# Cell Title: Step U5: Full CLI Smoke Test Across All Three Encoders

In [ ]:
import subprocess
import sys

from digikala_review_transformer import config

cmd = [
    sys.executable,
    str(config.ROOT / "scripts" / "train_transformer.py"),
    "--config",
    "configs/transformer/tooka_B.yaml",
    "configs/transformer/roberta_B.yaml",
    "configs/transformer/mdeberta_B.yaml",
    "--smoke",
    "--batch-size",
    "32",
]

print("Running:", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, check=False, cwd=config.ROOT, text=True, capture_output=True)
print(proc.stdout)
if proc.stderr:
    print("--- STDERR ---")
    print(proc.stderr)

if proc.returncode != 0:
    raise RuntimeError(f"[FAIL] CLI smoke test exited with code {proc.returncode}")
print("[PASS] Step U5: CLI smoke test passed across TookaBERT, RoBERTa, and mDeBERTa.")

Running: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/bin/python /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/scripts/train_transformer.py --config configs/transformer/tooka_B.yaml configs/transformer/roberta_B.yaml configs/transformer/mdeberta_B.yaml --smoke --batch-size 32
RUN [1/3]: config=configs/transformer/tooka_B.yaml (smoke=True, batch_size_override=32)
STARTING SMOKE TEST: run_name=tooka_B, model=TookaBERT-Base, variant=B, batch_size=32, max_length=128
Smoke output directory: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/artifacts/transformer_runs/smoke
[PASS] 1. Environment & CUDA: os=Linux 7.0.0-42-generic, python=3.12.16, torch=2.15.0+cu132, gpu='NVIDIA GeForce RTX 4060 Laptop GPU' (7.62 GB), bf16=True, free_disk=13.5 GB
[PASS] 2. Training pack integrity: fingerprint=9faa2a97e4502701, rows(train=2,860,236, val=922,794, test=922,805)
[PASS] 3. Model & tokenizer load:

RuntimeError: [FAIL] CLI smoke test exited with code 1

# Cell Title: Inspect: Tail of CLI Smoke Output (Read-Only)

In [4]:
for line in proc.stdout.splitlines():
    if any(tag in line for tag in ("RUN [", "STARTING SMOKE", "[PASS]", "[FAIL]", "[WARN]", "SMOKE RESULT")):
        print(line)
print("--- LAST 30 LINES OF STDOUT ---")
print("\n".join(proc.stdout.splitlines()[-30:]))
if proc.stderr:
    print("--- LAST 30 LINES OF STDERR ---")
    print("\n".join(proc.stderr.splitlines()[-30:]))

RUN [1/3]: config=configs/transformer/tooka_B.yaml (smoke=True, batch_size_override=32)
STARTING SMOKE TEST: run_name=tooka_B, model=TookaBERT-Base, variant=B, batch_size=32, max_length=128
[PASS] 1. Environment & CUDA: os=Linux 7.0.0-42-generic, python=3.12.16, torch=2.15.0+cu132, gpu='NVIDIA GeForce RTX 4060 Laptop GPU' (7.62 GB), bf16=True, free_disk=13.5 GB
[PASS] 2. Training pack integrity: fingerprint=9faa2a97e4502701, rows(train=2,860,236, val=922,794, test=922,805)
[PASS] 3. Model & tokenizer load: backend=hf_fast, model=TookaBERT-Base, params(total=122,319,366, trainable=122,319,366), ignored_unexpected=[]
[PASS] 4. Temporary token cache build & re-read: origin=(built -> cache), rows=64, tokens=928
[PASS] 5. Variant B/C batch order equivalence: B and C batch fingerprints match (ep0=b67e62e62448e664, ep1=18ef6322592a22a3), max_token_id=47308 fits embedding table
[PASS] 6. Worst-case batch (forward/backward/step): shape=(32, 128), loss=2.7747, peak_vram=2367.6 MB (2.31 GB)
[PASS

# Cell Title: Step U5: Full CLI Smoke Test Across All Three Encoders

In [5]:
import gc  # noqa: I001
import subprocess
import sys
import torch

from digikala_review_transformer import config

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

cmd = [
    sys.executable,
    str(config.ROOT / "scripts" / "train_transformer.py"),
    "--config",
    "configs/transformer/tooka_B.yaml",
    "configs/transformer/roberta_B.yaml",
    "configs/transformer/mdeberta_B.yaml",
    "--smoke",
    "--batch-size",
    "16",
]

print("Running:", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, check=False, cwd=config.ROOT, text=True, capture_output=True)

for line in proc.stdout.splitlines():
    if any(tag in line for tag in ("RUN [", "STARTING SMOKE", "[PASS]", "[FAIL]", "[WARN]", "SMOKE RESULT")):
        print(line)

if proc.returncode != 0:
    print("--- FULL STDOUT TAIL ---")
    print("\n".join(proc.stdout.splitlines()[-40:]))
    print("--- STDERR TAIL ---")
    print("\n".join(proc.stderr.splitlines()[-40:]))
    raise RuntimeError(f"[FAIL] CLI smoke test exited with code {proc.returncode}")

print("[PASS] Step U5: CLI smoke test passed across TookaBERT, RoBERTa, and mDeBERTa.")

Running: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/bin/python /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/scripts/train_transformer.py --config configs/transformer/tooka_B.yaml configs/transformer/roberta_B.yaml configs/transformer/mdeberta_B.yaml --smoke --batch-size 16
RUN [1/3]: config=configs/transformer/tooka_B.yaml (smoke=True, batch_size_override=16)
STARTING SMOKE TEST: run_name=tooka_B, model=TookaBERT-Base, variant=B, batch_size=16, max_length=128
[PASS] 1. Environment & CUDA: os=Linux 7.0.0-42-generic, python=3.12.16, torch=2.15.0+cu132, gpu='NVIDIA GeForce RTX 4060 Laptop GPU' (7.62 GB), bf16=True, free_disk=13.5 GB
[PASS] 2. Training pack integrity: fingerprint=9faa2a97e4502701, rows(train=2,860,236, val=922,794, test=922,805)
[PASS] 3. Model & tokenizer load: backend=hf_fast, model=TookaBERT-Base, params(total=122,319,366, trainable=122,319,366), ignored_unexpected=[]
[PASS] 4. Temporary token

# Cell Title: Step U4: Package data/training_pack into training_pack.zip and Clean Smoke Folder

In [6]:
import shutil
import zipfile
from pathlib import Path

from digikala_review_transformer import config
from digikala_review_transformer.training import pack

pack_dir = config.ROOT / "data" / "training_pack"
manifest, results = pack.verify_pack(pack_dir, check_content=False)
failed = [msg for ok, msg in results if not ok]
if failed:
    raise RuntimeError(f"[FAIL] Training pack check failed: {failed}")

expected_pack_files = [
    pack.MANIFEST_NAME,
    pack.pack_file_name("train"),
    pack.pack_file_name("validation"),
    pack.pack_file_name("test"),
]

zip_path = config.ROOT / "outputs_to_send" / "training_pack.zip"
zip_path.parent.mkdir(parents=True, exist_ok=True)
if zip_path.exists():
    zip_path.unlink()

# Store with ZIP_STORED because parquet files are already compressed; this is much faster
with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_STORED) as zf:
    for fname in expected_pack_files:
        fpath = pack_dir / fname
        arcname = (Path("data") / "training_pack" / fname).as_posix()
        zf.write(fpath, arcname=arcname)

with zipfile.ZipFile(zip_path, mode="r") as zf:
    names = zf.namelist()

size_mb = zip_path.stat().st_size / (1024**2)
print(f"[PASS] Created {zip_path} ({size_mb:.1f} MB) with entries: {names}")

# Clean up temporary smoke run directory created during Step U5
smoke_dir = config.ARTIFACTS / "transformer_runs" / "smoke"
if smoke_dir.exists():
    shutil.rmtree(smoke_dir)
    print(f"[PASS] Removed temporary smoke directory: {smoke_dir}")

[PASS] Created /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/outputs_to_send/training_pack.zip (170.0 MB) with entries: ['data/training_pack/manifest.json', 'data/training_pack/train.parquet', 'data/training_pack/validation.parquet', 'data/training_pack/test.parquet']
[PASS] Removed temporary smoke directory: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/artifacts/transformer_runs/smoke


---

# Cell Title: Step D1: Dev Fine-Tune of tooka_B and tooka_C on a Small Subset

In [7]:
import gc
import importlib.util
import time

import numpy as np
import pandas as pd
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import (
    checkpoint,
    data,
    loop,
    predict,
    run_config,
    tokenization,
)
from digikala_review_transformer.training import model as model_lib

# ---- Section: Dev settings (edit only here) ----
TRAIN_ROWS = 16_000
VAL_ROWS = 10_000
TEST_ROWS = 10_000
MONITOR_ROWS = 5_000
MAX_EPOCHS = 3
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 128
LOG_EVERY_STEPS = 100
DEV_RUNS_ROOT = "artifacts/transformer_runs_dev"
RUN_NAMES = ["tooka_B", "tooka_C"]

# ---- Section: Reuse the tested monitor-subset helper from the CLI script ----
cli_spec = importlib.util.spec_from_file_location(
    "train_transformer_cli", config.ROOT / "scripts" / "train_transformer.py"
)
cli = importlib.util.module_from_spec(cli_spec)
cli_spec.loader.exec_module(cli)

reference_ids: dict[str, np.ndarray] = {}


def free_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_dev(run_name: str) -> None:
    """Fine-tune one config on the small subset, using the same code path as the CLI."""
    cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / f"{run_name}.yaml")
    cfg["output"]["root"] = DEV_RUNS_ROOT
    training = cfg["training"]
    training["max_epochs"] = MAX_EPOCHS
    training["batch_size"] = BATCH_SIZE
    training["eval_batch_size"] = EVAL_BATCH_SIZE
    training["log_every_steps"] = LOG_EVERY_STEPS
    training["num_workers"] = 0
    training["train_subset_eval_rows"] = MONITOR_ROWS

    target_dir = run_config.run_dir(cfg)
    status = checkpoint.run_status(target_dir)
    if status == "complete":
        print(f"[PASS] {run_name}: already complete at {target_dir} -> skipping", flush=True)
        return
    print(f"[{run_name}] status={status}, output={target_dir}", flush=True)

    device = torch.device("cuda", torch.cuda.current_device()) if torch.cuda.is_available() else torch.device("cpu")
    train_split = val_split = test_split = monitor_split = model_instance = None
    started = time.perf_counter()
    try:
        pack_dir = run_config.project_path(cfg["data"]["pack_dir"])
        manifest = data.open_pack(pack_dir)
        bundle = tokenization.load_tokenizer(
            run_config.project_path(cfg["model"]["pretrained_path"]),
            backend=str(cfg["model"]["tokenizer_backend"]),
            max_length=int(cfg["model"]["max_length"]),
        )
        variant = str(training["variant"])

        def load(split_name: str, rows: int) -> data.SplitData:
            return data.load_split(
                split_name, bundle, pack_dir, manifest, variant=variant,
                subset_rows=rows, subset_seed=config.SPLIT_SEED,
                cache_dir=None, show_progress=False,
            )

        train_split = load("train", TRAIN_ROWS)
        val_split = load("validation", VAL_ROWS)
        test_split = load("test", TEST_ROWS)
        monitor_split = cli.build_monitor_subset(train_split, MONITOR_ROWS, config.SPLIT_SEED)

        # B and C must see exactly the same rows.
        for name, split in (("train", train_split), ("validation", val_split), ("test", test_split)):
            if name not in reference_ids:
                reference_ids[name] = split.ids.copy()
            elif not np.array_equal(reference_ids[name], split.ids):
                raise RuntimeError(f"[FAIL] {run_name}: '{name}' subset ids differ from the first run -> check subset seed")
        print(
            f"[PASS] {run_name}: subsets ready (train={train_split.n_rows:,}, val={val_split.n_rows:,}, "
            f"test={test_split.n_rows:,}, monitor={monitor_split.n_rows:,}, variant={variant})",
            flush=True,
        )

        model_instance, load_info = model_lib.build_model(cfg)
        max_token_id = max(int(s.stats["max_token_id"]) for s in (train_split, val_split, test_split))
        model_lib.check_token_ids_fit(model_instance, max_token_id=max_token_id, pad_id=bundle.pad_id)

        loop_result = loop.fit(
            cfg, model_instance, train_split, val_split, monitor_split,
            run_directory=target_dir, device=device,
        )
        finalized = predict.finalize_run(
            cfg, model_instance, bundle, loop_result,
            train_split=train_split, validation_split=val_split, test_split=test_split,
            train_subset_split=monitor_split, manifest=manifest,
            load_info=load_info, device=device,
        )
        print(
            f"[PASS] {run_name}: done in {(time.perf_counter() - started) / 60.0:.1f} min | "
            f"best_epoch={loop_result.best_epoch} | best_val_mean_macro_f1={loop_result.best_score:.4f} | "
            f"summary={finalized.summary_path}",
            flush=True,
        )
    finally:
        del model_instance, train_split, val_split, test_split, monitor_split
        free_memory()


free_memory()
overall_start = time.perf_counter()
for name in RUN_NAMES:
    run_dev(name)
print(f"[PASS] All dev runs finished in {(time.perf_counter() - overall_start) / 60.0:.1f} min", flush=True)

# ---- Section: Quick summary from history.csv (read-only) ----
show_columns = ["epoch", "val_mean_macro_f1", "val_macro_f1_rec", "val_macro_f1_band", "epoch_seconds", "is_best"]
for name in RUN_NAMES:
    run_directory = config.ROOT / DEV_RUNS_ROOT / name
    history = pd.read_csv(run_directory / predict.HISTORY_NAME)
    print(f"--- {name} (status={checkpoint.run_status(run_directory)}) ---")
    print(history[show_columns].to_string(index=False))

[tooka_B] status=fresh, output=/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/artifacts/transformer_runs_dev/tooka_B
[timer] tokenize_TookaBERT-Base_train: 0.1s
[timer] tokenize_TookaBERT-Base_validation: 0.1s
[timer] tokenize_TookaBERT-Base_test: 0.1s
[PASS] tooka_B: subsets ready (train=16,000, val=10,000, test=10,000, monitor=5,000, variant=B)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16286.79it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=tooka_B device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[optimizer] fused=True steps_per_epoch=500 total_steps=1500 warmup_steps=90
[train] run=tooka_B epoch=1/3 step=100/500 global_step=100/1500 progress=6.67% objective=2.158670 lr_encoder=1.986e-05 lr_heads=9.929e-05 samples_per_second=683.87 ETA=1.1 min
[train] run=tooka_B epoch=1/3 step=200/500 global_step=200/1500 progress=13.33% objective=1.898517 lr_encoder=1.844e-05 lr_heads=9.220e-05 samples_per_second=716.51 ETA=1.0 min
[train] run=tooka_B epoch=1/3 step=300/500 global_step=300/1500 progress=20.00% objective=1.810960 lr_encoder=1.702e-05 lr_heads=8.511e-05 samples_per_second=722.10 ETA=0.9 min
[train] run=tooka_B epoch=1/3 step=400/500 global_step=400/1500 progress=26.67% objective=1.749096 lr_encoder=1.560e-05 lr_heads=7.801e-05 samples_per_second=727.16 ETA=0.8 min
[train] run=tooka_B epoch=1/3 step=500/500 global_st

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 13708.24it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=tooka_C device=cuda:0 variant=C seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[optimizer] fused=True steps_per_epoch=500 total_steps=1500 warmup_steps=90
[train] run=tooka_C epoch=1/3 step=100/500 global_step=100/1500 progress=6.67% objective=2.134290 lr_encoder=1.986e-05 lr_heads=9.929e-05 samples_per_second=725.57 ETA=1.0 min
[train] run=tooka_C epoch=1/3 step=200/500 global_step=200/1500 progress=13.33% objective=1.841439 lr_encoder=1.844e-05 lr_heads=9.220e-05 samples_per_second=737.53 ETA=0.9 min
[train] run=tooka_C epoch=1/3 step=300/500 global_step=300/1500 progress=20.00% objective=1.749055 lr_encoder=1.702e-05 lr_heads=8.511e-05 samples_per_second=735.86 ETA=0.9 min
[train] run=tooka_C epoch=1/3 step=400/500 global_step=400/1500 progress=26.67% objective=1.695990 lr_encoder=1.560e-05 lr_heads=7.801e-05 samples_per_second=736.16 ETA=0.8 min
[train] run=tooka_C epoch=1/3 step=500/500 global_st

---

# Inspect: Dev Run Outputs and TF-IDF Prediction Files

In [8]:
# ---- Section: Imports and paths ----
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

from digikala_review_transformer import config

DEV_ROOT = config.ROOT / "artifacts" / "transformer_runs_dev"
BASE_PRED_DIR = config.ROOT / "reports" / "predictions"
PACK_DIR = config.ROOT / "data" / "training_pack"
RUN_NAMES = ["tooka_B", "tooka_C"]


# ---- Section: Helpers ----
def show_parquet(path: Path, n_head: int = 3) -> None:
    """Print schema, row count and a small head (the text column is skipped)."""
    pf = pq.ParquetFile(path)
    print(f"  file: {path.relative_to(config.ROOT)}")
    print(f"  rows: {pf.metadata.num_rows:,}")
    for field in pf.schema_arrow:
        print(f"    {field.name}: {field.type}")
    cols = [f.name for f in pf.schema_arrow if f.name != "body"]
    head = next(pf.iter_batches(batch_size=n_head, columns=cols)).to_pandas()
    print(head.to_string(index=False))


def show_json_structure(obj, indent: int = 2, depth: int = 0, max_depth: int = 2) -> None:
    """Print keys and value types (short values are printed too)."""
    pad = " " * indent * (depth + 1)
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, dict) and depth < max_depth:
                print(f"{pad}{key}: dict[{len(value)}]")
                show_json_structure(value, indent, depth + 1, max_depth)
            elif isinstance(value, (list, tuple)):
                print(f"{pad}{key}: list[{len(value)}]")
            else:
                text = repr(value)
                print(f"{pad}{key}: {type(value).__name__} = {text[:70]}")


# ---- Section: Dev run directories ----
for name in RUN_NAMES:
    run_dir = DEV_ROOT / name
    print("=" * 78)
    print(f"RUN: {name}")
    print("=" * 78)
    for item in sorted(run_dir.iterdir()):
        size = item.stat().st_size if item.is_file() else sum(
            f.stat().st_size for f in item.rglob("*") if f.is_file()
        )
        kind = "file" if item.is_file() else "dir "
        print(f"  [{kind}] {item.name:<28} {size / 1024:>12,.1f} KB")

    print("\n-- history.csv --")
    history = pd.read_csv(run_dir / "history.csv")
    print("  columns:", list(history.columns))
    print(history.head(2).T.to_string())

    print("\n-- run_summary.json (keys, depth 2) --")
    summary = json.loads((run_dir / "run_summary.json").read_text(encoding="utf-8"))
    show_json_structure(summary)

    print("\n-- val_probs_best.parquet --")
    show_parquet(run_dir / "val_probs_best.parquet")
    print("\n-- test_probs.parquet --")
    show_parquet(run_dir / "test_probs.parquet")

# ---- Section: TF-IDF baseline prediction files ----
print("=" * 78)
print("TF-IDF baseline files (tfidf12_*)")
print("=" * 78)
for path in sorted(BASE_PRED_DIR.glob("tfidf12_*")):
    print(f"  {path.name}")

for fname in ["tfidf12_rec_B__validation.parquet", "tfidf12_band_B__validation.parquet"]:
    print(f"\n-- {fname} --")
    show_parquet(BASE_PRED_DIR / fname)

# ---- Section: Validation pack schema ----
print("\n" + "=" * 78)
print("Pack validation.parquet")
print("=" * 78)
show_parquet(PACK_DIR / "validation.parquet")

RUN: tooka_B
  [file] best_model.pt                   477,877.5 KB
  [file] config_resolved.yaml                  0.6 KB
  [file] encoder_config.json                   1.1 KB
  [file] history.csv                           1.5 KB
  [file] run_summary.json                     10.7 KB
  [file] test_probs.parquet                  396.5 KB
  [dir ] tokenizer                         3,882.2 KB
  [file] train.log                             3.5 KB
  [file] train_log.csv                         3.0 KB
  [file] val_probs_best.parquet              397.5 KB

-- history.csv --
  columns: ['epoch', 'train_objective', 'train_plain_ce_rec', 'train_plain_ce_band', 'train_subset_plain_ce_rec', 'train_subset_plain_ce_band', 'val_plain_ce_rec', 'val_plain_ce_band', 'val_acc_rec', 'val_acc_band', 'val_macro_f1_rec', 'val_macro_f1_band', 'val_macro_f1_rec_raw', 'val_macro_f1_band_raw', 'train_subset_macro_f1_rec', 'train_subset_macro_f1_band', 'val_acc_rec_raw', 'val_acc_band_raw', 'val_mean_macro_f1', 'lr

#  Step D2: Dev Scaling Run of tooka_B (100k rows, more epochs)

In [1]:
import gc
import importlib.util
import time

import numpy as np
import pandas as pd
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import (
    checkpoint,
    data,
    loop,
    predict,
    run_config,
    tokenization,
)
from digikala_review_transformer.training import model as model_lib  # noqa: F811

# ---- Section: Dev settings (edit only here) ----
TRAIN_ROWS = 100_000
VAL_ROWS = 10_000
TEST_ROWS = 10_000
MONITOR_ROWS = 5_000
MAX_EPOCHS = 5
PATIENCE = 1   
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 128
LOG_EVERY_STEPS = 100
DEV_RUNS_ROOT = "artifacts/transformer_runs_dev_scale"   # separate root: D1 outputs stay untouched
RUN_NAMES = ["tooka_B", "tooka_C"]

# ---- Section: Reuse the tested monitor-subset helper from the CLI script ----
cli_spec = importlib.util.spec_from_file_location(
    "train_transformer_cli", config.ROOT / "scripts" / "train_transformer.py"
)
cli = importlib.util.module_from_spec(cli_spec)
cli_spec.loader.exec_module(cli)

reference_ids: dict[str, np.ndarray] = {}


def free_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_dev(run_name: str) -> None:
    """Fine-tune one config on the small subset, using the same code path as the CLI."""
    cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / f"{run_name}.yaml")
    cfg["output"]["root"] = DEV_RUNS_ROOT
    training = cfg["training"]
    training["max_epochs"] = MAX_EPOCHS
    training["batch_size"] = BATCH_SIZE
    training["eval_batch_size"] = EVAL_BATCH_SIZE
    training["log_every_steps"] = LOG_EVERY_STEPS
    training["num_workers"] = 0
    training["train_subset_eval_rows"] = MONITOR_ROWS
    cfg["training"]["early_stopping_patience"] = PATIENCE

    target_dir = run_config.run_dir(cfg)
    status = checkpoint.run_status(target_dir)
    if status == "complete":
        print(f"[PASS] {run_name}: already complete at {target_dir} -> skipping", flush=True)
        return
    print(f"[{run_name}] status={status}, output={target_dir}", flush=True)

    device = torch.device("cuda", torch.cuda.current_device()) if torch.cuda.is_available() else torch.device("cpu")
    train_split = val_split = test_split = monitor_split = model_instance = None
    started = time.perf_counter()
    try:
        pack_dir = run_config.project_path(cfg["data"]["pack_dir"])
        manifest = data.open_pack(pack_dir)
        bundle = tokenization.load_tokenizer(
            run_config.project_path(cfg["model"]["pretrained_path"]),
            backend=str(cfg["model"]["tokenizer_backend"]),
            max_length=int(cfg["model"]["max_length"]),
        )
        variant = str(training["variant"])

        def load(split_name: str, rows: int) -> data.SplitData:
            return data.load_split(
                split_name, bundle, pack_dir, manifest, variant=variant,
                subset_rows=rows, subset_seed=config.SPLIT_SEED,
                cache_dir=None, show_progress=False,
            )

        train_split = load("train", TRAIN_ROWS)
        val_split = load("validation", VAL_ROWS)
        test_split = load("test", TEST_ROWS)
        monitor_split = cli.build_monitor_subset(train_split, MONITOR_ROWS, config.SPLIT_SEED)

        # B and C must see exactly the same rows.
        for name, split in (("train", train_split), ("validation", val_split), ("test", test_split)):
            if name not in reference_ids:
                reference_ids[name] = split.ids.copy()
            elif not np.array_equal(reference_ids[name], split.ids):
                raise RuntimeError(f"[FAIL] {run_name}: '{name}' subset ids differ from the first run -> check subset seed")
        print(
            f"[PASS] {run_name}: subsets ready (train={train_split.n_rows:,}, val={val_split.n_rows:,}, "
            f"test={test_split.n_rows:,}, monitor={monitor_split.n_rows:,}, variant={variant})",
            flush=True,
        )

        model_instance, load_info = model_lib.build_model(cfg)
        max_token_id = max(int(s.stats["max_token_id"]) for s in (train_split, val_split, test_split))
        model_lib.check_token_ids_fit(model_instance, max_token_id=max_token_id, pad_id=bundle.pad_id)

        loop_result = loop.fit(
            cfg, model_instance, train_split, val_split, monitor_split,
            run_directory=target_dir, device=device,
        )
        finalized = predict.finalize_run(
            cfg, model_instance, bundle, loop_result,
            train_split=train_split, validation_split=val_split, test_split=test_split,
            train_subset_split=monitor_split, manifest=manifest,
            load_info=load_info, device=device,
        )
        print(
            f"[PASS] {run_name}: done in {(time.perf_counter() - started) / 60.0:.1f} min | "
            f"best_epoch={loop_result.best_epoch} | best_val_mean_macro_f1={loop_result.best_score:.4f} | "
            f"summary={finalized.summary_path}",
            flush=True,
        )
    finally:
        del model_instance, train_split, val_split, test_split, monitor_split
        free_memory()


free_memory()
overall_start = time.perf_counter()
for name in RUN_NAMES:
    run_dev(name)
print(f"[PASS] All dev runs finished in {(time.perf_counter() - overall_start) / 60.0:.1f} min", flush=True)

# ---- Section: Quick summary from history.csv (read-only) ----
show_columns = ["epoch", "val_mean_macro_f1", "val_macro_f1_rec", "val_macro_f1_band", "epoch_seconds", "is_best"]
for name in RUN_NAMES:
    run_directory = config.ROOT / DEV_RUNS_ROOT / name
    history = pd.read_csv(run_directory / predict.HISTORY_NAME)
    print(f"--- {name} (status={checkpoint.run_status(run_directory)}) ---")
    print(history[show_columns].to_string(index=False))

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[tooka_B] status=fresh, output=/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/artifacts/transformer_runs_dev_scale/tooka_B
[timer] tokenize_TookaBERT-Base_train: 1.0s
[timer] tokenize_TookaBERT-Base_validation: 0.1s
[timer] tokenize_TookaBERT-Base_test: 0.1s
[PASS] tooka_B: subsets ready (train=100,000, val=10,000, test=10,000, monitor=5,000, variant=B)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7047.69it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream ta

[run] start run=tooka_B device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[optimizer] fused=True steps_per_epoch=3125 total_steps=15625 warmup_steps=937
[train] run=tooka_B epoch=1/5 step=100/3125 global_step=100/15625 progress=0.64% objective=2.322957 lr_encoder=2.134e-06 lr_heads=1.067e-05 samples_per_second=697.59 ETA=11.9 min
[train] run=tooka_B epoch=1/5 step=200/3125 global_step=200/15625 progress=1.28% objective=2.197791 lr_encoder=4.269e-06 lr_heads=2.134e-05 samples_per_second=698.76 ETA=11.8 min
[train] run=tooka_B epoch=1/5 step=300/3125 global_step=300/15625 progress=1.92% objective=1.914356 lr_encoder=6.403e-06 lr_heads=3.202e-05 samples_per_second=709.79 ETA=11.5 min
[train] run=tooka_B epoch=1/5 step=400/3125 global_step=400/15625 progress=2.56% objective=1.824516 lr_encoder=8.538e-06 lr_heads=4.269e-05 samples_per_second=708.08 ETA=11.5 min
[train] run=tooka_B epoch=1/5 step=500/3

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16504.43it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=tooka_C device=cuda:0 variant=C seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[optimizer] fused=True steps_per_epoch=3125 total_steps=15625 warmup_steps=937
[train] run=tooka_C epoch=1/5 step=100/3125 global_step=100/15625 progress=0.64% objective=2.321009 lr_encoder=2.134e-06 lr_heads=1.067e-05 samples_per_second=752.46 ETA=11.0 min
[train] run=tooka_C epoch=1/5 step=200/3125 global_step=200/15625 progress=1.28% objective=2.187306 lr_encoder=4.269e-06 lr_heads=2.134e-05 samples_per_second=716.09 ETA=11.5 min
[train] run=tooka_C epoch=1/5 step=300/3125 global_step=300/15625 progress=1.92% objective=1.873309 lr_encoder=6.403e-06 lr_heads=3.202e-05 samples_per_second=716.68 ETA=11.4 min
[train] run=tooka_C epoch=1/5 step=400/3125 global_step=400/15625 progress=2.56% objective=1.772278 lr_encoder=8.538e-06 lr_heads=4.269e-05 samples_per_second=709.59 ETA=11.4 min
[train] run=tooka_C epoch=1/5 step=500/3

# Audit: Dev Scale Runs (ids, manual Macro F1 vs history, TF-IDF on same ids)

In [1]:
# ---- Section: Imports and constants ----
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from digikala_review_transformer import config

RUNS_ROOT = config.ROOT / "artifacts" / "transformer_runs_dev_scale"
RUN_NAMES = ["tooka_B", "tooka_C"]
PACK_DIR = config.ROOT / "data" / "training_pack"
BASE_DIR = config.ROOT / "reports" / "predictions"

HEAD_COLUMNS = {
    "rec": ["p_recommended", "p_not_recommended", "p_no_idea"],
    "band": ["p_low", "p_medium", "p_high"],
}
TOL = 1e-5      # tolerance for "same as history.csv"
N_BOOT = 300    # paired bootstrap resamples (kept small on purpose)

n_fail = 0


# ---- Section: Helpers ----
def check(ok: bool, pass_msg: str, fail_msg: str) -> None:
    """Print [PASS] or [FAIL] <reason> -> <fix> and count failures."""
    global n_fail
    if ok:
        print(f"[PASS] {pass_msg}")
    else:
        n_fail += 1
        print(f"[FAIL] {fail_msg}")


def macro_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Independent Macro F1 over the 3 classes (sklearn, not the project code)."""
    return float(f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0))


def valid_mask(series: pd.Series) -> np.ndarray:
    """True where the label is one of 0/1/2 (drops NaN / missing codes)."""
    return series.isin([0, 1, 2]).to_numpy()


# ---- Section: Pack (read without the body column) ----
pack_val = pd.read_parquet(
    PACK_DIR / "validation.parquet",
    columns=["id", "y_rec", "y_band", "y_rec_raw", "y_band_raw"],
)
pack_val["id"] = pack_val["id"].astype(str)
pack_val = pack_val.set_index("id")
pack_test_ids = set(pd.read_parquet(PACK_DIR / "test.parquet", columns=["id"])["id"].astype(str))
print(f"pack validation rows: {len(pack_val):,}\n")

# TF-IDF validation predictions (corrected labels + predicted class)
tfidf = {}
for head in HEAD_COLUMNS:
    frame = pd.read_parquet(
        BASE_DIR / f"tfidf12_{head}_B__validation.parquet",
        columns=["id", "y_true_corr", "pred"],
    )
    frame["id"] = frame["id"].astype(str)
    tfidf[head] = frame.drop_duplicates("id").set_index("id")

rng = np.random.default_rng(2026)

# ---- Section: Audit per run ----
for run_name in RUN_NAMES:
    run_dir = RUNS_ROOT / run_name
    print("=" * 78)
    print(f"RUN: {run_name}")
    print("=" * 78)

    history = pd.read_csv(run_dir / "history.csv")
    best_row = history[history["is_best"]].iloc[-1]
    last_row = history.iloc[-1]
    best_epoch, last_epoch = int(best_row["epoch"]), int(last_row["epoch"])
    print(f"history: best epoch = {best_epoch}, last epoch = {last_epoch}, "
          f"best val_mean = {best_row['val_mean_macro_f1']:.6f}, "
          f"last val_mean = {last_row['val_mean_macro_f1']:.6f}")

    # -- Check 1: ids --
    probs = pd.read_parquet(run_dir / "val_probs_best.parquet")
    probs["id"] = probs["id"].astype(str)
    check(probs["id"].is_unique, f"val ids are unique (n={len(probs):,})",
          "duplicate ids in val_probs_best -> bug in predict/evaluate ordering")
    in_pack = probs["id"].isin(pack_val.index)
    check(bool(in_pack.all()), "all val ids exist in pack validation.parquet",
          f"{int((~in_pack).sum())} ids not in validation split -> wrong data reached the model")
    test_ids = pd.read_parquet(run_dir / "test_probs.parquet", columns=["id"])["id"].astype(str)
    check(test_ids.is_unique and test_ids.isin(pack_test_ids).all(),
          f"test ids are unique and all exist in pack test.parquet (n={len(test_ids):,})",
          "test ids duplicated or not in test split -> check predict.finalize_run")

    labels = pack_val.reindex(probs["id"])
    corrected, raw, tf_cmp = {}, {}, {}

    for head, cols in HEAD_COLUMNS.items():
        pred = probs[cols].to_numpy().argmax(axis=1)

        # -- Check 2: manual Macro F1, corrected view (vs history best row) --
        y_corr = labels[f"y_{head}"]
        m = valid_mask(y_corr)
        corrected[head] = macro_f1(y_corr.to_numpy()[m], pred[m])
        hist_val = float(best_row[f"val_macro_f1_{head}"])
        check(abs(corrected[head] - hist_val) < TOL,
              f"{head} corrected F1 manual={corrected[head]:.6f} == history epoch {best_epoch} ({hist_val:.6f}), rows={int(m.sum()):,}",
              f"{head} corrected F1 manual={corrected[head]:.6f} != history {hist_val:.6f} -> wrong ids/labels/order or wrong checkpoint saved")

        # -- Check 3: raw view (history column found by name, never guessed blindly) --
        y_raw = labels[f"y_{head}_raw"]
        mr = valid_mask(y_raw)
        raw[head] = macro_f1(y_raw.to_numpy()[mr], pred[mr])
        cand = [c for c in history.columns if "raw" in c and f"f1_{head}" in c]
        if len(cand) == 1:
            hist_raw = float(best_row[cand[0]])
            check(abs(raw[head] - hist_raw) < TOL,
                  f"{head} raw F1 manual={raw[head]:.6f} == history {cand[0]} ({hist_raw:.6f})",
                  f"{head} raw F1 manual={raw[head]:.6f} != history {hist_raw:.6f} -> raw view computed from wrong labels")
        else:
            print(f"[SKIP] {head} raw: history columns found = {cand}; manual raw F1 = {raw[head]:.6f}")

        # -- Same-ids comparison with TF-IDF --
        tf = tfidf[head].reindex(probs["id"])
        ok_rows = m & tf["pred"].notna().to_numpy() & valid_mask(tf["y_true_corr"])
        both = m & valid_mask(tf["y_true_corr"])
        agree = float((tf["y_true_corr"].to_numpy()[both] == y_corr.to_numpy()[both]).mean())
        check(agree == 1.0, f"{head}: pack labels == TF-IDF file labels on shared ids",
              f"{head}: label agreement with TF-IDF file = {agree:.4f} -> label view mismatch between pack and baseline")
        yt = y_corr.to_numpy()[ok_rows]
        pt = pred[ok_rows]
        pb = tf["pred"].to_numpy()[ok_rows].astype(int)
        f_t, f_b = macro_f1(yt, pt), macro_f1(yt, pb)
        diffs = np.empty(N_BOOT)
        for i in range(N_BOOT):
            idx = rng.integers(0, len(yt), len(yt))
            diffs[i] = macro_f1(yt[idx], pt[idx]) - macro_f1(yt[idx], pb[idx])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        tf_cmp[head] = (len(yt), f_t, f_b, f_t - f_b, lo, hi)

    # -- Check 2b: mean + best-vs-last distinction --
    mean_manual = (corrected["rec"] + corrected["band"]) / 2
    check(abs(mean_manual - float(best_row["val_mean_macro_f1"])) < TOL,
          f"val_mean manual={mean_manual:.6f} == history best row ({float(best_row['val_mean_macro_f1']):.6f})",
          "val_mean manual != history best row -> selection metric computed differently")
    if best_epoch != last_epoch:
        check(abs(mean_manual - float(last_row["val_mean_macro_f1"])) > TOL,
              f"probabilities come from best epoch {best_epoch}, not last epoch {last_epoch}",
              "saved probabilities equal the LAST epoch -> best checkpoint/probs not saved from best epoch")

    # -- Report tables --
    print("\nManual Macro F1 on this run's validation subset:")
    print(f"  corrected: rec={corrected['rec']:.4f}  band={corrected['band']:.4f}")
    print(f"  raw      : rec={raw['rec']:.4f}  band={raw['band']:.4f}")
    print("\nTransformer vs TF-IDF on the SAME ids (corrected labels, paired bootstrap 95% CI of the difference):")
    for head, (n, f_t, f_b, d, lo, hi) in tf_cmp.items():
        print(f"  {head:<5} n={n:,}  transformer={f_t:.4f}  tfidf={f_b:.4f}  diff={d:+.4f}  CI=[{lo:+.4f}, {hi:+.4f}]")
    print()

# ---- Section: Final verdict ----
print("=" * 78)
if n_fail == 0:
    print("AUDIT RESULT: ALL CHECKS PASSED")
else:
    print(f"AUDIT RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above")

pack validation rows: 922,794

RUN: tooka_B
history: best epoch = 2, last epoch = 3, best val_mean = 0.646080, last val_mean = 0.640385
[PASS] val ids are unique (n=10,000)
[PASS] all val ids exist in pack validation.parquet
[PASS] test ids are unique and all exist in pack test.parquet (n=10,000)
[PASS] rec corrected F1 manual=0.694199 == history epoch 2 (0.694199), rows=8,541
[PASS] rec raw F1 manual=0.691690 == history val_macro_f1_rec_raw (0.691690)
[PASS] rec: pack labels == TF-IDF file labels on shared ids
[PASS] band corrected F1 manual=0.597961 == history epoch 2 (0.597961), rows=10,000
[PASS] band raw F1 manual=0.589627 == history val_macro_f1_band_raw (0.589627)
[PASS] band: pack labels == TF-IDF file labels on shared ids
[PASS] val_mean manual=0.646080 == history best row (0.646080)
[PASS] probabilities come from best epoch 2, not last epoch 3

Manual Macro F1 on this run's validation subset:
  corrected: rec=0.6942  band=0.5980
  raw      : rec=0.6917  band=0.5896

Transform

---

# Stage 1 / Cell 1: Mode Switch (dev vs real), Paths, Parameters and Run Audit

In [4]:
# ============================================================================
# Stage 1 / Cell 1: Mode switch, paths, parameters and run audit
# Change ONLY `STATE` below. Every later cell reads the variables built here.
# ============================================================================

# ---- Section: THE SWITCH ----
STATE = 1   # 0 = dev results (small subset)   |   1 = real results (friend's full runs)

# ---- Section: Imports ----
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import yaml

from digikala_review_transformer import config
from digikala_review_transformer.training import pack as pack_module

# ---- Section: Mode table (the only place where 0/1 matters) ----
MODES = {
    0: {
        "name": "dev",
        "tag": "DEV (100k-row training subset, 10k-row validation/test subsets)",
        "runs_root": "artifacts/transformer_runs_dev_scale",
        "run_names": ["tooka_B", "tooka_C"],
        "out_dir": "reports/transformer_stage1/dev",
        "full_data": False,
        # dotted-path -> expected value in config_resolved.yaml
        "expected_config": {"seed": config.SPLIT_SEED},
    },
    1: {
        "name": "full",
        "tag": "FULL (all train_supervised rows, full validation/test)",
        "runs_root": "artifacts/transformer_runs",
        "run_names": ["tooka_B", "tooka_C"],
        "out_dir": "reports/transformer_stage1/full",
        "full_data": True,
        "expected_config": {
            "seed": config.SPLIT_SEED,
            "model.max_length": 128,
            "model.tokenizer_backend": "hf_fast",
            "training.max_epochs": 4,
            "training.early_stopping_patience": 1,
            "training.batch_size": 128,
            "training.lr_encoder": 2e-5,
            "training.lr_heads": 1e-4,
            "training.weight_decay": 0.01,
            "training.warmup_ratio": 0.06,
            "training.label_smoothing": 0.1,
            "training.loss_weight_band": 1.0,
            "training.use_class_weights": True,
            "training.bf16": True,
        },
    },
}
if STATE not in MODES:
    raise ValueError(f"STATE must be 0 or 1, got {STATE!r}")
MODE = MODES[STATE]

# ---- Section: Paths derived from the mode ----
RUNS_ROOT = config.ROOT / MODE["runs_root"]
RUN_NAMES = MODE["run_names"]
OUT_DIR = config.ROOT / MODE["out_dir"]
TABLES_DIR = OUT_DIR / "tables"
FIGS_DIR = OUT_DIR / "figures"
for _d in (TABLES_DIR, FIGS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

PACK_DIR = config.ROOT / "data" / "training_pack"
BASELINE_DIR = config.ROOT / "reports" / "predictions"
BASELINE_IDS = {"rec": "tfidf12_rec_B", "band": "tfidf12_band_B"}   # files: <id>__<split>.parquet

# ---- Section: Analysis parameters (same in both modes; fixed by the plan) ----
HEADS = ("rec", "band")
VIEWS = ("raw", "corrected")                 # both are reported, neither is "main"
SELECTION_METRIC = "val_mean_macro_f1"       # mean of the two heads' Macro F1 (corrected validation)
C_OVER_B_THRESHOLD = 0.003                   # C wins only if score_C >= score_B + 0.003
N_BOOT = 1000                                # paired bootstrap resamples (used by later cells)
BOOT_SEED = config.SPLIT_SEED
FIG_DPI = 150
# Aurora Glass palette (same hex values as plot_style): raw = SLATE_LIGHT, corrected = BLUE (as in the baseline phase)
VIEW_COLORS = {"raw": "#94A3B8", "corrected": "#38BDF8"}
VARIANT_COLORS = {"B": "#2DD4BF", "C": "#FBBF24"}   # TURQUOISE / AMBER
MODE_TAG = MODE["tag"]                       # put this in every chart caption / table title

REQUIRED_FILES = [
    "config_resolved.yaml", "history.csv", "train_log.csv", "train.log",
    "run_summary.json", "val_probs_best.parquet", "test_probs.parquet", "encoder_config.json",
]
HISTORY_COLUMNS = [
    "epoch", "val_mean_macro_f1", "val_macro_f1_rec", "val_macro_f1_band", "epoch_seconds", "is_best",
]
TOL = 1e-6


# ---- Section: Helpers ----
n_fail = 0


def check(ok: bool, pass_msg: str, fail_msg: str) -> bool:
    """Print [PASS] or [FAIL] <reason> -> <fix>; count failures."""
    global n_fail
    if ok:
        print(f"[PASS] {pass_msg}")
    else:
        n_fail += 1
        print(f"[FAIL] {fail_msg}")
    return ok


def flatten(obj, prefix: str = "") -> dict:
    """Flatten nested dicts into {'a.b.c': value}."""
    flat = {}
    for key, value in obj.items():
        path = f"{prefix}{key}"
        if isinstance(value, dict):
            flat.update(flatten(value, path + "."))
        else:
            flat[path] = value
    return flat


def same_value(actual, expected) -> bool:
    """Numeric-tolerant equality (YAML may load 2e-05 as a string)."""
    if isinstance(expected, bool):
        return actual is expected or str(actual).lower() == str(expected).lower()
    if isinstance(expected, (int, float)):
        try:
            return abs(float(actual) - float(expected)) < 1e-12
        except (TypeError, ValueError):
            return False
    return actual == expected


def dig(obj: dict, *keys):
    """Nested lookup with a clear error that lists the keys that do exist."""
    cur = obj
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            have = list(cur.keys()) if isinstance(cur, dict) else type(cur).__name__
            raise KeyError(f"key '{'.'.join(keys)}' not found; available at that level: {have}")
        cur = cur[key]
    return cur


def parquet_rows(path: Path) -> int:
    return pq.ParquetFile(path).metadata.num_rows


# ---- Section: Pack facts ----
manifest = json.loads((PACK_DIR / pack_module.MANIFEST_NAME).read_text(encoding="utf-8"))
PACK_FINGERPRINT = pack_module.pack_fingerprint(manifest)
PACK_ROWS = {
    split: manifest["files"][pack_module.pack_file_name(split)]["rows"]
    for split in ("train", "validation", "test")
}

print("=" * 78)
print(f"MODE: STATE={STATE} -> {MODE_TAG}")
print(f"runs root : {RUNS_ROOT.relative_to(config.ROOT)}")
print(f"outputs   : {OUT_DIR.relative_to(config.ROOT)}")
print(f"pack      : fingerprint={PACK_FINGERPRINT}, rows={PACK_ROWS}")
print("=" * 78)

# ---- Section: Audit and load each run ----
RUNS = {}
for run_name in RUN_NAMES:
    print(f"\n--- {run_name} ---")
    run_dir = RUNS_ROOT / run_name
    if not check(run_dir.is_dir(), f"run directory exists: {run_dir.relative_to(config.ROOT)}",
                 f"missing run directory {run_dir} -> "
                 f"{'extract the friend zip into artifacts/transformer_runs/' if STATE == 1 else 'run the dev cell first'}"):
        continue

    # -- files --
    missing = [f for f in REQUIRED_FILES if not (run_dir / f).is_file() or (run_dir / f).stat().st_size == 0]
    check(not missing, "all required files exist and are non-empty",
          f"missing/empty files: {missing} -> re-package the run or re-run training")
    tok_dir = run_dir / "tokenizer"
    check(tok_dir.is_dir() and any(tok_dir.iterdir()), "tokenizer/ is non-empty",
          "tokenizer/ missing or empty -> re-package the run")
    check(not (run_dir / "last.pt").exists(), "no last.pt (run was finalized)",
          "last.pt still exists -> the run did not finish cleanly")
    print(f"[INFO] best_model.pt present: {(run_dir / 'best_model.pt').exists()} (not needed by the analysis)")
    if missing:
        continue

    # -- load --
    history = pd.read_csv(run_dir / "history.csv")
    summary = json.loads((run_dir / "run_summary.json").read_text(encoding="utf-8"))
    cfg = yaml.safe_load((run_dir / "config_resolved.yaml").read_text(encoding="utf-8"))
    flat_cfg = flatten(cfg)

    # -- config --
    check(cfg.get("run_name") == run_name and cfg["training"]["variant"] == run_name.split("_")[-1],
          f"config run_name/variant match the folder ({run_name}, variant {cfg['training']['variant']})",
          f"config says run_name={cfg.get('run_name')}, variant={cfg['training']['variant']} -> wrong folder/zip")
    wrong = {k: (flat_cfg.get(k), v) for k, v in MODE["expected_config"].items() if not same_value(flat_cfg.get(k), v)}
    check(not wrong, f"config matches the expected {MODE['name']} settings ({len(MODE['expected_config'])} keys)",
          f"config differs from the plan (key: (actual, expected)): {wrong} -> run was made with other settings")

    # -- history --
    miss_cols = [c for c in HISTORY_COLUMNS if c not in history.columns]
    if not check(not miss_cols, f"history.csv has the required columns ({len(history)} epochs)",
                 f"history.csv lacks columns {miss_cols} -> check training/loop.py history writer"):
        continue
    best_row = history[history["is_best"]].iloc[-1]
    check(best_row[SELECTION_METRIC] >= history[SELECTION_METRIC].max() - TOL,
          f"is_best marks the maximum {SELECTION_METRIC} (epoch {int(best_row['epoch'])})",
          f"is_best row is not the maximum of {SELECTION_METRIC} -> best-epoch selection bug")
    max_epochs = int(cfg["training"]["max_epochs"])
    if len(history) < max_epochs:
        check(not bool(history.iloc[-1]["is_best"]),
              f"stopped early after {len(history)}/{max_epochs} epochs with a non-best last epoch",
              "stopped early although the last epoch was best -> early-stopping logic bug")

    # -- summary keys (from the other-AI digest; reported clearly if missing) --
    try:
        s_best_epoch = int(dig(summary, "selection_and_stopping", "best_epoch"))
        s_best_score = float(dig(summary, "selection_and_stopping", "best_val_mean_macro_f1"))
        stop_reason = dig(summary, "selection_and_stopping", "stop_reason")
        s_fingerprint = dig(summary, "pack", "fingerprint")
        check(s_best_epoch == int(best_row["epoch"]) and abs(s_best_score - float(best_row[SELECTION_METRIC])) < TOL,
              f"run_summary best epoch/score agree with history (epoch {s_best_epoch}, {s_best_score:.6f})",
              f"summary says epoch {s_best_epoch}/{s_best_score:.6f}, history says "
              f"epoch {int(best_row['epoch'])}/{float(best_row[SELECTION_METRIC]):.6f} -> inconsistent run files")
        check(s_fingerprint == PACK_FINGERPRINT, f"pack fingerprint matches ({s_fingerprint})",
              f"run pack fingerprint {s_fingerprint} != current pack {PACK_FINGERPRINT} -> trained on a different pack")
        dig(summary, "validation_best")
        dig(summary, "test_quick")
    except KeyError as exc:
        check(False, "", f"run_summary.json key problem: {exc} -> send this line, I will fix the key names")
        continue

    # -- probability file rows --
    n_val = parquet_rows(run_dir / "val_probs_best.parquet")
    n_test = parquet_rows(run_dir / "test_probs.parquet")
    if MODE["full_data"]:
        check(n_val == PACK_ROWS["validation"] and n_test == PACK_ROWS["test"],
              f"probability files cover the full splits (val {n_val:,}, test {n_test:,})",
              f"rows val {n_val:,}/test {n_test:,} != pack {PACK_ROWS['validation']:,}/{PACK_ROWS['test']:,} "
              "-> a subset run was packaged as a real run")
    else:
        check(n_val < PACK_ROWS["validation"] and n_test < PACK_ROWS["test"],
              f"dev probability files are subsets (val {n_val:,}, test {n_test:,})",
              "dev run covers the full split -> STATE/runs_root mismatch")

    RUNS[run_name] = {
        "dir": run_dir, "cfg": cfg, "summary": summary, "history": history,
        "variant": cfg["training"]["variant"], "best_epoch": s_best_epoch,
        "best_score": s_best_score, "stop_reason": stop_reason,
        "n_val": n_val, "n_test": n_test,
    }

# ---- Section: B vs C must differ only in variant and run_name ----
if all(name in RUNS for name in RUN_NAMES) and len(RUN_NAMES) == 2:
    flat_a, flat_b = (flatten(RUNS[n]["cfg"]) for n in RUN_NAMES)
    diff_keys = {k for k in set(flat_a) | set(flat_b) if flat_a.get(k) != flat_b.get(k)}
    check(diff_keys == {"training.variant", "run_name"},
          "B and C configs differ only in training.variant and run_name",
          f"B/C configs differ in {sorted(diff_keys)} -> unfair comparison, runs must share all settings")
    check(RUNS[RUN_NAMES[0]]["n_val"] == RUNS[RUN_NAMES[1]]["n_val"]
          and RUNS[RUN_NAMES[0]]["n_test"] == RUNS[RUN_NAMES[1]]["n_test"],
          "B and C evaluated on the same number of validation/test rows",
          "B and C have different row counts -> different evaluation sets")

# ---- Section: Compact overview ----
print("\n" + "=" * 78)
overview = pd.DataFrame([
    {"run": n, "variant": r["variant"], "epochs_done": len(r["history"]), "best_epoch": r["best_epoch"],
     "best_val_mean_macro_f1": round(r["best_score"], 6), "stop_reason": r["stop_reason"]}
    for n, r in RUNS.items()
])
print(overview.to_string(index=False) if len(overview) else "(no runs loaded)")

ANALYSIS_READY = (n_fail == 0) and len(RUNS) == len(RUN_NAMES)
print("=" * 78)
print("CELL 1 RESULT: ALL CHECKS PASSED" if ANALYSIS_READY
      else f"CELL 1 RESULT: {n_fail} CHECK(S) FAILED -> fix before running the next cells")

MODE: STATE=1 -> FULL (all train_supervised rows, full validation/test)
runs root : artifacts/transformer_runs
outputs   : reports/transformer_stage1/full
pack      : fingerprint=9faa2a97e4502701, rows={'train': 2860236, 'validation': 922794, 'test': 922805}

--- tooka_B ---
[PASS] run directory exists: artifacts/transformer_runs/tooka_B
[PASS] all required files exist and are non-empty
[PASS] tokenizer/ is non-empty
[PASS] no last.pt (run was finalized)
[INFO] best_model.pt present: False (not needed by the analysis)
[PASS] config run_name/variant match the folder (tooka_B, variant B)
[PASS] config matches the expected full settings (14 keys)
[PASS] history.csv has the required columns (4 epochs)
[PASS] is_best marks the maximum val_mean_macro_f1 (epoch 4)
[PASS] run_summary best epoch/score agree with history (epoch 4, 0.663465)
[PASS] pack fingerprint matches (9faa2a97e4502701)
[PASS] probability files cover the full splits (val 922,794, test 922,805)

--- tooka_C ---
[PASS] run dir

# Inspect: plot_style and plots (palette, public API, usage in earlier notebooks)

In [4]:
# ---- Section: Imports and limits ----
import importlib
import inspect
import json
import types

from digikala_review_transformer import config

PKG = "digikala_review_transformer"
MODULE_NAMES = ["plot_style", "plots"]
MAX_SOURCE_LINES = 160      # source dump limit for plot_style
MAX_USAGE_LINES = 30        # usage examples limit (total)


# ---- Section: Public API of each module ----
for name in MODULE_NAMES:
    print("=" * 78)
    print(f"MODULE: {PKG}.{name}")
    print("=" * 78)
    try:
        mod = importlib.import_module(f"{PKG}.{name}")
    except Exception as exc:                      # read-only inspection: report and go on  # noqa: BLE001
        print(f"  import failed: {type(exc).__name__}: {exc}")
        continue

    src_path = inspect.getsourcefile(mod)
    print(f"  file: {src_path}")
    doc = (inspect.getdoc(mod) or "").strip().splitlines()
    print(f"  module doc: {doc[0] if doc else '(none)'}")

    print("\n  -- functions / classes --")
    for attr, obj in sorted(vars(mod).items()):
        if attr.startswith("_") or getattr(obj, "__module__", None) != mod.__name__:
            continue
        if inspect.isfunction(obj) or inspect.isclass(obj):
            try:
                sig = str(inspect.signature(obj))
            except (TypeError, ValueError):
                sig = "(?)"
            first = (inspect.getdoc(obj) or "").strip().splitlines()
            print(f"  {attr}{sig}")
            print(f"      doc: {first[0] if first else '(none)'}")

    print("\n  -- constants (colours, dicts, lists, strings, numbers) --")
    for attr, obj in sorted(vars(mod).items()):
        if attr.startswith("_") or isinstance(obj, (types.ModuleType, types.FunctionType, type)):
            continue
        if isinstance(obj, (dict, list, tuple, str, int, float, bool)):
            print(f"  {attr} = {repr(obj)[:220]}")

# ---- Section: plot_style source (truncated) ----
print("\n" + "=" * 78)
print(f"SOURCE: {PKG}.plot_style (first {MAX_SOURCE_LINES} lines)")
print("=" * 78)
try:
    style_mod = importlib.import_module(f"{PKG}.plot_style")
    lines = inspect.getsource(style_mod).splitlines()
    print("\n".join(lines[:MAX_SOURCE_LINES]))
    if len(lines) > MAX_SOURCE_LINES:
        print(f"... ({len(lines) - MAX_SOURCE_LINES} more lines not shown)")
except Exception as exc:  # noqa: BLE001
    print(f"  could not read source: {type(exc).__name__}: {exc}")

# ---- Section: How earlier notebooks call them (06_* is skipped on purpose) ----
print("\n" + "=" * 78)
print("USAGE in earlier notebooks (lines mentioning plot_style / plots.)")
print("=" * 78)
shown = 0
for nb_path in sorted((config.ROOT / "notebooks").glob("*.ipynb")):
    if nb_path.name.startswith("06_") or nb_path.stat().st_size == 0 or nb_path.name.startswith("10_"):
        continue
    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8"))
    except Exception:  # noqa: BLE001, S112
        continue
    seen = set()
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        for line in "".join(cell.get("source", [])).splitlines():
            text = line.strip()
            if ("plot_style" in text or "plots." in text) and text not in seen and shown < MAX_USAGE_LINES:
                seen.add(text)
                print(f"  [{nb_path.name[:22]}] {text[:150]}")
                shown += 1
print("\n(read-only cell: no [PASS]/[FAIL] lines, only source/info dumps)")

MODULE: digikala_review_transformer.plot_style
  file: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/src/digikala_review_transformer/plot_style.py
  module doc: Shared 'Aurora Glass' chart style (no purple): canvas, glass tiles, fonts, Persian text helper, safe save.

  -- functions / classes --
  add_tile(fig, left, bottom, width, height, tile_title=None, pad_left=0.05, pad_right=0.025, pad_bottom=0.09, pad_top=None)
      doc: Glass tile (white alpha 0.09, border alpha 0.28) and a transparent axes inside it.
  fa(text)
      doc: Persian text for matplotlib. Diagnosis (matplotlib 3.11.2, Vazirmatn) showed raw text is drawn
  glass_legend(ax, handles=None, labels=None, **legend_kwargs)
      doc: Legend with a dark translucent frame; place it outside the data area via bbox_to_anchor.
  init_style(font_dir)
      doc: Register Vazirmatn (and Poppins if present), set rcParams. Call once per notebook session.
  make_background(width=1600, height=900, see

# Stage 1 / Cell 2: Learning Curves of B and C (val_mean, rec, band)

In [29]:
# ============================================================================
# Stage 1 / Cell 2: Learning curves of B and C
# Reads only RUNS[...]["history"] built by cell 1 (no STATE here).
# Layout is MEASURED from the rendered figure, not guessed.
# ============================================================================

# ---- Section: Imports and guard ----
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter

from digikala_review_transformer import config, plot_style

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")

n_fail = 0   # reset the failure counter of the shared check() helper

# ---- Section: Constants ----
CURVE_METRICS = [
    ("val_mean_macro_f1", "Mean of both heads"),
    ("val_macro_f1_rec", "rec head"),
    ("val_macro_f1_band", "band head"),
]
FIGURE_NAME = "01_stage1_learning_curves.png"
FONT_DIR = config.ROOT / "assets" / "fonts"

TILE_MARGIN = 0.035       # thin margin between the card edge and its content (fraction of card size)
LEGEND_GAP = 0.012        # gap between the x label and the legend (fraction of figure height)
FIT_ITERATIONS = 8        # layout refinement passes
FIT_TOLERANCE = 1e-4      # stop when the axes move less than this (figure fraction)
CHECK_TOLERANCE = 2e-3    # tolerance for the "inside the card" check (figure fraction)

Y_PAD_BOTTOM = 0.25       # free space below the lowest value (fraction of the data span), for labels
Y_PAD_TOP = 0.25          # free space above the highest value (fraction of the data span), for labels
LABEL_OFFSET = 15         # distance between a star and its value label (points)
NOTE_COLOR = "#9AA5B5"    # neutral colour for legend outline and neutral keys


# ---- Section: Tables (long format: one row per run and epoch) ----
frames = []
for run_name, run in RUNS.items():
    frame = run["history"].copy()
    frame.insert(0, "variant", run["variant"])
    frame.insert(0, "run", run_name)
    frames.append(frame)
curves = pd.concat(frames, ignore_index=True)
curves_path = TABLES_DIR / "stage1_learning_curves.csv"
curves.to_csv(curves_path, index=False)

summary_rows = []
for run_name, run in RUNS.items():
    history = run["history"]
    best = history[history["is_best"]].iloc[-1]
    last = history.iloc[-1]
    summary_rows.append({
        "run": run_name,
        "epochs_done": len(history),
        "best_epoch": int(best["epoch"]),
        "best_val_mean": round(float(best["val_mean_macro_f1"]), 6),
        "best_rec": round(float(best["val_macro_f1_rec"]), 6),
        "best_band": round(float(best["val_macro_f1_band"]), 6),
        "last_val_mean": round(float(last["val_mean_macro_f1"]), 6),
        "best_minus_last": round(float(best["val_mean_macro_f1"] - last["val_mean_macro_f1"]), 6),
        "mean_epoch_seconds": round(float(history["epoch_seconds"].mean()), 1),
    })
learning_summary = pd.DataFrame(summary_rows)
learning_summary.to_csv(TABLES_DIR / "stage1_learning_summary.csv", index=False)
print(learning_summary.to_string(index=False))

check(len(curves) == sum(len(r["history"]) for r in RUNS.values()),
      f"curves table saved ({len(curves)} rows): {curves_path.relative_to(config.ROOT)}",
      "curves table row count differs from the histories -> concat problem")


# ---- Section: Layout helpers (measure the rendered figure, then fit the axes into the card) ----
def tile_axes(fig, rect, title):
    """Create a glass tile with tiny paddings and return its axes (accepts an Axes or a tuple containing one).
    The real position of the axes is set later by fit_axes_to_tiles(), so these paddings are not final."""
    result = plot_style.add_tile(fig, *rect, tile_title=title,
                                 pad_left=0.004, pad_right=0.004, pad_bottom=0.004)
    candidates = result if isinstance(result, (tuple, list)) else (result,)
    found = [c for c in candidates if hasattr(c, "plot") and hasattr(c, "set_xlim")]
    if len(found) != 1:
        raise RuntimeError(f"add_tile returned {type(result).__name__} without exactly one Axes "
                           "-> send this message, I will adapt the call")
    return found[0]


def measure(fig, axes_list):
    """Render once and return, per axes: tight box of everything attached to it (ticks, labels, legend),
    its current position and the x-label box. All in figure fractions."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_fig = fig.transFigure.inverted()
    out = []
    for ax in axes_list:
        tight = ax.get_tightbbox(renderer).transformed(to_fig)
        xlabel = ax.xaxis.label.get_window_extent(renderer).transformed(to_fig)
        out.append((tight, ax.get_position(), xlabel))
    return out


def anchor_legend(ax, pos, xlabel, height):
    """Put the legend (top edge) LEGEND_GAP below the x label, centred under the axes."""
    drop = (pos.y0 - xlabel.y0) + LEGEND_GAP           # distance below the axes bottom, figure fraction
    ax.get_legend().set_bbox_to_anchor((0.5, -drop / height), transform=ax.transAxes)


def fit_axes_to_tiles(fig, items):
    """items: list of (ax, tile_rect). Moves every axes so that axes + ticks + labels + legend fill the card
    (minus a thin margin). The same left/right/bottom room is used for all cards, so the plots are identical in size."""
    axes_list = [ax for ax, _ in items]
    tops = [ax.get_position().y1 for ax in axes_list]    # keep the space reserved for the card title
    for _ in range(FIT_ITERATIONS):
        measured = measure(fig, axes_list)
        for ax, (_, pos, xlabel) in zip(axes_list, measured):   # make the legend position exact first
            anchor_legend(ax, pos, xlabel, pos.height)
        measured = measure(fig, axes_list)
        need_left = max(pos.x0 - tight.x0 for tight, pos, _ in measured)
        need_right = max(tight.x1 - pos.x1 for tight, pos, _ in measured)
        need_bottom = max(pos.y0 - tight.y0 for tight, pos, _ in measured)
        moved = 0.0
        for (ax, rect), top, (_, pos, xlabel) in zip(items, tops, measured):
            left, bottom, width, height = rect
            x0 = left + TILE_MARGIN * width + need_left
            x1 = left + width - TILE_MARGIN * width - need_right
            y0 = bottom + TILE_MARGIN * height + need_bottom
            new_pos = [x0, y0, x1 - x0, top - y0]
            if min(new_pos[2], new_pos[3]) <= 0:
                raise RuntimeError("card too small for its content -> shorten labels or legend text")
            moved = max(moved, abs(x0 - pos.x0), abs(y0 - pos.y0), abs(new_pos[2] - pos.width))
            ax.set_position(new_pos)
            anchor_legend(ax, pos, xlabel, new_pos[3])          # offsets below the axes are absolute: reuse them
        if moved < FIT_TOLERANCE:
            break


# ---- Section: Label and legend helpers ----
def best_row(history):
    """Row of the best epoch of one run."""
    return history[history["is_best"]].iloc[-1]


def label_best_points(ax, column):
    """Value label next to the star of each run, as a coloured badge.
    The run with the higher value gets its label above the star, the next one below it,
    so two labels never sit on top of each other and never hide under a curve."""
    entries = []
    for run in RUNS.values():
        best = best_row(run["history"])
        entries.append((float(best[column]), float(best["epoch"]), VARIANT_COLORS[run["variant"]]))
    entries.sort(key=lambda e: e[0], reverse=True)

    for rank, (value, epoch, color) in enumerate(entries):
        above = rank % 2 == 0
        ax.annotate(
            f"{value:.4f}", (epoch, value),
            textcoords="offset points",
            xytext=(0, LABEL_OFFSET if above else -LABEL_OFFSET),
            ha="center", va="bottom" if above else "top",
            color="white", fontsize=10, fontweight="bold", zorder=8,
            bbox={"boxstyle": "round,pad=0.25,rounding_size=0.6",
                  "facecolor": color, "edgecolor": "none", "alpha": 0.95},
        )


def add_legend(ax, idx, column, text_color):
    """A different legend per card, outline only (no filled background)."""
    kind = idx % 3
    if kind == 0:
        # Card 1: which line is which run
        title = "Runs"
        handles = [Line2D([], [], color=VARIANT_COLORS[run["variant"]], linewidth=2.4, marker="o",
                          label=f"{run['variant']}: {'no weights' if run['variant'] == 'B' else 'Step-2 weights'}")
                   for run in RUNS.values()]
    elif kind == 1:
        # Card 2: what the markers mean
        title = "Markers"
        handles = [
            Line2D([], [], color=NOTE_COLOR, linewidth=2.0, marker="o", markersize=7,
                   label="validation score per epoch"),
            Line2D([], [], color=NOTE_COLOR, linestyle="", marker="*", markersize=15,
                   markeredgecolor="white", markeredgewidth=0.8, label="best epoch (badge = value)"),
        ]
    else:
        # Card 3: the numbers of the best epoch of every run
        title = "Best epoch of each run"
        handles = []
        for run in RUNS.values():
            best = best_row(run["history"])
            handles.append(Line2D([], [], color=VARIANT_COLORS[run["variant"]], linestyle="",
                                  marker="s", markersize=9,
                                  label=f"{run['variant']}: epoch {int(best['epoch'])}  ->  {best[column]:.4f}"))

    leg = ax.legend(handles=handles, title=title, loc="upper center", bbox_to_anchor=(0.5, -0.2),
                    ncol=1, frameon=True, fancybox=True,
                    facecolor="none", edgecolor=to_rgba(NOTE_COLOR, 0.7),
                    fontsize=9.5, title_fontsize=9.5,
                    handlelength=1.8, borderpad=0.7, labelspacing=0.5)
    leg.get_frame().set_linewidth(1.0)
    leg.get_title().set_color(text_color)
    for text in leg.get_texts():
        text.set_color(text_color)


# ---- Section: Figure ----
plot_style.init_style(FONT_DIR)                       # idempotent
fig = plot_style.new_canvas(
    title="Stage 1: learning curves of B and C",
    subtitle="Validation Macro F1 per epoch (corrected labels). Star = best epoch of each run.",
    footer=f"{MODE_TAG}. Single seed; no test data is used here.",
)
rects = plot_style.split_row(len(CURVE_METRICS))      # equal widths
max_epoch = int(curves["epoch"].max())
tile_items = []

for idx, (rect, (column, tile_title)) in enumerate(zip(rects, CURVE_METRICS)):
    ax = tile_axes(fig, rect, tile_title)
    plot_style.style_axes(ax)

    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Macro F1", fontsize=12)
    text_color = ax.yaxis.label.get_color()           # follows the theme, no hard-coded colour

    for run_name, run in RUNS.items():
        history = run["history"]
        color = VARIANT_COLORS[run["variant"]]
        ax.plot(history["epoch"], history[column], color=color, linewidth=2.4,
                marker="o", markersize=7)
        best = best_row(history)
        ax.plot([best["epoch"]], [best[column]], linestyle="", marker="*", markersize=20,
                color=color, markeredgecolor="white", markeredgewidth=1.0, zorder=5)

    label_best_points(ax, column)

    values = curves[column].to_numpy()
    span = max(values.max() - values.min(), 1e-3)
    ax.set_ylim(values.min() - Y_PAD_BOTTOM * span, values.max() + Y_PAD_TOP * span)
    ax.set_xlim(0.6, max_epoch + 0.4)
    ax.set_xticks(range(1, max_epoch + 1))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))   # same tick-label width in every card

    add_legend(ax, idx, column, text_color)           # exact position is set by the fit
    tile_items.append((ax, rect))

fit_axes_to_tiles(fig, tile_items)

# ---- Section: Layout check (content must be inside its card, margins reported) ----
final = measure(fig, [ax for ax, _ in tile_items])
for index, ((ax, rect), (tight, pos, _)) in enumerate(zip(tile_items, final), start=1):
    left, bottom, width, height = rect
    margin_left = (tight.x0 - left) / width
    margin_right = (left + width - tight.x1) / width
    margin_bottom = (tight.y0 - bottom) / height
    inside = min(margin_left * width, margin_right * width, margin_bottom * height) >= -CHECK_TOLERANCE
    check(inside,
          f"card {index}: content inside the card (margins L {margin_left:.1%}, R {margin_right:.1%}, "
          f"B {margin_bottom:.1%} of card size)",
          f"card {index}: content leaves the card (margins L {margin_left:.1%}, R {margin_right:.1%}, "
          f"B {margin_bottom:.1%}) -> shorten the legend labels or lower FIT tolerance")
sizes = {(round(pos.width, 4), round(pos.height, 4)) for _, pos, _ in final}
check(len(sizes) == 1, "all three plot areas have the same size",
      f"plot areas differ in size: {sizes} -> fit_axes_to_tiles did not converge")

# ---- Section: Save (remove our own previous file first: same name, same folder) ----
fig_path = FIGS_DIR / FIGURE_NAME
if fig_path.exists():
    fig_path.unlink()
plot_style.save_figure(fig, fig_path)
check(fig_path.is_file() and fig_path.stat().st_size > 0,
      f"figure saved: {fig_path.relative_to(config.ROOT)}",
      "figure file missing or empty -> save_figure failed")
plt.show()
plt.close(fig)

# ---- Section: Result ----
print("=" * 78)
print("CELL 2 RESULT: ALL CHECKS PASSED" if n_fail == 0
      else f"CELL 2 RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above")

    run  epochs_done  best_epoch  best_val_mean  best_rec  best_band  last_val_mean  best_minus_last  mean_epoch_seconds
tooka_B            4           4       0.663465  0.720016   0.606914       0.663465              0.0              1813.8
tooka_C            4           4       0.663744  0.719830   0.607659       0.663744              0.0              2024.5
[PASS] curves table saved (8 rows): reports/transformer_stage1/full/tables/stage1_learning_curves.csv
[PASS] card 1: content inside the card (margins L 3.5%, R 3.5%, B 3.5% of card size)
[PASS] card 2: content inside the card (margins L 4.0%, R 3.5%, B 3.5% of card size)
[PASS] card 3: content inside the card (margins L 3.5%, R 3.5%, B 3.5% of card size)
[PASS] all three plot areas have the same size
[PASS] figure saved: reports/transformer_stage1/full/figures/01_stage1_learning_curves.png
CELL 2 RESULT: ALL CHECKS PASSED


# Stage 1 / Cell 2b: Figure layout toolkit (any aspect ratio, measured, legend-free)

In [20]:
# ============================================================================
# Stage 1 / Cell 2b: Figure layout toolkit (any aspect ratio, measured, no legend needed)
# Run after cell 2 and before cells 3b / 6.
# Defines: make_canvas, content_area, grid_rects, fit_cards, solve_figure_height,
#          layout_check (fixed, replaces the broken one), check_saved_image.
# ============================================================================

# ---- Section: Imports and guards ----
import matplotlib.pyplot as plt
import matplotlib.text as mtext
import numpy as np
from PIL import Image

from digikala_review_transformer import plot_style

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")

# ---- Section: Parameters (everything else is derived from the figure itself) ----
CARD_MARGIN = 0.025        # inner margin content -> card edge, as a fraction of the card's shorter side
FIT_PASSES = 12            # max measure-and-move passes in fit_cards
FIT_MOVE_TOL_IN = 0.002    # stop when no plot area moves more than this (inches)
CARD_CHECK_TOL_IN = 0.01   # tolerance of the layout check (inches)
SQUARE_TOL = 0.02          # allowed relative width/height difference of a "square" plot
HEIGHT_TOL = 0.004         # relative tolerance of the figure-height solver
HEIGHT_ROUNDS = 8


# ---- Section: Small helpers ----
def fig_size_in(fig):
    """Figure size in inches as floats (width, height)."""
    w_in, h_in = fig.get_size_inches()
    return float(w_in), float(h_in)


def _renderer(fig):
    """Draw once (applies aspect/layout) and return a renderer for measuring."""
    fig.canvas.draw()
    try:
        return fig.canvas.get_renderer()
    except AttributeError:
        return fig._get_renderer()


def _to_fig(fig, bbox):
    """Display-pixel bbox -> figure fractions."""
    return bbox.transformed(fig.transFigure.inverted())


def _visible_texts(fig):
    return [t for t in fig.findobj(mtext.Text) if t.get_visible() and t.get_text().strip()]


def _fraction_texts(fig):
    """Texts placed in figure fractions (directly or through a full-figure axes)."""
    found = []
    for t in _visible_texts(fig):
        transform, ax = t.get_transform(), t.axes
        on_figure = transform == fig.transFigure
        on_full_axes = (ax is not None and transform == ax.transAxes
                        and np.allclose(ax.get_position().bounds, (0, 0, 1, 1)))
        if on_figure or on_full_axes:
            found.append(t)
    return found


# ---- Section: Canvas with a free height (title/footer keep their distance in inches from the edges) ----
def make_canvas(title, subtitle, footer, height_in=None):
    """plot_style canvas; optionally a different height (portrait/landscape) with header/footer pinned to the edges."""
    fig = plot_style.new_canvas(title=title, subtitle=subtitle, footer=footer)
    if height_in is not None:
        old_w, old_h = fig_size_in(fig)
        movable = _fraction_texts(fig)
        fig.set_size_inches(old_w, height_in)
        for t in movable:
            x, y = t.get_position()
            t.set_position((x, 1 - (1 - y) * old_h / height_in) if y > 0.5 else (x, y * old_h / height_in))
    return fig


def content_area(fig):
    """Free region between the measured header texts and footer texts, plus the style's own card gap."""
    renderer = _renderer(fig)
    w_in, h_in = fig_size_in(fig)
    boxes = [_to_fig(fig, t.get_window_extent(renderer)) for t in _visible_texts(fig)]
    header = [b for b in boxes if b.y0 >= 0.5]
    footer = [b for b in boxes if b.y1 < 0.5]
    if not header or not footer:
        raise RuntimeError("could not find the canvas title/footer texts -> send the source of plot_style.new_canvas")
    ref = plot_style.split_row(2)                       # the style's own side margins and card gap
    left, right = ref[0][0], ref[1][0] + ref[1][2]
    gap_in = (ref[1][0] - (ref[0][0] + ref[0][2])) * w_in
    top = min(b.y0 for b in header) - gap_in / h_in
    bottom = max(b.y1 for b in footer) + gap_in / h_in
    return {"rect": (left, bottom, right - left, top - bottom), "gap_in": gap_in}


def grid_rects(fig, area, n_rows, n_cols):
    """Equal cards in a grid, row-major from the top row; the gap is the same in inches both ways."""
    w_in, h_in = fig_size_in(fig)
    left, bottom, width, height = area["rect"]
    gap_x, gap_y = area["gap_in"] / w_in, area["gap_in"] / h_in
    card_w = (width - (n_cols - 1) * gap_x) / n_cols
    card_h = (height - (n_rows - 1) * gap_y) / n_rows
    rects = []
    for row in range(n_rows):
        top = bottom + height - row * (card_h + gap_y)
        for col in range(n_cols):
            rects.append((left + col * (card_w + gap_x), top - card_h, card_w, card_h))
    return rects


# ---- Section: Measure and fit ----
def card_measure(fig, axes_list):
    """For each axes: (tight bbox incl. every label/text/legend, active position, original position)."""
    renderer = _renderer(fig)
    return [(_to_fig(fig, ax.get_tightbbox(renderer)), ax.get_position(), ax.get_position(original=True))
            for ax in axes_list]


def fit_cards(fig, items, margin=CARD_MARGIN, passes=FIT_PASSES):
    """Place every plot area so that its content (ticks, labels, texts) touches the same thin margin of its card.
    All plot areas get the same size (the largest needed label space is used for every card)."""
    w_in, h_in = fig_size_in(fig)
    axes_list = [ax for ax, _ in items]
    top_gaps = [rect[1] + rect[3] - ax.get_position(original=True).y1 for ax, rect in items]   # card-title zone
    moved = 0.0
    for _ in range(passes):
        measured = card_measure(fig, axes_list)
        need_l = max(0.0, max(pos.x0 - tight.x0 for tight, pos, _ in measured))
        need_r = max(0.0, max(tight.x1 - pos.x1 for tight, pos, _ in measured))
        need_b = max(0.0, max(pos.y0 - tight.y0 for tight, pos, _ in measured))
        moved = 0.0
        for (ax, rect), top_gap in zip(items, top_gaps):
            left, bottom, width, height = rect
            m_in = margin * min(width * w_in, height * h_in)
            mx, my = m_in / w_in, m_in / h_in
            x0, y0 = left + mx + need_l, bottom + my + need_b
            x1, y1 = left + width - mx - need_r, bottom + height - top_gap
            old = ax.get_position(original=True)
            moved = max(moved, abs(x0 - old.x0) * w_in, abs(y0 - old.y0) * h_in,
                        abs((x1 - x0) - old.width) * w_in, abs((y1 - y0) - old.height) * h_in)
            ax.set_position([x0, y0, x1 - x0, y1 - y0])
        if moved < FIT_MOVE_TOL_IN:
            break
    return moved


def solve_figure_height(render, rows):
    """For square plots: find the figure height at which the cards have exactly the shape of their content.
    render(height_in or None) -> (fig, items). Returns the final (fig, items)."""
    height = None
    for _ in range(HEIGHT_ROUNDS):
        fig, items = render(height)
        w_in, h_in = fig_size_in(fig)
        origs = [ax.get_position(original=True) for ax, _ in items]
        excess = float(np.mean([o.width * w_in - o.height * h_in for o in origs]))   # width minus height of the free box
        new_height = max(h_in + rows * excess, 0.5 * h_in)
        if abs(new_height - h_in) / h_in < HEIGHT_TOL:
            return fig, items
        plt.close(fig)
        height = new_height
    return render(height)


# ---- Section: Checks ----
def layout_check(fig, items, square=False) -> None:
    """Content must stay inside its card; all plot areas equal; optionally every plot area square."""
    w_in, h_in = fig_size_in(fig)
    measured = card_measure(fig, [ax for ax, _ in items])
    for index, ((ax, rect), (tight, pos, _)) in enumerate(zip(items, measured), start=1):
        left, bottom, width, height = rect
        m_l, m_r = (tight.x0 - left) * w_in, (left + width - tight.x1) * w_in
        m_b, m_t = (tight.y0 - bottom) * h_in, (bottom + height - tight.y1) * h_in
        text = f"margins L {m_l:.2f}, R {m_r:.2f}, B {m_b:.2f}, T {m_t:.2f} in"
        check(min(m_l, m_r, m_b, m_t) >= -CARD_CHECK_TOL_IN,
              f"card {index}: content inside the card ({text})",
              f"card {index}: content leaves the card ({text}) -> shorten the labels")
    sizes = {(round(pos.width * w_in, 2), round(pos.height * h_in, 2)) for _, pos, _ in measured}
    check(len(sizes) == 1, "all plot areas have the same size",
          f"plot areas differ in size: {sizes} -> fit_cards did not converge")
    if square:
        worst = max(abs(pos.width * w_in - pos.height * h_in) / (pos.width * w_in) for _, pos, _ in measured)
        check(worst <= SQUARE_TOL, f"every plot area is square (worst deviation {worst:.2%})",
              f"plot areas are not square (worst deviation {worst:.2%}) -> solve_figure_height did not converge")


def check_saved_image(fig, path) -> None:
    """The PNG on disk must have the same aspect as the figure (save_figure must not force a fixed size)."""
    with Image.open(path) as image:
        w_px, h_px = image.size
    w_in, h_in = fig_size_in(fig)
    check(abs(w_px / h_px - w_in / h_in) / (w_in / h_in) < 0.01,
          f"saved image {w_px}x{h_px} px matches the figure aspect",
          f"saved image is {w_px}x{h_px} px but the figure aspect is {w_in / h_in:.3f} "
          "-> save_figure forces a fixed size; send its source")

# ---- Section: Saving figures whose aspect differs from the style's fixed canvas ----
def save_figure_any_size(fig, path):
    """plot_style.save_figure for 16:9 figures; same width in pixels (same dpi) for any other aspect."""
    from pathlib import Path

    w_in, h_in = fig_size_in(fig)
    expected_w, expected_h = plot_style.EXPECTED_PIXELS
    if abs(w_in / h_in - expected_w / expected_h) < 1e-4:
        return plot_style.save_figure(fig, path)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=expected_w / w_in, facecolor=fig.get_facecolor())   # no bbox_inches="tight": it would change the size
    with Image.open(path) as image:
        width_px, height_px = image.size
    if width_px != expected_w:
        raise ValueError(f"unexpected image width {width_px} vs {expected_w}")
    return {"path": str(path), "pixels": (width_px, height_px), "size_mb": round(path.stat().st_size / 1e6, 2)}

print("Layout toolkit ready: make_canvas, content_area, grid_rects, fit_cards, solve_figure_height, "
      "layout_check, save_figure_any_size")

Layout toolkit ready: make_canvas, content_area, grid_rects, fit_cards, solve_figure_height, layout_check, save_figure_any_size


# Stage 1 / Cell 3: Transformer vs TF-IDF (both label views, paired bootstrap)

### Cell 3a: Analyzed and saved tables

In [16]:

# ============================================================================
# Stage 1 / Cell 3a: Transformer vs TF-IDF vs Dummy baseline - ANALYSIS ONLY
# Reads probability parquets, pack labels and baseline prediction files.
# Writes tables to TABLES_DIR (reports/transformer_stage1/dev/tables).
# Needs cell 1 (RUNS, paths, parameters). Does not need cell 2.
# ============================================================================

# ---- Section: Imports and guards ----
import time

import numpy as np
import pandas as pd

T0 = time.time()
from digikala_review_transformer import config

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")

n_fail = 0   # reset the failure counter of the shared check() helper

# ---- Section: Constants ----
PROB_FILES = {"validation": "val_probs_best.parquet", "test": "test_probs.parquet"}
SPLITS = tuple(PROB_FILES)
HEAD_COLUMNS = {
    "rec": ["p_recommended", "p_not_recommended", "p_no_idea"],
    "band": ["p_low", "p_medium", "p_high"],
}
BASELINE_LABEL_COLUMNS = {"corrected": "y_true_corr", "raw": "y_true_raw"}
CHECK_TOL = 1e-5
TABLE_FILE = "stage1_vs_tfidf.csv"
PCT_FILE = "stage1_vs_tfidf_pct.csv"

# Dummy baseline
DUMMY_PATH = config.ROOT / "reports" / "baseline_tfidf" / "tables" / "dummy_baseline.csv"
DUMMY_VARIANT = "B"   # the file has A and B; they must be identical (checked below)


# ---- Section: Helpers ----
def confusion_to_f1(conf: np.ndarray) -> np.ndarray:
    """Macro F1 from confusion matrices shaped (..., true, pred). Classes with no support/predictions count as 0."""
    conf = np.asarray(conf, dtype=float)
    tp = np.diagonal(conf, axis1=-2, axis2=-1)
    denom = conf.sum(axis=-1) + conf.sum(axis=-2)
    f1 = np.divide(2.0 * tp, denom, out=np.zeros_like(tp), where=denom > 0)
    return f1.mean(axis=-1)


def load_pack_labels(split: str) -> pd.DataFrame:
    frame = pd.read_parquet(
        PACK_DIR / f"{split}.parquet",
        columns=["id", "y_rec", "y_band", "y_rec_raw", "y_band_raw"]
    )
    frame["id"] = frame["id"].astype(str)
    return frame.set_index("id")


def load_baseline(head: str, split: str) -> pd.DataFrame:
    frame = pd.read_parquet(
        BASELINE_DIR / f"{BASELINE_IDS[head]}__{split}.parquet",
        columns=["id", "y_true_raw", "y_true_corr", "pred"]
    )
    frame["id"] = frame["id"].astype(str)
    return frame.drop_duplicates("id").set_index("id")


def rel_pct(new: float, ref: float) -> float:
    """Relative change in percent: (new - ref) / ref * 100."""
    return 100.0 * (new - ref) / ref if ref else float("nan")


# ---- Section: Load labels and baseline predictions ----
pack_labels = {split: load_pack_labels(split) for split in PROB_FILES}
baselines = {
    (head, split): load_baseline(head, split)
    for head in HEADS
    for split in PROB_FILES
}
rng = np.random.default_rng(BOOT_SEED)

# ---- Section: Load the Dummy baseline (majority class, saved by the baseline phase) ----
dummy_all = pd.read_csv(DUMMY_PATH)

_idx = ["split", "head", "label_view"]

dummy = (
    dummy_all[dummy_all["variant"] == DUMMY_VARIANT]
    .set_index(_idx)
    .sort_index()
)

_dummy_a = (
    dummy_all[dummy_all["variant"] == "A"]
    .set_index(_idx)["macro_f1"]
    .sort_index()
)

check(
    len(dummy) == len(SPLITS) * len(HEADS) * len(VIEWS)
    and dummy.index.is_unique
    and float((_dummy_a - dummy["macro_f1"]).abs().max()) < 1e-12,
    f"dummy_baseline.csv: {len(dummy)} rows for variant {DUMMY_VARIANT}, identical to variant A",
    "dummy_baseline.csv rows missing/duplicated or A differs from B -> send me this line"
)


# ---- Section: Metrics per run / split / head / view ----
rows, agreements, history_gaps, dummy_n_gaps = [], [], [], []
reference_ids = {}

for split in SPLITS:
    labels_all = pack_labels[split]

    for run_name, run in RUNS.items():
        probs = pd.read_parquet(run["dir"] / PROB_FILES[split])
        ids = probs["id"].astype(str).to_numpy()
        assert len(ids) == len(probs), "[FAIL] ids and probs lengths differ -> the 'ids = ...' line is missing or stale"
        print(f"[t={time.time() - T0:.0f}s] {split}/{run_name}: probs read ({len(ids):,} rows) "
              f"file={(run['dir'] / PROB_FILES[split]).name} probs_shape={probs.shape}", flush=True)

        check(
            len(set(ids)) == len(ids)
            and bool(pd.Index(ids).isin(labels_all.index).all()),
            f"{run_name}/{split}: ids unique and all in the pack split (n={len(ids):,})",
            f"{run_name}/{split}: duplicate ids or ids missing from the pack -> wrong file or bug in predict.py"
        )

        if split not in reference_ids:
            reference_ids[split] = ids
        else:
            check(
                np.array_equal(ids, reference_ids[split]),
                f"{run_name}/{split}: same ids in the same order as the first run",
                f"{run_name}/{split}: ids differ from the first run -> B and C were evaluated on different rows"
            )

        labels = labels_all.reindex(ids)
        print(f"[t={time.time() - T0:.0f}s] {split}/{run_name}: labels reindexed", flush=True)

        for head in HEADS:
            pred_t = (
                probs[HEAD_COLUMNS[head]]
                .to_numpy()
                .argmax(axis=1)
                .astype(np.int64)
            )

            base = baselines[(head, split)].reindex(ids)
            has_base = base["pred"].notna().to_numpy()
            pred_b = base["pred"].fillna(-1).to_numpy().astype(np.int64)

            for view in VIEWS:
                pack_col = f"y_{head}" if view == "corrected" else f"y_{head}_raw"
                y_series = labels[pack_col].fillna(-1)
                valid = y_series.isin([0, 1, 2]).to_numpy()
                y = y_series.to_numpy().astype(np.int64)

                # Labels in the pack must equal the labels stored with baseline predictions
                base_y = base[BASELINE_LABEL_COLUMNS[view]]
                both = valid & has_base & base_y.isin([0, 1, 2]).to_numpy()

                agree = (
                    float((base_y.to_numpy()[both] == y[both]).mean())
                    if both.any() else 1.0
                )
                agreements.append((split, head, view, agree))

                # Transformer alone, on every row with a valid label
                f1_own = float(confusion_to_f1(
                    np.bincount(
                        y[valid] * 3 + pred_t[valid],
                        minlength=9
                    ).reshape(3, 3)
                ))

                if split == "validation" and view == "corrected":
                    best = run["history"][run["history"]["is_best"]].iloc[-1]
                    history_gaps.append(
                        abs(f1_own - float(best[f"val_macro_f1_{head}"]))
                    )

                # Paired comparison on the rows both models predict
                common = valid & has_base
                n_common = int(common.sum())

                joint = np.bincount(
                    y[common] * 9 + pred_t[common] * 3 + pred_b[common],
                    minlength=27
                ).reshape(3, 3, 3).astype(float)

                # [true, pred_transformer, pred_tfidf]
                f1_t = float(confusion_to_f1(joint.sum(axis=2)))
                f1_b = float(confusion_to_f1(joint.sum(axis=1)))

                draws = rng.multinomial(
                    n_common,
                    joint.ravel() / n_common,
                    size=N_BOOT
                )
                draws = draws.reshape(N_BOOT, 3, 3, 3).astype(float)

                diffs = (
                    confusion_to_f1(draws.sum(axis=3))
                    - confusion_to_f1(draws.sum(axis=2))
                )

                lo, hi = np.percentile(diffs, [2.5, 97.5])

                # Dummy baseline: same split, head and label view
                dummy_row = dummy.loc[(split, head, view)]

                if int(dummy_row["n"]) != n_common:
                    dummy_n_gaps.append(
                        (run_name, split, head, view,
                         int(dummy_row["n"]), n_common)
                    )

                rows.append({
                    "run": run_name,
                    "variant": run["variant"],
                    "split": split,
                    "head": head,
                    "view": view,
                    "n_own": int(valid.sum()),
                    "f1_own": f1_own,
                    "n_common": n_common,
                    "f1_transformer": f1_t,
                    "f1_tfidf": f1_b,
                    "f1_dummy": float(dummy_row["macro_f1"]),
                    "diff": f1_t - f1_b,
                    "ci_lo": float(lo),
                    "ci_hi": float(hi),
                    "ci_excludes_0": bool(lo > 0 or hi < 0),
                })

                print(f"[t={time.time() - T0:.0f}s] {split}/{run_name}/{head}/{view}: done", flush=True)


# ---- Section: Checks on the numbers ----
worst = min(agreements, key=lambda item: item[3])

check(
    worst[3] == 1.0,
    "pack labels equal the baseline file labels on shared rows (all splits/heads/views)",
    f"label agreement {worst[3]:.4f} at {worst[:3]} -> the baseline and the pack use different label views"
)

check(
    max(history_gaps) < CHECK_TOL,
    f"validation corrected F1 recomputed here equals history.csv (max gap {max(history_gaps):.2e})",
    f"gap to history.csv = {max(history_gaps):.2e} -> F1 formula or ids/labels differ from training evaluation"
)

check(
    not dummy_n_gaps,
    "Dummy n equals n_common on every split/head/view (Dummy is on the same rows as the models)",
    f"Dummy n differs from n_common, e.g. {dummy_n_gaps[:3]} -> send me this line; Dummy must be recomputed on shared rows"
)


# ---- Section: Save table 1 (full comparison with bootstrap CI) ----
table = (
    pd.DataFrame(rows)
    .sort_values(["split", "head", "view", "run"])
    .reset_index(drop=True)
)

float_cols = [
    "f1_own", "f1_transformer", "f1_tfidf", "f1_dummy",
    "diff", "ci_lo", "ci_hi"
]

table[float_cols] = table[float_cols].round(6)

table_path = TABLES_DIR / TABLE_FILE
table.to_csv(table_path, index=False)

check(
    table_path.is_file()
    and len(table) == len(RUNS) * len(SPLITS) * len(HEADS) * len(VIEWS),
    f"comparison table saved ({len(table)} rows): {table_path.relative_to(config.ROOT)}",
    "comparison table has an unexpected number of rows -> loop problem"
)

# B and C must see the same TF-IDF numbers (same rows)
tfidf_spread = (
    table.groupby(["split", "head", "view"])["f1_tfidf"]
    .agg(lambda s: s.max() - s.min())
    .max()
)

check(
    tfidf_spread < 1e-9,
    "TF-IDF numbers are identical for B and C (same rows)",
    f"TF-IDF numbers differ between runs by {tfidf_spread:.2e} -> runs were evaluated on different rows"
)


# ---- Section: Save table 2 (chart table: macro F1 and relative changes in %) ----
run_names = list(RUNS)

model_names = (
    ["Dummy", "TF-IDF"]
    + [RUNS[name]["variant"] for name in run_names]
)

TFIDF_IDX, first_idx, last_idx = 1, 2, len(model_names) - 1

pct_rows = []

for split in SPLITS:
    for head in HEADS:
        f1 = {}

        for view in VIEWS:
            part = (
                table[
                    (table["split"] == split)
                    & (table["head"] == head)
                    & (table["view"] == view)
                ]
                .set_index("run")
            )

            f1[view] = [
                float(part.loc[run_names[0], "f1_dummy"]),
                float(part.loc[run_names[0], "f1_tfidf"]),
            ] + [
                float(part.loc[name, "f1_transformer"])
                for name in run_names
            ]

        for view in VIEWS:
            for i, model in enumerate(model_names):  # noqa: F402
                pct_rows.append({
                    "split": split,
                    "head": head,
                    "view": view,
                    "model_order": i,
                    "model": model,
                    "macro_f1": round(f1[view][i], 6),

                    # Transformer vs TF-IDF, same label view
                    "vs_tfidf_pct": (
                        np.nan
                        if i <= TFIDF_IDX
                        else round(
                            rel_pct(f1[view][i], f1[view][TFIDF_IDX]),
                            3
                        )
                    ),

                    # Corrected vs raw labels, same model
                    "corr_vs_raw_pct": round(
                        rel_pct(f1["corrected"][i], f1["raw"][i]),
                        3
                    ),

                    # Last run vs first run, same label view
                    "last_vs_first_pct": (
                        round(
                            rel_pct(
                                f1[view][last_idx],
                                f1[view][first_idx]
                            ),
                            3
                        )
                        if (i == last_idx and last_idx > first_idx)
                        else np.nan
                    ),
                })

pct_table = pd.DataFrame(pct_rows)

pct_path = TABLES_DIR / PCT_FILE
pct_table.to_csv(pct_path, index=False)

check(
    pct_path.is_file()
    and len(pct_table) == len(SPLITS) * len(HEADS) * len(VIEWS) * len(model_names),
    f"chart table saved ({len(pct_table)} rows): {pct_path.relative_to(config.ROOT)}",
    "chart table has an unexpected number of rows -> loop problem"
)


# ---- Section: Print ----
show_cols = [
    "run", "split", "head", "view", "n_common",
    "f1_transformer", "f1_tfidf", "f1_dummy",
    "diff", "ci_lo", "ci_hi", "ci_excludes_0"
]

print("\nMacro F1 on rows shared with TF-IDF (diff = transformer - TF-IDF, paired bootstrap 95% CI):")
print(table[show_cols].to_string(index=False))

# ---- Section: Result ----
print("=" * 78)
print(
    "CELL 3a RESULT: ALL CHECKS PASSED"
    if n_fail == 0
    else f"CELL 3a RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above"
)


[PASS] dummy_baseline.csv: 8 rows for variant B, identical to variant A
[t=1s] validation/tooka_B: probs read (922,794 rows) file=val_probs_best.parquet probs_shape=(922794, 7)
[PASS] tooka_B/validation: ids unique and all in the pack split (n=922,794)
[t=3s] validation/tooka_B: labels reindexed
[t=3s] validation/tooka_B/rec/raw: done
[t=3s] validation/tooka_B/rec/corrected: done
[t=3s] validation/tooka_B/band/raw: done
[t=3s] validation/tooka_B/band/corrected: done
[t=3s] validation/tooka_C: probs read (922,794 rows) file=val_probs_best.parquet probs_shape=(922794, 7)
[PASS] tooka_C/validation: ids unique and all in the pack split (n=922,794)
[PASS] tooka_C/validation: same ids in the same order as the first run
[t=6s] validation/tooka_C: labels reindexed
[t=6s] validation/tooka_C/rec/raw: done
[t=6s] validation/tooka_C/rec/corrected: done
[t=6s] validation/tooka_C/band/raw: done
[t=6s] validation/tooka_C/band/corrected: done
[t=6s] test/tooka_B: probs read (922,805 rows) file=test_pr


### Stage 1 / Cell 3b: Transformer vs TF-IDF vs Dummy baseline, figure only

In [28]:
# ============================================================================
# Stage 1 / Cell 3b: Transformer vs TF-IDF vs Dummy baseline - FIGURE ONLY
# Reads only the chart table written by cell 3a (no analysis here).
# Needs cell 1 (RUNS, VARIANT_COLORS, paths), cell 2 (tile_axes) and cell 2b (layout toolkit).
# ============================================================================

# ---- Section: Imports and guards ----
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FormatStrFormatter, MaxNLocator

from digikala_review_transformer import config, plot_style

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")

for _name in ("tile_axes", "make_canvas", "content_area", "grid_rects", "fit_cards",
              "layout_check", "check_saved_image"):
    if _name not in globals():
        raise RuntimeError(f"'{_name}' is missing -> run cells 2 and 2b first in this kernel")

n_fail = 0   # reset the failure counter of the shared check() helper


# ---- Section: Constants ----
CHART_SPLIT = "test"
PCT_FILE = "stage1_vs_tfidf_pct.csv"
FIGURE_NAME = "02_stage1_vs_tfidf.png"
FONT_DIR = config.ROOT / "assets" / "fonts"

BAR_WIDTH, BAR_GAP = 0.36, 0.06
VIEW_OFFSETS = {
    "raw": -(BAR_WIDTH + BAR_GAP) / 2,
    "corrected": (BAR_WIDTH + BAR_GAP) / 2
}
VIEW_SHORT = {"raw": "raw", "corrected": "corr."}

TFIDF_COLOR = "#8E99A8"
DUMMY_COLOR = "#5E6B7E"   # darker slate; change only if it is hard to tell apart from TF-IDF

UP_COLOR, DOWN_COLOR, NOTE_COLOR = "#2FA878", "#D9534F", "#9AA5B5"

ROW_Y = (0.95, 0.84, 0.73)       # rows of the percent pills (axes fraction), more spacing
BAR_TOP_FRACTION = 0.60          # the tallest bar reaches this fraction of the axes height
STAGGER = {"raw": +0.022, "corrected": -0.022}   # vertical offset to separate neighbouring pills
MIN_INSIDE_FRAC = 0.25           # bars shorter than this: value label goes above the bar

SUBTITLE = (
    f"{CHART_SPLIT.capitalize()} Macro F1 on shared rows  |  "
    "solid = corrected, hatched = raw  |  badges = % change"
)

FOOTER = (
    f"{MODE_TAG}. Rows shared by all models; test is never used for decisions. "
)


# ---- Section: Load the chart table written by cell 3a ----
pct_path = TABLES_DIR / PCT_FILE

if not pct_path.is_file():
    raise FileNotFoundError(f"{pct_path} not found -> run cell 3a first")

chart = pd.read_csv(pct_path)
chart = chart[chart["split"] == CHART_SPLIT]

check(
    len(chart) > 0,
    f"chart table loaded for split '{CHART_SPLIT}' ({len(chart)} rows): {pct_path.relative_to(config.ROOT)}",
    f"no rows for split '{CHART_SPLIT}' in {PCT_FILE} -> rerun cell 3a"
)

model_labels = list(
    chart.sort_values("model_order")["model"].unique()
)

check(
    model_labels[:2] == ["Dummy", "TF-IDF"],
    "model order is Dummy, TF-IDF, then the transformer runs",
    f"unexpected model order {model_labels} -> rerun cell 3a after the patch"
)

MODEL_COLORS = {
    "Dummy": DUMMY_COLOR,
    "TF-IDF": TFIDF_COLOR,
    **{name: VARIANT_COLORS[name] for name in model_labels[2:]}
}

first_variant, last_variant = model_labels[2], model_labels[-1]
last_idx, first_idx = len(model_labels) - 1, 2


# ---- Section: Helpers ----
def column(head: str, view: str, name: str) -> np.ndarray:
    """One column of the chart table for a head and view, ordered as the models on the x axis."""
    part = (
        chart[
            (chart["head"] == head)
            & (chart["view"] == view)
        ]
        .sort_values("model_order")
    )
    return part[name].to_numpy(dtype=float)


def pill(ax, x, row, text, value, dy=0.0):
    """Rounded badge in a fixed row (axes fraction), green if value >= 0 else red."""
    ax.text(
        x, ROW_Y[row] + dy, text,
        transform=ax.get_xaxis_transform(),
        ha="center", va="center",
        fontsize=8.5, fontweight="bold",
        color="white", zorder=7,
        bbox={
            "boxstyle": "round,pad=0.22,rounding_size=0.7",
            "facecolor": UP_COLOR if value >= 0 else DOWN_COLOR,
            "edgecolor": "none",
            "alpha": 0.93
        }
    )


def note(ax, x, row, text):
    """Muted row caption placed in an empty slot."""
    ax.text(
        x, ROW_Y[row], text,
        transform=ax.get_xaxis_transform(),
        ha="center", va="center",
        fontsize=9, fontstyle="italic",
        color=NOTE_COLOR, zorder=7
    )


def zoomed_axis(values):
    """Y axis from 0 (Dummy is far below the models); headroom above the bars is kept for the badges."""
    top = float(values.max())
    ticks = MaxNLocator(
        nbins=5,
        steps=[1, 2, 2.5, 5, 10]
    ).tick_values(0.0, top)

    y_hi = top / BAR_TOP_FRACTION
    shown = [
        float(t) for t in ticks
        if -1e-12 <= t <= top + 1e-9
    ]

    return 0.0, y_hi, shown, 2


def add_legend(ax, idx, text_color):
    """A different legend per tile, with no filled background (outline only)."""
    kind = idx % 3
    if kind == 0:
        title = "Label view"
        handles = [
            Patch(facecolor=NOTE_COLOR, edgecolor=NOTE_COLOR, label="corrected (solid)"),
            Patch(facecolor=to_rgba(NOTE_COLOR, 0.30), edgecolor=NOTE_COLOR,
                  hatch="////", label="raw (hatched)"),
        ]
        ncol = 2
    elif kind == 1:
        title = "Badge colour = % change"
        handles = [
            Line2D([0], [0], marker="s", ls="", ms=9, color=UP_COLOR, label="improvement"),
            Line2D([0], [0], marker="s", ls="", ms=9, color=DOWN_COLOR, label="drop"),
        ]
        ncol = 2
    else:
        title = "Models  (B: no weights, C: Step-2 weights)"
        handles = [Patch(facecolor=MODEL_COLORS[m], edgecolor=MODEL_COLORS[m], label=m)
                   for m in model_labels]
        ncol = min(len(model_labels), 4)

    leg = ax.legend(
        handles=handles, title=title,
        loc="upper center", bbox_to_anchor=(0.5, -0.20),
        ncol=ncol, frameon=True, fancybox=True,
        facecolor="none", edgecolor=to_rgba(NOTE_COLOR, 0.7),
        fontsize=9, title_fontsize=9,
        handlelength=1.6, columnspacing=1.4, borderpad=0.7
    )
    leg.get_frame().set_linewidth(1.0)
    leg.get_title().set_color(text_color)
    for t in leg.get_texts():
        t.set_color(text_color)


# ---- Section: Figure (grouped bars: colour = model, fill = label view, percent badges on top) ----
plot_style.init_style(FONT_DIR)                       # idempotent

fig = make_canvas(
    title="Stage 1: transformer vs TF-IDF baseline",
    subtitle=SUBTITLE,
    footer=FOOTER
)

rects = grid_rects(fig, content_area(fig), 1, len(HEADS))
tile_items = []

for idx, (head, rect) in enumerate(zip(HEADS, rects)):
    ax = tile_axes(fig, rect, f"{head} head")
    plot_style.style_axes(ax)

    ax.set_ylabel("Macro F1", fontsize=12)
    text_color = ax.yaxis.label.get_color()      # follows the theme, no hard-coded colour

    series = {view: column(head, view, "macro_f1") for view in VIEWS}
    y_lo, y_hi, y_ticks, decimals = zoomed_axis(np.concatenate(list(series.values())))
    span = y_hi - y_lo

    # Soft separators between the model groups
    for boundary in np.arange(len(model_labels) - 1) + 0.5:
        ax.axvline(boundary, color=NOTE_COLOR, alpha=0.18, linewidth=1.0,
                   linestyle=(0, (2, 4)), zorder=1)

    # Bars + value labels (inside if the bar is tall, above the bar if it is short)
    for view in VIEWS:
        solid = view == "corrected"
        for i, value in enumerate(series[view]):
            color = MODEL_COLORS[model_labels[i]]
            x = i + VIEW_OFFSETS[view]

            ax.bar(x, value - y_lo, bottom=y_lo, width=BAR_WIDTH,
                   color=color if solid else to_rgba(color, 0.30),
                   edgecolor=color, linewidth=1.4,
                   hatch=None if solid else "////", zorder=3)

            inside = (value - y_lo) / span >= MIN_INSIDE_FRAC
            if inside:
                ax.text(x, value - 0.015 * span, f"{value:.3f}",
                        rotation=90, ha="center", va="top",
                        color="white", fontsize=10, fontweight="bold", zorder=6,
                        path_effects=[pe.withStroke(linewidth=2.4, foreground=(0, 0, 0, 0.45))])
            else:
                ax.text(x, value + 0.015 * span, f"{value:.3f}",
                        rotation=90, ha="center", va="bottom",
                        color=text_color, fontsize=9.5, fontweight="bold", zorder=6)

    # Row 1: corrected vs raw (one pill per model, centred on the group)
    gaps = column(head, "corrected", "corr_vs_raw_pct")
    for i, gap in enumerate(gaps):
        pill(ax, i, 0, f"raw→corr. {gap:+.1f}%", gap)

    # Row 2: each transformer vs TF-IDF (raw / corrected staggered vertically)
    note(ax, 1, 1, "vs TF-IDF")
    for view in VIEWS:
        changes = column(head, view, "vs_tfidf_pct")
        for i in range(first_idx, len(model_labels)):
            pill(ax, i + VIEW_OFFSETS[view], 1, f"{changes[i]:+.1f}%",
                 changes[i], dy=STAGGER[view])

    # Row 3: last run vs first run
    if last_idx > first_idx:
        note(ax, first_idx, 2, f"{last_variant} vs {first_variant}")
        for view in VIEWS:
            change = column(head, view, "last_vs_first_pct")[last_idx]
            pill(ax, last_idx + VIEW_OFFSETS[view], 2, f"{change:+.1f}%",
                 change, dy=STAGGER[view])

    # Axes
    ax.set_ylim(y_lo, y_hi)
    ax.set_yticks(y_ticks)
    ax.yaxis.set_major_formatter(FormatStrFormatter(f"%.{decimals}f"))
    ax.set_xlim(-0.55, len(model_labels) - 0.45)

    ax.set_xticks(range(len(model_labels)))
    ax.set_xticklabels(model_labels, fontsize=12, fontweight="bold")
    for label, name in zip(ax.get_xticklabels(), model_labels):
        label.set_color(MODEL_COLORS[name])
    ax.tick_params(axis="x", which="major", length=0, pad=22)

    minor = sorted((i + VIEW_OFFSETS[v], VIEW_SHORT[v])
                   for i in range(len(model_labels)) for v in VIEWS)
    ax.set_xticks([x for x, _ in minor], minor=True)
    ax.set_xticklabels([l for _, l in minor], minor=True)
    ax.tick_params(axis="x", which="minor", length=0, pad=4, labelsize=9)

    ax.set_xlabel("")                      # the old shared label is replaced by the per-tile legend
    add_legend(ax, idx, text_color)

    tile_items.append((ax, rect))


# ---- Section: Layout checks ----
fit_cards(fig, tile_items)
layout_check(fig, tile_items)


# ---- Section: Save (remove our own previous file first: same name, same folder) ----
fig_path = FIGS_DIR / FIGURE_NAME

if fig_path.exists():
    fig_path.unlink()

plot_style.save_figure(fig, fig_path)

check(
    fig_path.is_file() and fig_path.stat().st_size > 0,
    f"figure saved: {fig_path.relative_to(config.ROOT)}",
    "figure file missing or empty -> save_figure failed"
)

check_saved_image(fig, fig_path)

plt.show()
plt.close(fig)


# ---- Section: Result ----
print("=" * 78)

print(
    "CELL 3b RESULT: ALL CHECKS PASSED"
    if n_fail == 0
    else f"CELL 3b RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above"
)

[PASS] chart table loaded for split 'test' (16 rows): reports/transformer_stage1/full/tables/stage1_vs_tfidf_pct.csv
[PASS] model order is Dummy, TF-IDF, then the transformer runs
[PASS] card 1: content inside the card (margins L 0.17, R 0.17, B 0.17, T 0.76 in)
[PASS] card 2: content inside the card (margins L 0.17, R 0.17, B 0.17, T 0.76 in)
[PASS] all plot areas have the same size
[PASS] figure saved: reports/transformer_stage1/full/figures/02_stage1_vs_tfidf.png
[PASS] saved image 3200x1800 px matches the figure aspect
CELL 3b RESULT: ALL CHECKS PASSED


# Stage 1 / Cell 4: B vs C Decision (pre-registered rule, validation only)

In [25]:
# ============================================================================
# Stage 1 / Cell 4: B vs C decision
# Rule fixed BEFORE seeing results: C wins only if
#     val_mean_macro_f1(C) >= val_mean_macro_f1(B) + 0.003, otherwise B.
# Uses ONLY the validation history (best epoch of each run). Test is never read here.
# Needs cell 1 (RUNS, paths, parameters) in this kernel.
# ============================================================================

# ---- Section: Imports and guard ----
import json
from datetime import UTC, datetime, timezone  # noqa: F401

import pandas as pd

from digikala_review_transformer import config

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")

n_fail = 0   # reset the failure counter of the shared check() helper

# ---- Section: Constants ----
CONFIG_DIR = config.ROOT / "configs" / "transformer"
STAGE2_ENCODERS = ["roberta", "mdeberta"]
DECISION_PATH = TABLES_DIR / "stage1_decision.json"
COMPARISON_PATH = TABLES_DIR / "stage1_b_vs_c.csv"
EPS = 1e-12                                   # float safety for the ">=" comparison
IS_REAL = bool(MODE["full_data"])
DECISION_LABEL = "REAL DECISION" if IS_REAL else "DEV (not a real decision)"
RULE_TEXT = (f"C wins only if {SELECTION_METRIC}(C) >= {SELECTION_METRIC}(B) + {C_OVER_B_THRESHOLD}; "
             "otherwise B. Validation, corrected labels, best epoch of each run.")


# ---- Section: Pick the two runs by variant (not by folder name) ----
by_variant = {run["variant"]: name for name, run in RUNS.items()}
check(set(by_variant) == {"B", "C"} and len(RUNS) == 2,
      f"found one run per variant: {by_variant}",
      f"expected exactly variants B and C, found {by_variant} -> check RUN_NAMES in cell 1")
if n_fail:
    raise RuntimeError("cannot decide without both variants -> see [FAIL] above")


def best_metrics(run: dict) -> dict:
    """Best-epoch row of one run (last row with is_best=True), taken from history.csv."""
    history = run["history"]
    best = history[history["is_best"]].iloc[-1]
    return {
        "run": run["dir"].name,
        "best_epoch": int(best["epoch"]),
        "epochs_done": len(history),
        "val_mean": float(best[SELECTION_METRIC]),
        "val_rec": float(best["val_macro_f1_rec"]),
        "val_band": float(best["val_macro_f1_band"]),
        "stop_reason": run["stop_reason"],
    }


metrics = {variant: best_metrics(RUNS[name]) for variant, name in by_variant.items()}

# ---- Section: Sanity checks on the inputs of the rule ----
for variant, m in metrics.items():
    mean_check = (m["val_rec"] + m["val_band"]) / 2
    check(abs(mean_check - m["val_mean"]) < 1e-6,
          f"{variant}: {SELECTION_METRIC} equals the mean of the two head F1 values ({m['val_mean']:.6f})",
          f"{variant}: mean of heads {mean_check:.6f} != {SELECTION_METRIC} {m['val_mean']:.6f} "
          "-> selection metric is not what the plan says")
    check(abs(RUNS[by_variant[variant]]["best_score"] - m["val_mean"]) < 1e-6,
          f"{variant}: run_summary best score equals history best row",
          f"{variant}: run_summary and history disagree -> inconsistent run files")

fingerprints = {dig(RUNS[name]["summary"], "pack", "fingerprint") for name in RUNS}
check(len(fingerprints) == 1 and fingerprints == {PACK_FINGERPRINT},
      f"B and C were trained on the same pack ({PACK_FINGERPRINT})",
      f"pack fingerprints differ: {fingerprints} -> B and C are not comparable")
check(RUNS[by_variant["B"]]["n_val"] == RUNS[by_variant["C"]]["n_val"],
      f"B and C were validated on the same number of rows ({RUNS[by_variant['B']]['n_val']:,})",
      "different validation row counts -> B and C are not comparable")

# ---- Section: Apply the rule ----
score_b, score_c = metrics["B"]["val_mean"], metrics["C"]["val_mean"]
difference = score_c - score_b
c_wins = difference >= C_OVER_B_THRESHOLD - EPS
winner = "C" if c_wins else "B"
margin = difference - C_OVER_B_THRESHOLD              # >= 0 means C passed the threshold

# ---- Section: Comparison table ----
table = pd.DataFrame([
    {"variant": "B (no weights)", **{k: metrics["B"][k] for k in ("run", "best_epoch", "epochs_done", "val_mean", "val_rec", "val_band")}},
    {"variant": "C (Step-2 weights)", **{k: metrics["C"][k] for k in ("run", "best_epoch", "epochs_done", "val_mean", "val_rec", "val_band")}},
])
delta = {"variant": "C minus B", "run": "", "best_epoch": None, "epochs_done": None,
         "val_mean": difference,
         "val_rec": metrics["C"]["val_rec"] - metrics["B"]["val_rec"],
         "val_band": metrics["C"]["val_band"] - metrics["B"]["val_band"]}
table = pd.concat([table, pd.DataFrame([delta])], ignore_index=True)
for col in ("val_mean", "val_rec", "val_band"):
    table[col] = table[col].round(6)
table.to_csv(COMPARISON_PATH, index=False)
check(COMPARISON_PATH.is_file(), f"comparison table saved: {COMPARISON_PATH.relative_to(config.ROOT)}",
      "comparison table not saved -> check TABLES_DIR")

print("\nValidation Macro F1 at each run's best epoch (corrected labels):")
print(table.to_string(index=False))

# ---- Section: Stage 2 configs for the winner ----
stage2_names = [f"{encoder}_{winner}" for encoder in STAGE2_ENCODERS]
STAGE2_CONFIGS = [f"configs/transformer/{name}.yaml" for name in stage2_names]
missing_cfg = [path for path in STAGE2_CONFIGS if not (config.ROOT / path).is_file()]
check(not missing_cfg, f"Stage 2 config files exist: {STAGE2_CONFIGS}",
      f"missing config files {missing_cfg} -> create them from base.yaml before Stage 2")

# ---- Section: Save the decision (JSON) ----
decision = {
    "decision_label": DECISION_LABEL,
    "is_real_decision": IS_REAL,
    "mode": MODE["name"],
    "rule": RULE_TEXT,
    "threshold": C_OVER_B_THRESHOLD,
    "winner": winner,
    "score_B": round(score_b, 6),
    "score_C": round(score_c, 6),
    "difference_C_minus_B": round(difference, 6),
    "margin_to_threshold": round(margin, 6),
    "per_variant": {v: {k: (round(x, 6) if isinstance(x, float) else x) for k, x in m.items()}
                    for v, m in metrics.items()},
    "stage2_configs": STAGE2_CONFIGS,
    "pack_fingerprint": PACK_FINGERPRINT,
    "created_utc": datetime.now(UTC).isoformat(timespec="seconds"),
}
if DECISION_PATH.exists():
    previous = json.loads(DECISION_PATH.read_text(encoding="utf-8"))
    if previous.get("winner") != winner:
        print(f"[INFO] previous decision file said winner={previous.get('winner')}; it is now {winner}")
DECISION_PATH.write_text(json.dumps(decision, indent=2, ensure_ascii=False), encoding="utf-8")
check(DECISION_PATH.is_file(), f"decision saved: {DECISION_PATH.relative_to(config.ROOT)}",
      "decision file not saved -> check TABLES_DIR")

STAGE1_WINNER = winner      # used by later cells

# ---- Section: Report ----
print("\n" + "=" * 78)
print(f"{DECISION_LABEL}")
print(f"  rule      : {RULE_TEXT}")
print(f"  B = {score_b:.6f}   C = {score_c:.6f}   C - B = {difference:+.6f}   "
      f"threshold = +{C_OVER_B_THRESHOLD}   margin = {margin:+.6f}")
print(f"  WINNER    : {winner}")
if IS_REAL:
    print("  Stage 2 command for the friend (PowerShell):")
    print("    uv run python scripts/train_transformer.py --config "
          + " ".join(STAGE2_CONFIGS))
else:
    print("  (dev numbers: 100k-row subset, 10k-row validation, single seed; do NOT use for Stage 2)")
print("=" * 78)
print("CELL 4 RESULT: ALL CHECKS PASSED" if n_fail == 0
      else f"CELL 4 RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above")

[PASS] found one run per variant: {'B': 'tooka_B', 'C': 'tooka_C'}
[PASS] B: val_mean_macro_f1 equals the mean of the two head F1 values (0.663465)
[PASS] B: run_summary best score equals history best row
[PASS] C: val_mean_macro_f1 equals the mean of the two head F1 values (0.663744)
[PASS] C: run_summary best score equals history best row
[PASS] B and C were trained on the same pack (9faa2a97e4502701)
[PASS] B and C were validated on the same number of rows (922,794)
[PASS] comparison table saved: reports/transformer_stage1/full/tables/stage1_b_vs_c.csv

Validation Macro F1 at each run's best epoch (corrected labels):
           variant     run best_epoch epochs_done  val_mean   val_rec  val_band
    B (no weights) tooka_B          4           4  0.663465  0.720016  0.606914
C (Step-2 weights) tooka_C          4           4  0.663744  0.719830  0.607659
         C minus B               None        None  0.000280 -0.000186  0.000745
[PASS] Stage 2 config files exist: ['configs/transfo

# Stage 1 / Cell 5: Test Metrics of B and C (raw and corrected labels, report only)

In [9]:
# ============================================================================
# Stage 1 / Cell 5: Test metrics of B and C in both label views
# REPORT ONLY: no decision uses these numbers.
# Needs cell 1 (RUNS, paths, parameters) and cell 3 (loaders) in this kernel.
# ============================================================================

# ---- Section: Imports and guards ----
import numpy as np
import pandas as pd

from digikala_review_transformer import config

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")
for _name in ("load_pack_labels", "load_baseline", "HEAD_COLUMNS", "PROB_FILES"):
    if _name not in globals():
        raise RuntimeError(f"'{_name}' is missing -> run cell 3 first in this kernel")

n_fail = 0   # reset the failure counter of the shared check() helper

# ---- Section: Constants ----
TEST_SPLIT = "test"
TABLE_PATH = TABLES_DIR / "stage1_test_metrics.csv"
CELL3_TABLE_PATH = TABLES_DIR / "stage1_vs_tfidf.csv"
METRIC_COLUMNS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_precision", "macro_recall"]
CELL3_TOL = 1e-5                       # cell 3 CSV is rounded to 6 decimals
REFERENCE_TOL = 5e-4                   # reference values below have 4 decimals
# TF-IDF test Macro F1 from the baseline phase (full test split): (head, view) -> value
TFIDF_TEST_REFERENCE = {
    ("rec", "raw"): 0.6849, ("rec", "corrected"): 0.6876,
    ("band", "raw"): 0.5776, ("band", "corrected"): 0.5852,
}
TFIDF_NAME = "tfidf (same rows)"


# ---- Section: Helpers ----
def metrics_from_confusion(conf: np.ndarray) -> dict:
    """Metrics from a 3x3 confusion matrix shaped (true, pred). Empty classes count as 0."""
    conf = np.asarray(conf, dtype=float)
    tp = np.diag(conf)
    support = conf.sum(axis=1)                   # rows of each true class
    predicted = conf.sum(axis=0)                 # rows predicted as each class
    n = conf.sum()
    recall = np.divide(tp, support, out=np.zeros(3), where=support > 0)
    precision = np.divide(tp, predicted, out=np.zeros(3), where=predicted > 0)
    denom = support + predicted
    f1 = np.divide(2.0 * tp, denom, out=np.zeros(3), where=denom > 0)
    return {
        "n": int(n),
        "accuracy": float(tp.sum() / n),
        "macro_f1": float(f1.mean()),
        "balanced_accuracy": float(recall.mean()),
        "macro_precision": float(precision.mean()),
        "macro_recall": float(recall.mean()),
    }


def view_column(head: str, view: str) -> str:
    return f"y_{head}" if view == "corrected" else f"y_{head}_raw"


def confusion(y_series: pd.Series, pred: np.ndarray, extra_mask: np.ndarray = None) -> np.ndarray:
    """3x3 confusion over rows with a valid label (and extra_mask if given)."""
    y_filled = y_series.fillna(-1)
    valid = y_filled.isin([0, 1, 2]).to_numpy()
    if extra_mask is not None:
        valid &= extra_mask
    y = y_filled.to_numpy().astype(np.int64)
    return np.bincount(y[valid] * 3 + pred[valid], minlength=9).reshape(3, 3)


# ---- Section: Transformer metrics on test ----
labels_all = load_pack_labels(TEST_SPLIT)
rows, reference_ids = [], None

for run_name, run in RUNS.items():
    probs = pd.read_parquet(run["dir"] / PROB_FILES[TEST_SPLIT])
    ids = probs["id"].astype(str).to_numpy()
    check(len(set(ids)) == len(ids) and bool(np.isin(ids, labels_all.index.to_numpy()).all()),
          f"{run_name}: test ids unique and all in the pack test split (n={len(ids):,})",
          f"{run_name}: duplicate test ids or ids missing from the pack -> bug in predict.py")
    if reference_ids is None:
        reference_ids = ids
    else:
        check(np.array_equal(ids, reference_ids), f"{run_name}: same test ids/order as the first run",
              f"{run_name}: test ids differ from the first run -> B and C evaluated on different rows")
    labels = labels_all.reindex(ids)
    for head in HEADS:
        pred = probs[HEAD_COLUMNS[head]].to_numpy().argmax(axis=1).astype(np.int64)
        for view in VIEWS:
            conf = confusion(labels[view_column(head, view)], pred)
            rows.append({"model": f"transformer_{run['variant']}", "variant": run["variant"],
                         "head": head, "view": view, **metrics_from_confusion(conf)})

# ---- Section: TF-IDF on exactly the same rows ----
labels = labels_all.reindex(reference_ids)
for head in HEADS:
    base = load_baseline(head, TEST_SPLIT).reindex(reference_ids)
    has_pred = base["pred"].notna().to_numpy()
    pred_b = base["pred"].fillna(-1).to_numpy().astype(np.int64)
    for view in VIEWS:
        conf = confusion(labels[view_column(head, view)], pred_b, extra_mask=has_pred)
        rows.append({"model": TFIDF_NAME, "variant": "-", "head": head, "view": view,
                     **metrics_from_confusion(conf)})

table = pd.DataFrame(rows)
table[METRIC_COLUMNS] = table[METRIC_COLUMNS].round(6)
table = table.sort_values(["head", "view", "model"]).reset_index(drop=True)
table.to_csv(TABLE_PATH, index=False)
check(TABLE_PATH.is_file() and len(table) == (len(RUNS) + 1) * len(HEADS) * len(VIEWS),
      f"test metrics table saved ({len(table)} rows): {TABLE_PATH.relative_to(config.ROOT)}",
      "unexpected number of rows in the test metrics table -> loop problem")

# ---- Section: Cross-checks ----
# (1) transformer Macro F1 here == f1_own of cell 3 (same rows, same formula, separate code path)
cell3 = pd.read_csv(CELL3_TABLE_PATH)
cell3 = cell3[cell3["split"] == TEST_SPLIT][["variant", "head", "view", "f1_own"]]
merged = table[table["variant"] != "-"].merge(cell3, on=["variant", "head", "view"], how="left")
gap = float((merged["macro_f1"] - merged["f1_own"]).abs().max())
check(merged["f1_own"].notna().all() and gap < CELL3_TOL,
      f"transformer test Macro F1 equals cell 3 (max gap {gap:.2e})",
      f"gap to cell 3 = {gap:.2e} (or missing rows) -> re-run cell 3 with the same STATE, then this cell")

# (2) TF-IDF numbers vs the baseline phase (full test split only)
if MODE["full_data"]:
    for (head, view), expected in TFIDF_TEST_REFERENCE.items():
        value = float(table[(table["model"] == TFIDF_NAME) & (table["head"] == head)
                            & (table["view"] == view)]["macro_f1"].iloc[0])
        check(abs(value - expected) < REFERENCE_TOL,
              f"TF-IDF test Macro F1 {head}/{view} = {value:.4f} matches the baseline phase ({expected:.4f})",
              f"TF-IDF {head}/{view} = {value:.4f} != baseline phase {expected:.4f} "
              "-> different rows or different label view")
else:
    print("[INFO] dev mode: TF-IDF reference check skipped (dev rows are a subset of test)")

# ---- Section: Report ----
print(f"\n{MODE_TAG}")
print("Test metrics (report only; both label views shown, none is 'main'):")
for head in HEADS:
    for view in VIEWS:
        part = table[(table["head"] == head) & (table["view"] == view)]
        shown = part[["model", "n"] + METRIC_COLUMNS].copy()
        shown[METRIC_COLUMNS] = shown[METRIC_COLUMNS].round(4)
        print(f"\n[{head} head, {view} labels]")
        print(shown.to_string(index=False))

print("\n" + "=" * 78)
print("CELL 5 RESULT: ALL CHECKS PASSED" if n_fail == 0
      else f"CELL 5 RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above")

[PASS] tooka_B: test ids unique and all in the pack test split (n=10,000)
[PASS] tooka_C: test ids unique and all in the pack test split (n=10,000)
[PASS] tooka_C: same test ids/order as the first run
[PASS] test metrics table saved (12 rows): reports/transformer_stage1/dev/tables/stage1_test_metrics.csv
[PASS] transformer test Macro F1 equals cell 3 (max gap 0.00e+00)
[INFO] dev mode: TF-IDF reference check skipped (dev rows are a subset of test)

DEV (100k-row training subset, 10k-row validation/test subsets)
Test metrics (report only; both label views shown, none is 'main'):

[rec head, raw labels]
            model    n  accuracy  macro_f1  balanced_accuracy  macro_precision  macro_recall
tfidf (same rows) 8543    0.8176    0.6818             0.7391           0.6511        0.7391
    transformer_B 8543    0.8193    0.6877             0.7329           0.6710        0.7329
    transformer_C 8543    0.8221    0.6823             0.7344           0.6514        0.7344

[rec head, correct

# Stage 1 / Cell 6: Confusion Matrices and Per-Class Metrics on Test (both label views, report only)

In [17]:
# ============================================================================
# Stage 1 / Cell 6: Confusion matrices and per-class metrics on test
# REPORT ONLY: no decision uses these numbers.
# Needs cells 1, 2, 3 and 5 in this kernel (RUNS, layout helpers, loaders, cell-5 table).
# ============================================================================

# ---- Section: Imports and guards ----
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch  # noqa: F811

from digikala_review_transformer import config, plot_style

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")
for _name in ("tile_axes", "fit_axes_to_tiles", "layout_check", "load_pack_labels", "confusion",
              "view_column", "HEAD_COLUMNS", "PROB_FILES", "TABLE_PATH"):
    if _name not in globals():
        raise RuntimeError(f"'{_name}' is missing -> run cells 2, 3 and 5 first in this kernel")

n_fail = 0   # reset the failure counter of the shared check() helper

# ---- Section: Constants ----
TEST_SPLIT = "test"
FONT_DIR = config.ROOT / "assets" / "fonts"
CLASS_NAMES = {head: [col[2:] for col in cols] for head, cols in HEAD_COLUMNS.items()}   # p_xxx -> xxx
SHORT_NAMES = {"recommended": "rec", "not_recommended": "not_rec"}                      # short tick labels
FIGURE_NAMES = {
    "raw": "03_stage1_confusion_raw.png",
    "corrected": "04_stage1_confusion_corrected.png",
    "per_class": "05_stage1_per_class_f1.png",
}
CONF_TABLE_PATH = TABLES_DIR / "stage1_confusion_matrices.csv"
PER_CLASS_TABLE_PATH = TABLES_DIR / "stage1_per_class_metrics.csv"
CROSS_TOL = 1e-5
HEATMAP_CMAP = LinearSegmentedColormap.from_list("aurora_heat", ["#16213A", plot_style.TURQUOISE])
DARK_TEXT = "#0B1020"
VARIANT_MARKERS = {"B": "o", "C": "D"}
VARIANT_OFFSETS = {"B": -0.2, "C": 0.2}           # horizontal offset of each variant inside a class group
VIEW_NUDGE = {"raw": -0.06, "corrected": 0.06}    # tiny extra offset so the two views do not overlap


# ---- Section: Helpers ----
def per_class_metrics(conf: np.ndarray) -> pd.DataFrame:
    """Per-class support, predicted count, precision, recall and F1 from a (true, pred) confusion matrix."""
    conf = np.asarray(conf, dtype=float)
    tp = np.diag(conf)
    support, predicted = conf.sum(axis=1), conf.sum(axis=0)
    recall = np.divide(tp, support, out=np.zeros(3), where=support > 0)
    precision = np.divide(tp, predicted, out=np.zeros(3), where=predicted > 0)
    f1 = np.divide(2.0 * tp, support + predicted, out=np.zeros(3), where=(support + predicted) > 0)
    return pd.DataFrame({"support": support.astype(int), "predicted": predicted.astype(int),
                         "precision": precision, "recall": recall, "f1": f1})


def short(name: str) -> str:
    return SHORT_NAMES.get(name, name)


# ---- Section: Confusion matrices from the saved test probabilities ----
labels_all = load_pack_labels(TEST_SPLIT)
runs_by_variant = {run["variant"]: run for run in RUNS.values()}
VARIANTS = sorted(runs_by_variant)                  # ["B", "C"]
CONF, conf_rows, class_rows = {}, [], []

for variant in VARIANTS:
    probs = pd.read_parquet(runs_by_variant[variant]["dir"] / PROB_FILES[TEST_SPLIT])
    ids = probs["id"].astype(str).to_numpy()
    labels = labels_all.reindex(ids)
    for head in HEADS:
        pred = probs[HEAD_COLUMNS[head]].to_numpy().argmax(axis=1).astype(np.int64)
        for view in VIEWS:
            conf = confusion(labels[view_column(head, view)], pred)
            CONF[(variant, head, view)] = conf
            totals = conf.sum(axis=1)
            for i, true_name in enumerate(CLASS_NAMES[head]):
                for j, pred_name in enumerate(CLASS_NAMES[head]):
                    conf_rows.append({"variant": variant, "head": head, "view": view,
                                      "true_class": true_name, "pred_class": pred_name,
                                      "count": int(conf[i, j]),
                                      "pct_of_true": round(100.0 * conf[i, j] / totals[i], 4) if totals[i] else 0.0})
            pc = per_class_metrics(conf)
            pc.insert(0, "class", CLASS_NAMES[head])
            pc.insert(0, "view", view)
            pc.insert(0, "head", head)
            pc.insert(0, "variant", variant)
            class_rows.append(pc)

conf_table = pd.DataFrame(conf_rows)
class_table = pd.concat(class_rows, ignore_index=True)
class_table[["precision", "recall", "f1"]] = class_table[["precision", "recall", "f1"]].round(6)
conf_table.to_csv(CONF_TABLE_PATH, index=False)
class_table.to_csv(PER_CLASS_TABLE_PATH, index=False)
check(CONF_TABLE_PATH.is_file() and PER_CLASS_TABLE_PATH.is_file(),
      f"tables saved: {CONF_TABLE_PATH.relative_to(config.ROOT)}, {PER_CLASS_TABLE_PATH.relative_to(config.ROOT)}",
      "tables not saved -> check TABLES_DIR")

# ---- Section: Checks ----
cell5 = pd.read_csv(TABLE_PATH)
cell5 = cell5[cell5["variant"].isin(VARIANTS)].set_index(["variant", "head", "view"])
worst_f1_gap, n_mismatch = 0.0, 0
for (variant, head, view), conf in CONF.items():
    reference = cell5.loc[(variant, head, view)]
    mean_f1 = per_class_metrics(conf)["f1"].mean()
    worst_f1_gap = max(worst_f1_gap, abs(mean_f1 - float(reference["macro_f1"])))
    n_mismatch += int(conf.sum() != int(reference["n"]))
check(worst_f1_gap < CROSS_TOL,
      f"mean per-class F1 equals Macro F1 of cell 5 (max gap {worst_f1_gap:.2e})",
      f"gap to cell 5 = {worst_f1_gap:.2e} -> re-run cell 5 with the same STATE, then this cell")
check(n_mismatch == 0, "confusion matrix totals equal n of cell 5 in every case",
      f"{n_mismatch} matrices have a different total than cell 5 -> different rows were used")
collapsed = [(v, h, w) for (v, h, w), conf in CONF.items() if (conf.sum(axis=0) == 0).any()]
check(not collapsed, "every class is predicted at least once by every model (no collapse)",
      f"some class is never predicted in {collapsed} -> model collapsed onto fewer classes; "
      "look at the training log of that run")


# ---- Section: Figure builders ----
def draw_confusion(ax, conf: np.ndarray, head: str) -> None:
    """Row-normalised heatmap: top number = % of the true class, bottom number = row count."""
    totals = conf.sum(axis=1, keepdims=True)
    pct = 100.0 * np.divide(conf, totals, out=np.zeros_like(conf, dtype=float), where=totals > 0)
    ax.imshow(pct, cmap=HEATMAP_CMAP, vmin=0, vmax=100, aspect="auto")
    for i in range(3):
        for j in range(3):
            color = DARK_TEXT if pct[i, j] > 50 else plot_style.TEXT
            ax.text(j, i - 0.12, f"{pct[i, j]:.1f}%", ha="center", va="center",
                    color=color, fontsize=13, fontweight="bold")
            ax.text(j, i + 0.24, f"{int(conf[i, j]):,}", ha="center", va="center", color=color, fontsize=9)
    names = [short(n) for n in CLASS_NAMES[head]]
    ax.set_xticks(range(3))
    ax.set_xticklabels(names)
    ax.set_yticks(range(3))
    ax.set_yticklabels(names)
    ax.tick_params(length=0)
    ax.grid(False)
    ax.set_xlabel("Predicted class", fontsize=12)
    ax.set_ylabel("True class", fontsize=12)
    handles = [Patch(facecolor=plot_style.TURQUOISE, label="top: % of true class"),
               Line2D([], [], linestyle="", label="bottom: row count")]
    plot_style.glass_legend(ax, handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=1)


def save_and_show(fig, name: str) -> None:
    """Remove our own previous file (same name, same folder), save, show."""
    path = FIGS_DIR / name
    if path.exists():
        path.unlink()
    save_figure_any_size(fig, path)
    check(path.is_file() and path.stat().st_size > 0, f"figure saved: {path.relative_to(config.ROOT)}",
          f"figure file missing or empty -> save_figure failed ({name})")
    plt.show()
    plt.close(fig)


plot_style.init_style(FONT_DIR)                       # idempotent

# ---- Section: Confusion matrix figures (portrait, rows = head, columns = variant, square matrices) ----
VARIANTS = [RUNS[name]["variant"] for name in RUN_NAMES]
CONFUSION_GRID = [(variant, head) for head in HEADS for variant in VARIANTS]

_missing = [(v, h, w) for v, h in CONFUSION_GRID for w in VIEWS if (v, h, w) not in CONF]
check(not _missing, "CONF has every (variant, head, view) key used by the figures",
      f"CONF is missing keys {_missing[:3]} (e.g. existing: {sorted(CONF)[:3]}) -> adjust CONFUSION_GRID keys")


def render_confusion(view, height_in):
    """One figure for a label view; height_in=None means the canvas' native height."""
    fig = make_canvas(
        title=f"Stage 1: test confusion matrices, {view} labels",
        subtitle="Row-normalised (% of the true class) with counts  |  rows = head, columns = variant",
        footer=f"{MODE_TAG}. Test split only; never used for decisions.",
        height_in=height_in,
    )
    rects = grid_rects(fig, content_area(fig), len(HEADS), len(VARIANTS))
    items = []
    for (variant, head), rect in zip(CONFUSION_GRID, rects):
        ax = tile_axes(fig, rect, f"{head} head | {variant}")
        plot_style.style_axes(ax)
        draw_confusion(ax, CONF[(variant, head, view)], head)
        ax.set_aspect("equal", adjustable="box")
        items.append((ax, rect))
    fit_cards(fig, items)
    return fig, items


for view in VIEWS:
    fig, tile_items = solve_figure_height(
        lambda h, v=view: render_confusion(v, h),
        rows=len(HEADS),
    )
    layout_check(fig, tile_items, square=True)
    save_and_show(fig, FIGURE_NAMES[view])
    check_saved_image(fig, FIGS_DIR / FIGURE_NAMES[view])
    
# ---- Section: Per-class F1 figure (colour = label view, marker = variant) ----
fig = plot_style.new_canvas(
    title="Stage 1: per-class F1 on test",
    subtitle="Colour = label view (raw / corrected). Marker = variant (B circle, C diamond). Report only.",
    footer=f"{MODE_TAG}. Report only; no decision uses test. Zoomed y axis.",
)
rects = plot_style.split_row(len(HEADS))
tile_items = []
for idx, (head, rect) in enumerate(zip(HEADS, rects)):
    ax = tile_axes(fig, rect, f"{head} head")
    plot_style.style_axes(ax)

    ax.set_ylabel("Macro F1", fontsize=12)
    text_color = ax.yaxis.label.get_color()      # follows the theme, no hard-coded colour

    series = {view: column(head, view, "macro_f1") for view in VIEWS}
    y_lo, y_hi, y_ticks, decimals = zoomed_axis(np.concatenate(list(series.values())))
    span = y_hi - y_lo

    for boundary in np.arange(len(model_labels) - 1) + 0.5:
        ax.axvline(boundary, color=NOTE_COLOR, alpha=0.18, linewidth=1.0,
                   linestyle=(0, (2, 4)), zorder=1)

    # Bars + value labels (inside if the bar is tall, above the bar if it is short)
    for view in VIEWS:
        solid = view == "corrected"
        for i, value in enumerate(series[view]):
            color = MODEL_COLORS[model_labels[i]]
            x = i + VIEW_OFFSETS[view]

            ax.bar(x, value - y_lo, bottom=y_lo, width=BAR_WIDTH,
                   color=color if solid else to_rgba(color, 0.30),
                   edgecolor=color, linewidth=1.4,
                   hatch=None if solid else "////", zorder=3)

            inside = (value - y_lo) / span >= MIN_INSIDE_FRAC
            if inside:
                ax.text(x, value - 0.015 * span, f"{value:.3f}",
                        rotation=90, ha="center", va="top",
                        color="white", fontsize=10, fontweight="bold", zorder=6,
                        path_effects=[pe.withStroke(linewidth=2.4, foreground=(0, 0, 0, 0.45))])
            else:
                ax.text(x, value + 0.015 * span, f"{value:.3f}",
                        rotation=90, ha="center", va="bottom",
                        color=text_color, fontsize=9.5, fontweight="bold", zorder=6)

    # Row 1: corrected vs raw (one pill per model, centred on the group)
    gaps = column(head, "corrected", "corr_vs_raw_pct")
    for i, gap in enumerate(gaps):
        pill(ax, i, 0, f"raw→corr. {gap:+.1f}%", gap)

    # Row 2: each transformer vs TF-IDF (raw / corrected staggered vertically)
    note(ax, 1, 1, "vs TF-IDF")
    for view in VIEWS:
        changes = column(head, view, "vs_tfidf_pct")
        for i in range(first_idx, len(model_labels)):
            pill(ax, i + VIEW_OFFSETS[view], 1, f"{changes[i]:+.1f}%",
                 changes[i], dy=STAGGER[view])

    # Row 3: last run vs first run
    if last_idx > first_idx:
        note(ax, first_idx, 2, f"{last_variant} vs {first_variant}")
        for view in VIEWS:
            change = column(head, view, "last_vs_first_pct")[last_idx]
            pill(ax, last_idx + VIEW_OFFSETS[view], 2, f"{change:+.1f}%",
                 change, dy=STAGGER[view])

    # Axes
    ax.set_ylim(y_lo, y_hi)
    ax.set_yticks(y_ticks)
    ax.yaxis.set_major_formatter(FormatStrFormatter(f"%.{decimals}f"))
    ax.set_xlim(-0.55, len(model_labels) - 0.45)

    ax.set_xticks(range(len(model_labels)))
    ax.set_xticklabels(model_labels, fontsize=12, fontweight="bold")
    for label, name in zip(ax.get_xticklabels(), model_labels):
        label.set_color(MODEL_COLORS[name])
    ax.tick_params(axis="x", which="major", length=0, pad=22)

    minor = sorted((i + VIEW_OFFSETS[v], VIEW_SHORT[v])
                   for i in range(len(model_labels)) for v in VIEWS)
    ax.set_xticks([x for x, _ in minor], minor=True)
    ax.set_xticklabels([l for _, l in minor], minor=True)
    ax.tick_params(axis="x", which="minor", length=0, pad=4, labelsize=9)

    ax.set_xlabel("")                      # the old shared label is replaced by the per-tile legend
    add_legend(ax, idx, text_color)

    tile_items.append((ax, rect))
    
fit_axes_to_tiles(fig, tile_items)
layout_check(fig, tile_items)
save_and_show(fig, FIGURE_NAMES["per_class"])

# ---- Section: Report ----
print(f"\n{MODE_TAG}")
print("Per-class test metrics (report only; both views shown, none is 'main'):")
shown = class_table.copy()
shown[["precision", "recall", "f1"]] = shown[["precision", "recall", "f1"]].round(4)
print(shown.sort_values(["head", "view", "variant"]).to_string(index=False))

print("\n" + "=" * 78)
print("CELL 6 RESULT: ALL CHECKS PASSED" if n_fail == 0
      else f"CELL 6 RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above")

RuntimeError: 'confusion' is missing -> run cells 2, 3 and 5 first in this kernel

# Stage 1 / Inspect 7: Timing and resource keys (read-only)

In [18]:
# ============================================================================
# Stage 1 / Inspect 7: timing / GPU / memory fields of a finished run (read-only, no checks)
# Needs cell 1 (RUNS_ROOT, RUN_NAMES). Prints facts only.
# ============================================================================
import json
from pathlib import Path

import pandas as pd

KEY_WORDS = ("time", "second", "sec", "speed", "throughput", "sample", "gpu", "vram", "mem", "device",
             "cuda", "torch", "peak", "bf16", "fused", "worker", "host", "platform", "python", "duration")


def walk(node, prefix=""):
    """Yield (dotted key, value) for every leaf of a nested dict/list."""
    if isinstance(node, dict):
        for key, value in node.items():
            yield from walk(value, f"{prefix}.{key}" if prefix else str(key))
    elif isinstance(node, list) and node and isinstance(node[0], (dict, list)):
        for i, value in enumerate(node[:3]):
            yield from walk(value, f"{prefix}[{i}]")
    else:
        yield prefix, node


name = RUN_NAMES[0]
run_dir = Path(RUNS_ROOT) / name
print(f"run dir: {run_dir}")

print("\n--- files (name, size KB) ---")
for path in sorted(run_dir.iterdir()):
    print(f"  {path.name:28s} {path.stat().st_size / 1024:10.1f}" if path.is_file() else f"  {path.name}/")

summary = json.loads((run_dir / "run_summary.json").read_text(encoding="utf-8"))
print("\n--- top-level keys of run_summary.json ---")
print(list(summary))

print("\n--- every leaf whose key looks like timing/resource ---")
for key, value in walk(summary):
    if any(word in key.lower() for word in KEY_WORDS):
        print(f"  {key} = {str(value)[:90]}")

print("\n--- history.csv columns ---")
history = pd.read_csv(run_dir / "history.csv")
print(list(history.columns))
print(history[[c for c in history.columns if "second" in c or c == "epoch"]].to_string(index=False))

print("\n--- train_log.csv columns + first 2 rows ---")
train_log = pd.read_csv(run_dir / "train_log.csv")
print(list(train_log.columns))
print(train_log.head(2).to_string(index=False))

print("\n--- train.log lines mentioning GPU / VRAM / peak / samples/s (max 12) ---")
hits = [line.rstrip() for line in (run_dir / "train.log").read_text(encoding="utf-8", errors="replace").splitlines()
        if any(w in line.lower() for w in ("gpu", "vram", "peak", "samples/s", "memory", "cuda"))]
for line in hits[:12]:
    print(" ", line[:160])
print(f"({len(hits)} matching lines in total)")

run dir: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/artifacts/transformer_runs_dev_scale/tooka_B

--- files (name, size KB) ---
  best_model.pt                  477877.5
  config_resolved.yaml                0.6
  encoder_config.json                 1.1
  history.csv                         1.5
  run_summary.json                   10.8
  test_probs.parquet                391.5
  tokenizer/
  train.log                          17.9
  train_log.csv                      18.1
  val_probs_best.parquet            391.8

--- top-level keys of run_summary.json ---
['status', 'run_name', 'seed', 'variant', 'config_hash', 'config', 'model', 'parameter_counts', 'tokenizer_backend', 'tokenizer', 'token_stats', 'encoder_load_info', 'split_rows', 'training_schedule', 'selection_and_stopping', 'validation_best', 'test_quick', 'timing_seconds', 'library_versions', 'hardware', 'pack', 'artifact_sha256']

--- every leaf whose key looks like timing/resource ---
  conf

# Stage 1 / Cell 7: Time and resource usage (table + figure)

In [19]:
# ============================================================================
# Stage 1 / Cell 7: Time and resource usage (table + figure)
# Needs cell 1 (RUNS, RUNS_ROOT, RUN_NAMES, VARIANT_COLORS, TABLES_DIR, FIGS_DIR, MODE_TAG, check),
#       cell 2 (tile_axes) and cell 2b (layout toolkit).
# Reads run_summary.json (timing_seconds, hardware, config), history.csv, train_log.csv.
# ============================================================================

# ---- Section: Imports and guards ----
import json
from pathlib import Path

import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgba

from digikala_review_transformer import config, plot_style

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")
for _name in ("tile_axes", "make_canvas", "content_area", "grid_rects", "fit_cards",
              "layout_check", "check_saved_image", "save_figure_any_size"):
    if _name not in globals():
        raise RuntimeError(f"'{_name}' is missing -> run cells 2 and 2b first in this kernel")

n_fail = 0   # reset the failure counter of the shared check() helper

# ---- Section: Constants ----
TABLE_NAME = "stage1_time_resources.csv"
FIGURE_NAME = "06_stage1_time_resources.png"
FONT_DIR = config.ROOT / "assets" / "fonts"
NOTE_COLOR = "#9AA5B5"
TIME_TOL_S = 0.5            # tolerance when comparing wall-clock sums (seconds)
BAR_FILL_FRACTION = 0.92    # bar width relative to its slot
GROUP_WIDTH = 0.8           # width of one epoch group on the x axis
Y_HEADROOM = 1.12           # room above the tallest bar for its value label

# Keys read from run_summary.json (dotted paths; verified by the Inspect cell)
SUMMARY_KEYS = {
    "train_only_seconds": "timing_seconds.train_Pure_epoch_sum",
    "loop_wall_seconds": "timing_seconds.loop_wall",
    "total_wall_seconds": "timing_seconds.total_wall",
    "best_epoch_seconds_summary": "selection_and_stopping.best_epoch_history.epoch_seconds",
    "gpu_name": "hardware.gpu_name",
    "gpu_total_mb": "hardware.gpu_total_memory_mb",
    "peak_allocated_mb": "hardware.gpu_max_memory_allocated_mb",
    "peak_reserved_mb": "hardware.gpu_max_memory_reserved_mb",
    "bf16_supported": "hardware.bf16_supported",
    "cuda_version": "hardware.cuda_version",
    "platform": "hardware.platform",
    "torch": "library_versions.torch",
    "bf16_config": "config.training.bf16",
    "max_epochs": "config.training.max_epochs",
    "num_workers": "config.training.num_workers",
    "optimizer_fused": "training_schedule.optimizer_fused",
}


def fetch(summary: dict, dotted: str):
    """Value at a dotted path of the nested summary dict, or None if any part is missing."""
    node = summary
    for part in dotted.split("."):
        if not isinstance(node, dict) or part not in node:
            return None
        node = node[part]
    return node


# ---- Section: Read every run and check the numbers ----
rows, epoch_rows = [], []
for name in RUN_NAMES:
    run_dir = Path(RUNS_ROOT) / name
    summary = json.loads((run_dir / "run_summary.json").read_text(encoding="utf-8"))
    history = pd.read_csv(run_dir / "history.csv")
    train_log = pd.read_csv(run_dir / "train_log.csv")

    values = {label: fetch(summary, path) for label, path in SUMMARY_KEYS.items()}
    missing = [path for label, path in SUMMARY_KEYS.items() if values[label] is None]
    check(not missing, f"{name}: all {len(SUMMARY_KEYS)} summary keys present",
          f"{name}: missing keys {missing} -> run the Inspect cell again and fix SUMMARY_KEYS")
    check("epoch_seconds" in history.columns and "samples_per_second" in train_log.columns,
          f"{name}: epoch_seconds and samples_per_second columns present",
          f"{name}: timing columns missing -> history {list(history.columns)[:5]}...")
    if missing:
        continue

    variant, best_epoch = RUNS[name]["variant"], int(RUNS[name]["best_epoch"])
    epochs_run = len(history)
    epoch_sum = float(history["epoch_seconds"].sum())

    check(history["epoch"].tolist() == list(range(1, epochs_run + 1)),
          f"{name}: history epochs are 1..{epochs_run}", f"{name}: history epochs are not consecutive")
    check(1 <= epochs_run <= int(values["max_epochs"]),
          f"{name}: epochs run {epochs_run} within max_epochs {values['max_epochs']}",
          f"{name}: epochs run {epochs_run} exceeds max_epochs {values['max_epochs']} -> check history.csv")
    # epoch_seconds contains validation: pure training <= sum of epochs <= loop <= total
    ordered = (values["train_only_seconds"] <= epoch_sum + TIME_TOL_S
               and epoch_sum <= values["loop_wall_seconds"] + TIME_TOL_S
               and values["loop_wall_seconds"] <= values["total_wall_seconds"] + TIME_TOL_S)
    check(ordered,
          f"{name}: train-only {values['train_only_seconds']:.1f} <= epochs {epoch_sum:.1f} "
          f"<= loop {values['loop_wall_seconds']:.1f} <= total {values['total_wall_seconds']:.1f} s",
          f"{name}: time ordering broken (train-only {values['train_only_seconds']:.1f}, epochs {epoch_sum:.1f}, "
          f"loop {values['loop_wall_seconds']:.1f}, total {values['total_wall_seconds']:.1f}) "
          "-> epoch_seconds may not include validation; revisit the table column names")
    best_row = history[history["epoch"] == best_epoch]
    check(len(best_row) == 1 and abs(float(best_row["epoch_seconds"].iloc[0]) - values["best_epoch_seconds_summary"]) < 1e-3,
          f"{name}: best-epoch seconds in the summary match history.csv",
          f"{name}: best-epoch seconds differ between summary and history -> check best_epoch")
    speeds = train_log["samples_per_second"].to_numpy(dtype=float)
    check(len(speeds) > 0 and np.isfinite(speeds).all() and (speeds > 0).all(),
          f"{name}: {len(speeds)} throughput samples, all finite and positive",
          f"{name}: throughput has non-positive or non-finite values")
    check(0 < values["peak_allocated_mb"] <= values["peak_reserved_mb"] <= values["gpu_total_mb"] + 1,
          f"{name}: VRAM allocated {values['peak_allocated_mb']:.0f} <= reserved {values['peak_reserved_mb']:.0f} "
          f"<= total {values['gpu_total_mb']:.0f} MB",
          f"{name}: VRAM numbers are not ordered allocated <= reserved <= total")
    check(not values["bf16_config"] or values["bf16_supported"],
          f"{name}: bf16 requested and supported", f"{name}: bf16 requested but the GPU does not support it")

    rows.append({
        "run": name, "variant": variant, "gpu": values["gpu_name"],
        "epochs_run": epochs_run, "max_epochs": int(values["max_epochs"]), "best_epoch": best_epoch,
        "stop_reason": RUNS[name]["stop_reason"],
        "epoch_seconds_mean": epoch_sum / epochs_run,
        "epoch_seconds_min": float(history["epoch_seconds"].min()),
        "epoch_seconds_max": float(history["epoch_seconds"].max()),
        "train_only_seconds": values["train_only_seconds"],
        "validation_and_overhead_seconds": epoch_sum - values["train_only_seconds"],
        "loop_wall_seconds": values["loop_wall_seconds"],
        "total_wall_seconds": values["total_wall_seconds"],
        "total_wall_hours": values["total_wall_seconds"] / 3600,
        "samples_per_second_median": float(np.median(speeds)),
        "peak_allocated_mb": values["peak_allocated_mb"],
        "peak_reserved_mb": values["peak_reserved_mb"],
        "gpu_total_mb": values["gpu_total_mb"],
        "reserved_pct_of_gpu": 100 * values["peak_reserved_mb"] / values["gpu_total_mb"],
        "bf16": bool(values["bf16_config"]), "optimizer_fused": bool(values["optimizer_fused"]),
        "num_workers": int(values["num_workers"]), "torch": values["torch"],
        "cuda_version": values["cuda_version"], "platform": values["platform"],
    })
    for _, row in history.iterrows():
        epoch_rows.append({"run": name, "variant": variant, "epoch": int(row["epoch"]),
                           "seconds": float(row["epoch_seconds"]), "is_best": int(row["epoch"]) == best_epoch})

check(len(rows) == len(RUN_NAMES), f"{len(rows)} runs read", "some runs could not be read -> see [FAIL] lines above")
table, epoch_df = pd.DataFrame(rows), pd.DataFrame(epoch_rows)
check(table["gpu"].nunique() == 1 and table["platform"].nunique() == 1,
      f"all runs used the same machine: {table['gpu'].iloc[0]}",
      f"runs used different machines: {table['gpu'].unique()} -> runs are not comparable")

TABLES_DIR.mkdir(parents=True, exist_ok=True)
table.to_csv(TABLES_DIR / TABLE_NAME, index=False)
check((TABLES_DIR / TABLE_NAME).is_file(), f"table saved: {(TABLES_DIR / TABLE_NAME).relative_to(config.ROOT)}",
      "table not saved")

# ---- Section: Print the table (English only) ----
print("-" * 78)
print(table.set_index("run").T.to_string())
print("-" * 78)
print(f"Stage-1 total wall time (runs are sequential): {table['total_wall_seconds'].sum() / 3600:.2f} h")
print("Note: epochs_run can be below max_epochs (early stopping); total time scales with epochs_run.")

# ---- Section: Figure (left: seconds per epoch, right: peak GPU memory; no legend, direct labels) ----
STROKE = [pe.withStroke(linewidth=2.4, foreground=(0, 0, 0, 0.45))]
n_runs = len(RUN_NAMES)
bar_slot = GROUP_WIDTH / n_runs
gpu_name, bf16 = table["gpu"].iloc[0], table["bf16"].iloc[0]

plot_style.init_style(FONT_DIR)                       # idempotent
fig = make_canvas(
    title="Stage 1: training time and GPU memory",
    subtitle=f"{gpu_name}  |  bf16 = {bf16}  |  white outline = best epoch",
    footer=f"{MODE_TAG}. Epoch time includes validation; memory = PyTorch peak allocated / reserved.",
)
left_rect, right_rect = grid_rects(fig, content_area(fig), 1, 2)

# left card: one group per epoch, one bar per run (colour = variant, run name written inside the bar)
ax_time = tile_axes(fig, left_rect, "Seconds per epoch")
plot_style.style_axes(ax_time)
top_seconds = float(epoch_df["seconds"].max())
for r, name in enumerate(RUN_NAMES):
    part = epoch_df[epoch_df["run"] == name]
    color = VARIANT_COLORS[part["variant"].iloc[0]]
    xs = part["epoch"].to_numpy(dtype=float) + (r - (n_runs - 1) / 2) * bar_slot
    ax_time.bar(xs, part["seconds"], width=bar_slot * BAR_FILL_FRACTION, color=color,
                edgecolor=["white" if best else color for best in part["is_best"]],
                linewidth=[2.4 if best else 1.0 for best in part["is_best"]], zorder=3)
    for x, seconds in zip(xs, part["seconds"]):
        ax_time.text(x, seconds + 0.012 * top_seconds, f"{seconds:.0f}", ha="center", va="bottom",
                     color=plot_style.TEXT, fontsize=10, fontweight="bold", zorder=6)
        ax_time.text(x, 0.02 * top_seconds, name, rotation=90, ha="center", va="bottom",
                     color="white", fontsize=10, fontweight="bold", zorder=6, path_effects=STROKE)
ax_time.set_ylim(0, top_seconds * Y_HEADROOM)
ax_time.set_xlim(0.5 - (1 - GROUP_WIDTH) / 2, int(epoch_df["epoch"].max()) + 0.5 + (1 - GROUP_WIDTH) / 2)
ax_time.set_xticks(sorted(epoch_df["epoch"].unique()))
ax_time.set_xlabel("Epoch", fontsize=12)
ax_time.set_ylabel("Seconds per epoch (training + validation)", fontsize=12)

# right card: per run two bars, allocated (solid) and reserved (light, hatched); dashed line = GPU total
ax_mem = tile_axes(fig, right_rect, "Peak GPU memory")
plot_style.style_axes(ax_mem)
gpu_total = float(table["gpu_total_mb"].max())
mem_slot = 0.36
for r, row in table.iterrows():
    color = VARIANT_COLORS[row["variant"]]
    for offset, key, solid, tag in ((-mem_slot / 2, "peak_allocated_mb", True, "allocated"),
                                    (mem_slot / 2, "peak_reserved_mb", False, "reserved")):
        x, value = r + offset, float(row[key])
        ax_mem.bar(x, value, width=mem_slot * BAR_FILL_FRACTION, color=color if solid else to_rgba(color, 0.30),
                   edgecolor=color, linewidth=1.4, hatch=None if solid else "////", zorder=3)
        ax_mem.text(x, value + 0.012 * gpu_total, f"{value:.0f}", ha="center", va="bottom",
                    color=plot_style.TEXT, fontsize=10, fontweight="bold", zorder=6)
        ax_mem.text(x, 0.02 * gpu_total, tag, rotation=90, ha="center", va="bottom",
                    color="white", fontsize=10, fontweight="bold", zorder=6, path_effects=STROKE)
ax_mem.axhline(gpu_total, color=NOTE_COLOR, linewidth=1.4, linestyle="--", zorder=2)
ax_mem.text(0.98, gpu_total, f"GPU total {gpu_total:.0f} MB", transform=ax_mem.get_yaxis_transform(),
            ha="right", va="bottom", color=NOTE_COLOR, fontsize=10, fontstyle="italic", zorder=6)
ax_mem.set_ylim(0, gpu_total * Y_HEADROOM)
ax_mem.set_xlim(-0.5, n_runs - 0.5)
ax_mem.set_xticks(range(n_runs))
ax_mem.set_xticklabels(list(table["run"]), fontsize=12, fontweight="bold")
for label, variant in zip(ax_mem.get_xticklabels(), table["variant"]):
    label.set_color(VARIANT_COLORS[variant])
ax_mem.tick_params(axis="x", length=0)
ax_mem.set_xlabel("Run", fontsize=12)
ax_mem.set_ylabel("Peak GPU memory (MB)", fontsize=12)

tile_items = [(ax_time, left_rect), (ax_mem, right_rect)]
fit_cards(fig, tile_items)
layout_check(fig, tile_items)

# ---- Section: Save (remove our own previous file first: same name, same folder) ----
fig_path = FIGS_DIR / FIGURE_NAME
if fig_path.exists():
    fig_path.unlink()
save_figure_any_size(fig, fig_path)
check(fig_path.is_file() and fig_path.stat().st_size > 0,
      f"figure saved: {fig_path.relative_to(config.ROOT)}", "figure file missing or empty -> save failed")
check_saved_image(fig, fig_path)
plt.show()
plt.close(fig)

# ---- Section: Result ----
print("=" * 78)
print("CELL 7 RESULT: ALL CHECKS PASSED" if n_fail == 0
      else f"CELL 7 RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above")

[PASS] tooka_B: all 16 summary keys present
[PASS] tooka_B: epoch_seconds and samples_per_second columns present
[PASS] tooka_B: history epochs are 1..3
[PASS] tooka_B: epochs run 3 within max_epochs 5
[PASS] tooka_B: train-only 414.5 <= epochs 422.4 <= loop 427.1 <= total 429.1 s
[PASS] tooka_B: best-epoch seconds in the summary match history.csv
[PASS] tooka_B: 96 throughput samples, all finite and positive
[PASS] tooka_B: VRAM allocated 3209 <= reserved 3552 <= total 7808 MB
[PASS] tooka_B: bf16 requested and supported
[PASS] tooka_C: all 16 summary keys present
[PASS] tooka_C: epoch_seconds and samples_per_second columns present
[PASS] tooka_C: history epochs are 1..4
[PASS] tooka_C: epochs run 4 within max_epochs 5
[PASS] tooka_C: train-only 552.4 <= epochs 562.8 <= loop 570.1 <= total 572.1 s
[PASS] tooka_C: best-epoch seconds in the summary match history.csv
[PASS] tooka_C: 128 throughput samples, all finite and positive
[PASS] tooka_C: VRAM allocated 3209 <= reserved 3552 <= to

# Stage 1 / Cell 8: Final summary and Stage-2 handoff file

In [24]:
# ============================================================================
# Stage 1 / Cell 8: Final summary and Stage-2 handoff file
# Needs cell 1 (RUNS, RUN_NAMES, paths, MODE_TAG, PACK_FINGERPRINT, C_OVER_B_THRESHOLD, check),
#       cell 4 (STAGE1_WINNER, STAGE2_CONFIGS) and cell 7 (stage1_time_resources.csv).
# Writes: <TABLES_DIR>/stage1_handoff.json
# ============================================================================

# ---- Section: Imports and guards ----
import hashlib
import json
from pathlib import Path

import pandas as pd

from digikala_review_transformer import config

if not ANALYSIS_READY:
    raise RuntimeError("Cell 1 did not end with ALL CHECKS PASSED -> fix cell 1 first")
for _name in ("STAGE1_WINNER", "STAGE2_CONFIGS"):
    if _name not in globals():
        raise RuntimeError(f"'{_name}' is missing -> run cell 4 first in this kernel")

n_fail = 0   # reset the failure counter of the shared check() helper

# ---- Section: Constants ----
HANDOFF_NAME = "stage1_handoff.json"
DECISION_NAME = "stage1_decision.json"
SCORE_TOL = 1e-6

EXPECTED_TABLES = [
    "stage1_learning_curves.csv", "stage1_learning_summary.csv",
    "stage1_vs_tfidf.csv", "stage1_vs_tfidf_pct.csv",
    "stage1_b_vs_c.csv", DECISION_NAME,
    "stage1_test_metrics.csv",
    "stage1_confusion_matrices.csv", "stage1_per_class_metrics.csv",
    "stage1_time_resources.csv",
]
EXPECTED_FIGURES = [
    "01_stage1_learning_curves.png", "02_stage1_vs_tfidf.png",
    "03_stage1_confusion_raw.png", "04_stage1_confusion_corrected.png",
    "05_stage1_per_class_f1.png", "06_stage1_time_resources.png",
]


def fetch(node, dotted):
    """Value at a dotted path of a nested dict, or None if any part is missing."""
    for part in dotted.split("."):
        if not isinstance(node, dict) or part not in node:
            return None
        node = node[part]
    return node


def sha256_of(path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


# ---- Section: All Stage-1 outputs exist ----
missing_tables = [n for n in EXPECTED_TABLES if not (TABLES_DIR / n).is_file()]
missing_figures = [n for n in EXPECTED_FIGURES if not (FIGS_DIR / n).is_file()]
check(not missing_tables, f"{len(EXPECTED_TABLES)} tables exist in {TABLES_DIR.relative_to(config.ROOT)}",
      f"missing tables {missing_tables} -> run the cell that writes them (3a, 4, 5, 6, 7)")
check(not missing_figures, f"{len(EXPECTED_FIGURES)} figures exist in {FIGS_DIR.relative_to(config.ROOT)}",
      f"missing figures {missing_figures} -> run the cell that draws them (2, 3b, 6, 7)")

# ---- Section: Re-derive the B vs C rule from the runs and compare with cell 4 ----
by_variant = {RUNS[name]["variant"]: name for name in RUN_NAMES}
check(set(by_variant) == {"B", "C"}, f"runs by variant: {by_variant}",
      f"expected exactly variants B and C, got {sorted(by_variant)} -> check RUN_NAMES")
score_b, score_c = float(RUNS[by_variant["B"]]["best_score"]), float(RUNS[by_variant["C"]]["best_score"])
margin = score_c - score_b
expected_winner = "C" if margin >= C_OVER_B_THRESHOLD - 1e-12 else "B"
check(str(STAGE1_WINNER).endswith(expected_winner),
      f"rule re-derived here (C - B = {margin:+.6f}, threshold {C_OVER_B_THRESHOLD}) gives {expected_winner} = cell 4",
      f"cell 4 says {STAGE1_WINNER!r} but the rule gives {expected_winner} (C - B = {margin:+.6f}) "
      "-> rerun cell 4 and compare its scores")

decision = json.loads((TABLES_DIR / DECISION_NAME).read_text(encoding="utf-8"))
decision_label = decision.get("decision_label")
check(decision_label is not None, f"decision label: {decision_label}",
      f"'decision_label' not in {DECISION_NAME} (keys: {list(decision)}) -> tell me the real key")
is_real = decision_label is not None and "DEV" not in str(decision_label).upper()

# ---- Section: Cross-check the scores against each run_summary.json ----
run_info = {}
for name in RUN_NAMES:
    run_dir = Path(RUNS_ROOT) / name
    summary = json.loads((run_dir / "run_summary.json").read_text(encoding="utf-8"))
    best_val = fetch(summary, "validation_best.corrected.mean_macro_f1")
    ok = best_val is not None and abs(float(best_val) - float(RUNS[name]["best_score"])) < SCORE_TOL
    check(ok, f"{name}: best score in run_summary.json equals the score used by the rule ({RUNS[name]['best_score']:.6f})",
          f"{name}: summary value {best_val} differs from RUNS best_score {RUNS[name]['best_score']} "
          "-> check validation_best vs selection_and_stopping")
    run_info[name] = {
        "variant": RUNS[name]["variant"],
        "run_dir": str(run_dir.resolve().relative_to(Path(config.ROOT).resolve())),
        "config_hash": summary.get("config_hash"),
        "best_epoch": int(RUNS[name]["best_epoch"]),
        "stop_reason": RUNS[name]["stop_reason"],
        "val_corrected": fetch(summary, "validation_best.corrected"),
        "val_raw": fetch(summary, "validation_best.raw"),
    }

# ---- Section: Stage-2 configs exist ----
stage2 = [str(p) for p in STAGE2_CONFIGS]
absent = [p for p in stage2 if not (config.ROOT / p).is_file()]
check(len(stage2) > 0 and not absent, f"{len(stage2)} Stage-2 configs exist: {stage2}",
      f"Stage-2 configs missing or empty list: {absent or stage2} -> check cell 4 / configs/transformer/")
winner_tag = f"_{expected_winner}.yaml"
check(all(p.endswith(winner_tag) for p in stage2), f"all Stage-2 configs use variant {expected_winner}",
      f"some Stage-2 config does not end with '{winner_tag}': {stage2} -> cell 4 chose the wrong configs")

# ---- Section: Time totals from cell 7 ----
time_table = pd.read_csv(TABLES_DIR / "stage1_time_resources.csv")
check(set(time_table["run"]) == set(RUN_NAMES), "time table covers both runs",
      f"time table runs {sorted(time_table['run'])} differ from {RUN_NAMES} -> rerun cell 7")
total_hours = float(time_table["total_wall_seconds"].sum() / 3600)
for name in RUN_NAMES:
    row = time_table[time_table["run"] == name].iloc[0]
    run_info[name].update({"epochs_run": int(row["epochs_run"]),
                           "total_wall_hours": float(row["total_wall_hours"]),
                           "peak_reserved_mb": float(row["peak_reserved_mb"]),
                           "gpu": row["gpu"]})

# ---- Section: Write the handoff file ----
handoff = {
    "stage": 1,
    "mode_tag": MODE_TAG,
    "is_real_decision": bool(is_real),
    "decision_label": decision_label,
    "pack_fingerprint": PACK_FINGERPRINT,
    "selection_metric": SELECTION_METRIC,
    "rule": f"C wins only if val_mean_macro_f1 >= B + {C_OVER_B_THRESHOLD}, else B",
    "scores": {"B": score_b, "C": score_c, "C_minus_B": margin},
    "winner_variant": expected_winner,
    "stage2_configs": stage2,
    "stage2_command_powershell": "uv run python scripts/train_transformer.py --config " + " ".join(stage2),
    "runs": run_info,
    "stage1_total_wall_hours": total_hours,
    "decision_file_sha256": sha256_of(TABLES_DIR / DECISION_NAME),
}
handoff_path = TABLES_DIR / HANDOFF_NAME
handoff_path.write_text(json.dumps(handoff, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
reloaded = json.loads(handoff_path.read_text(encoding="utf-8"))
check(reloaded["winner_variant"] == expected_winner and reloaded["stage2_configs"] == stage2,
      f"handoff written and re-read: {handoff_path.relative_to(config.ROOT)}",
      "handoff file does not round-trip -> check json writing")

# ---- Section: Final summary (English only) ----
print("-" * 78)
print(f"Mode          : {MODE_TAG}")
print(f"Decision      : {decision_label}")
print(f"Pack          : {PACK_FINGERPRINT}")
for name in RUN_NAMES:
    info = run_info[name]
    print(f"{name:10s}: best epoch {info['best_epoch']} of {info['epochs_run']} run, "
          f"val_mean_macro_f1 {RUNS[name]['best_score']:.6f}, stop = {info['stop_reason']}, "
          f"{info['total_wall_hours']:.2f} h, peak reserved {info['peak_reserved_mb']:.0f} MB")
print(f"C - B         : {margin:+.6f}  (threshold {C_OVER_B_THRESHOLD})  -> winner {expected_winner}")
print(f"Stage-1 time  : {total_hours:.2f} h")
print("Stage-2 command (PowerShell, friend's machine):")
print("  " + handoff["stage2_command_powershell"])
if not is_real:
    print("NOTE: this is a DEV result; the handoff is marked is_real_decision = false. "
          "Set STATE = 1 in cell 1 and re-run cells 1-8 once the real runs arrive.")
print("=" * 78)
print("CELL 8 RESULT: ALL CHECKS PASSED" if n_fail == 0
      else f"CELL 8 RESULT: {n_fail} CHECK(S) FAILED -> read the [FAIL] lines above")

[PASS] 10 tables exist in reports/transformer_stage1/dev/tables
[PASS] 6 figures exist in reports/transformer_stage1/dev/figures
[PASS] runs by variant: {'B': 'tooka_B', 'C': 'tooka_C'}
[PASS] rule re-derived here (C - B = +0.002839, threshold 0.003) gives B = cell 4
[PASS] decision label: DEV (not a real decision)
[PASS] tooka_B: best score in run_summary.json equals the score used by the rule (0.646080)
[PASS] tooka_C: best score in run_summary.json equals the score used by the rule (0.648919)
[PASS] 2 Stage-2 configs exist: ['configs/transformer/roberta_B.yaml', 'configs/transformer/mdeberta_B.yaml']
[PASS] all Stage-2 configs use variant B
[PASS] time table covers both runs
[PASS] handoff written and re-read: reports/transformer_stage1/dev/tables/stage1_handoff.json
------------------------------------------------------------------------------
Mode          : DEV (100k-row training subset, 10k-row validation/test subsets)
Decision      : DEV (not a real decision)
Pack          : 9f

# Stage 2 / Inspect 0: What exists on disk for Stage 2 (read-only)

In [ ]:
# ============================================================================
# Stage 2 / Inspect 0: existing runs, configs and train_transformer.py options (read-only, no checks)
# ============================================================================
import json
import re
from pathlib import Path

from digikala_review_transformer import config

ROOT = Path(config.ROOT)

print("--- run folders under artifacts/ (name | variant | model | max_length | max_epochs | epochs run | status) ---")
for art in sorted((ROOT / "artifacts").glob("*")):
    if not art.is_dir():
        continue
    runs = [p for p in sorted(art.iterdir()) if p.is_dir()]
    print(f"{art.name}/  ({len(runs)} sub-folders)")
    for run in runs:
        summary_path = run / "run_summary.json"
        if not summary_path.is_file():
            print(f"   {run.name:24s} no run_summary.json; files: {sorted(x.name for x in run.iterdir())[:6]}")
            continue
        s = json.loads(summary_path.read_text(encoding="utf-8"))
        cfg = s.get("config", {})
        model_cfg, train_cfg = cfg.get("model", {}), cfg.get("training", {})
        hist = run / "history.csv"
        n_epochs = max(sum(1 for _ in hist.open(encoding="utf-8")) - 1, 0) if hist.is_file() else "?"
        print(f"   {run.name:24s} {s.get('variant')} | {Path(str(model_cfg.get('pretrained_path'))).name} | "
              f"{model_cfg.get('max_length')} | {train_cfg.get('max_epochs')} | {n_epochs} | {s.get('status')}")
        split_rows = s.get("split_rows")
        if split_rows:
            print(f"   {'':24s} split_rows = {split_rows}")

print("\n--- configs/transformer/ ---")
for path in sorted((ROOT / "configs" / "transformer").glob("*.yaml")):
    print(f"  {path.name}")

print("\n--- scripts/train_transformer.py: command-line options ---")
source = (ROOT / "scripts" / "train_transformer.py").read_text(encoding="utf-8")
for match in re.finditer(r"add_argument\((.*?)\)\s*\n", source, flags=re.DOTALL):
    print("  ", " ".join(match.group(1).split())[:200])

print("\n--- notebooks/ ---")
for path in sorted((ROOT / "notebooks").glob("*.ipynb")):
    print(f"  {path.name:55s} {path.stat().st_size / 1024:8.1f} KB")

--- run folders under artifacts/ (name | variant | model | max_length | max_epochs | epochs run | status) ---
Step1_label_correction/  (3 sub-folders)
   rating_band              no run_summary.json; files: ['label_correction_bands_rating_band.html']
   rating_band_more_precise no run_summary.json; files: ['rating_band_more_precise.html']
   recommendation_status    no run_summary.json; files: ['label_correction_bands.html']
baseline_tfidf/  (0 sub-folders)
step2_weighting/  (0 sub-folders)
transformer_runs/  (0 sub-folders)
transformer_runs_dev/  (2 sub-folders)
   tooka_B                  B | TookaBERT-Base | 128 | 3 | 3 | complete
                            split_rows = {'train': 16000, 'validation': 10000, 'test': 10000, 'train_subset': 5000}
   tooka_C                  C | TookaBERT-Base | 128 | 3 | 3 | complete
                            split_rows = {'train': 16000, 'validation': 10000, 'test': 10000, 'train_subset': 5000}
transformer_runs_dev_scale/  (2 sub-folders)
   tooka_

---

# Unit testing with mock implementations (in a temporary notebook cell, using the project config):

In [1]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

from digikala_review_transformer import config


def make_fake_run(root, run_name, variant, score, status="complete"):
    run_dir = Path(root) / run_name
    run_dir.mkdir(parents=True)
    summary = {
        "status": status, "variant": variant,
        "validation_best": {"corrected": {"mean_macro_f1": score}},
        "selection_and_stopping": {"best_val_mean_macro_f1": score, "best_epoch": 2},
    }
    (run_dir / "run_summary.json").write_text(json.dumps(summary), encoding="utf-8")

def run_pick(root):
    result = subprocess.run(
        [sys.executable, "scripts/pick_variant.py", "--runs-root", str(root)],
        check=False, capture_output=True, text=True, cwd=config.ROOT)
    return result.returncode, result.stdout.strip()

cases = [  # (label, score_B, score_C, status_C, expected (exit code, stdout))
    ("C wins by +0.0031",        0.6400, 0.6431, "complete", (0, "C")),
    ("B wins, C only +0.0029",   0.6400, 0.6429, "complete", (0, "B")),
    ("exactly +0.003 -> C",      0.6400, 0.6430, "complete", (0, "C")),
    ("C worse than B",           0.6400, 0.6300, "complete", (0, "B")),
    ("C still running -> wait",  0.6400, 0.6500, "running",   (3, "")),
]
all_ok = True
for label, score_b, score_c, status_c, expected in cases:
    with tempfile.TemporaryDirectory() as tmp:
        make_fake_run(tmp, "tooka_B", "B", score_b)
        make_fake_run(tmp, "tooka_C", "C", score_c, status=status_c)
        got = run_pick(tmp)
    ok = got == expected
    all_ok &= ok
    print(f"[{'PASS' if ok else 'FAIL'}] {label}: got {got}, expected {expected}")
print("[PASS] all pick_variant cases" if all_ok else "[FAIL] some pick_variant cases -> fix before use")

[PASS] C wins by +0.0031: got (0, 'C'), expected (0, 'C')
[PASS] B wins, C only +0.0029: got (0, 'B'), expected (0, 'B')
[PASS] exactly +0.003 -> C: got (0, 'C'), expected (0, 'C')
[PASS] C worse than B: got (0, 'B'), expected (0, 'B')
[PASS] C still running -> wait: got (3, ''), expected (3, '')
[PASS] all pick_variant cases


# Decision on the user's laptop (Target: less than 5 minutes)

In [3]:
from pathlib import Path

from digikala_review_transformer import config

RUNS_DIR = Path(config.ROOT) / "artifacts" / "transformer_runs"
RUN_NAMES_TO_CHECK = ["tooka_B", "tooka_C"]
REQUIRED_FILES = ["run_summary.json", "history.csv", "train_log.csv", "train.log",
                  "config_resolved.yaml", "val_probs_best.parquet", "test_probs.parquet",
                  "encoder_config.json"]
REQUIRED_DIRS = ["tokenizer"]

all_ok = True
for run_name in RUN_NAMES_TO_CHECK:
    run_dir = RUNS_DIR / run_name
    if (run_dir / run_name).exists():
        print(f"[FAIL] {run_name}: nested folder {run_dir / run_name} -> move its content one level up")
        all_ok = False
        continue
    missing = [name for name in REQUIRED_FILES if not (run_dir / name).is_file()]
    missing += [name for name in REQUIRED_DIRS if not (run_dir / name).is_dir()]
    if missing:
        print(f"[FAIL] {run_name}: missing {missing} -> ask the friend to re-send the zip")
        all_ok = False
    else:
        print(f"[PASS] {run_name}: all expected files are present")
print("[PASS] layout ok" if all_ok else "[FAIL] fix the layout before running Cell 1")

[PASS] tooka_B: all expected files are present
[PASS] tooka_C: all expected files are present
[PASS] layout ok


# Cross-control with script

In [6]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(Path(config.ROOT) / "scripts" / "pick_variant.py"),
     "--runs-root", str(RUNS_DIR)],
    check=False, capture_output=True, text=True, cwd=config.ROOT)
script_choice = result.stdout.strip()
print(result.stderr.strip())
if result.returncode == 0 and script_choice == STAGE1_WINNER:
    print(f"[PASS] Cell 4 and pick_variant.py agree: {STAGE1_WINNER}")
else:
    print(f"[FAIL] Cell 4 says {STAGE1_WINNER!r}, script says {script_choice!r} "
          f"(exit {result.returncode}) -> do NOT announce a decision; inspect run_summary.json")

[INFO] B=0.663465 (epoch 4), C=0.663744 (epoch 4), C-B=+0.000280, threshold=0.003
[PASS] Cell 4 and pick_variant.py agree: B


---

# Inspect 5/9/10: fields for run_comparison, dummy baseline files, metrics.evaluate(ordinal=True)

In [7]:
# ============================================================================
# Inspect 5/9/10: fields for run_comparison, dummy baseline files, metrics.evaluate(ordinal=True)
# Read-only, no checks. Needs Cell 1 (RUNS_ROOT, RUN_NAMES).
# ============================================================================
import json
from pathlib import Path

import numpy as np

from digikala_review_transformer import config, metrics

ROOT = Path(config.ROOT)
run_dir = Path(RUNS_ROOT) / RUN_NAMES[0]
summary = json.loads((run_dir / "run_summary.json").read_text(encoding="utf-8"))

for key in ("model", "parameter_counts", "tokenizer_backend", "tokenizer", "encoder_load_info", "training_schedule"):
    print(f"--- run_summary['{key}'] ---")
    print(json.dumps(summary.get(key), indent=1, default=str)[:900])

print("\n--- models/pretrained/*: files and 'license' lines in README.md (first 60 lines) ---")
for model_dir in sorted((ROOT / "models" / "pretrained").glob("*")):
    if not model_dir.is_dir():
        continue
    print(f"{model_dir.name}: {sorted(p.name for p in model_dir.iterdir())[:14]}")
    readme = model_dir / "README.md"
    if readme.is_file():
        lines = readme.read_text(encoding="utf-8", errors="replace").splitlines()[:60]
        print("   ", [line.strip() for line in lines if "licen" in line.lower()][:3])

print("\n--- baseline prediction files in reports/predictions/ ---")
for path in sorted((ROOT / "reports" / "predictions").glob("*"))[:40]:
    print(" ", path.name)

print("\n--- artifacts/ run folders ---")
for art in sorted((ROOT / "artifacts").glob("*")):
    if art.is_dir():
        print(f"  {art.name}: {sorted(p.name for p in art.iterdir() if p.is_dir())}")

print("\n--- metrics.evaluate on a toy input (ordinal=True) ---")
out = metrics.evaluate(np.array([0, 1, 2, 0, 1, 2]), np.array([0, 2, 2, 1, 1, 0]), n_classes=3, ordinal=True)
print(type(out).__name__)
print(json.dumps(out, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))[:1500])

--- run_summary['model'] ---
{
 "encoder_class": "BertModel",
 "hidden_size": 768,
 "head_dropout": 0.1,
 "classes": {
  "rec": [
   "recommended",
   "not_recommended",
   "no_idea"
  ],
  "band": [
   "low",
   "medium",
   "high"
  ]
 },
 "dtype": "torch.float32",
 "parameters": {
  "total": 122319366,
  "trainable": 122319366,
  "encoder": 122314752,
  "heads": 4614
 },
 "load_info": {
  "unexpected_keys": 5,
  "pooler_keys_missing": 2
 }
}
--- run_summary['parameter_counts'] ---
{
 "total": 122319366,
 "trainable": 122319366,
 "encoder": 122314752,
 "heads": 4614
}
--- run_summary['tokenizer_backend'] ---
"hf_fast"
--- run_summary['tokenizer'] ---
{
 "backend": "hf_fast",
 "source_model": "TookaBERT-Base",
 "max_length": 128,
 "special_ids": {
  "first": 2,
  "last": 3,
  "pad": 0,
  "unk": 1
 },
 "structure": "tokenizer adds its own special tokens; truncation keeps the end token",
 "fingerprint": "d1a1da1f5d72c29284a049412da210f8a4c0640dcf8acb67574e8786676d2aa1",
 "name": "TookaB

### Inspect 11: use_class_weights and variant keys in both tooka runs (read-only)

In [8]:
# ============================================================================
# Inspect 11: use_class_weights and variant keys in both tooka runs (read-only)
# ============================================================================
import json
from pathlib import Path

for name in RUN_NAMES:
    s = json.loads((Path(RUNS_ROOT) / name / "run_summary.json").read_text(encoding="utf-8"))
    print(name, "| variant:", s.get("variant"),
          "| use_class_weights:", s.get("training_schedule", {}).get("use_class_weights"),
          "| keys with weight/variant:", [k for k in s if "weight" in k.lower() or "variant" in k.lower()])

tooka_B | variant: B | use_class_weights: True | keys with weight/variant: ['variant']
tooka_C | variant: C | use_class_weights: True | keys with weight/variant: ['variant']


# cell training_settings.csv

In [10]:
# ============================================================================
# Cell: training_settings.csv (settings table with reasons)
# Needs Cell 1 (RUNS_ROOT, RUN_NAMES, utils) and TABLES_DIR (tables folder).
# ============================================================================
import json
import math  # noqa: F811
from pathlib import Path

import pandas as pd

from digikala_review_transformer import utils

_missing = [n for n in ("RUNS_ROOT", "RUN_NAMES", "TABLES_DIR") if n not in globals()]

assert not _missing, f"[FAIL] undefined names {_missing} -> run Cell 1 or send me the real tables-folder variable name"

# ---- Section: the settings table (values and reasons) ----
ROWS = [
    ("Optimizer", "AdamW",
     "Standard for Transformer fine-tuning; it applies weight decay correctly alongside adaptive updates."),
    ("Encoder learning rate", "2e-5",
     "Typical safe range (1e-5 to 5e-5) for BERT-sized encoders; small enough to preserve pretrained knowledge. Not tuned, as there was no time for a search."),
    ("Head learning rate", "1e-4",
     "Heads start from random initialization and need larger steps than the pretrained encoder. The project requires encoder LR to be smaller than head LR."),
    ("Weight decay", "0.01",
     "A mild, commonly used regularizer. Biases and normalization weights are excluded because decaying them is not useful."),
    ("Scheduler", "6% linear warmup, then linear decay to zero",
     "Warmup prevents large early updates from damaging pretrained weights; linear decay makes the final steps small and stable. A standard choice."),
    ("Gradient clipping", "1.0",
     "Guards against occasional loss spikes, which are more likely with noisy labels."),
    ("Precision", "bf16 autocast",
     "Cuts memory and speeds up training on the training GPU. Unlike fp16, it needs no loss scaling and is more stable, which matters for DeBERTa-style models."),
    ("Batch size", "128 (TookaBERT, RoBERTa) / 64 (mDeBERTa); no gradient accumulation",
     "128 is large enough for stable gradients and good GPU utilization on a 16 GB card. mDeBERTa at 160 tokens with batch 128 peaked at 23.06 GB in the smoke test, above the 16 GB card, so it was trained with batch 64; learning rates were not changed. Accumulation is unnecessary, which keeps training simple."),
    ("Max epochs / patience", "4 / 1",
     "A fixed budget of 4 epochs for all encoders, not tuned. The best epoch of both TookaBERT runs (B and C) was the last one (4), so the cap may have limited the results."),
    ("Checkpoint criterion", "Mean Macro F1 of both heads on corrected validation labels",
     "Macro F1 treats minority classes fairly. The encoder is shared, so one checkpoint must serve both heads."),
    ("Label smoothing", "0.1",
     "Chosen in the planning document to reduce the influence of residual label noise. It lowers maximum predicted probabilities, which affects how many pseudo-labels pass the 0.90 threshold later."),
    ("Dropout", "Encoder default plus 0.1 before each head",
     "The encoder keeps its built-in dropout; the extra head dropout is a mild safeguard against memorizing noisy labels."),
    ("Class weights", "balanced, from training labels only",
     "Both tasks are imbalanced. Weights keep minority classes visible, consistent with the TF-IDF baseline and the Macro F1 objective."),
    ("Sample weights", "Variant B: all 1; Variant C: Step-2 weights",
     "Step-2 weights down-weight likely mislabeled rows. Comparing B and C tests whether this helps."),
    ("Task loss weight", "1.0 (rec and band equal)",
     "Neutral default; there is no evidence yet to favor either task."),
    ("Pooling", "First token representation",
     "Standard sentence representation for classification and available in all three encoders."),
    ("max_length", "128 (TookaBERT) / 128 (RoBERTa) / 160 (mDeBERTa)",
     "Derived from the earlier token-length analysis. Values differ per model because the tokenizers split text differently."),
    ("Batching", "Length-bucketed shuffle; order depends only on seed and epoch",
     "Texts are short (median about 6 words), so grouping similar lengths minimizes padding. Identical order for B and C makes the comparison fair."),
    ("Data used", "Full train_supervised; full validation each epoch",
     "The hardware allows it and it avoids quality loss from subsampling."),
    ("Seed", "2026",
     "One project-wide seed keeps all phases consistent and reproducible (value taken from the smoke-test log)."),
]

# ---- Section: checks against each existing run_summary.json ----
def _is(a, b):
    return math.isclose(float(a), float(b), rel_tol=1e-9) if isinstance(b, float) else a == b

CHECKS = [  # (setting, label, getter, expected(run_name))
    ("Optimizer", "optimizer", lambda s: s["training_schedule"]["optimizer"], lambda n: "AdamW"),
    ("Encoder learning rate", "lr_encoder", lambda s: s["training_schedule"]["lr_encoder"], lambda n: 2e-5),
    ("Head learning rate", "lr_heads", lambda s: s["training_schedule"]["lr_heads"], lambda n: 1e-4),
    ("Weight decay", "weight_decay", lambda s: s["training_schedule"]["weight_decay"], lambda n: 0.01),
    ("Scheduler", "warmup_ratio", lambda s: s["training_schedule"]["warmup_ratio"], lambda n: 0.06),
    ("Gradient clipping", "max_grad_norm", lambda s: s["training_schedule"]["max_grad_norm"], lambda n: 1.0),
    ("Precision", "bf16", lambda s: s["training_schedule"]["bf16"], lambda n: True),
    ("Batch size", "batch_size", lambda s: s["training_schedule"]["batch_size"], lambda n: 64 if n.startswith("mdeberta") else 128),
    ("Batch size", "gradient_accumulation_steps", lambda s: s["training_schedule"]["gradient_accumulation_steps"], lambda n: 1),
    ("Label smoothing", "label_smoothing", lambda s: s["training_schedule"]["label_smoothing"], lambda n: 0.1),
    ("Dropout", "head_dropout", lambda s: s["model"]["head_dropout"], lambda n: 0.1),
    ("Class weights", "use_class_weights", lambda s: s["training_schedule"]["use_class_weights"], lambda n: True),
    ("Sample weights", "variant", lambda s: s["variant"], lambda n: n.split("_")[-1]),
    ("Task loss weight", "loss_weight_band", lambda s: s["training_schedule"]["loss_weight_band"], lambda n: 1.0),
    ("max_length", "tokenizer.max_length", lambda s: s["tokenizer"]["max_length"], lambda n: 160 if n.startswith("mdeberta") else 128),
]

with utils.timer("training_settings"):
    failed, checked_runs, verified = [], [], set()
    for name in RUN_NAMES:
        path = Path(RUNS_ROOT) / name / "run_summary.json"
        if not path.is_file():
            print(f"[WARN] {name}: run_summary.json not found, not checked")
            continue
        summary = json.loads(path.read_text(encoding="utf-8"))
        checked_runs.append(name)
        for setting, label, getter, expected in CHECKS:
            try:
                actual = getter(summary)
            except KeyError as exc:
                failed.append(f"{name}: {label} key {exc} missing")
                continue
            if _is(actual, expected(name)):
                verified.add(setting)
            else:
                failed.append(f"{name}: {label} is {actual!r}, expected {expected(name)!r}")

    # file is saved only if no check failed, so every setting in `verified` passed
    if failed:
        for line in failed:
            print(f"[FAIL] {line} -> do not save; send me this output")
    else:
        print(f"[PASS] all {len(CHECKS)} checks agree with run_summary.json for runs {checked_runs}")
        table = pd.DataFrame(ROWS, columns=["setting", "value", "reason"])
        table["verified_in_run_summary"] = table["setting"].map(lambda s: "yes" if s in verified else "no")
        assert len(table) == 20, f"[FAIL] expected 20 rows, got {len(table)} -> check ROWS"
        out_path = Path(TABLES_DIR) / "training_settings.csv"
        table.to_csv(out_path, index=False, encoding="utf-8")
        print(f"[PASS] saved {out_path} ({len(table)} rows)")
        print(table[["setting", "verified_in_run_summary"]].to_string(index=False))

[PASS] all 15 checks agree with run_summary.json for runs ['tooka_B', 'tooka_C']
[PASS] saved /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/reports/transformer_stage1/full/tables/training_settings.csv (20 rows)
              setting verified_in_run_summary
            Optimizer                     yes
Encoder learning rate                     yes
   Head learning rate                     yes
         Weight decay                     yes
            Scheduler                     yes
    Gradient clipping                     yes
            Precision                     yes
           Batch size                     yes
Max epochs / patience                      no
 Checkpoint criterion                      no
      Label smoothing                     yes
              Dropout                     yes
        Class weights                     yes
       Sample weights                     yes
     Task loss weight                     yes
              Pooli

# Next step: Checking the read-only Dummy

In [11]:
# ============================================================================
# Inspect 12: dummy_baseline.csv structure (read-only)
# ============================================================================
from pathlib import Path

import pandas as pd

from digikala_review_transformer import config

ROOT = Path(config.ROOT)
tables = ROOT / "reports" / "baseline_tfidf" / "tables"
print("--- files in reports/baseline_tfidf/tables ---")
for p in sorted(tables.glob("*")):
    print(" ", p.name)

dummy_path = tables / "dummy_baseline.csv"
print(f"\n--- {dummy_path.name}: exists={dummy_path.is_file()} ---")
if dummy_path.is_file():
    df = pd.read_csv(dummy_path)
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print(df.to_string(index=False))

--- files in reports/baseline_tfidf/tables ---
  ab_comparison.csv
  artifact_table.csv
  band_by_rate_zero.csv
  confidence_table.csv
  confusion_long.csv
  data_checks.csv
  dummy_baseline.csv
  final_fit.csv
  high_conf_errors.csv
  metrics_per_class.csv
  metrics_summary.csv
  overlap_with_train.csv
  reproducibility_check.json
  step1_changed_rows_check.csv
  timing.csv
  top_ngrams.csv
  tuning_results.csv
  vocab_probe.csv

--- dummy_baseline.csv: exists=True ---
shape: (16, 15)
columns: ['model_id', 'head', 'variant', 'split', 'majority_class', 'n', 'accuracy', 'macro_f1', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'label_view', 'mean_abs_distance', 'neighbor_error_pct', 'extreme_error_pct']
    model_id head variant      split majority_class      n  accuracy  macro_f1  balanced_accuracy  macro_precision  macro_recall label_view  mean_abs_distance  neighbor_error_pct  extreme_error_pct
 dummy_rec_A  rec       A validation    recommended 788677  0.789572  0.294138  